### Trial Data Acquisition

In [ ]:
# Set up the TrialSignal workspace and save files to Google Drive.
from google.colab import drive
drive.mount("/content/portfolio-drive")

from pathlib import Path
import requests
import pandas as pd
import json
import time

DATA_DIR = Path("./TrialSignal/data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

API_URL = "https://clinicaltrials.gov/api/v2/studies"

print("Data folder:", DATA_DIR)

Mounted at /content/portfolio-drive
Data folder: ./TrialSignal/data


In [ ]:
# Pull every ClinicalTrials.gov study returned for a disease, following all API pages.
def fetch_trials_by_condition(condition, page_size=100):
    studies = []

    params = {
        "query.cond": condition,
        "pageSize": page_size,
        "format": "json"
    }

    while True:
        response = requests.get(API_URL, params=params, timeout=60)
        response.raise_for_status()

        data = response.json()
        studies.extend(data.get("studies", []))

        next_token = data.get("nextPageToken")

        if not next_token:
            break

        params["pageToken"] = next_token
        time.sleep(0.1)

    return studies

In [ ]:
# Retrieve HAE studies and keep only interventional clinical trials.
hae_all = fetch_trials_by_condition("Hereditary Angioedema")

hae_interventional = [
    study
    for study in hae_all
    if (
        study
        .get("protocolSection", {})
        .get("designModule", {})
        .get("studyType")
        == "INTERVENTIONAL"
    )
]
print("All HAE studies returned:", len(hae_all))
print("Interventional HAE studies:", len(hae_interventional))

All HAE studies returned: 190
Interventional HAE studies: 133


In [ ]:
# Save the untouched HAE API records before performing any cleaning or restructuring.
raw_path = DATA_DIR / "hae_interventional_raw.json"

with open(raw_path, "w", encoding="utf-8") as f:
    json.dump(hae_interventional, f, ensure_ascii=False, indent=2)

print("Saved:", raw_path)

Saved: ./TrialSignal/data/hae_interventional_raw.json


In [ ]:
# Convert the nested ClinicalTrials.gov records into one study-level row per trial.
def flatten_study(study, disease_label):
    protocol = study.get("protocolSection", {})

    identification = protocol.get("identificationModule", {})
    status = protocol.get("statusModule", {})
    sponsor = protocol.get("sponsorCollaboratorsModule", {})
    design = protocol.get("designModule", {})
    conditions = protocol.get("conditionsModule", {})
    interventions = protocol.get("armsInterventionsModule", {})
    outcomes = protocol.get("outcomesModule", {})
    eligibility = protocol.get("eligibilityModule", {})

    lead_sponsor = sponsor.get("leadSponsor", {})
    enrollment = design.get("enrollmentInfo", {})
    design_info = design.get("designInfo", {})

    intervention_list = interventions.get("interventions", [])

    intervention_names = [
        item.get("name")
        for item in intervention_list
        if item.get("name")
    ]

    intervention_types = [
        item.get("type")
        for item in intervention_list
        if item.get("type")
    ]

    return {
        "disease": disease_label,
        "nct_id": identification.get("nctId"),
        "brief_title": identification.get("briefTitle"),
        "official_title": identification.get("officialTitle"),

        "study_type": design.get("studyType"),
        "overall_status": status.get("overallStatus"),
        "has_results": study.get("hasResults"),

        "phases": "; ".join(design.get("phases", [])),
        "primary_purpose": design_info.get("primaryPurpose"),

        "lead_sponsor": lead_sponsor.get("name"),
        "funder_type": lead_sponsor.get("class"),

        "enrollment": enrollment.get("count"),
        "enrollment_type": enrollment.get("type"),

        "start_date": (
            status.get("startDateStruct", {}).get("date")
        ),
        "primary_completion_date": (
            status.get("primaryCompletionDateStruct", {}).get("date")
        ),
        "completion_date": (
            status.get("completionDateStruct", {}).get("date")
        ),

        "conditions": "; ".join(conditions.get("conditions", [])),
        "intervention_names": "; ".join(intervention_names),
        "intervention_types": "; ".join(intervention_types),

        "sex": eligibility.get("sex"),
        "minimum_age": eligibility.get("minimumAge"),
        "maximum_age": eligibility.get("maximumAge"),
        "age_groups": "; ".join(eligibility.get("stdAges", [])),

        # Keep the complete outcome structures so no endpoint text is lost.
        "primary_outcomes": json.dumps(
            outcomes.get("primaryOutcomes", []),
            ensure_ascii=False
        ),
        "secondary_outcomes": json.dumps(
            outcomes.get("secondaryOutcomes", []),
            ensure_ascii=False
        )
    }


hae_trials = pd.DataFrame(
    [
        flatten_study(study, "Hereditary Angioedema")
        for study in hae_interventional
    ]
)

print("HAE study table shape:", hae_trials.shape)
hae_trials.head()

HAE study table shape: (133, 25)


,disease,nct_id,brief_title,official_title,study_type,overall_status,has_results,phases,primary_purpose,lead_sponsor,...,completion_date,conditions,intervention_names,intervention_types,sex,minimum_age,maximum_age,age_groups,primary_outcomes,secondary_outcomes
0,Hereditary Angioedema,NCT01912456,A Study to Evaluate the Clinical Efficacy and ...,"A Double-blind, Randomized, Placebo-controlled...",INTERVENTIONAL,COMPLETED,True,PHASE3,PREVENTION,CSL Behring,...,2015-10,Hereditary Angioedema Types I and II,Low-volume C1-esterase inhibitor; Higher-volum...,BIOLOGICAL; BIOLOGICAL; BIOLOGICAL; BIOLOGICAL,ALL,12 Years,None,CHILD; ADULT; OLDER_ADULT,"[{""measure"": ""The Time-normalized Number of He...","[{""measure"": ""Percentage of Subjects With a ≥ ..."
1,Hereditary Angioedema,NCT03873116,Study to Evaluate the Efficacy and Safety of B...,"A Phase 3, Randomized, Double-blind, Placebo-c...",INTERVENTIONAL,COMPLETED,True,PHASE3,PREVENTION,BioCryst Pharmaceuticals,...,2021-07-08,"Hereditary Angioedema, HAE",BCX7353 capsules; BCX7353 capsules; Placebo or...,DRUG; DRUG; DRUG,ALL,12 Years,None,CHILD; ADULT; OLDER_ADULT,"[{""measure"": ""Part 1: The Rate of Expert-confi...","[{""measure"": ""Part 1: Proportion of Days With ..."
2,Hereditary Angioedema,NCT00456508,Safety and Efficacy Study of Repeated Doses of...,Open-label Patient Continuation of DX-88 (Ecal...,INTERVENTIONAL,COMPLETED,True,PHASE3,TREATMENT,Shire,...,2010-09-01,Hereditary Angioedema (HAE),ecallantide,DRUG,ALL,10 Years,None,CHILD; ADULT; OLDER_ADULT,"[{""measure"": ""Change From Baseline in Mean Sym...","[{""measure"": ""Treatment Outcome Score (TOS) at..."
3,Hereditary Angioedema,NCT04618211,Dose-ranging Study of Oral PHA-022121 for Acut...,"A Phase II, Double-blind, Placebo-controlled, ...",INTERVENTIONAL,COMPLETED,True,PHASE2,TREATMENT,Pharvaris Netherlands B.V.,...,2023-03-01,Hereditary Angioedema; Hereditary Angioedema T...,Deucrictibant; Placebo,DRUG; DRUG,ALL,18 Years,75 Years,ADULT; OLDER_ADULT,"[{""measure"": ""Change of the 3-symptom Composit...","[{""measure"": ""Time to Onset of Symptom Relief ..."
4,Hereditary Angioedema,NCT01923207,A Single Increasing Dose Study to Assess Safet...,"A Phase 1, Double-Blind, Single Ascending Dose...",INTERVENTIONAL,COMPLETED,False,PHASE1,TREATMENT,Shire,...,2014-01-07,Hereditary Angioedema (HAE),DX-2930; Placebo,DRUG; DRUG,ALL,18 Years,55 Years,ADULT,"[{""measure"": ""Proportion of subjects with adve...","[{""measure"": ""DX-2930 plasma level"", ""timeFram..."


In [ ]:
# Check whether the HAE trials contain the endpoint information TrialSignal needs.
def has_outcomes(json_text):
    try:
        return len(json.loads(json_text)) > 0
    except:
        return False


hae_trials["has_primary_outcome"] = (
    hae_trials["primary_outcomes"].apply(has_outcomes)
)

hae_trials["has_secondary_outcome"] = (
    hae_trials["secondary_outcomes"].apply(has_outcomes)
)

print("Total interventional trials:", len(hae_trials))
print(
    "Trials with a primary outcome:",
    hae_trials["has_primary_outcome"].sum()
)
print(
    "Trials with a secondary outcome:",
    hae_trials["has_secondary_outcome"].sum()
)
print(
    "Trials missing a primary outcome:",
    (~hae_trials["has_primary_outcome"]).sum()
)

Total interventional trials: 133
Trials with a primary outcome: 132
Trials with a secondary outcome: 119
Trials missing a primary outcome: 1


In [ ]:
# Inspect representative trial metadata before doing any filtering.
display(
    hae_trials[
        [
            "nct_id",
            "brief_title",
            "overall_status",
            "phases",
            "primary_purpose",
            "intervention_types",
            "has_primary_outcome",
            "has_secondary_outcome"
        ]
    ].head(10)
)

,nct_id,brief_title,overall_status,phases,primary_purpose,intervention_types,has_primary_outcome,has_secondary_outcome
0,NCT01912456,A Study to Evaluate the Clinical Efficacy and ...,COMPLETED,PHASE3,PREVENTION,BIOLOGICAL; BIOLOGICAL; BIOLOGICAL; BIOLOGICAL,True,True
1,NCT03873116,Study to Evaluate the Efficacy and Safety of B...,COMPLETED,PHASE3,PREVENTION,DRUG; DRUG; DRUG,True,True
2,NCT00456508,Safety and Efficacy Study of Repeated Doses of...,COMPLETED,PHASE3,TREATMENT,DRUG,True,True
3,NCT04618211,Dose-ranging Study of Oral PHA-022121 for Acut...,COMPLETED,PHASE2,TREATMENT,DRUG; DRUG,True,True
4,NCT01923207,A Single Increasing Dose Study to Assess Safet...,COMPLETED,PHASE1,TREATMENT,DRUG; DRUG,True,True
5,NCT00851409,A Study of the Safety and Immunogenicity of Re...,COMPLETED,PHASE2,TREATMENT,DRUG,True,True
6,NCT05047185,Dose-ranging Study of Oral PHA-022121 for Prop...,TERMINATED,PHASE2,PREVENTION,DRUG; DRUG; DRUG,True,True
7,NCT05055258,A Trial to Evaluate the Efficacy and Safety of...,TERMINATED,PHASE2,PREVENTION,DRUG; DRUG,True,True
8,NCT05819775,CSL312_3003 Safety and Pharmacokinetic Study i...,COMPLETED,PHASE3,TREATMENT,BIOLOGICAL,True,True
9,NCT04654351,A Study of Icatibant (TAK-667) in Japanese Chi...,COMPLETED,PHASE3,TREATMENT,DRUG,True,True


In [ ]:
# Save the study-level HAE dataset so it can be reused without another API call.
hae_csv_path = DATA_DIR / "hae_trials_study_level.csv"

hae_trials.to_csv(hae_csv_path, index=False)

print("Saved:", hae_csv_path)

Saved: ./TrialSignal/data/hae_trials_study_level.csv


In [ ]:
# Inspect the endpoint structure and identify the one trial without a primary outcome.
missing_primary = hae_trials[
    ~hae_trials["has_primary_outcome"]
][
    [
        "nct_id",
        "brief_title",
        "overall_status",
        "phases",
        "intervention_types"
    ]
]

print("Trial missing a primary outcome:")
display(missing_primary)

print("\nExample primary outcomes:")
print(
    json.dumps(
        json.loads(hae_trials.loc[0, "primary_outcomes"]),
        indent=2
    )
)

Trial missing a primary outcome:


,nct_id,brief_title,overall_status,phases,intervention_types
17,NCT00004694,Study of Heparin Prophylaxis of Hereditary Ang...,COMPLETED,NA,DRUG



Example primary outcomes:
[
  {
    "measure": "The Time-normalized Number of Hereditary Angioedema Attacks",
    "description": "The time normalized number of HAE attacks as reported by the investigator per subject was calculated as: The total number of HAE attacks per subject and per treatment period / length of stay of subject in treatment period (days), Where length of stay of subject in treatment period was calculated as: Date of last day of subject in treatment period - date of first day of Week 3 of subject in treatment period + 1.",
    "timeFrame": "During the treatment phase, up to 28 weeks."
  }
]


In [ ]:
# Convert each trial's primary and secondary outcomes into individual endpoint rows.
def build_endpoint_table(trials_df):
    endpoint_rows = []

    for _, row in trials_df.iterrows():

        for endpoint_type, column in [
            ("Primary", "primary_outcomes"),
            ("Secondary", "secondary_outcomes")
        ]:
            try:
                outcomes = json.loads(row[column])
            except:
                outcomes = []

            for endpoint_number, outcome in enumerate(outcomes, start=1):

                endpoint_name = outcome.get("measure", "")
                endpoint_description = outcome.get("description", "")
                endpoint_timeframe = outcome.get("timeFrame", "")

                # Combine the endpoint name and description for later semantic analysis.
                endpoint_text = " ".join(
                    part.strip()
                    for part in [endpoint_name, endpoint_description]
                    if part and part.strip()
                )

                endpoint_rows.append({
                    "disease": row["disease"],
                    "nct_id": row["nct_id"],
                    "brief_title": row["brief_title"],
                    "overall_status": row["overall_status"],
                    "phases": row["phases"],
                    "primary_purpose": row["primary_purpose"],
                    "lead_sponsor": row["lead_sponsor"],
                    "funder_type": row["funder_type"],
                    "enrollment": row["enrollment"],
                    "start_date": row["start_date"],
                    "intervention_names": row["intervention_names"],
                    "intervention_types": row["intervention_types"],
                    "endpoint_type": endpoint_type,
                    "endpoint_number": endpoint_number,
                    "endpoint_name": endpoint_name,
                    "endpoint_description": endpoint_description,
                    "endpoint_timeframe": endpoint_timeframe,
                    "endpoint_text": endpoint_text
                })

    return pd.DataFrame(endpoint_rows)


hae_endpoints = build_endpoint_table(hae_trials)

print("Endpoint-level rows:", len(hae_endpoints))
print("Studies represented:", hae_endpoints["nct_id"].nunique())
print()
print("Endpoint types:")
print(hae_endpoints["endpoint_type"].value_counts())

hae_endpoints.head()

Endpoint-level rows: 1136
Studies represented: 132

Endpoint types:
endpoint_type
Secondary    795
Primary      341
Name: count, dtype: int64


,disease,nct_id,brief_title,overall_status,phases,primary_purpose,lead_sponsor,funder_type,enrollment,start_date,intervention_names,intervention_types,endpoint_type,endpoint_number,endpoint_name,endpoint_description,endpoint_timeframe,endpoint_text
0,Hereditary Angioedema,NCT01912456,A Study to Evaluate the Clinical Efficacy and ...,COMPLETED,PHASE3,PREVENTION,CSL Behring,INDUSTRY,90,2014-01,Low-volume C1-esterase inhibitor; Higher-volum...,BIOLOGICAL; BIOLOGICAL; BIOLOGICAL; BIOLOGICAL,Primary,1,The Time-normalized Number of Hereditary Angio...,The time normalized number of HAE attacks as r...,"During the treatment phase, up to 28 weeks.",The Time-normalized Number of Hereditary Angio...
1,Hereditary Angioedema,NCT01912456,A Study to Evaluate the Clinical Efficacy and ...,COMPLETED,PHASE3,PREVENTION,CSL Behring,INDUSTRY,90,2014-01,Low-volume C1-esterase inhibitor; Higher-volum...,BIOLOGICAL; BIOLOGICAL; BIOLOGICAL; BIOLOGICAL,Secondary,1,Percentage of Subjects With a ≥ 50% Reduction ...,The percentage reduction (%) in the time norma...,"During the treatment phase, up to 28 weeks.",Percentage of Subjects With a ≥ 50% Reduction ...
2,Hereditary Angioedema,NCT01912456,A Study to Evaluate the Clinical Efficacy and ...,COMPLETED,PHASE3,PREVENTION,CSL Behring,INDUSTRY,90,2014-01,Low-volume C1-esterase inhibitor; Higher-volum...,BIOLOGICAL; BIOLOGICAL; BIOLOGICAL; BIOLOGICAL,Secondary,2,Time-Normalized Number of Uses of Rescue Medic...,The time-normalized number of uses of rescue m...,"During the treatment phase, up to 28 weeks.",Time-Normalized Number of Uses of Rescue Medic...
3,Hereditary Angioedema,NCT01912456,A Study to Evaluate the Clinical Efficacy and ...,COMPLETED,PHASE3,PREVENTION,CSL Behring,INDUSTRY,90,2014-01,Low-volume C1-esterase inhibitor; Higher-volum...,BIOLOGICAL; BIOLOGICAL; BIOLOGICAL; BIOLOGICAL,Secondary,3,Percentage of Subjects With Adverse Events (AE...,,Within 24 hours of C1-esterase inhibitor or pl...,Percentage of Subjects With Adverse Events (AE...
4,Hereditary Angioedema,NCT01912456,A Study to Evaluate the Clinical Efficacy and ...,COMPLETED,PHASE3,PREVENTION,CSL Behring,INDUSTRY,90,2014-01,Low-volume C1-esterase inhibitor; Higher-volum...,BIOLOGICAL; BIOLOGICAL; BIOLOGICAL; BIOLOGICAL,Secondary,4,Percentage of Subjects With AEs or Other Speci...,The percentage of subjects experiencing the fo...,"During the treatment phase, up to 32 weeks.",Percentage of Subjects With AEs or Other Speci...


In [ ]:
# Check whether the endpoint rows contain usable text for later semantic matching.
print(
    "Endpoints with non-empty text:",
    (hae_endpoints["endpoint_text"].str.strip() != "").sum()
)

print(
    "Endpoints missing text:",
    (hae_endpoints["endpoint_text"].str.strip() == "").sum()
)

display(
    hae_endpoints[
        [
            "nct_id",
            "endpoint_type",
            "endpoint_name",
            "endpoint_description"
        ]
    ].head(10)
)

Endpoints with non-empty text: 1136
Endpoints missing text: 0


,nct_id,endpoint_type,endpoint_name,endpoint_description
0,NCT01912456,Primary,The Time-normalized Number of Hereditary Angio...,The time normalized number of HAE attacks as r...
1,NCT01912456,Secondary,Percentage of Subjects With a ≥ 50% Reduction ...,The percentage reduction (%) in the time norma...
2,NCT01912456,Secondary,Time-Normalized Number of Uses of Rescue Medic...,The time-normalized number of uses of rescue m...
3,NCT01912456,Secondary,Percentage of Subjects With Adverse Events (AE...,
4,NCT01912456,Secondary,Percentage of Subjects With AEs or Other Speci...,The percentage of subjects experiencing the fo...
5,NCT01912456,Secondary,Percentage of Subjects Experiencing Solicited ...,The percentage of subjects experiencing solici...
6,NCT01912456,Secondary,Injections Resulting in Solicited AEs (Injecti...,The rate/injection of injections of C1-esteras...
7,NCT03873116,Primary,Part 1: The Rate of Expert-confirmed HAE Attac...,The angioedema event rate and the treatment co...
8,NCT03873116,Primary,Part 2: To Evaluate the Long-term Safety and T...,The safety data was assessed for the safety po...
9,NCT03873116,Primary,Part 3: To Evaluate the Long-term Safety and T...,The safety data was assessed for the safety po...


In [ ]:
# Save the endpoint-level HAE dataset for later TrialSignal analysis.
hae_endpoint_path = DATA_DIR / "hae_endpoints_raw.csv"

hae_endpoints.to_csv(hae_endpoint_path, index=False)

print("Saved:", hae_endpoint_path)

Saved: ./TrialSignal/data/hae_endpoints_raw.csv


In [ ]:
# Summarize the HAE trial population before deciding which studies belong in the alignment analysis.
print("Primary purpose:")
print(hae_trials["primary_purpose"].value_counts(dropna=False))

print("\nPhase:")
print(hae_trials["phases"].value_counts(dropna=False))

print("\nIntervention types:")
print(hae_trials["intervention_types"].value_counts(dropna=False).head(15))

Primary purpose:
primary_purpose
TREATMENT        88
PREVENTION       35
OTHER             6
BASIC_SCIENCE     2
DIAGNOSTIC        1
None              1
Name: count, dtype: int64

Phase:
phases
PHASE3            56
PHASE2            26
PHASE1            18
PHASE4            11
PHASE2; PHASE3     9
PHASE1; PHASE2     6
NA                 6
EARLY_PHASE1       1
Name: count, dtype: int64

Intervention types:
intervention_types
DRUG                                                             43
DRUG; DRUG                                                       35
BIOLOGICAL                                                       15
DRUG; DRUG; DRUG                                                  6
DRUG; DRUG; DRUG; DRUG                                            5
BIOLOGICAL; BIOLOGICAL                                            5
BIOLOGICAL; DRUG                                                  4
DRUG; OTHER                                                       3
BIOLOGICAL; OTHER           

In [ ]:
# Inspect studies that may not represent therapeutic efficacy/effectiveness trials.
review_trials = hae_trials[
    hae_trials["primary_purpose"].isin(
        ["BASIC_SCIENCE", "DIAGNOSTIC", "SCREENING", "SUPPORTIVE_CARE"]
    )
    | hae_trials["primary_purpose"].isna()
    | (hae_trials["phases"] == "PHASE1")
][
    [
        "nct_id",
        "brief_title",
        "phases",
        "primary_purpose",
        "intervention_types"
    ]
]

print("Studies flagged for review:", len(review_trials))
display(review_trials)

Studies flagged for review: 22


,nct_id,brief_title,phases,primary_purpose,intervention_types
4,NCT01923207,A Single Increasing Dose Study to Assess Safet...,PHASE1,TREATMENT,DRUG; DRUG
29,NCT05477160,A Study of STAR-0215 in Healthy Adult Particip...,PHASE1,TREATMENT,DRUG; DRUG; DRUG; DRUG
30,NCT02892682,The Role of the Coagulation Pathways in Recurr...,NA,BASIC_SCIENCE,OTHER
32,NCT05178355,"A Single and Multiple Doses Safety, Tolerabili...",PHASE1,TREATMENT,DRUG; DRUG
33,NCT02093923,"A Double-Blind, Multiple Ascending Dose Study ...",PHASE1,TREATMENT,DRUG; DRUG
40,NCT00432510,Pharmacokinetics of C1 Esterase Inhibitor in H...,PHASE1,OTHER,DRUG
46,NCT02218294,Study to Determine How BCX4161 is Metabolized ...,PHASE1,TREATMENT,DRUG
49,NCT07293364,A Study to Learn About the C1-Inhibitor Functi...,NA,DIAGNOSTIC,DIAGNOSTIC_TEST
54,NCT00748202,Berinert P Study of Subcutaneous Versus Intrav...,PHASE3,None,DRUG
60,NCT02448264,"First-in-Human Study to Evaluate the Safety, T...",PHASE1,TREATMENT,DRUG; DRUG


In [ ]:
# Identify studies that are appropriate candidates for the core TrialSignal alignment analysis.
included_phases = {
    "PHASE1; PHASE2",
    "PHASE2",
    "PHASE2; PHASE3",
    "PHASE3",
    "PHASE4"
}

included_purposes = {
    "TREATMENT",
    "PREVENTION"
}

hae_trials["analysis_candidate"] = (
    hae_trials["phases"].isin(included_phases)
    & hae_trials["primary_purpose"].isin(included_purposes)
)

print("Core analysis candidates:", hae_trials["analysis_candidate"].sum())
print("Not automatically included:", (~hae_trials["analysis_candidate"]).sum())

Core analysis candidates: 105
Not automatically included: 28


In [ ]:
# Separate clearly out-of-scope early/safety studies from records that need a quick manual review.
def classify_trial_scope(row):
    if row["phases"] in {"PHASE1", "EARLY_PHASE1"}:
        return "Exclude - early phase"

    if row["primary_purpose"] in {"BASIC_SCIENCE", "DIAGNOSTIC"}:
        return "Exclude - non-therapeutic purpose"

    if row["analysis_candidate"]:
        return "Include"

    return "Manual review"


hae_trials["scope_decision"] = hae_trials.apply(classify_trial_scope, axis=1)

print(hae_trials["scope_decision"].value_counts())

display(
    hae_trials[
        hae_trials["scope_decision"] == "Manual review"
    ][
        [
            "nct_id",
            "brief_title",
            "phases",
            "primary_purpose",
            "intervention_types"
        ]
    ]
)

scope_decision
Include                              105
Exclude - early phase                 19
Manual review                          6
Exclude - non-therapeutic purpose      3
Name: count, dtype: int64


,nct_id,brief_title,phases,primary_purpose,intervention_types
17,NCT00004694,Study of Heparin Prophylaxis of Hereditary Ang...,NA,TREATMENT,DRUG
54,NCT00748202,Berinert P Study of Subcutaneous Versus Intrav...,PHASE3,None,DRUG
67,NCT04032002,Involvement of Monocytic B1 and B2 Receptors i...,NA,OTHER,OTHER; OTHER
77,NCT04334031,Deployment o the Multidisciplinary Prospective...,NA,OTHER,GENETIC; OTHER
104,NCT01426763,A Study to Evaluate the Safety and Pharmacolog...,PHASE2,OTHER,BIOLOGICAL
131,NCT01095497,"A Study to Evaluate the Safety, Pharmacokineti...",PHASE2,OTHER,BIOLOGICAL


In [ ]:
# Resolve the six manually reviewed HAE studies.
manual_exclusions = {
    "NCT00004694": "Exclude - missing registered outcome data",
    "NCT00748202": "Exclude - PK/safety study",
    "NCT04032002": "Exclude - mechanistic/non-therapeutic study",
    "NCT04334031": "Exclude - multi-disease cohort/biobank",
    "NCT01426763": "Exclude - safety/PK/PD study",
    "NCT01095497": "Exclude - safety/PK/PD study"
}

for nct_id, decision in manual_exclusions.items():
    hae_trials.loc[
        hae_trials["nct_id"] == nct_id,
        "scope_decision"
    ] = decision

print(hae_trials["scope_decision"].value_counts())

scope_decision
Include                                        105
Exclude - early phase                           19
Exclude - non-therapeutic purpose                3
Exclude - safety/PK/PD study                     2
Exclude - missing registered outcome data        1
Exclude - PK/safety study                        1
Exclude - mechanistic/non-therapeutic study      1
Exclude - multi-disease cohort/biobank           1
Name: count, dtype: int64


In [ ]:
# Create the final HAE study-level cohort for TrialSignal.
hae_trials_analysis = hae_trials[
    hae_trials["scope_decision"] == "Include"
].copy()

print("HAE studies retrieved:", len(hae_trials))
print("HAE studies included in analysis:", len(hae_trials_analysis))
print("HAE studies excluded:", len(hae_trials) - len(hae_trials_analysis))

HAE studies retrieved: 133
HAE studies included in analysis: 105
HAE studies excluded: 28


In [ ]:
# Keep endpoints only from studies included in the final HAE analysis cohort.
included_hae_ids = set(hae_trials_analysis["nct_id"])

hae_endpoints_analysis = hae_endpoints[
    hae_endpoints["nct_id"].isin(included_hae_ids)
].copy()

print("Studies represented:", hae_endpoints_analysis["nct_id"].nunique())
print("Total endpoints:", len(hae_endpoints_analysis))

print("\nEndpoint types:")
print(hae_endpoints_analysis["endpoint_type"].value_counts())

Studies represented: 105
Total endpoints: 911

Endpoint types:
endpoint_type
Secondary    673
Primary      238
Name: count, dtype: int64


In [ ]:
# Save the finalized HAE study and endpoint datasets.
hae_trials_analysis.to_csv(
    DATA_DIR / "hae_trials_analysis.csv",
    index=False
)

hae_endpoints_analysis.to_csv(
    DATA_DIR / "hae_endpoints_analysis.csv",
    index=False
)

print("Saved finalized HAE analysis files.")

Saved finalized HAE analysis files.


Collection trouble shooting is concluded, now we'll move on to creating a generic disease collection function

In [ ]:
# Reusable pipeline for collecting and structuring one disease from ClinicalTrials.gov.
def collect_disease_data(condition_query, disease_label):
    # Pull all studies matching the condition.
    all_studies = fetch_trials_by_condition(condition_query)

    # Keep interventional studies only.
    interventional = [
        study
        for study in all_studies
        if (
            study
            .get("protocolSection", {})
            .get("designModule", {})
            .get("studyType")
            == "INTERVENTIONAL"
        )
    ]

    # Build study-level table.
    trials = pd.DataFrame(
        [
            flatten_study(study, disease_label)
            for study in interventional
        ]
    )

    # Identify whether endpoint information exists.
    trials["has_primary_outcome"] = (
        trials["primary_outcomes"].apply(has_outcomes)
    )

    trials["has_secondary_outcome"] = (
        trials["secondary_outcomes"].apply(has_outcomes)
    )

    # Apply our preliminary scope rule.
    trials["analysis_candidate"] = (
        trials["phases"].isin(included_phases)
        & trials["primary_purpose"].isin(included_purposes)
    )

    trials["scope_decision"] = trials.apply(
        classify_trial_scope,
        axis=1
    )

    # Build endpoint-level table.
    endpoints = build_endpoint_table(trials)

    print(disease_label)
    print("-" * len(disease_label))
    print("All studies returned:", len(all_studies))
    print("Interventional studies:", len(interventional))
    print("Studies with primary outcomes:", trials["has_primary_outcome"].sum())
    print("Endpoint rows:", len(endpoints))

    print("\nScope decisions:")
    print(trials["scope_decision"].value_counts())

    return interventional, trials, endpoints

In [ ]:
# Save raw, study-level, and endpoint-level files for one disease.
def save_disease_data(
    raw_studies,
    trials_df,
    endpoints_df,
    file_prefix
):
    raw_path = DATA_DIR / f"{file_prefix}_interventional_raw.json"
    trials_path = DATA_DIR / f"{file_prefix}_trials_study_level.csv"
    endpoints_path = DATA_DIR / f"{file_prefix}_endpoints_raw.csv"

    with open(raw_path, "w", encoding="utf-8") as f:
        json.dump(
            raw_studies,
            f,
            ensure_ascii=False,
            indent=2
        )

    trials_df.to_csv(trials_path, index=False)
    endpoints_df.to_csv(endpoints_path, index=False)

    print("Saved:")
    print(raw_path)
    print(trials_path)
    print(endpoints_path)

In [ ]:
fa_raw, fa_trials, fa_endpoints = collect_disease_data(
    condition_query="Friedreich Ataxia",
    disease_label="Friedreich's Ataxia"
)

Friedreich's Ataxia
-------------------
All studies returned: 112
Interventional studies: 78
Studies with primary outcomes: 76
Endpoint rows: 674

Scope decisions:
scope_decision
Include                              49
Exclude - early phase                15
Manual review                        11
Exclude - non-therapeutic purpose     3
Name: count, dtype: int64


In [ ]:
save_disease_data(
    fa_raw,
    fa_trials,
    fa_endpoints,
    file_prefix="friedreich_ataxia"
)

Saved:
./TrialSignal/data/friedreich_ataxia_interventional_raw.json
./TrialSignal/data/friedreich_ataxia_trials_study_level.csv
./TrialSignal/data/friedreich_ataxia_endpoints_raw.csv


In [ ]:
fa_manual_review = fa_trials[
    fa_trials["scope_decision"] == "Manual review"
][
    [
        "nct_id",
        "brief_title",
        "phases",
        "primary_purpose",
        "intervention_types",
        "has_primary_outcome"
    ]
]

print("Manual review studies:", len(fa_manual_review))
display(fa_manual_review)

Manual review studies: 11


,nct_id,brief_title,phases,primary_purpose,intervention_types,has_primary_outcome
17,NCT01921868,An Open-label Study of the Effects of Acetyl-L...,NA,TREATMENT,DRUG,True
19,NCT07444333,Cardiac Output and Fatigue in Friedreich's Ataxia,NA,TREATMENT,BEHAVIORAL,True
20,NCT04192136,NAD+ and Exercise in FA,NA,OTHER,DIETARY_SUPPLEMENT; DIETARY_SUPPLEMENT; OTHER,True
27,NCT06539598,Respiratory Training in Friedreich's Ataxia,NA,OTHER,OTHER,True
34,NCT00824512,Efficacy of EGb761 in Patients Suffering From ...,PHASE2,SUPPORTIVE_CARE,DRUG; DRUG,True
37,NCT06692296,Efficacy of Stabilometric Platform to Improve ...,NA,TREATMENT,BEHAVIORAL; BEHAVIORAL,True
40,NCT07072676,The Use of Assistive Gait Devices Can Reduce t...,NA,PREVENTION,DEVICE,True
52,NCT04817111,NAD+ Precursor Supplementation in Friedreich's...,PHASE2,OTHER,DRUG,True
56,NCT01716221,An Objective Double-blind Evaluation of Buprop...,PHASE4,SUPPORTIVE_CARE,DRUG; DRUG; DRUG; DRUG,True
70,NCT04564495,Home Based Tele-exercise for People With Chron...,NA,TREATMENT,BEHAVIORAL,True


In [ ]:
print("\nPrimary purpose:")
print(fa_trials["primary_purpose"].value_counts(dropna=False))

print("\nPhase:")
print(fa_trials["phases"].value_counts(dropna=False))


Primary purpose:
primary_purpose
TREATMENT          69
OTHER               3
DIAGNOSTIC          2
SUPPORTIVE_CARE     2
PREVENTION          1
BASIC_SCIENCE       1
Name: count, dtype: int64

Phase:
phases
PHASE2            25
PHASE3            14
PHASE1            13
NA                11
PHASE1; PHASE2     8
PHASE2; PHASE3     3
EARLY_PHASE1       2
PHASE4             2
Name: count, dtype: int64


In [ ]:
# Inspect primary endpoints for the remaining ambiguous FA studies.
fa_ambiguous_ids = {
    "NCT07444333",
    "NCT04192136",
    "NCT04817111",
    "NCT01716221"
}

display(
    fa_trials[
        fa_trials["nct_id"].isin(fa_ambiguous_ids)
    ][
        [
            "nct_id",
            "brief_title",
            "phases",
            "primary_purpose",
            "intervention_names",
            "intervention_types",
            "enrollment"
        ]
    ]
)

print("\nPrimary endpoints:\n")

display(
    fa_endpoints[
        fa_endpoints["nct_id"].isin(fa_ambiguous_ids)
        & (fa_endpoints["endpoint_type"] == "Primary")
    ][
        [
            "nct_id",
            "endpoint_name",
            "endpoint_description",
            "endpoint_timeframe"
        ]
    ]
)

,nct_id,brief_title,phases,primary_purpose,intervention_names,intervention_types,enrollment
19,NCT07444333,Cardiac Output and Fatigue in Friedreich's Ataxia,NA,TREATMENT,Aerobic Exercise,BEHAVIORAL,30
20,NCT04192136,NAD+ and Exercise in FA,NA,OTHER,Nicotinamide Riboside; Placebo; Exercise Inter...,DIETARY_SUPPLEMENT; DIETARY_SUPPLEMENT; OTHER,75
52,NCT04817111,NAD+ Precursor Supplementation in Friedreich's...,PHASE2,OTHER,MIB-626,DRUG,7
56,NCT01716221,An Objective Double-blind Evaluation of Buprop...,PHASE4,SUPPORTIVE_CARE,bupropion & Citalopram; Bupropion & Placebo; P...,DRUG; DRUG; DRUG; DRUG,1



Primary endpoints:



,nct_id,endpoint_name,endpoint_description,endpoint_timeframe
177,NCT07444333,Adverse events (Primary outcome for Aim 1),Adverse events are defined as exercise-related...,"0, 3, 6 months"
178,NCT07444333,Maximal Oxygen Consumption (primary outcome me...,Maximal oxygen consumption (VO2max) will be de...,"0, 3-, 6-months"
179,NCT07444333,Fatigue Severity scale (primary outcome measur...,A commonly used measure to assess fatigue in a...,"0, 3-, and 6-months"
187,NCT04192136,Within-Participant Change in Peak V02 (Maximal...,Peak V02 will be assessed by completion of an ...,Baseline to 12 Weeks
551,NCT04817111,Percentage of Individuals With Treatment-emerg...,Safety will be monitored through collection of...,14 Days
563,NCT01716221,International Cooperative Ataxia Rating Scale ...,The ICARS is a 19 item rating scale of ataxia ...,Assessements are performed in 5 different stat...
564,NCT01716221,Friedreich Ataxia Rating Scale (FARS),A rating scale developed for Friedreich ataxia...,Assessements are performed in 5 different stat...


In [ ]:
# Resolve the manually reviewed Friedreich's ataxia studies.
fa_manual_decisions = {
    "NCT01921868": "Include",
    "NCT07444333": "Include",
    "NCT04192136": "Include",
    "NCT06539598": "Include",
    "NCT00824512": "Include",
    "NCT06692296": "Include",
    "NCT01716221": "Include",

    "NCT07072676": "Exclude - broad multi-disease study",
    "NCT04564495": "Exclude - broad multi-disease study",
    "NCT03120013": "Exclude - broad ataxia study",
    "NCT04817111": "Exclude - safety-focused study"
}

for nct_id, decision in fa_manual_decisions.items():
    fa_trials.loc[
        fa_trials["nct_id"] == nct_id,
        "scope_decision"
    ] = decision

print(fa_trials["scope_decision"].value_counts())

scope_decision
Include                                56
Exclude - early phase                  15
Exclude - non-therapeutic purpose       3
Exclude - broad multi-disease study     2
Exclude - safety-focused study          1
Exclude - broad ataxia study            1
Name: count, dtype: int64


In [ ]:
# Keep all studies approved for the final FA alignment analysis.
fa_trials_analysis = fa_trials[
    fa_trials["scope_decision"] == "Include"
].copy()

included_fa_ids = set(fa_trials_analysis["nct_id"])

fa_endpoints_analysis = fa_endpoints[
    fa_endpoints["nct_id"].isin(included_fa_ids)
].copy()

print("FA studies retrieved:", len(fa_trials))
print("FA studies included:", len(fa_trials_analysis))
print("FA studies excluded:", len(fa_trials) - len(fa_trials_analysis))

print("\nStudies represented:", fa_endpoints_analysis["nct_id"].nunique())
print("Total endpoints:", len(fa_endpoints_analysis))

print("\nEndpoint types:")
print(fa_endpoints_analysis["endpoint_type"].value_counts())

FA studies retrieved: 78
FA studies included: 56
FA studies excluded: 22

Studies represented: 56
Total endpoints: 440

Endpoint types:
endpoint_type
Secondary    320
Primary      120
Name: count, dtype: int64


In [ ]:
# Save the finalized Friedreich's ataxia analysis datasets.
fa_trials_analysis.to_csv(
    DATA_DIR / "friedreich_ataxia_trials_analysis.csv",
    index=False
)

fa_endpoints_analysis.to_csv(
    DATA_DIR / "friedreich_ataxia_endpoints_analysis.csv",
    index=False
)

print("Saved finalized FA analysis files.")

Saved finalized FA analysis files.


### Pull FSHD

In [ ]:
fshd_raw, fshd_trials, fshd_endpoints = collect_disease_data(
    condition_query="Facioscapulohumeral Muscular Dystrophy",
    disease_label="Facioscapulohumeral Muscular Dystrophy"
)

Facioscapulohumeral Muscular Dystrophy
--------------------------------------
All studies returned: 93
Interventional studies: 59
Studies with primary outcomes: 57
Endpoint rows: 540

Scope decisions:
scope_decision
Manual review                        26
Include                              21
Exclude - non-therapeutic purpose     8
Exclude - early phase                 4
Name: count, dtype: int64


In [ ]:
save_disease_data(
    fshd_raw,
    fshd_trials,
    fshd_endpoints,
    file_prefix="fshd"
)

Saved:
./TrialSignal/data/fshd_interventional_raw.json
./TrialSignal/data/fshd_trials_study_level.csv
./TrialSignal/data/fshd_endpoints_raw.csv


In [ ]:
fshd_manual_review = fshd_trials[
    fshd_trials["scope_decision"] == "Manual review"
][
    [
        "nct_id",
        "brief_title",
        "phases",
        "primary_purpose",
        "intervention_names",
        "intervention_types",
        "enrollment",
        "has_primary_outcome"
    ]
]

print("Manual review studies:", len(fshd_manual_review))
display(fshd_manual_review)

Manual review studies: 26


,nct_id,brief_title,phases,primary_purpose,intervention_names,intervention_types,enrollment,has_primary_outcome
1,NCT06363357,"The Effect of a Muscle-mimicking, Fabric-type ...",NA,SUPPORTIVE_CARE,Shoulder orthosis,DEVICE,8,True
4,NCT05453461,ADVANCED FSHD-COM: New Clinical Outcome Measur...,NA,OTHER,Validation of new COMs for non ambulant FSHD p...,DIAGNOSTIC_TEST,30,True
7,NCT04038138,Clinical Trial Readiness Network FSHD France: ...,NA,OTHER,Validation of new COA for FSHD patients,DIAGNOSTIC_TEST,100,True
9,NCT06378203,Rehabilitation in Muscular Dystrophies From th...,NA,TREATMENT,tele-rehabilitation system with virtual realit...,OTHER,10,True
13,NCT05902884,New Biomarkers in Facioscapulohumeral Muscular...,NA,OTHER,MSOT Acuity Echo,DEVICE,20,True
14,NCT00821548,Electrostimulation of Shoulder Girdle and Quad...,NA,TREATMENT,electrostimulation,PROCEDURE,9,True
15,NCT02861911,Effects of NMES on Muscle Function of Patients...,NA,SUPPORTIVE_CARE,Current flow between sensory and motor thresho...,OTHER; DEVICE,9,True
16,NCT06425445,Quantitative Assessment of Orofacial Muscle Fu...,NA,OTHER,Orofacial function assessment tests,PROCEDURE,22,True
20,NCT05812144,New Clinical Outcome Measures to Remotely Eval...,NA,OTHER,Remote monitoring program,OTHER,70,True
21,NCT02948244,Effect of Creatine Monohydrate on Functional M...,NA,TREATMENT,Creatine Monohydrate; Placebo,DIETARY_SUPPLEMENT; DIETARY_SUPPLEMENT,11,True


In [ ]:
print("Primary purpose:")
print(fshd_trials["primary_purpose"].value_counts(dropna=False))

print("\nPhase:")
print(fshd_trials["phases"].value_counts(dropna=False))

Primary purpose:
primary_purpose
TREATMENT          36
OTHER               8
SUPPORTIVE_CARE     6
BASIC_SCIENCE       5
DIAGNOSTIC          3
PREVENTION          1
Name: count, dtype: int64

Phase:
phases
NA                34
PHASE1; PHASE2    10
PHASE2             9
PHASE1             4
PHASE3             2
Name: count, dtype: int64


In [ ]:
# Resolve all manually reviewed FSHD studies.
fshd_manual_decisions = {
    "NCT06363357": "Exclude - broad multi-disease study",
    "NCT05453461": "Exclude - outcome measure validation",
    "NCT04038138": "Exclude - outcome measure validation",
    "NCT06378203": "Exclude - broad multi-disease study",
    "NCT05902884": "Exclude - biomarker study",

    "NCT00821548": "Include",
    "NCT02861911": "Include",

    "NCT06425445": "Exclude - assessment study",
    "NCT05812144": "Exclude - outcome measure validation",

    "NCT02948244": "Include",

    "NCT00027391": "Exclude - missing registered outcome data",

    "NCT01116570": "Include",

    "NCT07072676": "Exclude - broad multi-disease study",

    "NCT04267354": "Include",
    "NCT01596803": "Include",

    "NCT00004685": "Exclude - missing registered outcome data",

    "NCT06708468": "Exclude - broad multi-disease study",

    "NCT02159963": "Include",

    "NCT02622438": "Exclude - natural history/usual care",
    "NCT04694456": "Exclude - mechanistic study",
    "NCT06079567": "Exclude - natural history study",
    "NCT07478172": "Exclude - broad multi-disease study",
    "NCT06600308": "Exclude - assessment study",

    "NCT01618331": "Include",

    "NCT02810028": "Exclude - broad multi-disease study",

    "NCT07543016": "Include"
}

for nct_id, decision in fshd_manual_decisions.items():
    fshd_trials.loc[
        fshd_trials["nct_id"] == nct_id,
        "scope_decision"
    ] = decision

In [ ]:
print(fshd_trials["scope_decision"].value_counts())

print(
    "\nRemaining manual review studies:",
    (fshd_trials["scope_decision"] == "Manual review").sum()
)

scope_decision
Include                                      30
Exclude - non-therapeutic purpose             8
Exclude - broad multi-disease study           6
Exclude - early phase                         4
Exclude - outcome measure validation          3
Exclude - missing registered outcome data     2
Exclude - assessment study                    2
Exclude - biomarker study                     1
Exclude - natural history/usual care          1
Exclude - mechanistic study                   1
Exclude - natural history study               1
Name: count, dtype: int64

Remaining manual review studies: 0


In [ ]:
# Create the final FSHD study-level cohort for TrialSignal.
fshd_trials_analysis = fshd_trials[
    fshd_trials["scope_decision"] == "Include"
].copy()

included_fshd_ids = set(fshd_trials_analysis["nct_id"])

fshd_endpoints_analysis = fshd_endpoints[
    fshd_endpoints["nct_id"].isin(included_fshd_ids)
].copy()

print("FSHD studies retrieved:", len(fshd_trials))
print("FSHD studies included:", len(fshd_trials_analysis))
print("FSHD studies excluded:", len(fshd_trials) - len(fshd_trials_analysis))

print("\nStudies represented:", fshd_endpoints_analysis["nct_id"].nunique())
print("Total endpoints:", len(fshd_endpoints_analysis))

print("\nEndpoint types:")
print(fshd_endpoints_analysis["endpoint_type"].value_counts())

FSHD studies retrieved: 59
FSHD studies included: 30
FSHD studies excluded: 29

Studies represented: 30
Total endpoints: 291

Endpoint types:
endpoint_type
Secondary    191
Primary      100
Name: count, dtype: int64


In [ ]:
# Save the finalized FSHD analysis datasets.
fshd_trials_analysis.to_csv(
    DATA_DIR / "fshd_trials_analysis.csv",
    index=False
)

fshd_endpoints_analysis.to_csv(
    DATA_DIR / "fshd_endpoints_analysis.csv",
    index=False
)

print("Saved finalized FSHD analysis files.")

Saved finalized FSHD analysis files.


#### Pull Sickle Cell Disease

In [ ]:
scd_raw, scd_trials, scd_endpoints = collect_disease_data(
    condition_query="Sickle Cell Disease",
    disease_label="Sickle Cell Disease"
)

Sickle Cell Disease
-------------------
All studies returned: 1211
Interventional studies: 866
Studies with primary outcomes: 844
Endpoint rows: 7074

Scope decisions:
scope_decision
Include                              417
Manual review                        280
Exclude - early phase                126
Exclude - non-therapeutic purpose     43
Name: count, dtype: int64


In [ ]:
save_disease_data(
    scd_raw,
    scd_trials,
    scd_endpoints,
    file_prefix="sickle_cell_disease"
)

Saved:
./TrialSignal/data/sickle_cell_disease_interventional_raw.json
./TrialSignal/data/sickle_cell_disease_trials_study_level.csv
./TrialSignal/data/sickle_cell_disease_endpoints_raw.csv


In [ ]:
print("Primary purpose:")
print(scd_trials["primary_purpose"].value_counts(dropna=False))

print("\nPhase:")
print(scd_trials["phases"].value_counts(dropna=False))

Primary purpose:
primary_purpose
TREATMENT                   595
PREVENTION                   67
SUPPORTIVE_CARE              60
OTHER                        52
BASIC_SCIENCE                32
DIAGNOSTIC                   23
HEALTH_SERVICES_RESEARCH     22
SCREENING                     8
None                          7
Name: count, dtype: int64

Phase:
phases
NA                287
PHASE2            207
PHASE1            111
PHASE3            102
PHASE1; PHASE2     82
PHASE4             45
PHASE2; PHASE3     17
EARLY_PHASE1       15
Name: count, dtype: int64


In [ ]:
scd_manual_review = scd_trials[
    scd_trials["scope_decision"] == "Manual review"
][
    [
        "nct_id",
        "brief_title",
        "phases",
        "primary_purpose",
        "intervention_names",
        "intervention_types",
        "enrollment",
        "has_primary_outcome"
    ]
]

print("Manual review studies:", len(scd_manual_review))

pd.set_option("display.max_rows", None)
display(scd_manual_review)

Manual review studies: 280


,nct_id,brief_title,phases,primary_purpose,intervention_names,intervention_types,enrollment,has_primary_outcome
1,NCT02698761,A Comprehensive Care Plan for Pediatric Patien...,NA,SUPPORTIVE_CARE,Comprehensive Care Plan,OTHER,3.0,True
2,NCT07442851,SickleFit Exercise and Nutrition Study,NA,TREATMENT,SickleFit Exercise plus Nutrition,BEHAVIORAL,40.0,True
3,NCT03805139,Role of Ajwa Derived Polyphenols in Dyslipidae...,NA,TREATMENT,Ajwa Date (phoenix dactylifera),DIETARY_SUPPLEMENT,60.0,True
4,NCT03027258,Point-of-Delivery Prenatal Test Results Throug...,NA,OTHER,mHealth,OTHER,300.0,True
10,NCT02384590,Cognitive Behavioral Therapy and Real-time Sel...,NA,TREATMENT,Computerized cognitive behavioral therapy,BEHAVIORAL,30.0,True
11,NCT03383913,HRV-B for Symptom Management in Sickle Cell Pa...,NA,TREATMENT,Heart Rate Variability Biofeedback,BEHAVIORAL,15.0,True
15,NCT06397118,Prevention of Malnutrition in Children With Si...,NA,PREVENTION,Maternal Nutrition Education; Standard Care,BEHAVIORAL; BEHAVIORAL,146.0,True
16,NCT05200338,Preservation and Transfer of HBV Immunity Afte...,NA,OTHER,Engerix-B,DRUG,30.0,True
20,NCT05732922,ORCHARD- Optimising Home Assessment of Rural P...,NA,SUPPORTIVE_CARE,"A sphygmanometer, a digital thermometer, a pul...",DEVICE,12.0,True
22,NCT02501447,Feasibility of a Stress Reduction Intervention...,NA,TREATMENT,Guided audio-visual relaxation,BEHAVIORAL,28.0,True


In [ ]:
# Refine the SCD scope decisions for studies without a conventional trial phase.
def classify_scd_second_pass(row):
    # Preserve decisions already made by the original classifier.
    if row["scope_decision"] != "Manual review":
        return row["scope_decision"]

    # A study cannot be scored if no registered primary outcome is available.
    if not row["has_primary_outcome"]:
        return "Exclude - missing registered outcome data"

    # These study purposes do not evaluate direct therapeutic patient benefit.
    if row["primary_purpose"] in {
        "SCREENING",
        "HEALTH_SERVICES_RESEARCH"
    }:
        return "Exclude - non-therapeutic purpose"

    # Phase is often not applicable to behavioral, device, rehabilitation,
    # nutrition, and other non-drug interventions.
    if row["primary_purpose"] in {
        "TREATMENT",
        "PREVENTION"
    }:
        return "Include"

    # SUPPORTIVE_CARE, OTHER, and missing purposes still need human review.
    return "Manual review"


scd_trials["scope_decision"] = scd_trials.apply(
    classify_scd_second_pass,
    axis=1
)

print(scd_trials["scope_decision"].value_counts())

print(
    "\nRemaining manual review studies:",
    (scd_trials["scope_decision"] == "Manual review").sum()
)

scope_decision
Include                                      557
Exclude - early phase                        126
Manual review                                107
Exclude - non-therapeutic purpose             72
Exclude - missing registered outcome data      4
Name: count, dtype: int64

Remaining manual review studies: 107


In [ ]:
# Add the registered primary endpoint names to the remaining manual-review table.
scd_primary_endpoint_names = (
    scd_endpoints[
        scd_endpoints["endpoint_type"] == "Primary"
    ]
    .groupby("nct_id")["endpoint_name"]
    .apply(lambda x: " | ".join(x.astype(str)))
    .rename("primary_endpoint_names")
)

scd_manual_review_2 = (
    scd_trials[
        scd_trials["scope_decision"] == "Manual review"
    ][
        [
            "nct_id",
            "brief_title",
            "phases",
            "primary_purpose",
            "conditions",
            "intervention_names",
            "intervention_types",
            "enrollment",
            "has_primary_outcome"
        ]
    ]
    .merge(
        scd_primary_endpoint_names,
        on="nct_id",
        how="left"
    )
)

print("Remaining manual review studies:", len(scd_manual_review_2))

print("\nPurpose breakdown:")
print(
    scd_manual_review_2["primary_purpose"]
    .value_counts(dropna=False)
)

pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

display(scd_manual_review_2)

Remaining manual review studies: 107

Purpose breakdown:
primary_purpose
SUPPORTIVE_CARE    57
OTHER              46
None                4
Name: count, dtype: int64


,nct_id,brief_title,phases,primary_purpose,conditions,intervention_names,intervention_types,enrollment,has_primary_outcome,primary_endpoint_names
0,NCT02698761,A Comprehensive Care Plan for Pediatric Patients With Vaso-Occlusive Crises,NA,SUPPORTIVE_CARE,Sickle Cell Disease; Sickle Cell Crisis; Vaso-Occlusive Crises,Comprehensive Care Plan,OTHER,3.0,True,Length of Stay
1,NCT03027258,Point-of-Delivery Prenatal Test Results Through mHealth to Improve Birth Outcome,NA,OTHER,HIV; Sickle Cell Disease; Hepatitis B,mHealth,OTHER,300.0,True,Reach | Effectiveness | Adoption | Acceptability
2,NCT05200338,Preservation and Transfer of HBV Immunity After Allogeneic HSCT for SCD,NA,OTHER,Sickle Cell Disease,Engerix-B,DRUG,30.0,True,The proportion of SCD patients with a preserved anti-HBs response and HBV-specific cellular response following non-myeloablative allogeneic HSCT with an HBV naive MSD at 12 months post-transplantation as compared to SCD patients without HSCT.
3,NCT05732922,ORCHARD- Optimising Home Assessment of Rural Patients,NA,SUPPORTIVE_CARE,Organ Transplant; Cancer; Inborn Errors of Metabolism; Immunosuppression; COPD Asthma; Sickle Cell Disease,"A sphygmanometer, a digital thermometer, a pulse oximeter and a peak flow meter will be given to participants to use during their telemedicine consultations with health care providers.",DEVICE,12.0,True,Short questionnaire | Interviews
4,NCT06529874,Pressure Pain Tolerance in Relation to Balance and Strength in Children,NA,OTHER,Sickle Cell Anemia,Pressure pain tolerance measurement using Digital Pressure Algometer; Balance assessment using HUMAC balance system; Lower limb muscular strength using Lafayette hand held dynamometer,OTHER; OTHER; OTHER,80.0,True,Pressure pain tolerance | Balance assessment | Lower limb muscle strength
5,NCT02608580,"Survey in a Population of Sickle Cell Disease Patients to Evaluate the Transition Between the Queen Fabiola Children Hospital and the CHU Brugmann Hospital, and the Quality of the Hospital Care Within the CHU Brugmann Hospital.",NA,OTHER,Sickle Cell Disease,Survey,OTHER,30.0,True,Factors impacting the quality of transition between the pediatric and adult care system
6,NCT07402811,Mindfulness-Based Intervention for Pain and Sleep in Adolescents and Young Adults With Sickle Cell Disease,NA,SUPPORTIVE_CARE,Chronic Pain; Sickle Cell Disease; Sleep Disturbance,Mobile bindfulness-based intervention,BEHAVIORAL,25.0,True,Participant retention rate | Attrition rate and reasons for withdrawal | Weekly module completion rate | Acceptability of the Mindfulness-Based Intervention
7,NCT07719972,Universal Newborn Screening For Sickle Cell Disease In Mozambique,NA,OTHER,Sickle Cell Disease,Point-of-care testing (POCT) -Patient participants; Point-of-care testing (POCT) - Healthy control participants; Context Assessment for Community Health (COACH) survey; Semi-structured interview,OTHER; OTHER; OTHER; OTHER,6750.0,True,Percentage of eligible population screened for Sickle Cell Disease | Percentage of new SCD cases linked to care
8,NCT06296654,Effect of Empowerment-Based Interventions on Self-Efficacy and Self-Care Capacity Among Adults With Sickle Cell Disease,NA,SUPPORTIVE_CARE,Self-Efficacy; Self-Care,Empowerment-Based Interventions; Standard Health Education,BEHAVIORAL; BEHAVIORAL,68.0,True,Sickle Cell Self-Efficacy
9,NCT00860782,Parent Educational Program for Children With Sickle Cell Disease,NA,SUPPORTIVE_CARE,"Hemoglobin SC Disease; Anemia, Sickle Cell",Parent Educational Support,BEHAVIORAL,73.0,True,"Academic achievement as measured by the Woodcock Johnson Tests of Achievement, Third Edition"


In [ ]:
scd_manual_review_2.to_csv(
    DATA_DIR / "scd_manual_review_round2.csv",
    index=False
)

print("Saved SCD round-2 manual review table.")

Saved SCD round-2 manual review table.


In [ ]:
# Resolve all remaining manually reviewed SCD studies.
scd_manual_decisions = {
    "NCT02698761": "Include",
    "NCT03027258": "Exclude - broad/non-therapeutic study",
    "NCT05200338": "Exclude - ancillary immunogenicity study",
    "NCT05732922": "Exclude - broad multi-disease study",
    "NCT06529874": "Exclude - assessment study",
    "NCT02608580": "Exclude - survey/care-process study",
    "NCT07402811": "Exclude - feasibility-focused study",
    "NCT07719972": "Exclude - screening/implementation study",
    "NCT06296654": "Include",
    "NCT00860782": "Include",
    "NCT06930703": "Include",
    "NCT02475993": "Include",
    "NCT02620488": "Include",
    "NCT04892160": "Exclude - broad multi-disease study",
    "NCT05018728": "Include",
    "NCT05519111": "Include",
    "NCT01589926": "Include",
    "NCT00600665": "Include",
    "NCT06139510": "Exclude - assessment study",
    "NCT06619197": "Include",
    "NCT06351462": "Exclude - observational study",
    "NCT04606160": "Exclude - feasibility-focused study",
    "NCT04122378": "Exclude - feasibility-focused study",
    "NCT04289311": "Exclude - feasibility-focused study",
    "NCT05383911": "Exclude - feasibility-focused study",
    "NCT02222246": "Include",
    "NCT03006718": "Include",
    "NCT03938324": "Exclude - broad multi-disease study",
    "NCT06761560": "Include",
    "NCT06260891": "Include",
    "NCT01945073": "Exclude - education/care-process study",
    "NCT03417947": "Include",
    "NCT02186418": "Include",
    "NCT05457790": "Exclude - feasibility-focused study",
    "NCT04076683": "Include",
    "NCT00415727": "Include",
    "NCT02895841": "Include",
    "NCT07703566": "Include",
    "NCT06098755": "Exclude - broad multi-disease study",
    "NCT02651272": "Exclude - safety-focused study",
    "NCT04602728": "Include",
    "NCT02731157": "Include",
    "NCT04678037": "Exclude - outcome measure/feasibility study",
    "NCT04011345": "Include",
    "NCT03041909": "Exclude - safety-focused study",
    "NCT07562451": "Include",
    "NCT07797582": "Include",
    "NCT03977532": "Exclude - biomarker/assessment study",
    "NCT05228756": "Exclude - feasibility-focused study",
    "NCT00639392": "Include",
    "NCT07316478": "Include",
    "NCT02578017": "Include",
    "NCT03648710": "Include",
    "NCT06110754": "Include",
    "NCT04917120": "Exclude - observational study",
    "NCT06031714": "Exclude - mechanistic/broad study",
    "NCT01518218": "Include",
    "NCT03976180": "Include",
    "NCT06872333": "Exclude - broad multi-disease study",
    "NCT04181944": "Exclude - survey/assessment study",
    "NCT02769845": "Include",
    "NCT07530965": "Exclude - feasibility-focused study",
    "NCT02522104": "Exclude - PK study",
    "NCT06853158": "Exclude - feasibility-focused study",
    "NCT06797583": "Exclude - feasibility-focused study",
    "NCT03477552": "Exclude - procedural support study",
    "NCT00127543": "Exclude - broad education study",
    "NCT03585543": "Exclude - feasibility-focused study",
    "NCT06773715": "Include",
    "NCT06691867": "Include",
    "NCT03387033": "Exclude - feasibility-focused study",
    "NCT05153044": "Exclude - unrelated survey/serology study",
    "NCT05315908": "Exclude - broad multi-disease study",
    "NCT04173026": "Exclude - implementation study",
    "NCT03368261": "Exclude - epidemiology/assessment study",
    "NCT04579926": "Include",
    "NCT03694548": "Exclude - feasibility-focused study",
    "NCT07648043": "Exclude - feasibility-focused study",
    "NCT04208529": "Exclude - safety follow-up study",
    "NCT05572294": "Exclude - feasibility-focused study",
    "NCT04817345": "Exclude - collection/safety study",
    "NCT03240731": "Include",
    "NCT04756375": "Include",
    "NCT03353584": "Include",
    "NCT00276146": "Include",
    "NCT06823219": "Include",
    "NCT03984500": "Exclude - sickle cell trait education study",
    "NCT04121247": "Exclude - education/care-process study",
    "NCT03291613": "Exclude - usability/feasibility study",
    "NCT05632289": "Include",
    "NCT04191213": "Include",
    "NCT04691323": "Exclude - observational study",
    "NCT02596087": "Exclude - broad health-services study",
    "NCT07802210": "Include",
    "NCT07155031": "Exclude - mechanistic/kinetics study",
    "NCT04675645": "Exclude - feasibility-focused study",
    "NCT02987725": "Exclude - mechanistic laboratory study",
    "NCT07246265": "Exclude - PK/assessment study",
    "NCT04688411": "Exclude - feasibility-focused study",
    "NCT02394587": "Exclude - feasibility-focused study",
    "NCT06735625": "Exclude - biomarker study",
    "NCT03577691": "Exclude - decision-support/education study",
    "NCT05336201": "Include",
    "NCT03903133": "Include",
    "NCT02960503": "Include",
    "NCT04109820": "Exclude - mechanistic study",
    "NCT04772391": "Exclude - feasibility-focused study"
}

for nct_id, decision in scd_manual_decisions.items():
    scd_trials.loc[
        scd_trials["nct_id"] == nct_id,
        "scope_decision"
    ] = decision

print(scd_trials["scope_decision"].value_counts())

print(
    "\nRemaining manual review studies:",
    (scd_trials["scope_decision"] == "Manual review").sum()
)

scope_decision
Include                                        605
Exclude - early phase                          126
Exclude - non-therapeutic purpose               72
Exclude - feasibility-focused study             19
Exclude - broad multi-disease study              6
Exclude - missing registered outcome data        4
Exclude - observational study                    3
Exclude - education/care-process study           2
Exclude - assessment study                       2
Exclude - safety-focused study                   2
Exclude - broad/non-therapeutic study            1
Exclude - screening/implementation study         1
Exclude - survey/care-process study              1
Exclude - ancillary immunogenicity study         1
Exclude - outcome measure/feasibility study      1
Exclude - biomarker/assessment study             1
Exclude - mechanistic/broad study                1
Exclude - survey/assessment study                1
Exclude - PK study                               1
Exclude - proced

In [ ]:
# Create the final SCD study-level cohort.
scd_trials_analysis = scd_trials[
    scd_trials["scope_decision"] == "Include"
].copy()

included_scd_ids = set(scd_trials_analysis["nct_id"])

scd_endpoints_analysis = scd_endpoints[
    scd_endpoints["nct_id"].isin(included_scd_ids)
].copy()

print("SCD studies retrieved:", len(scd_trials))
print("SCD studies included:", len(scd_trials_analysis))
print("SCD studies excluded:", len(scd_trials) - len(scd_trials_analysis))

print("\nStudies represented:", scd_endpoints_analysis["nct_id"].nunique())
print("Total endpoints:", len(scd_endpoints_analysis))

print("\nEndpoint types:")
print(scd_endpoints_analysis["endpoint_type"].value_counts())

SCD studies retrieved: 866
SCD studies included: 605
SCD studies excluded: 261

Studies represented: 590
Total endpoints: 4858

Endpoint types:
endpoint_type
Secondary    3664
Primary      1194
Name: count, dtype: int64


In [ ]:
# Save the finalized SCD analysis datasets.
scd_trials_analysis.to_csv(
    DATA_DIR / "sickle_cell_disease_trials_analysis.csv",
    index=False
)

scd_endpoints_analysis.to_csv(
    DATA_DIR / "sickle_cell_disease_endpoints_analysis.csv",
    index=False
)

print("Saved finalized SCD analysis files.")

Saved finalized SCD analysis files.


In [ ]:
# Identify included SCD studies that produced no endpoint rows.
scd_endpoint_ids = set(scd_endpoints_analysis["nct_id"])

scd_missing_endpoint_studies = scd_trials_analysis[
    ~scd_trials_analysis["nct_id"].isin(scd_endpoint_ids)
][
    [
        "nct_id",
        "brief_title",
        "phases",
        "primary_purpose",
        "has_primary_outcome",
        "has_secondary_outcome"
    ]
]

print("Included studies with no endpoint rows:", len(scd_missing_endpoint_studies))
display(scd_missing_endpoint_studies)

Included studies with no endpoint rows: 15


,nct_id,brief_title,phases,primary_purpose,has_primary_outcome,has_secondary_outcome
149,NCT00000602,Pediatric Hydroxyurea in Sickle Cell Anemia (PED HUG),PHASE2,TREATMENT,False,False
166,NCT00066079,Home Based Massage and Relaxation for Sickle Cell Pain,PHASE2,TREATMENT,False,False
200,NCT00001716,Effects of Nitric Oxide and Nitroglycerin in Patients With Sickle Cell Anemia,PHASE2,TREATMENT,False,False
268,NCT00006182,Stroke Prevention in Sickle Cell Anemia (STOP 2),PHASE3,PREVENTION,False,False
310,NCT00040417,Bone Marrow Transplant From Donor Using Less Toxic Conditioning for Patient With High Risk Hemoglobinopathies,PHASE2,TREATMENT,False,False
351,NCT00004492,Phase I/II Randomized Study of Hydroxyurea With or Without Clotrimazole in Patients With Sickle Cell Anemia,PHASE1; PHASE2,TREATMENT,False,False
401,NCT00000592,Stroke Prevention in Sickle Cell Anemia (STOP 1),PHASE3,PREVENTION,False,False
415,NCT00005782,Mobilization and Handling of Stem Cells for Transplant From Healthy Volunteers With Sickle Cell Trait,PHASE3,TREATMENT,False,False
505,NCT00029731,Arginine Treatment of Acute Chest Syndrome (Pneumonia) in Sickle Cell Disease Patients,PHASE2,TREATMENT,False,False
591,NCT00040469,Bone Marrow Transplant From Related Donor for Patients With High Risk Hemoglobinopathies,PHASE2,TREATMENT,False,False


In [ ]:
print(
    "Included studies without a primary outcome:",
    (~scd_trials_analysis["has_primary_outcome"]).sum()
)

display(
    scd_trials_analysis[
        ~scd_trials_analysis["has_primary_outcome"]
    ][
        [
            "nct_id",
            "brief_title",
            "phases",
            "primary_purpose",
            "has_primary_outcome",
            "has_secondary_outcome"
        ]
    ]
)

Included studies without a primary outcome: 15


,nct_id,brief_title,phases,primary_purpose,has_primary_outcome,has_secondary_outcome
149,NCT00000602,Pediatric Hydroxyurea in Sickle Cell Anemia (PED HUG),PHASE2,TREATMENT,False,False
166,NCT00066079,Home Based Massage and Relaxation for Sickle Cell Pain,PHASE2,TREATMENT,False,False
200,NCT00001716,Effects of Nitric Oxide and Nitroglycerin in Patients With Sickle Cell Anemia,PHASE2,TREATMENT,False,False
268,NCT00006182,Stroke Prevention in Sickle Cell Anemia (STOP 2),PHASE3,PREVENTION,False,False
310,NCT00040417,Bone Marrow Transplant From Donor Using Less Toxic Conditioning for Patient With High Risk Hemoglobinopathies,PHASE2,TREATMENT,False,False
351,NCT00004492,Phase I/II Randomized Study of Hydroxyurea With or Without Clotrimazole in Patients With Sickle Cell Anemia,PHASE1; PHASE2,TREATMENT,False,False
401,NCT00000592,Stroke Prevention in Sickle Cell Anemia (STOP 1),PHASE3,PREVENTION,False,False
415,NCT00005782,Mobilization and Handling of Stem Cells for Transplant From Healthy Volunteers With Sickle Cell Trait,PHASE3,TREATMENT,False,False
505,NCT00029731,Arginine Treatment of Acute Chest Syndrome (Pneumonia) in Sickle Cell Disease Patients,PHASE2,TREATMENT,False,False
591,NCT00040469,Bone Marrow Transplant From Related Donor for Patients With High Risk Hemoglobinopathies,PHASE2,TREATMENT,False,False


In [ ]:
# Exclude included SCD studies that have no registered endpoint data.
missing_endpoint_ids = set(
    scd_missing_endpoint_studies["nct_id"]
)

scd_trials.loc[
    scd_trials["nct_id"].isin(missing_endpoint_ids),
    "scope_decision"
] = "Exclude - missing registered outcome data"

print(scd_trials["scope_decision"].value_counts())

scope_decision
Include                                        590
Exclude - early phase                          126
Exclude - non-therapeutic purpose               72
Exclude - missing registered outcome data       19
Exclude - feasibility-focused study             19
Exclude - broad multi-disease study              6
Exclude - observational study                    3
Exclude - education/care-process study           2
Exclude - assessment study                       2
Exclude - safety-focused study                   2
Exclude - broad/non-therapeutic study            1
Exclude - screening/implementation study         1
Exclude - survey/care-process study              1
Exclude - ancillary immunogenicity study         1
Exclude - outcome measure/feasibility study      1
Exclude - biomarker/assessment study             1
Exclude - mechanistic/broad study                1
Exclude - survey/assessment study                1
Exclude - PK study                               1
Exclude - proced

In [ ]:
# Rebuild the final SCD cohort after excluding studies with no endpoint data.
scd_trials_analysis = scd_trials[
    scd_trials["scope_decision"] == "Include"
].copy()

included_scd_ids = set(scd_trials_analysis["nct_id"])

scd_endpoints_analysis = scd_endpoints[
    scd_endpoints["nct_id"].isin(included_scd_ids)
].copy()

print("SCD studies retrieved:", len(scd_trials))
print("SCD studies included:", len(scd_trials_analysis))
print("SCD studies excluded:", len(scd_trials) - len(scd_trials_analysis))

print("\nStudies represented:", scd_endpoints_analysis["nct_id"].nunique())
print("Total endpoints:", len(scd_endpoints_analysis))

print("\nEndpoint types:")
print(scd_endpoints_analysis["endpoint_type"].value_counts())

SCD studies retrieved: 866
SCD studies included: 590
SCD studies excluded: 276

Studies represented: 590
Total endpoints: 4858

Endpoint types:
endpoint_type
Secondary    3664
Primary      1194
Name: count, dtype: int64


In [ ]:
# Save the corrected finalized SCD analysis datasets.
scd_trials_analysis.to_csv(
    DATA_DIR / "sickle_cell_disease_trials_analysis.csv",
    index=False
)

scd_endpoints_analysis.to_csv(
    DATA_DIR / "sickle_cell_disease_endpoints_analysis.csv",
    index=False
)

print("Saved corrected finalized SCD analysis files.")

Saved corrected finalized SCD analysis files.


#### Pull Dravet Syndrome

In [ ]:
dravet_raw, dravet_trials, dravet_endpoints = collect_disease_data(
    condition_query="Dravet Syndrome",
    disease_label="Dravet Syndrome"
)

Dravet Syndrome
---------------
All studies returned: 103
Interventional studies: 65
Studies with primary outcomes: 65
Endpoint rows: 654

Scope decisions:
scope_decision
Include                              49
Manual review                         8
Exclude - early phase                 6
Exclude - non-therapeutic purpose     2
Name: count, dtype: int64


In [ ]:
save_disease_data(
    dravet_raw,
    dravet_trials,
    dravet_endpoints,
    file_prefix="dravet_syndrome"
)

Saved:
./TrialSignal/data/dravet_syndrome_interventional_raw.json
./TrialSignal/data/dravet_syndrome_trials_study_level.csv
./TrialSignal/data/dravet_syndrome_endpoints_raw.csv


In [ ]:
print("Primary purpose:")
print(dravet_trials["primary_purpose"].value_counts(dropna=False))

print("\nPhase:")
print(dravet_trials["phases"].value_counts(dropna=False))

Primary purpose:
primary_purpose
TREATMENT          57
PREVENTION          2
OTHER               2
BASIC_SCIENCE       1
SUPPORTIVE_CARE     1
SCREENING           1
DIAGNOSTIC          1
Name: count, dtype: int64

Phase:
phases
PHASE3            27
PHASE2            11
PHASE1; PHASE2     8
NA                 8
PHASE4             5
PHASE1             5
EARLY_PHASE1       1
Name: count, dtype: int64


In [ ]:
dravet_manual_review = dravet_trials[
    dravet_trials["scope_decision"] == "Manual review"
][
    [
        "nct_id",
        "brief_title",
        "phases",
        "primary_purpose",
        "conditions",
        "intervention_names",
        "intervention_types",
        "enrollment",
        "has_primary_outcome"
    ]
]

print("Manual review studies:", len(dravet_manual_review))

pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

display(dravet_manual_review)

Manual review studies: 8


,nct_id,brief_title,phases,primary_purpose,conditions,intervention_names,intervention_types,enrollment,has_primary_outcome
14,NCT06447675,Efficacy and Safety Evaluation of Hyperthermic Baths in the Treatment of Seizures in CDKL5 Deficiency Disorder,NA,TREATMENT,CDKL5 Deficiency Disorder,Hyperthermic Baths,OTHER,8,True
23,NCT05472389,Neurodevelopmental Impact of Epilepsy on Autonomic Function in Dravet Syndrome,NA,OTHER,Dravet Syndrome; Epilepsy,Video-electroencephalography; Blood Samples,OTHER; OTHER,92,True
24,NCT05026398,Fenfluramine and Cognition,PHASE4,OTHER,Cognitive Function,Fenfluramine; Placebo,DRUG; OTHER,56,True
40,NCT06448663,Targeting the Gut to Improve Seizure Control in CDKL5 Deficiency Disorder (CDD),NA,TREATMENT,CDKL5,"alpha-lactalbumin, fructooligosaccharides, inulin; alpha-lactalbumin, sodium butyrate, fructooligosaccharides, inulin",DIETARY_SUPPLEMENT; DIETARY_SUPPLEMENT,20,True
43,NCT07225231,Clinical Utility of Reduced EEG Home Monitoring in Fenfluramine Titration for Dravet and LGS,NA,SUPPORTIVE_CARE,Dravet Syndrome (DS); Lennox Gastaut Syndrome (LGS),Byteflies Remote patient monitoring solution (with reduced EEG),DEVICE,20,True
48,NCT07013331,A PET-MRI Study of Serotoninergic Brainstem Pathway in Patients With Dravet Syndrome,NA,SCREENING,Epilepsy; Dravet Syndrome; Drug Resistant Epilepsy; Healthy Controls,[18F]MPPF PET-MRI,OTHER,30,True
51,NCT07602205,Efficacy and Safety of Daily Home-based Hyperthermic Baths for Reducing Epileptic Seizures in CDKL5 Deficiency Disorder: A PROBE Clinical Trial,NA,TREATMENT,CDKL5 Deficiency Disorder,Home-based hyperthermic Baths,BEHAVIORAL,34,True
54,NCT03254680,Turmeric as Treatment in Epilepsy,NA,TREATMENT,Epilepsy; Dravet Syndrome; Lennox-Gastaut Syndrome; Tuberous Sclerosis; Focal Seizures,Turmeric,DIETARY_SUPPLEMENT,0,True


In [ ]:
# Resolve all manually reviewed Dravet studies.
dravet_manual_decisions = {
    "NCT06447675": "Exclude - wrong disease population",
    "NCT05472389": "Exclude - mechanistic/assessment study",
    "NCT05026398": "Exclude - wrong study population",
    "NCT06448663": "Exclude - wrong disease population",
    "NCT07225231": "Exclude - monitoring/broad multi-disease study",
    "NCT07013331": "Exclude - mechanistic/screening study",
    "NCT07602205": "Exclude - wrong disease population",
    "NCT03254680": "Exclude - broad multi-disease study"
}

for nct_id, decision in dravet_manual_decisions.items():
    dravet_trials.loc[
        dravet_trials["nct_id"] == nct_id,
        "scope_decision"
    ] = decision

print(dravet_trials["scope_decision"].value_counts())

print(
    "\nRemaining manual review studies:",
    (dravet_trials["scope_decision"] == "Manual review").sum()
)

scope_decision
Include                                           49
Exclude - early phase                              6
Exclude - wrong disease population                 3
Exclude - non-therapeutic purpose                  2
Exclude - mechanistic/assessment study             1
Exclude - wrong study population                   1
Exclude - monitoring/broad multi-disease study     1
Exclude - mechanistic/screening study              1
Exclude - broad multi-disease study                1
Name: count, dtype: int64

Remaining manual review studies: 0


In [ ]:
# Create the final Dravet analysis cohort.
dravet_trials_analysis = dravet_trials[
    dravet_trials["scope_decision"] == "Include"
].copy()

included_dravet_ids = set(dravet_trials_analysis["nct_id"])

dravet_endpoints_analysis = dravet_endpoints[
    dravet_endpoints["nct_id"].isin(included_dravet_ids)
].copy()

print("Dravet studies retrieved:", len(dravet_trials))
print("Dravet studies included:", len(dravet_trials_analysis))
print("Dravet studies excluded:", len(dravet_trials) - len(dravet_trials_analysis))

print("\nStudies represented:", dravet_endpoints_analysis["nct_id"].nunique())
print("Total endpoints:", len(dravet_endpoints_analysis))

print("\nEndpoint types:")
print(dravet_endpoints_analysis["endpoint_type"].value_counts())

Dravet studies retrieved: 65
Dravet studies included: 49
Dravet studies excluded: 16

Studies represented: 49
Total endpoints: 527

Endpoint types:
endpoint_type
Secondary    331
Primary      196
Name: count, dtype: int64


In [ ]:
dravet_trials_analysis.to_csv(
    DATA_DIR / "dravet_syndrome_trials_analysis.csv",
    index=False
)

dravet_endpoints_analysis.to_csv(
    DATA_DIR / "dravet_syndrome_endpoints_analysis.csv",
    index=False
)

print("Saved finalized Dravet analysis files.")

Saved finalized Dravet analysis files.


#### Pull SMA

In [ ]:
sma_raw, sma_trials, sma_endpoints = collect_disease_data(
    condition_query="Spinal Muscular Atrophy",
    disease_label="Spinal Muscular Atrophy"
)

Spinal Muscular Atrophy
-----------------------
All studies returned: 466
Interventional studies: 285
Studies with primary outcomes: 281
Endpoint rows: 2532

Scope decisions:
scope_decision
Include                              121
Manual review                        118
Exclude - early phase                 24
Exclude - non-therapeutic purpose     22
Name: count, dtype: int64


In [ ]:
save_disease_data(
    sma_raw,
    sma_trials,
    sma_endpoints,
    file_prefix="spinal_muscular_atrophy"
)

Saved:
./TrialSignal/data/spinal_muscular_atrophy_interventional_raw.json
./TrialSignal/data/spinal_muscular_atrophy_trials_study_level.csv
./TrialSignal/data/spinal_muscular_atrophy_endpoints_raw.csv


In [ ]:
print("Primary purpose:")
print(sma_trials["primary_purpose"].value_counts(dropna=False))

print("\nPhase:")
print(sma_trials["phases"].value_counts(dropna=False))

print("\nScope decisions:")
print(sma_trials["scope_decision"].value_counts())

Primary purpose:
primary_purpose
TREATMENT                   201
OTHER                        26
SUPPORTIVE_CARE              14
BASIC_SCIENCE                12
DIAGNOSTIC                   11
PREVENTION                    7
DEVICE_FEASIBILITY            6
None                          5
SCREENING                     2
HEALTH_SERVICES_RESEARCH      1
Name: count, dtype: int64

Phase:
phases
NA                136
PHASE2             45
PHASE3             34
PHASE1; PHASE2     26
PHASE1             21
PHASE4             11
PHASE2; PHASE3      9
EARLY_PHASE1        3
Name: count, dtype: int64

Scope decisions:
scope_decision
Include                              121
Manual review                        118
Exclude - early phase                 24
Exclude - non-therapeutic purpose     22
Name: count, dtype: int64


In [ ]:
sma_manual_review = sma_trials[
    sma_trials["scope_decision"] == "Manual review"
][
    [
        "nct_id",
        "brief_title",
        "phases",
        "primary_purpose",
        "conditions",
        "intervention_names",
        "intervention_types",
        "enrollment",
        "has_primary_outcome",
        "has_secondary_outcome"
    ]
]

print("Manual review studies:", len(sma_manual_review))

pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

display(sma_manual_review)

Manual review studies: 118


,nct_id,brief_title,phases,primary_purpose,conditions,intervention_names,intervention_types,enrollment,has_primary_outcome,has_secondary_outcome
0,NCT06094205,Feasibility of the BrainGate2 Neural Interface System in Persons With Tetraplegia (BG-Speech-02),NA,OTHER,Anarthria; Dysarthria; Tetraplegia; Spinal Cord Injuries; Amyotrophic Lateral Sclerosis; Brain Stem Infarctions; Locked-in Syndrome; Muscular Dystrophies,BrainGate Neural Interface System,DEVICE,2.0,True,True
1,NCT00004553,Electromyography to Diagnose Neuromuscular Disorders,NA,None,Healthy; Neuromuscular Disease; Postpoliomyelitis Syndrome,Electromyography,PROCEDURE,30.0,False,False
2,NCT06363357,"The Effect of a Muscle-mimicking, Fabric-type Shoulder Orthosis on Functional Movements of the Upper Limb in Patients With Neuromuscular Disorder",NA,SUPPORTIVE_CARE,"Muscular Dystrophy, Duchenne; Orthotic Devices; Upper Extremity; Neuromuscular Diseases (NMD); Fascioscapulohumeral Muscular Dystrophy; Spinal Muscular Atrophy (SMA); ALS (Amyotrophic Lateral Sclerosis); LGMD; SCI - Spinal Cord Injury",Shoulder orthosis,DEVICE,8.0,True,False
3,NCT05544994,The Effect of Aerobic Exercise Training in Patients With Type III Spinal Muscular Atrophy,NA,TREATMENT,Neuromuscular Diseases; Spinal Muscular Atrophy Type 3,Aerobic Exercise Training; Home exercise program,OTHER; OTHER,20.0,True,True
4,NCT06765070,Efficacy and Usability of the EXPLORER Exoskeleton in Children With Neurodevelopmental Disorders in Natural Environments,NA,TREATMENT,Neuromuscular Disorders; Neurodevelopmental Disorders,EXPLORER,DEVICE,15.0,True,True
7,NCT05589987,"Spinal Muscular Atrophy on Wheels, Using Power Mobility",NA,TREATMENT,Spinal Muscular Atrophy,power mobility,DEVICE,20.0,True,True
9,NCT02341950,Clinical Trial of a Serious Game for Individuals With SCI/D,NA,PREVENTION,Spinal Cord Injury; Spinal Cord Involvement; Spina Bifida; Transverse Myelitis; Polio and Post-polio Syndrome; Syringomyelia; Spinal Stenosis; Spinal Neoplasms; Spinal Cord Diseases,SCI Hard,BEHAVIORAL,184.0,True,True
17,NCT02309983,Activity Dependent Rehabilitation Model to Improve Bone and Muscle Outcomes,NA,TREATMENT,"Disuse Osteoporosis; Atrophy, Disuse; Nervous System; Structural; Marker; Skeletal Muscle Hypertrophy",Electrical Stimulation; Stand Retraining with Body Weight Support (BWS),OTHER; OTHER,21.0,True,True
18,NCT05724173,Feasibility of the BrainGate2 Neural Interface System in Persons With Tetraplegia,NA,OTHER,Anarthria; Dysarthria; Tetraplegia; Spinal Cord Injuries; Amyotrophic Lateral Sclerosis; Brain Stem Infarctions; Locked in Syndrome; Muscular Dystrophies,BrainGate Neural Interface System,DEVICE,3.0,True,True
19,NCT06419322,"Acceptability, Feasibility, Safety and Efficacy of a Optimized Rehabilitation Program for Treated Patients With Spinal Muscular Atrophy (SMA).",NA,OTHER,Spinal Muscular Atrophy,Optimized rehabilitation program,OTHER,14.0,True,True


In [ ]:
print("Most common condition strings:")

print(
    sma_trials["conditions"]
    .value_counts()
    .head(20)
)

Most common condition strings:
conditions
Spinal Muscular Atrophy                                  54
Muscular Atrophy, Spinal                                 23
Spinal Muscular Atrophy (SMA)                            16
Spinal Muscular Atrophy Type I                            5
Spinal Muscular Atrophy Type 3                            4
Prostate Cancer                                           4
Spinal Muscular Atrophy 1                                 4
SMA - Spinal Muscular Atrophy                             4
Metastatic Castration Resistant Prostate Cancer           3
Post-polio Syndrome                                       3
Amyotrophic Lateral Sclerosis (ALS)                       3
Postpoliomyelitis Syndrome                                3
SMA                                                       3
Breast Cancer                                             3
Spinal and Bulbar Muscular Atrophy                        3
Infantile Spinal Muscular Atrophy                         

In [ ]:
# Reduce the SMA manual-review group by removing clearly out-of-scope records.
import re

def mentions_sma(row):
    text = (
        str(row["brief_title"]) + " " +
        str(row["conditions"])
    )

    return (
        re.search(
            r"spinal muscular atrophy|muscular atrophy,\s*spinal|\bSMA\b",
            text,
            flags=re.IGNORECASE
        )
        is not None
    )


def classify_sma_second_pass(row):
    # Preserve decisions already made.
    if row["scope_decision"] != "Manual review":
        return row["scope_decision"]

    # No registered outcomes means the study cannot be scored.
    if (
        not row["has_primary_outcome"]
        and not row["has_secondary_outcome"]
    ):
        return "Exclude - missing registered outcome data"

    # These purposes are not direct therapeutic-benefit trials.
    if row["primary_purpose"] in {
        "SCREENING",
        "HEALTH_SERVICES_RESEARCH",
        "DEVICE_FEASIBILITY"
    }:
        return "Exclude - non-therapeutic purpose"

    # Remove records returned by the search that are not actually SMA studies.
    if not mentions_sma(row):
        return "Exclude - wrong disease population"

    # Everything genuinely SMA-related stays for human review.
    return "Manual review"


sma_trials["scope_decision"] = sma_trials.apply(
    classify_sma_second_pass,
    axis=1
)

print(sma_trials["scope_decision"].value_counts())

print(
    "\nRemaining manual review studies:",
    (sma_trials["scope_decision"] == "Manual review").sum()
)

scope_decision
Include                                      121
Manual review                                 66
Exclude - wrong disease population            41
Exclude - non-therapeutic purpose             31
Exclude - early phase                         24
Exclude - missing registered outcome data      2
Name: count, dtype: int64

Remaining manual review studies: 66


In [ ]:
# Add primary endpoint names to the remaining SMA manual-review studies.
sma_primary_endpoint_names = (
    sma_endpoints[
        sma_endpoints["endpoint_type"] == "Primary"
    ]
    .groupby("nct_id")["endpoint_name"]
    .apply(lambda x: " | ".join(x.astype(str)))
    .rename("primary_endpoint_names")
)

sma_manual_review_2 = (
    sma_trials[
        sma_trials["scope_decision"] == "Manual review"
    ][
        [
            "nct_id",
            "brief_title",
            "phases",
            "primary_purpose",
            "conditions",
            "intervention_names",
            "intervention_types",
            "enrollment",
            "has_primary_outcome"
        ]
    ]
    .merge(
        sma_primary_endpoint_names,
        on="nct_id",
        how="left"
    )
)

print("Remaining manual review studies:", len(sma_manual_review_2))

print("\nPurpose breakdown:")
print(
    sma_manual_review_2["primary_purpose"]
    .value_counts(dropna=False)
)

pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

display(sma_manual_review_2)

Remaining manual review studies: 66

Purpose breakdown:
primary_purpose
TREATMENT          38
OTHER              15
SUPPORTIVE_CARE     9
PREVENTION          3
None                1
Name: count, dtype: int64


,nct_id,brief_title,phases,primary_purpose,conditions,intervention_names,intervention_types,enrollment,has_primary_outcome,primary_endpoint_names
0,NCT06363357,"The Effect of a Muscle-mimicking, Fabric-type Shoulder Orthosis on Functional Movements of the Upper Limb in Patients With Neuromuscular Disorder",NA,SUPPORTIVE_CARE,"Muscular Dystrophy, Duchenne; Orthotic Devices; Upper Extremity; Neuromuscular Diseases (NMD); Fascioscapulohumeral Muscular Dystrophy; Spinal Muscular Atrophy (SMA); ALS (Amyotrophic Lateral Sclerosis); LGMD; SCI - Spinal Cord Injury",Shoulder orthosis,DEVICE,8.0,True,Performance of the upper limb module 2.0 (PUL 2.0) | Active Range of motion | Functional workspace | Goal Attainment Scale (GAS) | Surface electromyography (sEMG) | Heart rate | Blood pressure | Numeral Rating Scale for Pain | Box and block test
1,NCT05544994,The Effect of Aerobic Exercise Training in Patients With Type III Spinal Muscular Atrophy,NA,TREATMENT,Neuromuscular Diseases; Spinal Muscular Atrophy Type 3,Aerobic Exercise Training; Home exercise program,OTHER; OTHER,20.0,True,Functional Capacity
2,NCT05589987,"Spinal Muscular Atrophy on Wheels, Using Power Mobility",NA,TREATMENT,Spinal Muscular Atrophy,power mobility,DEVICE,20.0,True,QUEST (Version 2.0) Quebec User Evaluation of Satisfaction with assistive Technology | YC-PEM (Young Children's Participation & Environment Measure) | Pediatric Evaluation of Disability Inventory-Computer Adaptive-test (PEDI-CAT) | Photovoice
3,NCT06419322,"Acceptability, Feasibility, Safety and Efficacy of a Optimized Rehabilitation Program for Treated Patients With Spinal Muscular Atrophy (SMA).",NA,OTHER,Spinal Muscular Atrophy,Optimized rehabilitation program,OTHER,14.0,True,Acceptability of an optimized rehabilitation program. Measured as number of eligible patients who accepted to participate in the study compared to the number of eligible patients who have refused to participate. Endpoint: 70% of acceptability. | Feasibility of an optimized rehabilitation program. Measured as the number of patients who complete the study compared to the number of included patients. Endpoint: 60% of feasibility.
4,NCT06368076,High-intensity Interval Training in Patients With Spinal Muscular Atrophy,NA,TREATMENT,Spinal Muscular Atrophy,Exercise,BEHAVIORAL,20.0,True,Questionnaire on fatigue | Questionnaire on pain. | Questionnaire on constipation | Questionnaire on quality of life | Questionnaire on sleep quality
5,NCT01611610,Clinical Assessment of Spinal Muscular Atrophy Type II and III (SMA Europe),NA,None,Spinal Muscular Atrophy,Functional tests; Functional tests,OTHER; OTHER,4.0,True,Change from baseline in total score of Motor Function Measure at one year
6,NCT04674618,Ultrasound-assisted vs Landmark Based Intrathecal Administration of Nusinersen,NA,TREATMENT,"Muscular Atrophy, Spinal; Ultrasound",US-assisted nusinersen administration,DEVICE,58.0,True,Number of attempts
7,NCT04798378,NuroSleeve Powered Brace & Stimulation System to Restore Arm Function,NA,SUPPORTIVE_CARE,Neurologic Diseases; Hemiparesis; Quadriplegia; Muscular Dystrophies; Arthrogryposis; Spinal Cord Injuries; Charcot-Marie-Tooth; Stroke; Weakness of Extremities as Sequela of Stroke; Weakness Due to Upper Motor Neuron Dysfunction; Amyotrophic Lateral Sclerosis; Spinal Muscular Atrophy; Arm Paralysis; Cerebral Palsy,Neurosleeve,DEVICE,20.0,True,Change in Canadian Occupational Performance Measure score at 8 weeks
8,NCT04159987,Monitoring to the Evolution of Motor Function in SMA Type II Adults Patients Treated With SPINRAZA®,NA,OTHER,Spinal Muscular Atrophy,Spinraza intrathecal injection,OTHER,20.0,True,"Change of the Motor Function Measure-32 (MFM-32) from Baseline (M0) to 1 month (M1), 3 months (M3), 7 months (M7), 15 months (M15) and 27 months (M27)"
9,NCT03395795,Trial Evaluating the Interest of Noninvasive Ventilation in NAVA Mode in Respiratory Decompensations Children With Infantile Spinal Muscular Atrophy Type II,NA,TREATMENT,Infantile 

In [ ]:
sma_manual_review_2.to_csv(
    DATA_DIR / "sma_manual_review_round2.csv",
    index=False
)

print("Saved SMA round-2 manual review table.")

Saved SMA round-2 manual review table.


In [ ]:
# Resolve all remaining manually reviewed SMA studies.
sma_manual_decisions = {
    "NCT06363357": "Exclude - broad multi-disease study",
    "NCT05544994": "Include",
    "NCT05589987": "Include",
    "NCT06419322": "Exclude - feasibility-focused study",
    "NCT06368076": "Include",
    "NCT01611610": "Exclude - assessment/natural history study",
    "NCT04674618": "Exclude - procedural administration study",
    "NCT04798378": "Exclude - broad multi-disease study",
    "NCT04159987": "Include",
    "NCT03395795": "Include",
    "NCT02235090": "Exclude - measurement/feasibility study",
    "NCT03056144": "Include",
    "NCT03689660": "Exclude - broad multi-disease study",
    "NCT07617779": "Include",
    "NCT05041114": "Exclude - broad multi-disease study",
    "NCT04690998": "Exclude - assessment/biomarker study",
    "NCT02341053": "Exclude - mechanistic/assessment study",
    "NCT06772402": "Include",
    "NCT07488338": "Include",
    "NCT05409079": "Exclude - broad multi-disease study",
    "NCT03223051": "Exclude - outcome measure validation",
    "NCT07356063": "Exclude - broad caregiver/multi-disease study",
    "NCT05878418": "Include",
    "NCT06167954": "Exclude - broad multi-disease study",
    "NCT02428673": "Exclude - broad multi-disease study",
    "NCT01862042": "Exclude - assessment/care-process study",
    "NCT04576494": "Include",
    "NCT05269693": "Exclude - broad multi-disease study",
    "NCT02003937": "Include",
    "NCT07578454": "Exclude - feasibility/device-development study",
    "NCT06191354": "Include",
    "NCT04813601": "Exclude - broad multi-disease study",
    "NCT06178653": "Include",
    "NCT07332702": "Exclude - genetic screening/assessment study",
    "NCT05632666": "Include",
    "NCT07072676": "Exclude - broad multi-disease study",
    "NCT05839145": "Exclude - validation/monitoring study",
    "NCT01522079": "Exclude - mechanistic study",
    "NCT02061189": "Include",
    "NCT05341453": "Include",
    "NCT06562283": "Exclude - outcome measure validation",
    "NCT05866419": "Exclude - procedural delivery study",
    "NCT06977269": "Include",
    "NCT05712330": "Exclude - observational/assessment study",
    "NCT05354414": "Exclude - procedural support study",
    "NCT07822711": "Include",
    "NCT06322654": "Include",
    "NCT05715749": "Include",
    "NCT05638750": "Include",
    "NCT05109637": "Exclude - outcome measure validation",
    "NCT07321977": "Exclude - broad multi-disease assessment",
    "NCT07478172": "Exclude - broad multi-disease study",
    "NCT05117827": "Exclude - broad multi-disease study",
    "NCT04335942": "Exclude - broad multi-disease study",
    "NCT06648486": "Include",
    "NCT06396325": "Include",
    "NCT01166022": "Include",
    "NCT06321965": "Exclude - natural history/phenotyping study",
    "NCT02227407": "Exclude - broad multi-disease study",
    "NCT04888702": "Exclude - monitoring/assessment study",
    "NCT07336602": "Include",
    "NCT04833348": "Exclude - outcome measure validation",
    "NCT07223320": "Exclude - feasibility-focused study",
    "NCT01233817": "Include",
    "NCT07547189": "Include",
    "NCT07838779": "Exclude - safety-focused study"
}

In [ ]:
# Confirm that every manually reviewed SMA study has exactly one decision.
manual_ids = set(sma_manual_review_2["nct_id"])
decision_ids = set(sma_manual_decisions)

print("Manual-review studies:", len(manual_ids))
print("Decisions entered:", len(decision_ids))
print("Missing decisions:", manual_ids - decision_ids)
print("Unexpected decisions:", decision_ids - manual_ids)

Manual-review studies: 66
Decisions entered: 66
Missing decisions: set()
Unexpected decisions: set()


In [ ]:
# Apply the manually reviewed SMA decisions.
for nct_id, decision in sma_manual_decisions.items():
    sma_trials.loc[
        sma_trials["nct_id"] == nct_id,
        "scope_decision"
    ] = decision

print(sma_trials["scope_decision"].value_counts())

print(
    "\nRemaining manual review studies:",
    (sma_trials["scope_decision"] == "Manual review").sum()
)

scope_decision
Include                                           149
Exclude - wrong disease population                 41
Exclude - non-therapeutic purpose                  31
Exclude - early phase                              24
Exclude - broad multi-disease study                14
Exclude - outcome measure validation                4
Exclude - feasibility-focused study                 2
Exclude - missing registered outcome data           2
Exclude - procedural administration study           1
Exclude - assessment/natural history study          1
Exclude - measurement/feasibility study             1
Exclude - assessment/biomarker study                1
Exclude - mechanistic/assessment study              1
Exclude - broad caregiver/multi-disease study       1
Exclude - assessment/care-process study             1
Exclude - feasibility/device-development study      1
Exclude - genetic screening/assessment study        1
Exclude - validation/monitoring study               1
Exclude - mec

In [ ]:
# Check included SMA studies for missing registered endpoint data.
sma_candidate = sma_trials[
    sma_trials["scope_decision"] == "Include"
].copy()

sma_missing_outcomes = sma_candidate[
    ~sma_candidate["has_primary_outcome"]
    & ~sma_candidate["has_secondary_outcome"]
]

print(
    "Included studies with no registered outcomes:",
    len(sma_missing_outcomes)
)

display(
    sma_missing_outcomes[
        [
            "nct_id",
            "brief_title",
            "phases",
            "primary_purpose"
        ]
    ]
)

Included studies with no registered outcomes: 1


,nct_id,brief_title,phases,primary_purpose
35,NCT00004771,Phase II Study of Leuprolide and Testosterone for Men With Kennedy's Disease or Other Motor Neuron Disease,PHASE2,TREATMENT


In [ ]:
sma_trials.loc[
    sma_trials["nct_id"].isin(sma_missing_outcomes["nct_id"]),
    "scope_decision"
] = "Exclude - missing registered outcome data"

In [ ]:
# Rebuild the finalized SMA cohort from the updated scope decisions.
sma_trials_analysis = sma_trials[
    sma_trials["scope_decision"] == "Include"
].copy()

included_sma_ids = set(sma_trials_analysis["nct_id"])

sma_endpoints_analysis = sma_endpoints[
    sma_endpoints["nct_id"].isin(included_sma_ids)
].copy()

print("SMA studies retrieved:", len(sma_trials))
print("SMA studies included:", len(sma_trials_analysis))
print("SMA studies excluded:", len(sma_trials) - len(sma_trials_analysis))

print("\nStudies represented:", sma_endpoints_analysis["nct_id"].nunique())
print("Total endpoints:", len(sma_endpoints_analysis))

print("\nEndpoint types:")
print(sma_endpoints_analysis["endpoint_type"].value_counts())

SMA studies retrieved: 285
SMA studies included: 148
SMA studies excluded: 137

Studies represented: 148
Total endpoints: 1529

Endpoint types:
endpoint_type
Secondary    1068
Primary       461
Name: count, dtype: int64


In [ ]:
# Save the finalized SMA analysis datasets.
sma_trials_analysis.to_csv(
    DATA_DIR / "spinal_muscular_atrophy_trials_analysis.csv",
    index=False
)

sma_endpoints_analysis.to_csv(
    DATA_DIR / "spinal_muscular_atrophy_endpoints_analysis.csv",
    index=False
)

print("Saved finalized SMA analysis files.")

Saved finalized SMA analysis files.


####Pull FSGS

In [ ]:
fsgs_raw, fsgs_trials, fsgs_endpoints = collect_disease_data(
    condition_query="Focal Segmental Glomerulosclerosis",
    disease_label="Primary FSGS"
)

Primary FSGS
------------
All studies returned: 116
Interventional studies: 87
Studies with primary outcomes: 85
Endpoint rows: 600

Scope decisions:
scope_decision
Include                              57
Exclude - early phase                16
Manual review                        12
Exclude - non-therapeutic purpose     2
Name: count, dtype: int64


In [ ]:
save_disease_data(
    fsgs_raw,
    fsgs_trials,
    fsgs_endpoints,
    file_prefix="primary_fsgs"
)

Saved:
./TrialSignal/data/primary_fsgs_interventional_raw.json
./TrialSignal/data/primary_fsgs_trials_study_level.csv
./TrialSignal/data/primary_fsgs_endpoints_raw.csv


In [ ]:
print("Primary purpose:")
print(fsgs_trials["primary_purpose"].value_counts(dropna=False))

print("\nPhase:")
print(fsgs_trials["phases"].value_counts(dropna=False))

print("\nScope decisions:")
print(fsgs_trials["scope_decision"].value_counts())

Primary purpose:
primary_purpose
TREATMENT        80
BASIC_SCIENCE     3
OTHER             2
PREVENTION        2
Name: count, dtype: int64

Phase:
phases
PHASE2            36
PHASE1            15
NA                13
PHASE3            10
PHASE4             6
PHASE2; PHASE3     4
PHASE1; PHASE2     2
EARLY_PHASE1       1
Name: count, dtype: int64

Scope decisions:
scope_decision
Include                              57
Exclude - early phase                16
Manual review                        12
Exclude - non-therapeutic purpose     2
Name: count, dtype: int64


In [ ]:
print("Most common condition strings:")

print(
    fsgs_trials["conditions"]
    .value_counts()
    .head(30)
)

Most common condition strings:
conditions
Focal Segmental Glomerulosclerosis                                                                                                                                                                          20
Primary Focal Segmental Glomerulosclerosis                                                                                                                                                                   4
Focal Segmental Glomerulosclerosis (FSGS)                                                                                                                                                                    4
FSGS                                                                                                                                                                                                         4
Focal Glomerulosclerosis                                                                                                          

In [ ]:
# Add primary endpoint names to FSGS manual-review studies.
fsgs_primary_endpoint_names = (
    fsgs_endpoints[
        fsgs_endpoints["endpoint_type"] == "Primary"
    ]
    .groupby("nct_id")["endpoint_name"]
    .apply(lambda x: " | ".join(x.astype(str)))
    .rename("primary_endpoint_names")
)

fsgs_manual_review = (
    fsgs_trials[
        fsgs_trials["scope_decision"] == "Manual review"
    ][
        [
            "nct_id",
            "brief_title",
            "phases",
            "primary_purpose",
            "conditions",
            "intervention_names",
            "intervention_types",
            "enrollment",
            "has_primary_outcome",
            "has_secondary_outcome"
        ]
    ]
    .merge(
        fsgs_primary_endpoint_names,
        on="nct_id",
        how="left"
    )
)

print("Manual review studies:", len(fsgs_manual_review))

pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

display(fsgs_manual_review)

Manual review studies: 12


,nct_id,brief_title,phases,primary_purpose,conditions,intervention_names,intervention_types,enrollment,has_primary_outcome,has_secondary_outcome,primary_endpoint_names
0,NCT04065438,Post Approval Study for Treatment of Drug-resistant Adult and Pediatric Primary FSGS Using the LIPOSORBER® LA-15 System,NA,TREATMENT,Focal Segmental Glomerulosclerosis,LIPOSORBER® LA-15,DEVICE,35,True,True,Evaluate the safety and probable benefit of the LIPOSORBER® LA-15 System in relieving nephrotic syndrome associated with primary FSGS at 1 month after the final apheresis treatment. partial remission at 1 month after the final apheresis treatment.
1,NCT01113385,Oral Galactose in Children With Steroid Resistant Nephrotic Syndrome,NA,TREATMENT,Focal Segmental Glomerulosclerosis; Steroid Resistant Nephrotic Syndrome,D-Galactose,DRUG,7,True,True,Focal Segmental Glomerulosclerosis Permeability Factor (FSPF)
2,NCT05583942,A Pilot Trial of taVNS for SRNS in Children (kidNEY-VNS),NA,TREATMENT,Nephrotic Syndrome in Children; Minimal Change Disease; Focal Segmental Glomerulosclerosis,trascutaneous auricular vagus nerve stimulation; Sham device,DEVICE; DEVICE,10,True,True,Success of Pilot Trial
3,NCT00801463,Prednisone Plus Tripterygium Wilfordii Treatment of Adult Patients With Idiopathic Focal Segmental Glomerulosclerosis,NA,TREATMENT,Proteinuria; Focal Segmental Glomerulosclerosis,tripterygium wilfordii (TW),DRUG,67,True,False,To assess the efficacy and safety of TW in the treatment of FSGS
4,NCT00302536,Tacrolimus Treatment of Patients With Idiopathic Focal Segmental Glomerulosclerosis,NA,TREATMENT,Focal Glomerulosclerosis,Tacrolimus,DRUG,0,True,True,To assess the efficacy of Tacrolimus Treatment Idiopathic Focal Segmental Glomerulosclerosis
5,NCT05588063,taVNS for FRNS in Children,NA,TREATMENT,Nephrotic Syndrome in Children; Minimal Change Disease; Focal Segmental Glomerulosclerosis,Transcutaneous auricular vagus nerve stimulation; Sham device,DEVICE; DEVICE,30,True,True,Success of Pilot Trial
6,NCT01370200,Regional Citrate Anticoagulation in Plasma Exchange Treatment,NA,TREATMENT,Focal Segmental Glomerulosclerosis; Guillain-Barre Syndrome,plasma exchange treatment,PROCEDURE,140,True,True,"The efficacy of plasma exchange, measuring immunoglobulins type G before and after the procedure, using 4% and 15% citrate as anticoagulants"
7,NCT04571658,NEPTUNE Match Study,NA,OTHER,Nephrotic Syndrome in Children; Focal Segmental Glomerulosclerosis; Minimal Change Disease; Minimal Change Nephrotic Syndrome; Membranous Nephropathy; FSGS; MCD; MCD - Minimal Change Disease; Alport Syndrome,Communication,OTHER,375,True,True,"Matched participants with at least one clinical trial receive a MNB assessment | Efficacy of communication methods at time of communication visit, measured by Teach Back 1 summary score | Kidney health endpoints that are specific to each individual trial"
8,NCT01835639,Vitamin D Supplementation in Glomerular Disease,NA,TREATMENT,Glomerular Disease,"Cholecalciferol, 2000 or 4000 IUs by mouth daily for 12 weeks",DIETARY_SUPPLEMENT,32,True,True,"To assess the efficacy of vitamin D supplementation, by the increase in total and free 25(OH)D levels. | To assess the safety of vitamin D supplementation, by serum Ionized calcium (ICal), Pi, and 25(OH)D, and urinary calcium excretion."
9,NCT01451489,The Efficacy and Safety of Tacrolimus in Treated Refractory Focal Segmental Glomerulosclerosis,NA,TREATMENT,FSGS,FK506; Cyclophosphamide,DRUG; DRUG,70,True,True,the rates of the complete remission during the induction phase


In [ ]:
fsgs_manual_review.to_csv(
    DATA_DIR / "fsgs_manual_review.csv",
    index=False
)

print("Saved FSGS manual review table.")

Saved FSGS manual review table.


In [ ]:
# Resolve the 12 manually reviewed FSGS studies.
fsgs_manual_decisions = {
    "NCT04065438": "Include",
    "NCT01113385": "Exclude - broad nephrotic syndrome study",
    "NCT05583942": "Exclude - broad nephrotic syndrome/feasibility study",
    "NCT00801463": "Include",
    "NCT00302536": "Include",
    "NCT05588063": "Exclude - broad nephrotic syndrome study",
    "NCT01370200": "Exclude - broad multi-disease/procedural study",
    "NCT04571658": "Exclude - non-therapeutic implementation study",
    "NCT01835639": "Exclude - broad glomerular disease study",
    "NCT01451489": "Include",
    "NCT02235857": "Include",
    "NCT00956059": "Include"
}

manual_ids = set(fsgs_manual_review["nct_id"])
decision_ids = set(fsgs_manual_decisions)

print("Manual-review studies:", len(manual_ids))
print("Decisions entered:", len(decision_ids))
print("Missing decisions:", manual_ids - decision_ids)
print("Unexpected decisions:", decision_ids - manual_ids)

for nct_id, decision in fsgs_manual_decisions.items():
    fsgs_trials.loc[
        fsgs_trials["nct_id"] == nct_id,
        "scope_decision"
    ] = decision

print("\nScope decisions:")
print(fsgs_trials["scope_decision"].value_counts())

print(
    "\nRemaining manual review:",
    (fsgs_trials["scope_decision"] == "Manual review").sum()
)

Manual-review studies: 12
Decisions entered: 12
Missing decisions: set()
Unexpected decisions: set()

Scope decisions:
scope_decision
Include                                                 63
Exclude - early phase                                   16
Exclude - broad nephrotic syndrome study                 2
Exclude - non-therapeutic purpose                        2
Exclude - broad nephrotic syndrome/feasibility study     1
Exclude - broad multi-disease/procedural study           1
Exclude - non-therapeutic implementation study           1
Exclude - broad glomerular disease study                 1
Name: count, dtype: int64

Remaining manual review: 0


In [ ]:
# Flag currently included studies that may not represent primary FSGS.
fsgs_scope_terms = [
    "transplant",
    "recurrent",
    "genetic",
    "APOL1",
    "secondary",
    "minimal change",
    "nephrotic syndrome",
    "IgA",
    "membranous",
    "Alport",
    "lupus",
    "diabetic",
    "proteinuric kidney",
    "chronic kidney",
    "healthy"
]

fsgs_include_audit = fsgs_trials[
    fsgs_trials["scope_decision"] == "Include"
].copy()

audit_text = (
    fsgs_include_audit["brief_title"].fillna("") + " " +
    fsgs_include_audit["conditions"].fillna("")
)

pattern = "|".join(fsgs_scope_terms)

fsgs_include_audit = fsgs_include_audit[
    audit_text.str.contains(
        pattern,
        case=False,
        regex=True
    )
].copy()

# Add primary endpoints so we can adjudicate them in one pass.
fsgs_include_audit = fsgs_include_audit.merge(
    fsgs_primary_endpoint_names,
    on="nct_id",
    how="left"
)

print("Included studies flagged for primary-FSGS audit:",
      len(fsgs_include_audit))

display(
    fsgs_include_audit[
        [
            "nct_id",
            "brief_title",
            "phases",
            "primary_purpose",
            "conditions",
            "intervention_names",
            "enrollment",
            "primary_endpoint_names"
        ]
    ]
)

Included studies flagged for primary-FSGS audit: 29


,nct_id,brief_title,phases,primary_purpose,conditions,intervention_names,enrollment,primary_endpoint_names
0,NCT02896270,Valproic Acid for Idiopathic Nephrotic Syndrome,PHASE2; PHASE3,TREATMENT,Idiopathic Nephrotic Syndrome; Focal Segmental Glomerulosclerosis; Minimal Change Disease,Valproic Acid,15,In remission group induction is the proportion of patients in complete remission | In remission maintenance group is the proportion of patients able to reduce maintenance
1,NCT01268033,Efficacy of Rituximab For the Treatment of Calcineurin Inhibitors Dependent Nephrotic Syndrome During Childhood,PHASE2; PHASE3,TREATMENT,Childhood Idiopathic Nephrotic Syndrome,Rituximab; Placebo,26,Proteinuria with relapse of nephrotic syndrome (Serum albumin < 30 g/L) within 5 months
2,NCT01129557,"Aldosterone Breakthrough During Diovan, Tekturna, and Combination Therapy in Patients With Proteinuric Kidney Disease",PHASE4,TREATMENT,"Proteinuric Kidney Disease; Diabetic Nephropathy; Hypertensive Nephrosclerosis; IgA Nephropathy; Focal Segmental Glomerulosclerosis; Glomerulopathy (Obesity-associated); Glomerulonephritis, Membranous",Aliskiren; Valsartan,46,Cumulative Incidence of Aldosterone Breakthrough in Subjects Who Completed the 9-month Study Protocol.
3,NCT02921789,Study to Assess the Efficacy and Safety of Bleselumab in Preventing the Recurrence of Focal Segmental Glomerulosclerosis in de Novo Kidney Transplant Recipients,PHASE2,TREATMENT,Kidney Transplantation; Primary Focal Segmental Glomerulosclerosis (FSGS),Bleselumab; Basiliximab; Mycophenolate Mofetil (MMF); Tacrolimus Capsules; Methylprednisone; Prednisone,67,Percentage of Participants With Recurrence of Focal Segmental Glomerulosclerosis (rFSGS) or Death or Graft Loss or Lost to Follow-up Through 3 Months Post Transplant
4,NCT06500702,"A Study to Evaluate the Efficacy and Safety of Frexalimab, Brivekimig, or Rilzabrutinib in Participants Aged 16 to 75 Years With Primary Focal Segmental Glomerulosclerosis or Minimal Change Disease",PHASE2,TREATMENT,Focal Segmental Glomerulosclerosis; Glomerulonephritis Minimal Lesion,frexalimab; brivekimig; rilzabrutinib; placebo,84,Percent reduction in urine protein to creatinine ratio (UPCR)
5,NCT03298698,Efficacy of Rituximab in Comparison to Continued Corticosteroid Treatment in Idiopathic Nephrotic Syndrome,PHASE3,TREATMENT,Idiopathic Nephrotic Syndrome; Minimal Change Disease; Focal Segmental Glomerulosclerosis,Rituximab; Prednisone,40,Complete remission
6,NCT05003986,Study of Sparsentan Treatment in Pediatrics With Proteinuric Glomerular Diseases,PHASE2,TREATMENT,Focal Segmental Glomerulosclerosis; Minimal Change Disease; Immunoglobulin A Nephropathy; IgA Vasculitis; Alport Syndrome,Sparsentan; Sparsentan; Sparsentan,67,"Incidence of treatment-emergent adverse events (TEAEs), serious adverse events (SAEs), AEs leading to treatment discontinuation, and adverse events of interest (AEOIs) | Urine protein/creatinine ratio (UP/C) at week 108"
7,NCT00981838,Rituximab in Multirelapsing Minimal Change Disease (MCD) or Focal Segmental Glomerulosclerosis (FSGS),PHASE3,TREATMENT,Nephrotic Syndrome,Rituximab,24,Recurrence of NS.
8,NCT04340362,Phase 2a Study of VX-147 in Adults With APOL1-mediated Focal Segmental Glomerulosclerosis,PHASE2,TREATMENT,"Glomerulosclerosis, Focal Segmental",VX-147,16,Percent Change From Baseline in UPCR
9,NCT05599815,Part 1 - A Clinical Trial in Patients With Frequently Relapsing and Steroid-Dependent Nephrotic Syndrome,PHASE2,TREATMENT,Nephrotic Syndrome,ADX-629,0,Adverse Event (AE) Query


In [ ]:
# Resolve the primary-FSGS audit.
fsgs_audit_decisions = {
    "NCT02896270": "Exclude - broad nephrotic syndrome study",
    "NCT01268033": "Exclude - broad nephrotic syndrome study",
    "NCT01129557": "Exclude - broad proteinuric kidney disease study",
    "NCT02921789": "Exclude - post-transplant recurrence study",
    "NCT06500702": "Exclude - mixed FSGS/MCD population",
    "NCT03298698": "Exclude - broad nephrotic syndrome study",
    "NCT05003986": "Exclude - broad glomerular disease study",
    "NCT00981838": "Exclude - mixed MCD/FSGS population",
    "NCT04340362": "Exclude - APOL1-mediated/genetic FSGS",
    "NCT05599815": "Exclude - broad nephrotic syndrome study",
    "NCT05237388": "Exclude - APOL1-associated kidney disease",
    "NCT06466135": "Exclude - broad glomerular disease study",

    "NCT03703908": "Include",

    "NCT00004990": "Include",

    "NCT00404833": "Exclude - mixed membranous/FSGS population",
    "NCT02592798": "Exclude - mixed FSGS/MCD population",
    "NCT01129284": "Exclude - broad nephrotic syndrome study",
    "NCT07219121": "Exclude - post-transplant/broad disease study",
    "NCT03366337": "Exclude - broad chronic kidney disease study",
    "NCT04387448": "Exclude - broad kidney disease study",
    "NCT05441826": "Exclude - mixed FSGS/MCD population",
    "NCT04573920": "Exclude - broad glomerular disease study",
    "NCT07499700": "Exclude - mixed FSGS/MCD population",
    "NCT06664814": "Exclude - broad podocyte disease study",
    "NCT05267262": "Exclude - mixed Alport/FSGS population",

    "NCT00001959": "Include",

    "NCT07614477": "Exclude - broad glomerular disease study",
    "NCT06635720": "Exclude - broad nephrotic syndrome study",
    "NCT02683889": "Exclude - post-transplant recurrence study"
}

audit_ids = set(fsgs_include_audit["nct_id"])
decision_ids = set(fsgs_audit_decisions)

print("Flagged studies:", len(audit_ids))
print("Decisions entered:", len(decision_ids))
print("Missing decisions:", audit_ids - decision_ids)
print("Unexpected decisions:", decision_ids - audit_ids)

for nct_id, decision in fsgs_audit_decisions.items():
    fsgs_trials.loc[
        fsgs_trials["nct_id"] == nct_id,
        "scope_decision"
    ] = decision

print("\nUpdated scope decisions:")
print(fsgs_trials["scope_decision"].value_counts())

Flagged studies: 29
Decisions entered: 29
Missing decisions: set()
Unexpected decisions: set()

Updated scope decisions:
scope_decision
Include                                                 37
Exclude - early phase                                   16
Exclude - broad nephrotic syndrome study                 8
Exclude - broad glomerular disease study                 5
Exclude - mixed FSGS/MCD population                      4
Exclude - post-transplant recurrence study               2
Exclude - non-therapeutic purpose                        2
Exclude - broad proteinuric kidney disease study         1
Exclude - mixed MCD/FSGS population                      1
Exclude - APOL1-mediated/genetic FSGS                    1
Exclude - APOL1-associated kidney disease                1
Exclude - broad nephrotic syndrome/feasibility study     1
Exclude - mixed membranous/FSGS population               1
Exclude - broad multi-disease/procedural study           1
Exclude - non-therapeutic implementati

In [ ]:
# Remove otherwise in-scope FSGS studies with no registered endpoint data.
fsgs_candidate = fsgs_trials[
    fsgs_trials["scope_decision"] == "Include"
].copy()

fsgs_missing_outcomes = fsgs_candidate[
    ~fsgs_candidate["has_primary_outcome"]
    & ~fsgs_candidate["has_secondary_outcome"]
]

print(
    "Included studies with no registered outcomes:",
    len(fsgs_missing_outcomes)
)

display(
    fsgs_missing_outcomes[
        [
            "nct_id",
            "brief_title",
            "phases",
            "primary_purpose"
        ]
    ]
)

Included studies with no registered outcomes: 2


,nct_id,brief_title,phases,primary_purpose
11,NCT00040508,Sirolimus for Focal Segmental Glomerulosclerosis,PHASE2,TREATMENT
42,NCT00004990,Once-A-Month Steroid Treatment for Patients With Focal Segmental Glomerulosclerosis,PHASE2,TREATMENT


In [ ]:
fsgs_trials.loc[
    fsgs_trials["nct_id"].isin(fsgs_missing_outcomes["nct_id"]),
    "scope_decision"
] = "Exclude - missing registered outcome data"

In [ ]:
# Build the final Primary FSGS cohort.
fsgs_trials_analysis = fsgs_trials[
    fsgs_trials["scope_decision"] == "Include"
].copy()

included_fsgs_ids = set(fsgs_trials_analysis["nct_id"])

fsgs_endpoints_analysis = fsgs_endpoints[
    fsgs_endpoints["nct_id"].isin(included_fsgs_ids)
].copy()

print("FSGS studies retrieved:", len(fsgs_trials))
print("FSGS studies included:", len(fsgs_trials_analysis))
print("FSGS studies excluded:", len(fsgs_trials) - len(fsgs_trials_analysis))

print("\nStudies represented:", fsgs_endpoints_analysis["nct_id"].nunique())
print("Total endpoints:", len(fsgs_endpoints_analysis))

print("\nEndpoint types:")
print(fsgs_endpoints_analysis["endpoint_type"].value_counts())

FSGS studies retrieved: 87
FSGS studies included: 35
FSGS studies excluded: 52

Studies represented: 35
Total endpoints: 276

Endpoint types:
endpoint_type
Secondary    178
Primary       98
Name: count, dtype: int64


In [ ]:
fsgs_trials_analysis.to_csv(
    DATA_DIR / "primary_fsgs_trials_analysis.csv",
    index=False
)

fsgs_endpoints_analysis.to_csv(
    DATA_DIR / "primary_fsgs_endpoints_analysis.csv",
    index=False
)

print("Saved finalized Primary FSGS analysis files.")

Saved finalized Primary FSGS analysis files.


In [ ]:
# Load the finalized study-level datasets from Drive.
study_files = [
    "hae_trials_analysis.csv",
    "friedreich_ataxia_trials_analysis.csv",
    "fshd_trials_analysis.csv",
    "sickle_cell_disease_trials_analysis.csv",
    "dravet_syndrome_trials_analysis.csv",
    "spinal_muscular_atrophy_trials_analysis.csv",
    "primary_fsgs_trials_analysis.csv"
]

all_trials_analysis = pd.concat(
    [pd.read_csv(DATA_DIR / file) for file in study_files],
    ignore_index=True
)

print("Total studies:", len(all_trials_analysis))

print("\nStudies by disease:")
print(all_trials_analysis["disease"].value_counts())

Total studies: 1013

Studies by disease:
disease
Sickle Cell Disease                       590
Spinal Muscular Atrophy                   148
Hereditary Angioedema                     105
Friedreich's Ataxia                        56
Dravet Syndrome                            49
Primary FSGS                               35
Facioscapulohumeral Muscular Dystrophy     30
Name: count, dtype: int64


In [ ]:
# Load the finalized endpoint-level datasets from Drive.
endpoint_files = [
    "hae_endpoints_analysis.csv",
    "friedreich_ataxia_endpoints_analysis.csv",
    "fshd_endpoints_analysis.csv",
    "sickle_cell_disease_endpoints_analysis.csv",
    "dravet_syndrome_endpoints_analysis.csv",
    "spinal_muscular_atrophy_endpoints_analysis.csv",
    "primary_fsgs_endpoints_analysis.csv"
]

all_endpoints_analysis = pd.concat(
    [pd.read_csv(DATA_DIR / file) for file in endpoint_files],
    ignore_index=True
)

print("Total endpoints:", len(all_endpoints_analysis))

print("\nEndpoints by disease:")
print(all_endpoints_analysis["disease"].value_counts())

print("\nEndpoint types:")
print(all_endpoints_analysis["endpoint_type"].value_counts())

Total endpoints: 8832

Endpoints by disease:
disease
Sickle Cell Disease                       4858
Spinal Muscular Atrophy                   1529
Hereditary Angioedema                      911
Dravet Syndrome                            527
Friedreich's Ataxia                        440
Facioscapulohumeral Muscular Dystrophy     291
Primary FSGS                               276
Name: count, dtype: int64

Endpoint types:
endpoint_type
Secondary    6425
Primary      2407
Name: count, dtype: int64


In [ ]:
# Make sure every final study contributes endpoint data.
study_ids = set(all_trials_analysis["nct_id"])
endpoint_ids = set(all_endpoints_analysis["nct_id"])

print("Studies without endpoints:", len(study_ids - endpoint_ids))
print("Unexpected endpoint-only studies:", len(endpoint_ids - study_ids))

# Check whether the same NCT ID appears in more than one final disease cohort.
duplicate_trials = (
    all_trials_analysis[
        all_trials_analysis.duplicated("nct_id", keep=False)
    ]
    .sort_values("nct_id")
)

print("Duplicate NCT IDs across final cohorts:", duplicate_trials["nct_id"].nunique())

if len(duplicate_trials) > 0:
    display(
        duplicate_trials[
            ["nct_id", "disease", "brief_title"]
        ]
    )

Studies without endpoints: 0
Unexpected endpoint-only studies: 0
Duplicate NCT IDs across final cohorts: 0


In [ ]:
all_trials_analysis.to_csv(
    DATA_DIR / "trialsignal_trials_master.csv",
    index=False
)

all_endpoints_analysis.to_csv(
    DATA_DIR / "trialsignal_endpoints_master.csv",
    index=False
)

print("Saved TrialSignal master datasets.")

Saved TrialSignal master datasets.


## Patient Priority Data

In [ ]:
PATIENT_DIR = DATA_DIR / "patient_reports"
PATIENT_DIR.mkdir(parents=True, exist_ok=True)

patient_sources = pd.DataFrame([
    {
        "source_id": "hae_vop",
        "disease": "Hereditary Angioedema",
        "source_type": "FDA-led PFDD",
        "meeting_date": "2017-09-25",
        "report_date": "2018-05",
        "filename": "hae_vop.pdf"
    },
    {
        "source_id": "fa_vop",
        "disease": "Friedreich's Ataxia",
        "source_type": "EL-PFDD",
        "meeting_date": "2017-06-02",
        "report_date": "2017-08-20",
        "filename": "fa_vop.pdf"
    },
    {
        "source_id": "fshd_vop",
        "disease": "Facioscapulohumeral Muscular Dystrophy",
        "source_type": "EL-PFDD",
        "meeting_date": "2020-06-29",
        "report_date": "2020-11-05",
        "filename": "fshd_vop.pdf"
    },
    {
        "source_id": "scd_vop",
        "disease": "Sickle Cell Disease",
        "source_type": "FDA-led PFDD",
        "meeting_date": "2014-02-07",
        "report_date": "2014-10",
        "filename": "scd_vop.pdf"
    },
    {
        "source_id": "dravet_vop",
        "disease": "Dravet Syndrome",
        "source_type": "EL-PFDD",
        "meeting_date": "2022-02-03",
        "report_date": "2022-05-31",
        "filename": "dravet_vop.pdf"
    },
    {
        "source_id": "sma_vop",
        "disease": "Spinal Muscular Atrophy",
        "source_type": "EL-PFDD",
        "meeting_date": "2017-04-18",
        "report_date": "2018-01-10",
        "filename": "sma_vop.pdf"
    },
    {
        "source_id": "fsgs_vop",
        "disease": "Primary FSGS",
        "source_type": "EL-PFDD",
        "meeting_date": "2020-08-28",
        "report_date": "2021-09-10",
        "filename": "fsgs_vop.pdf"
    }
])

patient_sources.to_csv(
    DATA_DIR / "patient_priority_source_manifest.csv",
    index=False
)

display(patient_sources)

,source_id,disease,source_type,meeting_date,report_date,filename
0,hae_vop,Hereditary Angioedema,FDA-led PFDD,2017-09-25,2018-05,hae_vop.pdf
1,fa_vop,Friedreich's Ataxia,EL-PFDD,2017-06-02,2017-08-20,fa_vop.pdf
2,fshd_vop,Facioscapulohumeral Muscular Dystrophy,EL-PFDD,2020-06-29,2020-11-05,fshd_vop.pdf
3,scd_vop,Sickle Cell Disease,FDA-led PFDD,2014-02-07,2014-10,scd_vop.pdf
4,dravet_vop,Dravet Syndrome,EL-PFDD,2022-02-03,2022-05-31,dravet_vop.pdf
5,sma_vop,Spinal Muscular Atrophy,EL-PFDD,2017-04-18,2018-01-10,sma_vop.pdf
6,fsgs_vop,Primary FSGS,EL-PFDD,2020-08-28,2021-09-10,fsgs_vop.pdf


In [ ]:
patient_sources["file_path"] = patient_sources["filename"].apply(
    lambda x: PATIENT_DIR / x
)

patient_sources["file_exists"] = patient_sources["file_path"].apply(
    lambda x: x.exists()
)

display(
    patient_sources[
        ["disease", "filename", "file_exists"]
    ]
)

print(
    "Reports found:",
    patient_sources["file_exists"].sum(),
    "of",
    len(patient_sources)
)

,disease,filename,file_exists
0,Hereditary Angioedema,hae_vop.pdf,True
1,Friedreich's Ataxia,fa_vop.pdf,True
2,Facioscapulohumeral Muscular Dystrophy,fshd_vop.pdf,True
3,Sickle Cell Disease,scd_vop.pdf,True
4,Dravet Syndrome,dravet_vop.pdf,True
5,Spinal Muscular Atrophy,sma_vop.pdf,True
6,Primary FSGS,fsgs_vop.pdf,True


Reports found: 7 of 7


In [ ]:
%pip -q install pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 69.0 MB/s eta 0:00:00


In [ ]:
import fitz
import re

page_rows = []

for _, source in patient_sources.iterrows():
    pdf_path = source["file_path"]
    doc = fitz.open(pdf_path)

    for page_index in range(len(doc)):
        page = doc[page_index]

        raw_text = page.get_text("text").strip()

        # A cleaned copy for search/retrieval.
        clean_text = re.sub(r"\s+", " ", raw_text).strip()

        page_rows.append({
            "disease": source["disease"],
            "source_id": source["source_id"],
            "source_type": source["source_type"],
            "filename": source["filename"],
            "pdf_page": page_index + 1,
            "page_text_raw": raw_text,
            "page_text_clean": clean_text
        })

    doc.close()

patient_report_pages = pd.DataFrame(page_rows)

print("Total report pages:", len(patient_report_pages))

display(
    patient_report_pages[
        ["disease", "source_id", "pdf_page", "page_text_clean"]
    ].head()
)

KeyError: 'file_path'

In [ ]:
patient_report_pages.to_csv(
    DATA_DIR / "patient_report_pages.csv",
    index=False
)

print("Saved page-level patient report corpus.")

Saved page-level patient report corpus.


In [ ]:
report_page_check = (
    patient_report_pages
    .groupby("disease")
    .agg(
        pages=("pdf_page", "max"),
        pages_with_text=("page_text_clean", lambda x: (x.str.len() > 0).sum())
    )
    .reset_index()
)

display(report_page_check)

print(
    "\nBlank pages:",
    (patient_report_pages["page_text_clean"].str.len() == 0).sum()
)

,disease,pages,pages_with_text
0,Dravet Syndrome,102,102
1,Facioscapulohumeral Muscular Dystrophy,140,140
2,Friedreich's Ataxia,83,83
3,Hereditary Angioedema,31,31
4,Primary FSGS,81,81
5,Sickle Cell Disease,31,31
6,Spinal Muscular Atrophy,149,148



Blank pages: 1


In [ ]:
priority_columns = [
    "disease",
    "priority_id",
    "priority_label",
    "priority_description",
    "priority_domain",
    "alignment_eligible",
    "evidence_text",
    "evidence_pdf_page",
    "source_id",
    "support_type",
    "review_status"
]

patient_priorities = pd.DataFrame(columns=priority_columns)

display(patient_priorities)

,disease,priority_id,priority_label,priority_description,priority_domain,alignment_eligible,evidence_text,evidence_pdf_page,source_id,support_type,review_status


In [ ]:
blank_report_pages = patient_report_pages[
    patient_report_pages["page_text_clean"].str.len() == 0
][
    ["disease", "source_id", "filename", "pdf_page"]
]

display(blank_report_pages)

,disease,source_id,filename,pdf_page
428,Spinal Muscular Atrophy,sma_vop,sma_vop.pdf,42


In [ ]:
def chunk_page_text(text, chunk_size=180, overlap=40):
    words = text.split()

    if not words:
        return []

    chunks = []
    start = 0

    while start < len(words):
        end = min(start + chunk_size, len(words))
        chunk = " ".join(words[start:end])

        chunks.append(chunk)

        if end == len(words):
            break

        start = end - overlap

    return chunks


chunk_rows = []

for _, row in patient_report_pages.iterrows():
    chunks = chunk_page_text(row["page_text_clean"])

    for chunk_number, chunk_text in enumerate(chunks, start=1):
        chunk_rows.append({
            "disease": row["disease"],
            "source_id": row["source_id"],
            "filename": row["filename"],
            "pdf_page": row["pdf_page"],
            "chunk_number": chunk_number,
            "chunk_id": f"{row['source_id']}_p{row['pdf_page']}_c{chunk_number}",
            "chunk_text": chunk_text
        })

patient_report_chunks = pd.DataFrame(chunk_rows)

print("Total chunks:", len(patient_report_chunks))

display(
    patient_report_chunks[
        [
            "disease",
            "pdf_page",
            "chunk_number",
            "chunk_text"
        ]
    ].head()
)

NameError: name 'patient_report_pages' is not defined

In [ ]:
patient_report_chunks.to_csv(
    DATA_DIR / "patient_report_chunks.csv",
    index=False
)

print("Saved patient report chunks.")

Saved patient report chunks.


In [ ]:
%pip -q install sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

chunk_embeddings = embedding_model.encode(
    patient_report_chunks["chunk_text"].tolist(),
    show_progress_bar=True,
    normalize_embeddings=True
)

print("Embedding matrix shape:", chunk_embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (5 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/56 [00:00<?, ?it/s]

Embedding matrix shape: (1771, 384)


In [ ]:
np.save(
    DATA_DIR / "patient_report_chunk_embeddings.npy",
    chunk_embeddings
)

print("Saved chunk embeddings.")

Saved chunk embeddings.


In [ ]:
def retrieve_patient_evidence(
    disease,
    query,
    top_k=8
):
    disease_mask = (
        patient_report_chunks["disease"] == disease
    ).values

    disease_indices = np.where(disease_mask)[0]

    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True
    )[0]

    scores = (
        chunk_embeddings[disease_indices]
        @ query_embedding
    )

    top_positions = np.argsort(scores)[::-1][:top_k]
    top_indices = disease_indices[top_positions]

    results = patient_report_chunks.iloc[top_indices].copy()
    results["similarity"] = scores[top_positions]

    return results[
        [
            "disease",
            "source_id",
            "pdf_page",
            "chunk_number",
            "chunk_id",
            "similarity",
            "chunk_text"
        ]
    ]

In [ ]:
test_results = retrieve_patient_evidence(
    disease="Hereditary Angioedema",
    query="Which symptoms, attacks, and disease impacts are most important to patients?",
    top_k=5
)

display(test_results)

,disease,source_id,pdf_page,chunk_number,chunk_id,similarity,chunk_text
51,Hereditary Angioedema,hae_vop,17,2,hae_vop_p17_c2,0.580803,signs that were named by both in-person and we...
19,Hereditary Angioedema,hae_vop,7,3,hae_vop_p7_c3,0.567168,"side effects, and ultimately lead to a cure. I..."
84,Hereditary Angioedema,hae_vop,31,2,hae_vop_p31_c2,0.517490,for acute attacks that might occur 53% 100% 19...
80,Hereditary Angioedema,hae_vop,29,2,hae_vop_p29_c2,0.503981,"a new treatment for your condition, which bene..."
49,Hereditary Angioedema,hae_vop,16,4,hae_vop_p16_c4,0.499601,never know-- it could travel to various places...


In [ ]:
hae_queries = [
    "What symptoms or complications of hereditary angioedema do patients say are most important, severe, or burdensome?",

    "How does hereditary angioedema affect patients' daily activities, work, school, independence, social life, or emotional well-being?",

    "What aspects of hereditary angioedema attacks do patients most want treatments to improve, such as attack frequency, severity, duration, or location?",

    "What outcomes or improvements do patients say they most want from treatment?",

    "What do patients say about speed of treatment response, complete relief, prevention of attacks, or control of symptoms?"
]

In [ ]:
hae_retrievals = []

for query_number, query in enumerate(hae_queries, start=1):
    results = retrieve_patient_evidence(
        disease="Hereditary Angioedema",
        query=query,
        top_k=8
    ).copy()

    results["query_number"] = query_number
    results["query"] = query

    hae_retrievals.append(results)

hae_retrievals = pd.concat(
    hae_retrievals,
    ignore_index=True
)

print("Retrieved rows:", len(hae_retrievals))
print(
    "Unique chunks:",
    hae_retrievals["chunk_id"].nunique()
)

Retrieved rows: 40
Unique chunks: 25


In [ ]:
hae_evidence_candidates = (
    hae_retrievals
    .sort_values("similarity", ascending=False)
    .drop_duplicates("chunk_id")
    .sort_values(
        ["pdf_page", "chunk_number"]
    )
    .reset_index(drop=True)
)

print(
    "Unique HAE evidence candidates:",
    len(hae_evidence_candidates)
)

display(
    hae_evidence_candidates[
        [
            "pdf_page",
            "chunk_number",
            "similarity",
            "chunk_text"
        ]
    ]
)

Unique HAE evidence candidates: 25


,pdf_page,chunk_number,similarity,chunk_text
0,1,1,0.683676,1 The Voice of the Patient A series of reports...
1,3,2,0.717156,the PFDD initiative under PDUFA V. More inform...
2,3,4,0.607662,HAE have been described: type 1 occurs in appr...
3,4,4,0.685491,"and frequency of attacks, they do not eliminat..."
4,7,3,0.502643,"side effects, and ultimately lead to a cure. I..."
5,8,2,0.536197,were reflected in these panelists’ stories. Pe...
6,15,1,0.446129,"15 “On a day-to-day basis, I have no social li..."
7,15,3,0.529840,on a treatment plan and it seemed to keep me p...
8,16,2,0.526695,question about the therapies used for the trea...
9,16,3,0.538333,"coming on [Appendix 3, Question 14]. For the w..."


In [ ]:
hae_evidence_candidates.to_csv(
    DATA_DIR / "hae_priority_evidence_candidates.csv",
    index=False
)

print("Saved HAE evidence candidates.")

Saved HAE evidence candidates.


## Set up the LLM

In [ ]:
# Install the OpenAI package for patient-priority extraction.
%pip -q install openai

In [ ]:
# Set up the LLM client using the API key stored in Colab Secrets.
from google.colab import userdata
from openai import OpenAI

client = OpenAI(
    api_key=userdata.get("OPENAI_API_KEY")
)

LLM_MODEL = "gpt-6-luna"

In [ ]:
# Combine the retrieved HAE chunks into one evidence package for the LLM.
hae_evidence_text = "\n\n".join(
    f"[{row['chunk_id']}] PDF page {row['pdf_page']}\n{row['chunk_text']}"
    for _, row in hae_evidence_candidates.iterrows()
)

print("Evidence characters:", len(hae_evidence_text))
print(hae_evidence_text[:3000])

Evidence characters: 23919
[hae_vop_p1_c1] PDF page 1
1 The Voice of the Patient A series of reports from the U.S. Food and Drug Administration’s Patient-Focused Drug Development Initiative Hereditary Angioedema Public Meeting: September 25, 2017 Report Date: May 2018 Center for Biologics Evaluation and Research (CBER) U.S. Food and Drug Administration (FDA)

[hae_vop_p3_c2] PDF page 3
the PFDD initiative under PDUFA V. More information on this initiative can be found at https://www.fda.gov/ForIndustry/UserFees/PrescriptionDrugUserFee/ucm347317.htm. Overview of Hereditary Angioedema Hereditary Angioedema is a rare genetic disorder known to be inherited in the autosomal dominant pattern. The condition usually presents with recurrent attacks of severe swelling (angioedema) that may involve any part of the body and frequently affects mucosal surfaces such as the lining of the gastrointestinal tract, the mouth, the tongue, the throat, the larynx, the genitourinary system, and others. Attac

In [ ]:
# Define the instructions for extracting structured patient priorities from retrieved evidence.
hae_priority_prompt = f"""
You are helping construct an auditable patient-priority dataset for a clinical-trial analysis.

DISEASE:
Hereditary Angioedema

TASK:
Using ONLY the retrieved evidence below, identify the distinct outcomes, symptoms, functional
impacts, or treatment effects that patients or caregivers indicate are meaningful or important.

The final project asks whether clinical-trial endpoints measure outcomes patients care about.

IMPORTANT RULES:
1. Use only the supplied evidence. Do not use outside medical knowledge.
2. A priority must be supported by patient/caregiver statements, patient polling, or the report's
   summary of patient/caregiver perspectives.
3. Ignore general disease background, epidemiology, meeting logistics, physician-only commentary,
   natural-history-study interest, and clinical-trial participation preferences.
4. Merge obvious synonyms, but keep clinically different outcomes separate.
   For example, frequency, severity, speed of response, and completeness of response should remain
   separate if the evidence treats them as distinct benefits.
5. Do not create an item merely because a symptom or concept appears in the report. It must be
   presented as meaningful, burdensome, important, or desired by patients/caregivers.
6. Do not rank priorities unless the evidence itself supports a ranking.
7. Return a concise set of distinct priorities rather than every possible phrase.

ALIGNMENT ELIGIBILITY:
Set alignment_eligible = true when the concept is a clinical, symptomatic, functional, or
quality-of-life outcome that could reasonably be measured as an endpoint in a therapeutic trial.

Set alignment_eligible = false for concerns such as:
- route of administration
- dosing convenience or frequency
- treatment access, insurance, or cost
- general treatment burden
- trial participation preferences
- generic desire for a cure
- treatment-risk preferences

These concerns can remain in the broader patient-priority database, but they will not be used
to penalize efficacy trials in the main alignment score.

PRIORITY DOMAINS:
Use one of:
- disease_control
- symptom_burden
- function_daily_life
- treatment_effectiveness
- treatment_burden
- treatment_safety
- access_convenience
- other

EVIDENCE:
For each priority, return the chunk IDs that support it.
Do NOT invent or paraphrase a quotation as evidence. Python will reconnect the chunk IDs to
the original report text afterward.

Return valid JSON in exactly this structure:

{{
  "priorities": [
    {{
      "priority_label": "short descriptive label",
      "priority_description": "one-sentence explanation of what matters to patients",
      "priority_domain": "one allowed domain",
      "alignment_eligible": true,
      "evidence_chunk_ids": ["chunk_id_1", "chunk_id_2"],
      "support_type": "polling", "discussion", or "polling_and_discussion",
      "confidence": "high", "medium", or "low"
    }}
  ]
}}

RETRIEVED EVIDENCE:
{hae_evidence_text}
"""

In [ ]:
# Run the HAE priority extraction and preserve the raw LLM response.
hae_response = client.responses.create(
    model=LLM_MODEL,
    input=hae_priority_prompt
)

hae_priority_raw = hae_response.output_text

print(hae_priority_raw)

with open(
    DATA_DIR / "hae_priority_llm_raw.json",
    "w",
    encoding="utf-8"
) as f:
    f.write(hae_priority_raw)

print("\nSaved raw HAE LLM output.")

{
  "priorities": [
    {
      "priority_label": "Reduction in attack frequency",
      "priority_description": "Patients value having fewer HAE attacks.",
      "priority_domain": "disease_control",
      "alignment_eligible": true,
      "evidence_chunk_ids": ["hae_vop_p17_c2", "hae_vop_p19_c4", "hae_vop_p29_c1"],
      "support_type": "polling_and_discussion",
      "confidence": "high"
    },
    {
      "priority_label": "Reduction in attack severity",
      "priority_description": "Patients value attacks being less severe.",
      "priority_domain": "disease_control",
      "alignment_eligible": true,
      "evidence_chunk_ids": ["hae_vop_p17_c2", "hae_vop_p29_c1"],
      "support_type": "polling_and_discussion",
      "confidence": "high"
    },
    {
      "priority_label": "Rapid response to acute treatment",
      "priority_description": "Patients value acute treatment that begins working quickly.",
      "priority_domain": "treatment_effectiveness",
      "alignment_eligibl

In [ ]:
# Parse the structured HAE priority candidates returned by the LLM.
import json

hae_priority_json = json.loads(hae_priority_raw)

hae_priority_candidates = pd.DataFrame(
    hae_priority_json["priorities"]
)

display(hae_priority_candidates)

,priority_label,priority_description,priority_domain,alignment_eligible,evidence_chunk_ids,support_type,confidence
0,Reduction in attack frequency,Patients value having fewer HAE attacks.,disease_control,True,"[hae_vop_p17_c2, hae_vop_p19_c4, hae_vop_p29_c1]",polling_and_discussion,high
1,Reduction in attack severity,Patients value attacks being less severe.,disease_control,True,"[hae_vop_p17_c2, hae_vop_p29_c1]",polling_and_discussion,high
2,Rapid response to acute treatment,Patients value acute treatment that begins wor...,treatment_effectiveness,True,"[hae_vop_p17_c2, hae_vop_p29_c1]",polling_and_discussion,high
3,Complete response to acute treatment,Patients value acute treatment that fully reso...,treatment_effectiveness,True,"[hae_vop_p17_c2, hae_vop_p29_c2]",polling_and_discussion,high
4,Throat or laryngeal swelling and breathing dif...,"Patients identify throat swelling, hoarseness,...",symptom_burden,True,[hae_vop_p27_c1],polling,high
5,Abdominal pain,Patients identify abdominal pain as a symptom ...,symptom_burden,True,[hae_vop_p27_c1],polling,high
6,Facial swelling,Patients identify facial swelling as having a ...,symptom_burden,True,[hae_vop_p27_c1],polling,high
7,Tongue swelling,Patients identify tongue swelling as having a ...,symptom_burden,True,[hae_vop_p27_c1],polling,high
8,"Arm, hand, leg, or foot swelling","Patients identify swelling of the arms, hands,...",symptom_burden,True,[hae_vop_p27_c1],polling,medium
9,Nausea and vomiting,Patients identify nausea or vomiting as sympto...,symptom_burden,True,[hae_vop_p27_c1],polling,medium


In [ ]:
# Verify that every evidence chunk cited by the LLM exists in the retrieved HAE evidence.
valid_hae_chunk_ids = set(
    hae_evidence_candidates["chunk_id"]
)

cited_hae_chunk_ids = {
    chunk_id
    for chunk_list in hae_priority_candidates["evidence_chunk_ids"]
    for chunk_id in chunk_list
}

invalid_hae_chunk_ids = (
    cited_hae_chunk_ids - valid_hae_chunk_ids
)

print("Cited chunks:", len(cited_hae_chunk_ids))
print("Invalid chunk IDs:", invalid_hae_chunk_ids)

Cited chunks: 8
Invalid chunk IDs: set()


In [ ]:
# Reconnect each extracted HAE priority to its original report pages and retrieved text.
hae_chunk_lookup = (
    hae_evidence_candidates
    .set_index("chunk_id")
    [["pdf_page", "chunk_text"]]
    .to_dict("index")
)

def attach_priority_evidence(chunk_ids):
    records = [
        hae_chunk_lookup[chunk_id]
        for chunk_id in chunk_ids
        if chunk_id in hae_chunk_lookup
    ]

    pages = sorted({
        record["pdf_page"]
        for record in records
    })

    evidence_text = "\n\n".join(
        record["chunk_text"]
        for record in records
    )

    return pd.Series({
        "evidence_pdf_pages": pages,
        "evidence_text": evidence_text
    })


hae_evidence_attached = (
    hae_priority_candidates["evidence_chunk_ids"]
    .apply(attach_priority_evidence)
)

hae_priority_candidates = pd.concat(
    [
        hae_priority_candidates,
        hae_evidence_attached
    ],
    axis=1
)

hae_priority_candidates.insert(
    0,
    "disease",
    "Hereditary Angioedema"
)

hae_priority_candidates["source_id"] = "hae_vop"

display(
    hae_priority_candidates[
        [
            "priority_label",
            "priority_domain",
            "alignment_eligible",
            "support_type",
            "confidence",
            "evidence_pdf_pages"
        ]
    ]
)

,priority_label,priority_domain,alignment_eligible,support_type,confidence,evidence_pdf_pages
0,Reduction in attack frequency,disease_control,True,polling_and_discussion,high,"[17, 19, 29]"
1,Reduction in attack severity,disease_control,True,polling_and_discussion,high,"[17, 29]"
2,Rapid response to acute treatment,treatment_effectiveness,True,polling_and_discussion,high,"[17, 29]"
3,Complete response to acute treatment,treatment_effectiveness,True,polling_and_discussion,high,"[17, 29]"
4,Throat or laryngeal swelling and breathing dif...,symptom_burden,True,polling,high,[27]
5,Abdominal pain,symptom_burden,True,polling,high,[27]
6,Facial swelling,symptom_burden,True,polling,high,[27]
7,Tongue swelling,symptom_burden,True,polling,high,[27]
8,"Arm, hand, leg, or foot swelling",symptom_burden,True,polling,medium,[27]
9,Nausea and vomiting,symptom_burden,True,polling,medium,[27]


In [ ]:
# Save the unedited HAE candidates so the original LLM extraction remains auditable.
hae_priority_candidates.to_json(
    DATA_DIR / "hae_priority_candidates.json",
    orient="records",
    indent=2
)

print("Saved HAE priority candidates.")

Saved HAE priority candidates.


In [ ]:
# Combine evidence from multiple raw HAE candidates into one validated priority.
def combine_hae_candidate_evidence(source_labels):
    subset = hae_priority_candidates[
        hae_priority_candidates["priority_label"].isin(source_labels)
    ]

    chunk_ids = []
    for ids in subset["evidence_chunk_ids"]:
        chunk_ids.extend(ids)

    chunk_ids = list(dict.fromkeys(chunk_ids))

    pages = sorted({
        hae_chunk_lookup[chunk_id]["pdf_page"]
        for chunk_id in chunk_ids
        if chunk_id in hae_chunk_lookup
    })

    evidence_text = "\n\n".join(
        hae_chunk_lookup[chunk_id]["chunk_text"]
        for chunk_id in chunk_ids
        if chunk_id in hae_chunk_lookup
    )

    support_types = set(subset["support_type"])

    if (
        "polling_and_discussion" in support_types
        or ("polling" in support_types and "discussion" in support_types)
    ):
        support_type = "polling_and_discussion"
    elif "polling" in support_types:
        support_type = "polling"
    else:
        support_type = "discussion"

    return {
        "evidence_chunk_ids": chunk_ids,
        "evidence_pdf_pages": pages,
        "evidence_text": evidence_text,
        "support_type": support_type
    }

In [ ]:
# Define the concept-level HAE priorities retained after human review.
hae_priority_plan = [
    {
        "priority_label": "Reduce attack frequency",
        "priority_description": "Reduce how often HAE attacks occur.",
        "priority_domain": "disease_control",
        "alignment_eligible": True,
        "source_labels": ["Reduction in attack frequency"]
    },
    {
        "priority_label": "Reduce attack severity",
        "priority_description": "Reduce the overall severity of HAE attacks.",
        "priority_domain": "disease_control",
        "alignment_eligible": True,
        "source_labels": ["Reduction in attack severity"]
    },
    {
        "priority_label": "Rapid response to acute treatment",
        "priority_description": "Provide relief quickly after treatment for an acute attack.",
        "priority_domain": "treatment_effectiveness",
        "alignment_eligible": True,
        "source_labels": ["Rapid response to acute treatment"]
    },
    {
        "priority_label": "Complete response to acute treatment",
        "priority_description": "Fully resolve the symptoms of an acute HAE attack.",
        "priority_domain": "treatment_effectiveness",
        "alignment_eligible": True,
        "source_labels": ["Complete response to acute treatment"]
    },
    {
        "priority_label": "Prevent or reduce airway-threatening swelling",
        "priority_description": "Reduce throat, laryngeal, or tongue swelling that can interfere with breathing.",
        "priority_domain": "symptom_burden",
        "alignment_eligible": True,
        "source_labels": [
            "Throat or laryngeal swelling and breathing difficulty",
            "Tongue swelling"
        ]
    },
    {
        "priority_label": "Reduce abdominal and gastrointestinal attack symptoms",
        "priority_description": "Reduce abdominal pain, nausea, and vomiting associated with HAE attacks.",
        "priority_domain": "symptom_burden",
        "alignment_eligible": True,
        "source_labels": [
            "Abdominal pain",
            "Nausea and vomiting"
        ]
    },
    {
        "priority_label": "Reduce pain during attacks",
        "priority_description": "Reduce the pain experienced during HAE attacks.",
        "priority_domain": "symptom_burden",
        "alignment_eligible": True,
        "source_labels": ["Pain during attacks"]
    },
    {
        "priority_label": "Reduce fatigue",
        "priority_description": "Reduce fatigue associated with HAE and its attacks.",
        "priority_domain": "symptom_burden",
        "alignment_eligible": True,
        "source_labels": ["Fatigue"]
    },
    {
        "priority_label": "Maintain daily functioning and independence",
        "priority_description": "Preserve the ability to work or attend school, care for oneself, and care for dependents.",
        "priority_domain": "function_daily_life",
        "alignment_eligible": True,
        "source_labels": [
            "Ability to work or attend school",
            "Ability to care for oneself",
            "Ability to care for children"
        ]
    },
    {
        "priority_label": "Maintain social and physical participation",
        "priority_description": "Preserve participation in family, social, recreational, and physical activities.",
        "priority_domain": "function_daily_life",
        "alignment_eligible": True,
        "source_labels": [
            "Participation in family and social activities",
            "Participation in sports"
        ]
    },
    {
        "priority_label": "Reduce anxiety related to HAE",
        "priority_description": "Reduce the anxiety and psychological burden associated with living with HAE.",
        "priority_domain": "function_daily_life",
        "alignment_eligible": True,
        "source_labels": ["Anxiety related to HAE"]
    },
    {
        "priority_label": "Avoid treatment side effects",
        "priority_description": "Minimize short- and long-term adverse effects of HAE treatment.",
        "priority_domain": "treatment_safety",
        "alignment_eligible": False,
        "source_labels": ["Avoiding treatment side effects"]
    }
]

In [ ]:
# Build the validated HAE priority records while preserving their original evidence.
hae_validated_rows = []

for item in hae_priority_plan:
    evidence = combine_hae_candidate_evidence(
        item["source_labels"]
    )

    hae_validated_rows.append({
        "disease": "Hereditary Angioedema",
        "priority_label": item["priority_label"],
        "priority_description": item["priority_description"],
        "priority_domain": item["priority_domain"],
        "alignment_eligible": item["alignment_eligible"],
        "evidence_chunk_ids": evidence["evidence_chunk_ids"],
        "evidence_pdf_pages": evidence["evidence_pdf_pages"],
        "evidence_text": evidence["evidence_text"],
        "source_id": "hae_vop",
        "support_type": evidence["support_type"],
        "review_status": "validated"
    })

In [ ]:
# Add important non-scoring HAE treatment concerns that the LLM omitted.
context_chunk_id = "hae_vop_p17_c3"

context_evidence = hae_chunk_lookup[context_chunk_id]

hae_context_priorities = [
    {
        "priority_label": "Less burdensome route of administration",
        "priority_description": "Patients prefer treatment routes that are less invasive and less traumatic.",
        "priority_domain": "access_convenience"
    },
    {
        "priority_label": "Less frequent treatment administration",
        "priority_description": "Patients value treatments that require less frequent administration.",
        "priority_domain": "access_convenience"
    },
    {
        "priority_label": "Better treatment access and availability",
        "priority_description": "Patients value reliable access to treatment, including affordability and regional availability.",
        "priority_domain": "access_convenience"
    }
]

for item in hae_context_priorities:
    hae_validated_rows.append({
        "disease": "Hereditary Angioedema",
        "priority_label": item["priority_label"],
        "priority_description": item["priority_description"],
        "priority_domain": item["priority_domain"],
        "alignment_eligible": False,
        "evidence_chunk_ids": [context_chunk_id],
        "evidence_pdf_pages": [context_evidence["pdf_page"]],
        "evidence_text": context_evidence["chunk_text"],
        "source_id": "hae_vop",
        "support_type": "discussion",
        "review_status": "validated"
    })

In [ ]:
# Create and save the final human-validated HAE patient-priority dataset.
hae_priorities_final = pd.DataFrame(
    hae_validated_rows
)

hae_priorities_final.insert(
    1,
    "priority_id",
    [
        f"HAE_P{i:02d}"
        for i in range(1, len(hae_priorities_final) + 1)
    ]
)

hae_priorities_final.to_json(
    DATA_DIR / "hae_priorities_final.json",
    orient="records",
    indent=2
)

hae_priorities_final.to_csv(
    DATA_DIR / "hae_priorities_final.csv",
    index=False
)

print("Total validated HAE priorities:", len(hae_priorities_final))
print(
    "Alignment-eligible:",
    hae_priorities_final["alignment_eligible"].sum()
)
print(
    "Context only:",
    (~hae_priorities_final["alignment_eligible"]).sum()
)

display(
    hae_priorities_final[
        [
            "priority_id",
            "priority_label",
            "priority_domain",
            "alignment_eligible",
            "support_type",
            "evidence_pdf_pages"
        ]
    ]
)

Total validated HAE priorities: 15
Alignment-eligible: 11
Context only: 4


,priority_id,priority_label,priority_domain,alignment_eligible,support_type,evidence_pdf_pages
0,HAE_P01,Reduce attack frequency,disease_control,True,polling_and_discussion,"[17, 19, 29]"
1,HAE_P02,Reduce attack severity,disease_control,True,polling_and_discussion,"[17, 29]"
2,HAE_P03,Rapid response to acute treatment,treatment_effectiveness,True,polling_and_discussion,"[17, 29]"
3,HAE_P04,Complete response to acute treatment,treatment_effectiveness,True,polling_and_discussion,"[17, 29]"
4,HAE_P05,Prevent or reduce airway-threatening swelling,symptom_burden,True,polling,[27]
5,HAE_P06,Reduce abdominal and gastrointestinal attack s...,symptom_burden,True,polling,[27]
6,HAE_P07,Reduce pain during attacks,symptom_burden,True,discussion,[15]
7,HAE_P08,Reduce fatigue,symptom_burden,True,discussion,[15]
8,HAE_P09,Maintain daily functioning and independence,function_daily_life,True,polling,[28]
9,HAE_P10,Maintain social and physical participation,function_daily_life,True,polling_and_discussion,"[15, 28]"


## Create a reusable VOTP Function

In [ ]:
# Create a reusable function for retrieving patient-priority evidence for each disease.
def build_priority_evidence_candidates(
    disease,
    queries,
    top_k=8
):
    retrievals = []

    for query_number, query in enumerate(queries, start=1):
        results = retrieve_patient_evidence(
            disease=disease,
            query=query,
            top_k=top_k
        ).copy()

        results["query_number"] = query_number
        results["query"] = query

        retrievals.append(results)

    retrievals = pd.concat(
        retrievals,
        ignore_index=True
    )

    candidates = (
        retrievals
        .sort_values("similarity", ascending=False)
        .drop_duplicates("chunk_id")
        .sort_values(
            ["pdf_page", "chunk_number"]
        )
        .reset_index(drop=True)
    )

    return retrievals, candidates

In [ ]:
# Define the standardized retrieval questions for Friedreich's Ataxia.
fa_queries = [
    "What symptoms, complications, or disease effects of Friedreich's Ataxia do patients and caregivers say are most important or burdensome?",

    "How does Friedreich's Ataxia affect mobility, balance, walking, falls, coordination, hand use, speech, fatigue, or other everyday abilities?",

    "How does Friedreich's Ataxia affect independence, work, school, social life, daily activities, or quality of life?",

    "What disease outcomes or improvements do patients say they most want treatments to achieve?",

    "What do patients say about slowing disease progression, preserving function, maintaining independence, or preventing serious complications?",

    "What treatment burdens, side effects, risks, or practical treatment concerns do patients and caregivers describe?"
]

In [ ]:
# Retrieve and deduplicate Friedreich's Ataxia patient-priority evidence.
fa_retrievals, fa_evidence_candidates = (
    build_priority_evidence_candidates(
        disease="Friedreich's Ataxia",
        queries=fa_queries,
        top_k=8
    )
)

print("Retrieved rows:", len(fa_retrievals))
print(
    "Unique evidence candidates:",
    len(fa_evidence_candidates)
)

display(
    fa_evidence_candidates[
        [
            "pdf_page",
            "chunk_number",
            "similarity",
            "chunk_text"
        ]
    ]
)

Retrieved rows: 48
Unique evidence candidates: 30


,pdf_page,chunk_number,similarity,chunk_text
0,7,1,0.814272,6 throughout the disease community. Additional...
1,7,2,0.567962,pursue hobbies. Loss of independence was noted...
2,7,3,0.698343,"issues, and it was noted that the surgeries we..."
3,7,4,0.641637,and cures for the disease. Many had already ta...
4,8,1,0.768931,"7 a treatment, and what risks they may be will..."
5,13,1,0.512370,12 significant impact on daily life. One FA pa...
6,15,1,0.564574,"14 In older patients, the isolation increases ..."
7,19,2,0.602591,trying a new treatment. The most highly ranked...
8,19,4,0.641327,"in the coming years. For both, medicine that r..."
9,19,5,0.569158,against time and sustain life as it is today.”...


In [ ]:
# Save the Friedreich's Ataxia evidence candidates before LLM extraction.
fa_evidence_candidates.to_csv(
    DATA_DIR / "fa_priority_evidence_candidates.csv",
    index=False
)

print("Saved Friedreich's Ataxia evidence candidates.")

Saved Friedreich's Ataxia evidence candidates.


In [ ]:
# Combine the retrieved FA chunks into one evidence package for the LLM.
fa_evidence_text = "\n\n".join(
    f"[{row['chunk_id']}] PDF page {row['pdf_page']}\n{row['chunk_text']}"
    for _, row in fa_evidence_candidates.iterrows()
)

print("Evidence characters:", len(fa_evidence_text))
print(fa_evidence_text[:3000])

Evidence characters: 29175
[fa_vop_p7_c1] PDF page 7
6 throughout the disease community. Additional groups of patients may have not been represented, and the demographics of the group was limited as noted above. The main themes that came out in the meeting, polling and testimonies were as follows:  Friedreich’s Ataxia is a devastating multisystem progressive disease. Patient symptoms include pervasive issues with loss of balance/coordination, issues with ambulation and falls, loss of manual dexterity and fatigue. These occur in nearly all patients. Over 50% of polling respondents also reported scoliosis, cardiac concerns, dysarthria and spasticity, with other symptoms represented in lower numbers of patients.  The disease has an enormous effect on quality of life, with younger patients discussing the social and physical concerns of navigating life with a disability, and older patients discussing how progression of specific symptoms limited their ability to work, have families and pur

In [ ]:
# Define the improved extraction instructions for Friedreich's Ataxia patient priorities.
fa_priority_prompt = f"""
You are helping construct an auditable patient-priority dataset for a clinical-trial analysis.

DISEASE:
Friedreich's Ataxia

TASK:
Using ONLY the retrieved evidence below, identify the distinct outcomes, symptoms, functional
impacts, treatment effects, and treatment concerns that patients or caregivers indicate are
meaningful or important.

The final project asks whether clinical-trial endpoints measure outcomes patients care about.

IMPORTANT RULES:
1. Use only the supplied evidence. Do not use outside medical knowledge.
2. A priority must be supported by patient/caregiver statements, patient polling, or the report's
   summary of patient/caregiver perspectives.
3. Ignore general disease background, meeting logistics, physician-only commentary, and generic
   clinical-trial participation interest.
4. Extract priorities at the CONCEPT level rather than turning every survey answer choice or symptom
   mention into its own row.
5. Merge closely related manifestations when they represent one broader patient concern.
6. However, keep concepts separate when the report clearly treats them as distinct meaningful
   outcomes. For example, balance, walking, fatigue, manual dexterity, cardiac function, and speech
   may remain separate if the evidence independently emphasizes them.
7. Do not create an item merely because a symptom appears in the report. It must be described as
   burdensome, important, concerning, or meaningful to improve.
8. Preserve broader functional priorities such as independence or ability to participate in daily
   life when patients clearly describe these as important.
9. Do not rank priorities unless the evidence itself supports a ranking.
10. Return a concise concept-level set rather than every possible phrase.

ALIGNMENT ELIGIBILITY:
Set alignment_eligible = true when the concept is a clinical, symptomatic, functional, survival,
or quality-of-life outcome that could reasonably be measured as an endpoint in a therapeutic trial.

Set alignment_eligible = false for concerns such as:
- route of administration
- treatment frequency or time burden
- travel burden
- treatment cost or access
- treatment inconvenience
- general willingness to participate in research
- generic desire for a cure
- treatment-risk preferences

Treatment safety itself may be retained as a contextual priority with alignment_eligible = false
for the main efficacy-alignment score.

PRIORITY DOMAINS:
Use one of:
- disease_control
- symptom_burden
- function_daily_life
- treatment_effectiveness
- survival_major_complications
- treatment_burden
- treatment_safety
- access_convenience
- other

EVIDENCE:
For each priority, return the chunk IDs that support it.
Do NOT invent or paraphrase a quotation as evidence. Python will reconnect the chunk IDs to
the original report text afterward.

Return valid JSON in exactly this structure:

{{
  "priorities": [
    {{
      "priority_label": "short descriptive label",
      "priority_description": "one-sentence explanation of what matters to patients",
      "priority_domain": "one allowed domain",
      "alignment_eligible": true,
      "evidence_chunk_ids": ["chunk_id_1", "chunk_id_2"],
      "support_type": "polling", "discussion", or "polling_and_discussion",
      "confidence": "high", "medium", or "low"
    }}
  ]
}}

RETRIEVED EVIDENCE:
{fa_evidence_text}
"""

In [ ]:
# Run the FA priority extraction and preserve the raw LLM response.
fa_response = client.responses.create(
    model=LLM_MODEL,
    input=fa_priority_prompt
)

fa_priority_raw = fa_response.output_text

print(fa_priority_raw)

with open(
    DATA_DIR / "fa_priority_llm_raw.json",
    "w",
    encoding="utf-8"
) as f:
    f.write(fa_priority_raw)

print("\nSaved raw FA LLM output.")

{
  "priorities": [
    {
      "priority_label": "Slow or stop disease progression",
      "priority_description": "Patients value slowing or stopping progression, even without functional improvement, to preserve their current abilities and plan their lives.",
      "priority_domain": "disease_control",
      "alignment_eligible": true,
      "evidence_chunk_ids": ["fa_vop_p7_c2", "fa_vop_p7_c3", "fa_vop_p19_c4", "fa_vop_p19_c5", "fa_vop_p21_c2", "fa_vop_p23_c2", "fa_vop_p75_c1"],
      "support_type": "polling_and_discussion",
      "confidence": "high"
    },
    {
      "priority_label": "Balance and standing",
      "priority_description": "Improving balance and the ability to stand is an important outcome for patients.",
      "priority_domain": "function_daily_life",
      "alignment_eligible": true,
      "evidence_chunk_ids": ["fa_vop_p7_c1", "fa_vop_p7_c3", "fa_vop_p20_c2", "fa_vop_p21_c2", "fa_vop_p23_c2", "fa_vop_p54_c1"],
      "support_type": "polling_and_discussion",
   

In [ ]:
# Parse the structured FA priority candidates returned by the LLM.
import json

fa_priority_json = json.loads(fa_priority_raw)

fa_priority_candidates = pd.DataFrame(
    fa_priority_json["priorities"]
)

display(fa_priority_candidates)

,priority_label,priority_description,priority_domain,alignment_eligible,evidence_chunk_ids,support_type,confidence
0,Slow or stop disease progression,Patients value slowing or stopping progression...,disease_control,True,"[fa_vop_p7_c2, fa_vop_p7_c3, fa_vop_p19_c4, fa...",polling_and_discussion,high
1,Balance and standing,Improving balance and the ability to stand is ...,function_daily_life,True,"[fa_vop_p7_c1, fa_vop_p7_c3, fa_vop_p20_c2, fa...",polling_and_discussion,high
2,Walking and fewer falls,Patients value being able to walk more freely ...,function_daily_life,True,"[fa_vop_p7_c1, fa_vop_p19_c4, fa_vop_p20_c2, f...",polling_and_discussion,high
3,Fatigue,Reducing fatigue matters because it burdens pa...,symptom_burden,True,"[fa_vop_p7_c1, fa_vop_p7_c2, fa_vop_p7_c4, fa_...",polling_and_discussion,high
4,Manual dexterity and fine motor function,Patients value retaining or improving hand fun...,function_daily_life,True,"[fa_vop_p7_c1, fa_vop_p7_c3, fa_vop_p20_c2, fa...",polling_and_discussion,high
5,Cardiac function and cardiomyopathy,Patients are particularly concerned about card...,survival_major_complications,True,"[fa_vop_p7_c2, fa_vop_p7_c3, fa_vop_p19_c4, fa...",polling_and_discussion,high
6,Speech,Patients identify improving speech as a meanin...,function_daily_life,True,"[fa_vop_p7_c1, fa_vop_p7_c3, fa_vop_p19_c4, fa...",polling_and_discussion,high
7,Scoliosis,Patients identify reducing or stopping scolios...,symptom_burden,True,"[fa_vop_p19_c4, fa_vop_p54_c1]",polling_and_discussion,medium
8,Independence and participation in daily and so...,Patients value maintaining independence and be...,function_daily_life,True,"[fa_vop_p7_c1, fa_vop_p7_c2, fa_vop_p15_c1, fa...",discussion,high
9,Pain,Patients describe persistent pain as a substan...,symptom_burden,True,[fa_vop_p13_c1],discussion,medium


In [ ]:
# Verify that every evidence chunk cited by the LLM actually came from the FA retrieval.
valid_fa_chunk_ids = set(
    fa_evidence_candidates["chunk_id"]
)

cited_fa_chunk_ids = {
    chunk_id
    for chunk_list in fa_priority_candidates["evidence_chunk_ids"]
    for chunk_id in chunk_list
}

invalid_fa_chunk_ids = (
    cited_fa_chunk_ids - valid_fa_chunk_ids
)

print("Cited chunks:", len(cited_fa_chunk_ids))
print("Invalid chunk IDs:", invalid_fa_chunk_ids)

Cited chunks: 18
Invalid chunk IDs: set()


In [ ]:
# Reattach the original FA report pages and evidence text to each extracted priority.
fa_chunk_lookup = (
    fa_evidence_candidates
    .set_index("chunk_id")
    [["pdf_page", "chunk_text"]]
    .to_dict("index")
)

def attach_fa_priority_evidence(chunk_ids):
    records = [
        fa_chunk_lookup[chunk_id]
        for chunk_id in chunk_ids
        if chunk_id in fa_chunk_lookup
    ]

    pages = sorted({
        record["pdf_page"]
        for record in records
    })

    evidence_text = "\n\n".join(
        record["chunk_text"]
        for record in records
    )

    return pd.Series({
        "evidence_pdf_pages": pages,
        "evidence_text": evidence_text
    })


fa_evidence_attached = (
    fa_priority_candidates["evidence_chunk_ids"]
    .apply(attach_fa_priority_evidence)
)

fa_priority_candidates = pd.concat(
    [
        fa_priority_candidates,
        fa_evidence_attached
    ],
    axis=1
)

fa_priority_candidates.insert(
    0,
    "disease",
    "Friedreich's Ataxia"
)

fa_priority_candidates["source_id"] = "fa_vop"

display(
    fa_priority_candidates[
        [
            "priority_label",
            "priority_domain",
            "alignment_eligible",
            "support_type",
            "confidence",
            "evidence_pdf_pages"
        ]
    ]
)

,priority_label,priority_domain,alignment_eligible,support_type,confidence,evidence_pdf_pages
0,Slow or stop disease progression,disease_control,True,polling_and_discussion,high,"[7, 19, 21, 23, 75]"
1,Balance and standing,function_daily_life,True,polling_and_discussion,high,"[7, 20, 21, 23, 54]"
2,Walking and fewer falls,function_daily_life,True,polling_and_discussion,high,"[7, 19, 20, 54, 56]"
3,Fatigue,symptom_burden,True,polling_and_discussion,high,"[7, 20, 21, 54, 56]"
4,Manual dexterity and fine motor function,function_daily_life,True,polling_and_discussion,high,"[7, 20, 21, 23, 54]"
5,Cardiac function and cardiomyopathy,survival_major_complications,True,polling_and_discussion,high,"[7, 19, 21, 23, 54]"
6,Speech,function_daily_life,True,polling_and_discussion,high,"[7, 19, 21, 23, 54]"
7,Scoliosis,symptom_burden,True,polling_and_discussion,medium,"[19, 54]"
8,Independence and participation in daily and so...,function_daily_life,True,discussion,high,"[7, 15, 20, 21, 68, 75]"
9,Pain,symptom_burden,True,discussion,medium,[13]


In [ ]:
# Save the unedited FA candidates so the original LLM extraction remains auditable.
fa_priority_candidates.to_json(
    DATA_DIR / "fa_priority_candidates.json",
    orient="records",
    indent=2
)

print("Saved Friedreich's Ataxia priority candidates.")

Saved Friedreich's Ataxia priority candidates.


In [ ]:
# Combine evidence from selected rows of the saved FA candidate extraction.
def combine_fa_candidate_evidence(source_rows):
    subset = fa_priority_candidates.loc[source_rows]

    chunk_ids = []
    for ids in subset["evidence_chunk_ids"]:
        chunk_ids.extend(ids)

    chunk_ids = list(dict.fromkeys(chunk_ids))

    pages = sorted({
        fa_chunk_lookup[chunk_id]["pdf_page"]
        for chunk_id in chunk_ids
        if chunk_id in fa_chunk_lookup
    })

    evidence_text = "\n\n".join(
        fa_chunk_lookup[chunk_id]["chunk_text"]
        for chunk_id in chunk_ids
        if chunk_id in fa_chunk_lookup
    )

    support_types = set(subset["support_type"])

    if (
        "polling_and_discussion" in support_types
        or ("polling" in support_types and "discussion" in support_types)
    ):
        support_type = "polling_and_discussion"
    elif "polling" in support_types:
        support_type = "polling"
    else:
        support_type = "discussion"

    return {
        "evidence_chunk_ids": chunk_ids,
        "evidence_pdf_pages": pages,
        "evidence_text": evidence_text,
        "support_type": support_type
    }

In [ ]:
# Define the concept-level FA priorities retained after human review.
fa_priority_plan = [
    {
        "priority_label": "Slow or stop disease progression",
        "priority_description": "Slow or halt progression of Friedreich's Ataxia and preserve the patient's current level of function.",
        "priority_domain": "disease_control",
        "alignment_eligible": True,
        "source_rows": [0]
    },
    {
        "priority_label": "Improve balance and standing ability",
        "priority_description": "Improve or preserve balance and the ability to stand.",
        "priority_domain": "function_daily_life",
        "alignment_eligible": True,
        "source_rows": [1]
    },
    {
        "priority_label": "Improve walking and reduce falls",
        "priority_description": "Preserve or improve walking ability while reducing falls.",
        "priority_domain": "function_daily_life",
        "alignment_eligible": True,
        "source_rows": [2]
    },
    {
        "priority_label": "Reduce fatigue",
        "priority_description": "Reduce the fatigue that limits daily activity and quality of life.",
        "priority_domain": "symptom_burden",
        "alignment_eligible": True,
        "source_rows": [3]
    },
    {
        "priority_label": "Preserve manual dexterity and fine motor function",
        "priority_description": "Preserve or improve hand and arm coordination, dexterity, and fine motor abilities.",
        "priority_domain": "function_daily_life",
        "alignment_eligible": True,
        "source_rows": [4]
    },
    {
        "priority_label": "Preserve cardiac function and reduce cardiomyopathy",
        "priority_description": "Prevent or slow cardiac complications, particularly progression of cardiomyopathy.",
        "priority_domain": "survival_major_complications",
        "alignment_eligible": True,
        "source_rows": [5]
    },
    {
        "priority_label": "Preserve or improve speech",
        "priority_description": "Preserve or improve speech affected by Friedreich's Ataxia.",
        "priority_domain": "function_daily_life",
        "alignment_eligible": True,
        "source_rows": [6]
    },
    {
        "priority_label": "Prevent or slow scoliosis",
        "priority_description": "Prevent or reduce progression of scoliosis associated with Friedreich's Ataxia.",
        "priority_domain": "symptom_burden",
        "alignment_eligible": True,
        "source_rows": [7]
    },
    {
        "priority_label": "Maintain independence and participation in daily life",
        "priority_description": "Preserve independence and the ability to work, pursue hobbies, maintain relationships, and participate in daily and social activities.",
        "priority_domain": "function_daily_life",
        "alignment_eligible": True,
        "source_rows": [8]
    }
]

In [ ]:
# Build the validated FA priorities while preserving all supporting evidence.
fa_validated_rows = []

for item in fa_priority_plan:
    evidence = combine_fa_candidate_evidence(
        item["source_rows"]
    )

    fa_validated_rows.append({
        "disease": "Friedreich's Ataxia",
        "priority_label": item["priority_label"],
        "priority_description": item["priority_description"],
        "priority_domain": item["priority_domain"],
        "alignment_eligible": item["alignment_eligible"],
        "evidence_chunk_ids": evidence["evidence_chunk_ids"],
        "evidence_pdf_pages": evidence["evidence_pdf_pages"],
        "evidence_text": evidence["evidence_text"],
        "source_id": "fa_vop",
        "support_type": evidence["support_type"],
        "review_status": "validated"
    })

In [ ]:
# Add prolonged life, which was supported by the patient poll but missed by the LLM.
survival_chunk_id = "fa_vop_p35_c2"
survival_evidence = fa_chunk_lookup[survival_chunk_id]

fa_validated_rows.append({
    "disease": "Friedreich's Ataxia",
    "priority_label": "Prolong life",
    "priority_description": "Extend survival and reduce the life-limiting consequences of Friedreich's Ataxia.",
    "priority_domain": "survival_major_complications",
    "alignment_eligible": True,
    "evidence_chunk_ids": [survival_chunk_id],
    "evidence_pdf_pages": [survival_evidence["pdf_page"]],
    "evidence_text": survival_evidence["chunk_text"],
    "source_id": "fa_vop",
    "support_type": "polling",
    "review_status": "validated"
})

In [ ]:
# Preserve the strong patient concern about serious treatment side effects as context only.
safety_evidence = combine_fa_candidate_evidence([10])

fa_validated_rows.append({
    "disease": "Friedreich's Ataxia",
    "priority_label": "Avoid serious treatment side effects",
    "priority_description": "Patients place substantial importance on avoiding serious risks and adverse effects from treatment.",
    "priority_domain": "treatment_safety",
    "alignment_eligible": False,
    "evidence_chunk_ids": safety_evidence["evidence_chunk_ids"],
    "evidence_pdf_pages": safety_evidence["evidence_pdf_pages"],
    "evidence_text": safety_evidence["evidence_text"],
    "source_id": "fa_vop",
    "support_type": safety_evidence["support_type"],
    "review_status": "validated"
})

In [ ]:
# Add distinct non-scoring FA treatment-burden and access concerns.
fa_context_priorities = [
    {
        "priority_label": "Reduce treatment time burden",
        "priority_description": "Patients value treatments that require less time, hospitalization, and fewer medical visits.",
        "priority_domain": "treatment_burden",
        "chunk_ids": ["fa_vop_p36_c2"]
    },
    {
        "priority_label": "Reduce travel burden",
        "priority_description": "Patients value treatments and research participation that require less travel.",
        "priority_domain": "access_convenience",
        "chunk_ids": ["fa_vop_p19_c2", "fa_vop_p36_c2", "fa_vop_p79_c1"]
    },
    {
        "priority_label": "Reduce treatment cost",
        "priority_description": "Patients identify treatment and related financial costs as an important consideration.",
        "priority_domain": "access_convenience",
        "chunk_ids": ["fa_vop_p19_c2", "fa_vop_p36_c2", "fa_vop_p79_c1"]
    },
    {
        "priority_label": "Less burdensome treatment administration",
        "priority_description": "Patients value treatment delivery methods that are less invasive and less burdensome.",
        "priority_domain": "treatment_burden",
        "chunk_ids": ["fa_vop_p19_c2", "fa_vop_p36_c2"]
    }
]

for item in fa_context_priorities:
    records = [
        fa_chunk_lookup[chunk_id]
        for chunk_id in item["chunk_ids"]
    ]

    fa_validated_rows.append({
        "disease": "Friedreich's Ataxia",
        "priority_label": item["priority_label"],
        "priority_description": item["priority_description"],
        "priority_domain": item["priority_domain"],
        "alignment_eligible": False,
        "evidence_chunk_ids": item["chunk_ids"],
        "evidence_pdf_pages": sorted({
            record["pdf_page"]
            for record in records
        }),
        "evidence_text": "\n\n".join(
            record["chunk_text"]
            for record in records
        ),
        "source_id": "fa_vop",
        "support_type": "polling_and_discussion",
        "review_status": "validated"
    })

In [ ]:
# Create and save the final human-validated Friedreich's Ataxia priority dataset.
fa_priorities_final = pd.DataFrame(
    fa_validated_rows
)

fa_priorities_final.insert(
    1,
    "priority_id",
    [
        f"FA_P{i:02d}"
        for i in range(1, len(fa_priorities_final) + 1)
    ]
)

fa_priorities_final.to_json(
    DATA_DIR / "fa_priorities_final.json",
    orient="records",
    indent=2
)

fa_priorities_final.to_csv(
    DATA_DIR / "fa_priorities_final.csv",
    index=False
)

print("Total validated FA priorities:", len(fa_priorities_final))
print(
    "Alignment-eligible:",
    fa_priorities_final["alignment_eligible"].sum()
)
print(
    "Context only:",
    (~fa_priorities_final["alignment_eligible"]).sum()
)

display(
    fa_priorities_final[
        [
            "priority_id",
            "priority_label",
            "priority_domain",
            "alignment_eligible",
            "support_type",
            "evidence_pdf_pages"
        ]
    ]
)

Total validated FA priorities: 15
Alignment-eligible: 10
Context only: 5


,priority_id,priority_label,priority_domain,alignment_eligible,support_type,evidence_pdf_pages
0,FA_P01,Slow or stop disease progression,disease_control,True,polling_and_discussion,"[7, 19, 21, 23, 75]"
1,FA_P02,Improve balance and standing ability,function_daily_life,True,polling_and_discussion,"[7, 20, 21, 23, 54]"
2,FA_P03,Improve walking and reduce falls,function_daily_life,True,polling_and_discussion,"[7, 19, 20, 54, 56]"
3,FA_P04,Reduce fatigue,symptom_burden,True,polling_and_discussion,"[7, 20, 21, 54, 56]"
4,FA_P05,Preserve manual dexterity and fine motor function,function_daily_life,True,polling_and_discussion,"[7, 20, 21, 23, 54]"
5,FA_P06,Preserve cardiac function and reduce cardiomyo...,survival_major_complications,True,polling_and_discussion,"[7, 19, 21, 23, 54]"
6,FA_P07,Preserve or improve speech,function_daily_life,True,polling_and_discussion,"[7, 19, 21, 23, 54]"
7,FA_P08,Prevent or slow scoliosis,symptom_burden,True,polling_and_discussion,"[19, 54]"
8,FA_P09,Maintain independence and participation in dai...,function_daily_life,True,discussion,"[7, 15, 20, 21, 68, 75]"
9,FA_P10,Prolong life,survival_major_complications,True,polling,[35]


In [ ]:
# Define the standardized retrieval questions for FSHD patient priorities.
fshd_queries = [
    "What symptoms, functional losses, or disease effects of FSHD do patients and caregivers say are most important or burdensome?",

    "How does FSHD affect mobility, walking, upper body function, arm and hand use, strength, fatigue, pain, or other physical abilities?",

    "How does FSHD affect independence, work, school, family life, social participation, daily activities, or quality of life?",

    "What outcomes or improvements do patients say they most want future treatments to achieve?",

    "What do patients say about slowing or stopping progression, preserving function, regaining function, or maintaining independence?",

    "What treatment burdens, side effects, risks, access concerns, or practical treatment preferences do patients and caregivers describe?"
]

In [ ]:
# Retrieve and deduplicate FSHD patient-priority evidence from the report.
fshd_retrievals, fshd_evidence_candidates = (
    build_priority_evidence_candidates(
        disease="Facioscapulohumeral Muscular Dystrophy",
        queries=fshd_queries,
        top_k=8
    )
)

print("Retrieved rows:", len(fshd_retrievals))
print(
    "Unique evidence candidates:",
    len(fshd_evidence_candidates)
)

display(
    fshd_evidence_candidates[
        [
            "pdf_page",
            "chunk_number",
            "similarity",
            "chunk_text"
        ]
    ]
)

Retrieved rows: 48
Unique evidence candidates: 38


,pdf_page,chunk_number,similarity,chunk_text
0,3,2,0.779237,3 3. Overview of FSHD and its management . . ....
1,3,5,0.583283,. . . . . . . . . . . . . . . . . . . . . . . ...
2,5,1,0.714249,F S H D S O C I E T Y | V O I C E O F T H E P ...
3,9,4,0.541264,"people with FSHD manage their disorder, and th..."
4,13,1,0.666715,F S H D S O C I E T Y | V O I C E O F T H E P ...
5,17,3,0.697587,was flipped – the increased pain in my legs. N...
6,17,4,0.764815,"but its consequences, including impaired mobil..."
7,18,3,0.613190,living the normal life of a teenager continued...
8,22,2,0.652495,the infection with. My mom fought this disease...
9,27,2,0.464106,"and cruel in many ways, but losing my independ..."


In [ ]:
# Save the FSHD evidence candidates before LLM extraction.
fshd_evidence_candidates.to_csv(
    DATA_DIR / "fshd_priority_evidence_candidates.csv",
    index=False
)

print("Saved FSHD evidence candidates.")

Saved FSHD evidence candidates.


In [ ]:
# Combine the retrieved FSHD chunks into one evidence package for the LLM.
fshd_evidence_text = "\n\n".join(
    f"[{row['chunk_id']}] PDF page {row['pdf_page']}\n{row['chunk_text']}"
    for _, row in fshd_evidence_candidates.iterrows()
)

print("Evidence characters:", len(fshd_evidence_text))
print(fshd_evidence_text[:3000])

Evidence characters: 29604
[fshd_vop_p3_c2] PDF page 3
3 3. Overview of FSHD and its management . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 4 4. Meeting overview . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 6 5. Topic 1: Symptoms of FSHD and daily impacts that matter most to patients . . . . . . . . . . . . . 10 Muscle weakness and other symptoms . . . . . . . . . . . . . . . . . . . . . . . .

[fshd_vop_p3_c5] PDF page 3
. . . . . . . . . . . . . . . . . . . . . . . . . . 27 Currently available treatments and their effectiveness . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 27 Hopes and expectations for future treatment . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 37 7. Incorporating patient input into a benefit-risk framework for FSHD . . . . . . . . . . . . . . . . . . . . . 40 8. Conclusions . . . .

In [ ]:
# Define the extraction instructions for concept-level FSHD patient priorities.
fshd_priority_prompt = f"""
You are helping construct an auditable patient-priority dataset for a clinical-trial analysis.

DISEASE:
Facioscapulohumeral Muscular Dystrophy (FSHD)

TASK:
Using ONLY the retrieved evidence below, identify the distinct outcomes, symptoms, functional
impacts, treatment effects, and treatment concerns that patients or caregivers indicate are
meaningful or important.

The final project asks whether clinical-trial endpoints measure outcomes patients care about.

IMPORTANT RULES:
1. Use only the supplied evidence. Do not use outside medical knowledge.
2. A priority must be supported by patient/caregiver statements, patient polling, or the report's
   summary of patient/caregiver perspectives.
3. Ignore general disease background, meeting logistics, physician-only commentary, and generic
   interest in clinical-trial participation.
4. Extract priorities at the CONCEPT level rather than turning every survey response or individual
   activity into its own row.
5. Merge closely related manifestations when they represent one broader patient concern.
6. Keep concepts separate when the report clearly treats them as distinct meaningful outcomes.
   For example, disease progression, mobility, upper-limb function, fatigue, pain, respiratory
   function, and independence may remain separate when independently emphasized.
7. Do not create an item merely because it appears in the report. It must be described as
   burdensome, important, concerning, or meaningful to improve.
8. Preserve broader functional outcomes such as independence and participation in daily life when
   patients clearly describe them as important.
9. Do not rank priorities unless the evidence itself supports a ranking.
10. Return a concise concept-level set rather than every possible symptom or activity.

ALIGNMENT ELIGIBILITY:
Set alignment_eligible = true when the concept is a clinical, symptomatic, functional,
respiratory, survival, or quality-of-life outcome that could reasonably be measured as an
endpoint in a therapeutic trial.

Set alignment_eligible = false for concerns such as:
- treatment cost or insurance coverage
- treatment access or availability
- time or effort required for treatment
- route or convenience of administration
- general willingness to participate in research
- generic desire for a cure
- treatment-risk preferences

Treatment safety itself may be retained as a contextual priority with alignment_eligible = false
for the main efficacy-alignment score.

PRIORITY DOMAINS:
Use one of:
- disease_control
- symptom_burden
- function_daily_life
- treatment_effectiveness
- survival_major_complications
- treatment_burden
- treatment_safety
- access_convenience
- other

EVIDENCE:
For each priority, return the chunk IDs that support it.
Do NOT invent or paraphrase a quotation as evidence. Python will reconnect the chunk IDs to
the original report text afterward.

Return valid JSON in exactly this structure:

{{
  "priorities": [
    {{
      "priority_label": "short descriptive label",
      "priority_description": "one-sentence explanation of what matters to patients",
      "priority_domain": "one allowed domain",
      "alignment_eligible": true,
      "evidence_chunk_ids": ["chunk_id_1", "chunk_id_2"],
      "support_type": "polling", "discussion", or "polling_and_discussion",
      "confidence": "high", "medium", or "low"
    }}
  ]
}}

RETRIEVED EVIDENCE:
{fshd_evidence_text}
"""

In [ ]:
# Run the FSHD priority extraction and preserve the raw LLM response.
fshd_response = client.responses.create(
    model=LLM_MODEL,
    input=fshd_priority_prompt
)

fshd_priority_raw = fshd_response.output_text

print(fshd_priority_raw)

with open(
    DATA_DIR / "fshd_priority_llm_raw.json",
    "w",
    encoding="utf-8"
) as f:
    f.write(fshd_priority_raw)

print("\nSaved raw FSHD LLM output.")

{
  "priorities": [
    {
      "priority_label": "Slowing progression and preserving or restoring muscle function",
      "priority_description": "Patients want treatment to slow or stop the loss of muscle function, stabilize their condition, or regain strength and muscle function.",
      "priority_domain": "treatment_effectiveness",
      "alignment_eligible": true,
      "evidence_chunk_ids": ["fshd_vop_p40_c1", "fshd_vop_p40_c2", "fshd_vop_p41_c1", "fshd_vop_p136_c2"],
      "support_type": "polling_and_discussion",
      "confidence": "high"
    },
    {
      "priority_label": "Mobility and walking",
      "priority_description": "Maintaining the ability to walk and move safely matters to patients, who describe mobility loss, difficulty with stairs, and falls as major impacts.",
      "priority_domain": "function_daily_life",
      "alignment_eligible": true,
      "evidence_chunk_ids": ["fshd_vop_p17_c3", "fshd_vop_p17_c4", "fshd_vop_p18_c3", "fshd_vop_p22_c2", "fshd_vop_p27_c2

In [ ]:
# Parse the structured FSHD priority candidates returned by the LLM.
fshd_priority_json = json.loads(fshd_priority_raw)

fshd_priority_candidates = pd.DataFrame(
    fshd_priority_json["priorities"]
)

display(fshd_priority_candidates)

,priority_label,priority_description,priority_domain,alignment_eligible,evidence_chunk_ids,support_type,confidence
0,Slowing progression and preserving or restorin...,Patients want treatment to slow or stop the lo...,treatment_effectiveness,True,"[fshd_vop_p40_c1, fshd_vop_p40_c2, fshd_vop_p4...",polling_and_discussion,high
1,Mobility and walking,Maintaining the ability to walk and move safel...,function_daily_life,True,"[fshd_vop_p17_c3, fshd_vop_p17_c4, fshd_vop_p1...",polling_and_discussion,high
2,Upper-limb and hand function,Patients value being able to use their arms an...,function_daily_life,True,"[fshd_vop_p13_c1, fshd_vop_p17_c3, fshd_vop_p1...",polling_and_discussion,high
3,Fatigue,Fatigue is reported as a substantial symptom t...,symptom_burden,True,"[fshd_vop_p5_c1, fshd_vop_p17_c3, fshd_vop_p17...",polling_and_discussion,high
4,Pain,Patients identify pain as a burdensome symptom...,symptom_burden,True,"[fshd_vop_p17_c3, fshd_vop_p18_c3, fshd_vop_p4...",polling_and_discussion,high
5,Respiratory function,Patients are concerned about preserving breath...,survival_major_complications,True,"[fshd_vop_p17_c4, fshd_vop_p27_c2, fshd_vop_p4...",polling_and_discussion,medium
6,Independence,Maintaining or regaining independence is impor...,function_daily_life,True,"[fshd_vop_p27_c2, fshd_vop_p33_c1, fshd_vop_p7...",polling_and_discussion,high
7,Participation in daily life and social activities,Patients value being able to pursue hobbies an...,function_daily_life,True,"[fshd_vop_p22_c2, fshd_vop_p45_c1, fshd_vop_p9...",polling_and_discussion,high
8,Uncertainty and emotional distress about progr...,The unpredictable course of the disease causes...,symptom_burden,True,"[fshd_vop_p27_c2, fshd_vop_p28_c3, fshd_vop_p4...",polling_and_discussion,high
9,Communication and swallowing function,Patients express concern about losing the abil...,function_daily_life,True,"[fshd_vop_p18_c3, fshd_vop_p27_c2, fshd_vop_p1...",polling,medium


In [ ]:
# Verify that every evidence chunk cited by the LLM came from the FSHD retrieval.
valid_fshd_chunk_ids = set(
    fshd_evidence_candidates["chunk_id"]
)

cited_fshd_chunk_ids = {
    chunk_id
    for chunk_list in fshd_priority_candidates["evidence_chunk_ids"]
    for chunk_id in chunk_list
}

invalid_fshd_chunk_ids = (
    cited_fshd_chunk_ids - valid_fshd_chunk_ids
)

print("Cited chunks:", len(cited_fshd_chunk_ids))
print("Invalid chunk IDs:", invalid_fshd_chunk_ids)

Cited chunks: 22
Invalid chunk IDs: set()


In [ ]:
# Reattach the original FSHD report pages and evidence text to each extracted priority.
fshd_chunk_lookup = (
    fshd_evidence_candidates
    .set_index("chunk_id")
    [["pdf_page", "chunk_text"]]
    .to_dict("index")
)

def attach_fshd_priority_evidence(chunk_ids):
    records = [
        fshd_chunk_lookup[chunk_id]
        for chunk_id in chunk_ids
        if chunk_id in fshd_chunk_lookup
    ]

    pages = sorted({
        record["pdf_page"]
        for record in records
    })

    evidence_text = "\n\n".join(
        record["chunk_text"]
        for record in records
    )

    return pd.Series({
        "evidence_pdf_pages": pages,
        "evidence_text": evidence_text
    })


fshd_evidence_attached = (
    fshd_priority_candidates["evidence_chunk_ids"]
    .apply(attach_fshd_priority_evidence)
)

fshd_priority_candidates = pd.concat(
    [
        fshd_priority_candidates,
        fshd_evidence_attached
    ],
    axis=1
)

fshd_priority_candidates.insert(
    0,
    "disease",
    "Facioscapulohumeral Muscular Dystrophy"
)

fshd_priority_candidates["source_id"] = "fshd_vop"

display(
    fshd_priority_candidates[
        [
            "priority_label",
            "priority_domain",
            "alignment_eligible",
            "support_type",
            "confidence",
            "evidence_pdf_pages"
        ]
    ]
)

,priority_label,priority_domain,alignment_eligible,support_type,confidence,evidence_pdf_pages
0,Slowing progression and preserving or restorin...,treatment_effectiveness,True,polling_and_discussion,high,"[40, 41, 136]"
1,Mobility and walking,function_daily_life,True,polling_and_discussion,high,"[17, 18, 22, 27, 45]"
2,Upper-limb and hand function,function_daily_life,True,polling_and_discussion,high,"[13, 17, 18]"
3,Fatigue,symptom_burden,True,polling_and_discussion,high,"[5, 17, 18]"
4,Pain,symptom_burden,True,polling_and_discussion,high,"[17, 18, 40]"
5,Respiratory function,survival_major_complications,True,polling_and_discussion,medium,"[17, 27, 40]"
6,Independence,function_daily_life,True,polling_and_discussion,high,"[27, 33, 78]"
7,Participation in daily life and social activities,function_daily_life,True,polling_and_discussion,high,"[22, 45, 90]"
8,Uncertainty and emotional distress about progr...,symptom_burden,True,polling_and_discussion,high,"[27, 28, 45, 58, 68]"
9,Communication and swallowing function,function_daily_life,True,polling,medium,"[18, 27, 135]"


In [ ]:
# Save the unedited FSHD candidates so the original LLM extraction remains auditable.
fshd_priority_candidates.to_json(
    DATA_DIR / "fshd_priority_candidates.json",
    orient="records",
    indent=2
)

print("Saved FSHD priority candidates.")

Saved FSHD priority candidates.


In [ ]:
# Combine evidence from selected FSHD candidate rows for human validation.
def combine_fshd_candidate_evidence(source_rows):
    subset = fshd_priority_candidates.loc[source_rows]

    chunk_ids = []
    for ids in subset["evidence_chunk_ids"]:
        chunk_ids.extend(ids)

    chunk_ids = list(dict.fromkeys(chunk_ids))

    pages = sorted({
        fshd_chunk_lookup[chunk_id]["pdf_page"]
        for chunk_id in chunk_ids
        if chunk_id in fshd_chunk_lookup
    })

    evidence_text = "\n\n".join(
        fshd_chunk_lookup[chunk_id]["chunk_text"]
        for chunk_id in chunk_ids
        if chunk_id in fshd_chunk_lookup
    )

    support_types = set(subset["support_type"])

    if (
        "polling_and_discussion" in support_types
        or ("polling" in support_types and "discussion" in support_types)
    ):
        support_type = "polling_and_discussion"
    elif "polling" in support_types:
        support_type = "polling"
    else:
        support_type = "discussion"

    return {
        "evidence_chunk_ids": chunk_ids,
        "evidence_pdf_pages": pages,
        "evidence_text": evidence_text,
        "support_type": support_type
    }

In [ ]:
# Define the concept-level FSHD priorities retained after human review.
fshd_priority_plan = [
    {
        "priority_label": "Slow or stop loss of muscle function",
        "priority_description": "Slow or halt the progressive loss of muscle function caused by FSHD.",
        "priority_domain": "disease_control",
        "alignment_eligible": True,
        "source_rows": [0]
    },
    {
        "priority_label": "Regain strength and muscle function",
        "priority_description": "Restore strength or muscle function that has already been lost.",
        "priority_domain": "treatment_effectiveness",
        "alignment_eligible": True,
        "source_rows": [0]
    },
    {
        "priority_label": "Preserve mobility and walking ability",
        "priority_description": "Maintain or improve walking, mobility, balance, and the ability to avoid falls.",
        "priority_domain": "function_daily_life",
        "alignment_eligible": True,
        "source_rows": [1]
    },
    {
        "priority_label": "Preserve upper-limb and hand function",
        "priority_description": "Maintain the ability to use the arms and hands for important daily tasks.",
        "priority_domain": "function_daily_life",
        "alignment_eligible": True,
        "source_rows": [2]
    },
    {
        "priority_label": "Reduce fatigue",
        "priority_description": "Reduce fatigue that limits activity and daily functioning.",
        "priority_domain": "symptom_burden",
        "alignment_eligible": True,
        "source_rows": [3]
    },
    {
        "priority_label": "Reduce pain",
        "priority_description": "Reduce pain associated with FSHD and its functional consequences.",
        "priority_domain": "symptom_burden",
        "alignment_eligible": True,
        "source_rows": [4]
    },
    {
        "priority_label": "Preserve respiratory function",
        "priority_description": "Maintain breathing and lung function as FSHD progresses.",
        "priority_domain": "survival_major_complications",
        "alignment_eligible": True,
        "source_rows": [5]
    },
    {
        "priority_label": "Maintain independence",
        "priority_description": "Preserve the ability to live and function independently.",
        "priority_domain": "function_daily_life",
        "alignment_eligible": True,
        "source_rows": [6]
    },
    {
        "priority_label": "Maintain participation in daily and social activities",
        "priority_description": "Preserve the ability to participate in hobbies, household tasks, social activities, travel, and other valued parts of daily life.",
        "priority_domain": "function_daily_life",
        "alignment_eligible": True,
        "source_rows": [7]
    },
    {
        "priority_label": "Reduce emotional distress about disease progression",
        "priority_description": "Reduce fear, anxiety, stress, and uncertainty caused by unpredictable disease progression.",
        "priority_domain": "symptom_burden",
        "alignment_eligible": True,
        "source_rows": [8]
    },
    {
        "priority_label": "Preserve communication and swallowing function",
        "priority_description": "Preserve the ability to communicate clearly and swallow safely.",
        "priority_domain": "function_daily_life",
        "alignment_eligible": True,
        "source_rows": [9]
    }
]

In [ ]:
# Build the validated FSHD alignment priorities while preserving source evidence.
fshd_validated_rows = []

for item in fshd_priority_plan:
    evidence = combine_fshd_candidate_evidence(
        item["source_rows"]
    )

    fshd_validated_rows.append({
        "disease": "Facioscapulohumeral Muscular Dystrophy",
        "priority_label": item["priority_label"],
        "priority_description": item["priority_description"],
        "priority_domain": item["priority_domain"],
        "alignment_eligible": item["alignment_eligible"],
        "evidence_chunk_ids": evidence["evidence_chunk_ids"],
        "evidence_pdf_pages": evidence["evidence_pdf_pages"],
        "evidence_text": evidence["evidence_text"],
        "source_id": "fshd_vop",
        "support_type": evidence["support_type"],
        "review_status": "validated"
    })

In [ ]:
# Preserve FSHD treatment burden and safety concerns as context-only priorities.
for source_row, label, description, domain in [
    (
        10,
        "Reduce treatment time and effort",
        "Patients value management approaches that require less time and physical effort.",
        "treatment_burden"
    ),
    (
        12,
        "Avoid treatment adverse effects",
        "Patients value treatments that minimize adverse effects.",
        "treatment_safety"
    )
]:
    evidence = combine_fshd_candidate_evidence([source_row])

    fshd_validated_rows.append({
        "disease": "Facioscapulohumeral Muscular Dystrophy",
        "priority_label": label,
        "priority_description": description,
        "priority_domain": domain,
        "alignment_eligible": False,
        "evidence_chunk_ids": evidence["evidence_chunk_ids"],
        "evidence_pdf_pages": evidence["evidence_pdf_pages"],
        "evidence_text": evidence["evidence_text"],
        "source_id": "fshd_vop",
        "support_type": evidence["support_type"],
        "review_status": "validated"
    })

In [ ]:
# Split FSHD treatment cost and treatment access into separate contextual concerns.
cost_access_evidence = combine_fshd_candidate_evidence([11])

for label, description in [
    (
        "Reduce treatment cost burden",
        "Patients identify high treatment costs and insufficient insurance coverage as important barriers."
    ),
    (
        "Improve treatment access and availability",
        "Patients value treatments that are more available and accessible."
    )
]:
    fshd_validated_rows.append({
        "disease": "Facioscapulohumeral Muscular Dystrophy",
        "priority_label": label,
        "priority_description": description,
        "priority_domain": "access_convenience",
        "alignment_eligible": False,
        "evidence_chunk_ids": cost_access_evidence["evidence_chunk_ids"],
        "evidence_pdf_pages": cost_access_evidence["evidence_pdf_pages"],
        "evidence_text": cost_access_evidence["evidence_text"],
        "source_id": "fshd_vop",
        "support_type": cost_access_evidence["support_type"],
        "review_status": "validated"
    })

In [ ]:
# Create and save the final human-validated FSHD patient-priority dataset.
fshd_priorities_final = pd.DataFrame(
    fshd_validated_rows
)

fshd_priorities_final.insert(
    1,
    "priority_id",
    [
        f"FSHD_P{i:02d}"
        for i in range(1, len(fshd_priorities_final) + 1)
    ]
)

fshd_priorities_final.to_json(
    DATA_DIR / "fshd_priorities_final.json",
    orient="records",
    indent=2
)

fshd_priorities_final.to_csv(
    DATA_DIR / "fshd_priorities_final.csv",
    index=False
)

print("Total validated FSHD priorities:", len(fshd_priorities_final))
print(
    "Alignment-eligible:",
    fshd_priorities_final["alignment_eligible"].sum()
)
print(
    "Context only:",
    (~fshd_priorities_final["alignment_eligible"]).sum()
)

display(
    fshd_priorities_final[
        [
            "priority_id",
            "priority_label",
            "priority_domain",
            "alignment_eligible",
            "support_type",
            "evidence_pdf_pages"
        ]
    ]
)

Total validated FSHD priorities: 15
Alignment-eligible: 11
Context only: 4


,priority_id,priority_label,priority_domain,alignment_eligible,support_type,evidence_pdf_pages
0,FSHD_P01,Slow or stop loss of muscle function,disease_control,True,polling_and_discussion,"[40, 41, 136]"
1,FSHD_P02,Regain strength and muscle function,treatment_effectiveness,True,polling_and_discussion,"[40, 41, 136]"
2,FSHD_P03,Preserve mobility and walking ability,function_daily_life,True,polling_and_discussion,"[17, 18, 22, 27, 45]"
3,FSHD_P04,Preserve upper-limb and hand function,function_daily_life,True,polling_and_discussion,"[13, 17, 18]"
4,FSHD_P05,Reduce fatigue,symptom_burden,True,polling_and_discussion,"[5, 17, 18]"
5,FSHD_P06,Reduce pain,symptom_burden,True,polling_and_discussion,"[17, 18, 40]"
6,FSHD_P07,Preserve respiratory function,survival_major_complications,True,polling_and_discussion,"[17, 27, 40]"
7,FSHD_P08,Maintain independence,function_daily_life,True,polling_and_discussion,"[27, 33, 78]"
8,FSHD_P09,Maintain participation in daily and social act...,function_daily_life,True,polling_and_discussion,"[22, 45, 90]"
9,FSHD_P10,Reduce emotional distress about disease progre...,symptom_burden,True,polling_and_discussion,"[27, 28, 45, 58, 68]"


#### Sickle Cell Disease VOTP

In [ ]:
# Define the standardized retrieval questions for Sickle Cell Disease patient priorities.
scd_queries = [
    "What symptoms, complications, or disease effects of sickle cell disease do patients and caregivers say are most important or burdensome?",

    "What do patients say about pain crises, chronic pain, fatigue, anemia, breathing problems, acute complications, or other major physical symptoms?",

    "How does sickle cell disease affect work, school, independence, family life, social participation, daily activities, or quality of life?",

    "What outcomes or improvements do patients say they most want treatments to achieve?",

    "What do patients say about preventing pain crises, reducing hospitalizations, avoiding serious complications, preserving organ function, or improving day-to-day health?",

    "What treatment burdens, side effects, risks, access concerns, or practical treatment preferences do patients and caregivers describe?"
]

In [ ]:
# Retrieve and deduplicate Sickle Cell Disease patient-priority evidence from the report.
scd_retrievals, scd_evidence_candidates = (
    build_priority_evidence_candidates(
        disease="Sickle Cell Disease",
        queries=scd_queries,
        top_k=8
    )
)

print("Retrieved rows:", len(scd_retrievals))
print(
    "Unique evidence candidates:",
    len(scd_evidence_candidates)
)

display(
    scd_evidence_candidates[
        [
            "pdf_page",
            "chunk_number",
            "similarity",
            "chunk_text"
        ]
    ]
)

Retrieved rows: 48
Unique evidence candidates: 37


,pdf_page,chunk_number,similarity,chunk_text
0,3,2,0.778913,disrupts the normal flow of the red blood cell...
1,4,4,0.496667,"disease health effects, impacts and treatment ..."
2,5,1,0.751282,5 The patient input summarized in this report ...
3,5,2,0.487193,"in school, pursue careers, have a family, and ..."
4,6,2,0.762775,Topic 1: The Effects of Sickle Cell Disease Th...
5,6,3,0.651995,age. The panelists described in depth the chal...
6,6,4,0.587029,"reporting chronic daily pain, fatigue,3 and di..."
7,7,1,0.585321,7 Pain crisis and chronic pain Acute pain cris...
8,7,3,0.549033,a particular challenge due to its effect on bl...
9,9,3,0.728673,like other children. As one participant explai...


In [ ]:
# Save the Sickle Cell Disease evidence candidates before LLM extraction.
scd_evidence_candidates.to_csv(
    DATA_DIR / "scd_priority_evidence_candidates.csv",
    index=False
)

print("Saved Sickle Cell Disease evidence candidates.")

Saved Sickle Cell Disease evidence candidates.


In [ ]:
# Combine the retrieved SCD chunks into one evidence package for the LLM.
scd_evidence_text = "\n\n".join(
    f"[{row['chunk_id']}] PDF page {row['pdf_page']}\n{row['chunk_text']}"
    for _, row in scd_evidence_candidates.iterrows()
)

print("Evidence characters:", len(scd_evidence_text))
print(scd_evidence_text[:3000])

Evidence characters: 40015
[scd_vop_p3_c2] PDF page 3
disrupts the normal flow of the red blood cells through the blood vessels of the body, ultimately preventing tissues from receiving adequate oxygen. Sickle cell disease affects approximately 100,000 people in the United States and millions of people worldwide. The disease occurs in 1 of 500 African-American births and in 1 of 36,000 Hispanic births. There are several different genetic variants of sickle cell disease, and disease complications and severity can vary widely across the disease spectrum. Sickle cell disease can affect every body system. Most patients experience both chronic and episodic pain, and acute pain crisis is the most common reason for emergency department use by patients with sickle cell disease. Other complications of sickle cell disease include susceptibility to infection, stroke (more common in children than adults), acute chest syndrome, fatigue, difficulty concentrating, leg ulcers, priapism (unwanted painf

In [ ]:
# Define the extraction instructions for concept-level Sickle Cell Disease patient priorities.
scd_priority_prompt = f"""
You are helping construct an auditable patient-priority dataset for a clinical-trial analysis.

DISEASE:
Sickle Cell Disease

TASK:
Using ONLY the retrieved evidence below, identify the distinct outcomes, symptoms, functional
impacts, treatment effects, and treatment concerns that patients or caregivers indicate are
meaningful or important.

The final project asks whether clinical-trial endpoints measure outcomes patients care about.

IMPORTANT RULES:
1. Use only the supplied evidence. Do not use outside medical knowledge.
2. A priority must be supported by patient/caregiver statements, patient polling, docket comments,
   or the report's summary of patient/caregiver perspectives.
3. Ignore general disease background, meeting logistics, physician-only commentary, and generic
   interest in clinical-trial participation.
4. Extract priorities at the CONCEPT level rather than turning every symptom, complication, or
   survey answer into its own row.
5. Merge closely related manifestations when they represent one broader patient concern.
6. Keep concepts separate when patients clearly experience or value them differently.
7. IMPORTANT: Acute pain crises and chronic daily pain are distinct patient experiences in this
   report. Do not combine them into one generic pain priority.
8. Do not create an item merely because a complication appears in background material. It must
   be described by patients/caregivers as burdensome, important, concerning, or meaningful to improve.
9. Preserve broad functional priorities involving school, work, self-care, family, relationships,
   daily activities, or quality of life when clearly supported.
10. Mechanistic concepts such as reducing sickling or anemia may be extracted only when the
    patient evidence explicitly identifies them as something desired from treatment.
11. Do not rank priorities unless the evidence itself supports a ranking.
12. Return a concise concept-level set rather than every possible symptom or complication.

ALIGNMENT ELIGIBILITY:
Set alignment_eligible = true when the concept is a clinical, symptomatic, functional,
cognitive, survival, disease-control, or quality-of-life outcome that could reasonably be
measured as an endpoint in a therapeutic trial.

Set alignment_eligible = false for concerns such as:
- route or convenience of administration
- ability to self-administer treatment
- treatment access or healthcare-system barriers
- cost or financial burden
- treatment inconvenience
- clinical-trial participation preferences
- trust in researchers or healthcare professionals
- generic desire for a cure
- treatment-risk preferences

Treatment side effects and safety concerns may be retained as contextual priorities with
alignment_eligible = false for the main efficacy-alignment score.

PRIORITY DOMAINS:
Use one of:
- disease_control
- symptom_burden
- function_daily_life
- cognitive_function
- treatment_effectiveness
- survival_major_complications
- emotional_wellbeing
- treatment_burden
- treatment_safety
- access_convenience
- other

EVIDENCE:
For each priority, return the chunk IDs that support it.
Do NOT invent or paraphrase a quotation as evidence. Python will reconnect the chunk IDs to
the original report text afterward.

Return valid JSON in exactly this structure:

{{
  "priorities": [
    {{
      "priority_label": "short descriptive label",
      "priority_description": "one-sentence explanation of what matters to patients",
      "priority_domain": "one allowed domain",
      "alignment_eligible": true,
      "evidence_chunk_ids": ["chunk_id_1", "chunk_id_2"],
      "support_type": "polling", "discussion", or "polling_and_discussion",
      "confidence": "high", "medium", or "low"
    }}
  ]
}}

RETRIEVED EVIDENCE:
{scd_evidence_text}
"""

In [ ]:
# Run the SCD priority extraction and preserve the raw LLM response.
scd_response = client.responses.create(
    model=LLM_MODEL,
    input=scd_priority_prompt
)

scd_priority_raw = scd_response.output_text

print(scd_priority_raw)

with open(
    DATA_DIR / "scd_priority_llm_raw.json",
    "w",
    encoding="utf-8"
) as f:
    f.write(scd_priority_raw)

print("\nSaved raw SCD LLM output.")

{
  "priorities": [
    {
      "priority_label": "Acute pain crises",
      "priority_description": "Patients want to prevent or stop sudden, severe pain crises that disrupt functioning and may require emergency care or hospitalization.",
      "priority_domain": "symptom_burden",
      "alignment_eligible": true,
      "evidence_chunk_ids": ["scd_vop_p5_c1", "scd_vop_p7_c1", "scd_vop_p10_c2", "scd_vop_p15_c3", "scd_vop_p20_c2"],
      "support_type": "discussion",
      "confidence": "high"
    },
    {
      "priority_label": "Chronic daily pain",
      "priority_description": "Patients value relief from persistent pain between crises, including aching or bone and joint pain that can limit daily activities.",
      "priority_domain": "symptom_burden",
      "alignment_eligible": true,
      "evidence_chunk_ids": ["scd_vop_p6_c3", "scd_vop_p7_c1", "scd_vop_p10_c2", "scd_vop_p18_c4"],
      "support_type": "polling_and_discussion",
      "confidence": "high"
    },
    {
      "priori

In [ ]:
# Parse the structured SCD priority candidates returned by the LLM.
scd_priority_json = json.loads(scd_priority_raw)

scd_priority_candidates = pd.DataFrame(
    scd_priority_json["priorities"]
)

display(scd_priority_candidates)

,priority_label,priority_description,priority_domain,alignment_eligible,evidence_chunk_ids,support_type,confidence
0,Acute pain crises,"Patients want to prevent or stop sudden, sever...",symptom_burden,True,"[scd_vop_p5_c1, scd_vop_p7_c1, scd_vop_p10_c2,...",discussion,high
1,Chronic daily pain,Patients value relief from persistent pain bet...,symptom_burden,True,"[scd_vop_p6_c3, scd_vop_p7_c1, scd_vop_p10_c2,...",polling_and_discussion,high
2,Fatigue and sleep difficulties,Patients want relief from ongoing or episodic ...,symptom_burden,True,"[scd_vop_p5_c1, scd_vop_p6_c3, scd_vop_p7_c3, ...",polling_and_discussion,high
3,Cognitive difficulties,Patients value improvement in difficulty conce...,cognitive_function,True,"[scd_vop_p5_c1, scd_vop_p6_c3, scd_vop_p6_c4, ...",polling_and_discussion,high
4,Acute chest syndrome and breathing difficulties,Patients consider acute chest syndrome and oth...,survival_major_complications,True,"[scd_vop_p5_c1, scd_vop_p6_c3, scd_vop_p19_c1,...",polling_and_discussion,high
5,Prevention of long-term tissue and organ damage,Patients want treatments that prevent or rever...,disease_control,True,"[scd_vop_p12_c2, scd_vop_p15_c2, scd_vop_p19_c...",discussion,high
6,Upstream control of sickling and anemia,Patients identified addressing the underlying ...,disease_control,True,"[scd_vop_p5_c2, scd_vop_p15_c2, scd_vop_p20_c2]",discussion,high
7,Daily functioning and participation,Patients value being able to manage self-care ...,function_daily_life,True,"[scd_vop_p5_c1, scd_vop_p5_c2, scd_vop_p9_c3, ...",discussion,high
8,Emotional wellbeing,"Patients describe emotional strain, including ...",emotional_wellbeing,True,"[scd_vop_p5_c1, scd_vop_p7_c1, scd_vop_p9_c3, ...",discussion,high
9,Avoiding early death,Patients and caregivers expressed concern abou...,survival_major_complications,True,"[scd_vop_p5_c1, scd_vop_p9_c3]",discussion,medium


In [ ]:
# Verify that every evidence chunk cited by the LLM came from the SCD retrieval.
valid_scd_chunk_ids = set(
    scd_evidence_candidates["chunk_id"]
)

cited_scd_chunk_ids = {
    chunk_id
    for chunk_list in scd_priority_candidates["evidence_chunk_ids"]
    for chunk_id in chunk_list
}

invalid_scd_chunk_ids = (
    cited_scd_chunk_ids - valid_scd_chunk_ids
)

print("Cited chunks:", len(cited_scd_chunk_ids))
print("Invalid chunk IDs:", invalid_scd_chunk_ids)

Cited chunks: 19
Invalid chunk IDs: set()


In [ ]:
# Reattach the original SCD report pages and evidence text to each extracted priority.
scd_chunk_lookup = (
    scd_evidence_candidates
    .set_index("chunk_id")
    [["pdf_page", "chunk_text"]]
    .to_dict("index")
)

def attach_scd_priority_evidence(chunk_ids):
    records = [
        scd_chunk_lookup[chunk_id]
        for chunk_id in chunk_ids
        if chunk_id in scd_chunk_lookup
    ]

    pages = sorted({
        record["pdf_page"]
        for record in records
    })

    evidence_text = "\n\n".join(
        record["chunk_text"]
        for record in records
    )

    return pd.Series({
        "evidence_pdf_pages": pages,
        "evidence_text": evidence_text
    })


scd_evidence_attached = (
    scd_priority_candidates["evidence_chunk_ids"]
    .apply(attach_scd_priority_evidence)
)

scd_priority_candidates = pd.concat(
    [
        scd_priority_candidates,
        scd_evidence_attached
    ],
    axis=1
)

scd_priority_candidates.insert(
    0,
    "disease",
    "Sickle Cell Disease"
)

scd_priority_candidates["source_id"] = "scd_vop"

display(
    scd_priority_candidates[
        [
            "priority_label",
            "priority_domain",
            "alignment_eligible",
            "support_type",
            "confidence",
            "evidence_pdf_pages"
        ]
    ]
)

,priority_label,priority_domain,alignment_eligible,support_type,confidence,evidence_pdf_pages
0,Acute pain crises,symptom_burden,True,discussion,high,"[5, 7, 10, 15, 20]"
1,Chronic daily pain,symptom_burden,True,polling_and_discussion,high,"[6, 7, 10, 18]"
2,Fatigue and sleep difficulties,symptom_burden,True,polling_and_discussion,high,"[5, 6, 7, 10, 15, 19]"
3,Cognitive difficulties,cognitive_function,True,polling_and_discussion,high,"[5, 6, 10, 15]"
4,Acute chest syndrome and breathing difficulties,survival_major_complications,True,polling_and_discussion,high,"[5, 6, 19]"
5,Prevention of long-term tissue and organ damage,disease_control,True,discussion,high,"[12, 15, 19, 20]"
6,Upstream control of sickling and anemia,disease_control,True,discussion,high,"[5, 15, 20]"
7,Daily functioning and participation,function_daily_life,True,discussion,high,"[5, 9, 12, 19]"
8,Emotional wellbeing,emotional_wellbeing,True,discussion,high,"[5, 7, 9, 12, 15]"
9,Avoiding early death,survival_major_complications,True,discussion,medium,"[5, 9]"


In [ ]:
# Save the unedited SCD candidates so the original LLM extraction remains auditable.
scd_priority_candidates.to_json(
    DATA_DIR / "scd_priority_candidates.json",
    orient="records",
    indent=2
)

print("Saved Sickle Cell Disease priority candidates.")

Saved Sickle Cell Disease priority candidates.


In [ ]:
# Combine evidence from selected SCD candidate rows for human validation.
def combine_scd_candidate_evidence(source_rows):
    subset = scd_priority_candidates.loc[source_rows]

    chunk_ids = []
    for ids in subset["evidence_chunk_ids"]:
        chunk_ids.extend(ids)

    chunk_ids = list(dict.fromkeys(chunk_ids))

    pages = sorted({
        scd_chunk_lookup[chunk_id]["pdf_page"]
        for chunk_id in chunk_ids
        if chunk_id in scd_chunk_lookup
    })

    evidence_text = "\n\n".join(
        scd_chunk_lookup[chunk_id]["chunk_text"]
        for chunk_id in chunk_ids
        if chunk_id in scd_chunk_lookup
    )

    support_types = set(subset["support_type"])

    if (
        "polling_and_discussion" in support_types
        or ("polling" in support_types and "discussion" in support_types)
    ):
        support_type = "polling_and_discussion"
    elif "polling" in support_types:
        support_type = "polling"
    else:
        support_type = "discussion"

    return {
        "evidence_chunk_ids": chunk_ids,
        "evidence_pdf_pages": pages,
        "evidence_text": evidence_text,
        "support_type": support_type
    }

In [ ]:
# Define the concept-level SCD priorities retained after human review.
scd_priority_plan = [
    {
        "priority_label": "Reduce acute pain crises",
        "priority_description": "Reduce the frequency and burden of severe acute sickle cell pain crises.",
        "priority_domain": "symptom_burden",
        "alignment_eligible": True,
        "source_rows": [0]
    },
    {
        "priority_label": "Reduce chronic daily pain",
        "priority_description": "Reduce persistent pain experienced outside acute pain crises.",
        "priority_domain": "symptom_burden",
        "alignment_eligible": True,
        "source_rows": [1]
    },
    {
        "priority_label": "Reduce fatigue",
        "priority_description": "Reduce persistent or severe fatigue that limits daily activity.",
        "priority_domain": "symptom_burden",
        "alignment_eligible": True,
        "source_rows": [2]
    },
    {
        "priority_label": "Improve sleep",
        "priority_description": "Reduce sleep disturbance and insomnia associated with sickle cell disease.",
        "priority_domain": "symptom_burden",
        "alignment_eligible": True,
        "source_rows": [2]
    },
    {
        "priority_label": "Improve cognitive function and concentration",
        "priority_description": "Reduce difficulties with concentration and other cognitive effects.",
        "priority_domain": "cognitive_function",
        "alignment_eligible": True,
        "source_rows": [3]
    },
    {
        "priority_label": "Prevent acute chest syndrome and breathing difficulties",
        "priority_description": "Reduce acute chest syndrome and other serious breathing complications.",
        "priority_domain": "survival_major_complications",
        "alignment_eligible": True,
        "source_rows": [4]
    },
    {
        "priority_label": "Prevent long-term tissue and organ damage",
        "priority_description": "Prevent or reduce progressive damage to organs, tissues, bones, and blood vessels.",
        "priority_domain": "survival_major_complications",
        "alignment_eligible": True,
        "source_rows": [5]
    },
    {
        "priority_label": "Reduce sickling and underlying anemia",
        "priority_description": "Address the underlying sickling process and anemia rather than only downstream complications.",
        "priority_domain": "disease_control",
        "alignment_eligible": True,
        "source_rows": [6]
    },
    {
        "priority_label": "Maintain daily functioning and participation",
        "priority_description": "Preserve the ability to work, attend school, care for family, maintain relationships, and participate in daily life.",
        "priority_domain": "function_daily_life",
        "alignment_eligible": True,
        "source_rows": [7]
    },
    {
        "priority_label": "Improve emotional wellbeing",
        "priority_description": "Reduce anxiety, stress, isolation, hopelessness, and other emotional effects of living with sickle cell disease.",
        "priority_domain": "emotional_wellbeing",
        "alignment_eligible": True,
        "source_rows": [8]
    },
    {
        "priority_label": "Reduce risk of early death",
        "priority_description": "Reduce the life-threatening consequences of sickle cell disease and improve survival.",
        "priority_domain": "survival_major_complications",
        "alignment_eligible": True,
        "source_rows": [9]
    }
]

In [ ]:
# Build the validated SCD alignment priorities while preserving source evidence.
scd_validated_rows = []

for item in scd_priority_plan:
    evidence = combine_scd_candidate_evidence(
        item["source_rows"]
    )

    scd_validated_rows.append({
        "disease": "Sickle Cell Disease",
        "priority_label": item["priority_label"],
        "priority_description": item["priority_description"],
        "priority_domain": item["priority_domain"],
        "alignment_eligible": item["alignment_eligible"],
        "evidence_chunk_ids": evidence["evidence_chunk_ids"],
        "evidence_pdf_pages": evidence["evidence_pdf_pages"],
        "evidence_text": evidence["evidence_text"],
        "source_id": "scd_vop",
        "support_type": evidence["support_type"],
        "review_status": "validated"
    })

In [ ]:
# Preserve treatment side effects and serious risks as a context-only SCD concern.
safety_evidence = combine_scd_candidate_evidence([10])

scd_validated_rows.append({
    "disease": "Sickle Cell Disease",
    "priority_label": "Avoid treatment side effects and serious risks",
    "priority_description": "Patients value treatments with fewer burdensome side effects and serious risks.",
    "priority_domain": "treatment_safety",
    "alignment_eligible": False,
    "evidence_chunk_ids": safety_evidence["evidence_chunk_ids"],
    "evidence_pdf_pages": safety_evidence["evidence_pdf_pages"],
    "evidence_text": safety_evidence["evidence_text"],
    "source_id": "scd_vop",
    "support_type": safety_evidence["support_type"],
    "review_status": "validated"
})

In [ ]:
# Add important SCD treatment-delivery and healthcare-access concerns missed by the LLM.
scd_context_priorities = [
    {
        "priority_label": "Easier self-administered treatment",
        "priority_description": "Patients value treatment options that can be managed at home and reduce reliance on emergency care.",
        "priority_domain": "treatment_burden",
        "chunk_ids": ["scd_vop_p15_c3"]
    },
    {
        "priority_label": "Better access to appropriate sickle cell care",
        "priority_description": "Patients emphasize reliable access to knowledgeable, respectful, and timely healthcare.",
        "priority_domain": "access_convenience",
        "chunk_ids": [
            "scd_vop_p17_c5",
            "scd_vop_p18_c1"
        ]
    }
]

for item in scd_context_priorities:
    records = [
        scd_chunk_lookup[chunk_id]
        for chunk_id in item["chunk_ids"]
    ]

    scd_validated_rows.append({
        "disease": "Sickle Cell Disease",
        "priority_label": item["priority_label"],
        "priority_description": item["priority_description"],
        "priority_domain": item["priority_domain"],
        "alignment_eligible": False,
        "evidence_chunk_ids": item["chunk_ids"],
        "evidence_pdf_pages": sorted({
            record["pdf_page"]
            for record in records
        }),
        "evidence_text": "\n\n".join(
            record["chunk_text"]
            for record in records
        ),
        "source_id": "scd_vop",
        "support_type": "discussion",
        "review_status": "validated"
    })

In [ ]:
# Create and save the final human-validated Sickle Cell Disease priority dataset.
scd_priorities_final = pd.DataFrame(
    scd_validated_rows
)

scd_priorities_final.insert(
    1,
    "priority_id",
    [
        f"SCD_P{i:02d}"
        for i in range(1, len(scd_priorities_final) + 1)
    ]
)

scd_priorities_final.to_json(
    DATA_DIR / "scd_priorities_final.json",
    orient="records",
    indent=2
)

scd_priorities_final.to_csv(
    DATA_DIR / "scd_priorities_final.csv",
    index=False
)

print("Total validated SCD priorities:", len(scd_priorities_final))
print(
    "Alignment-eligible:",
    scd_priorities_final["alignment_eligible"].sum()
)
print(
    "Context only:",
    (~scd_priorities_final["alignment_eligible"]).sum()
)

display(
    scd_priorities_final[
        [
            "priority_id",
            "priority_label",
            "priority_domain",
            "alignment_eligible",
            "support_type",
            "evidence_pdf_pages"
        ]
    ]
)

Total validated SCD priorities: 14
Alignment-eligible: 11
Context only: 3


,priority_id,priority_label,priority_domain,alignment_eligible,support_type,evidence_pdf_pages
0,SCD_P01,Reduce acute pain crises,symptom_burden,True,discussion,"[5, 7, 10, 15, 20]"
1,SCD_P02,Reduce chronic daily pain,symptom_burden,True,polling_and_discussion,"[6, 7, 10, 18]"
2,SCD_P03,Reduce fatigue,symptom_burden,True,polling_and_discussion,"[5, 6, 7, 10, 15, 19]"
3,SCD_P04,Improve sleep,symptom_burden,True,polling_and_discussion,"[5, 6, 7, 10, 15, 19]"
4,SCD_P05,Improve cognitive function and concentration,cognitive_function,True,polling_and_discussion,"[5, 6, 10, 15]"
5,SCD_P06,Prevent acute chest syndrome and breathing dif...,survival_major_complications,True,polling_and_discussion,"[5, 6, 19]"
6,SCD_P07,Prevent long-term tissue and organ damage,survival_major_complications,True,discussion,"[12, 15, 19, 20]"
7,SCD_P08,Reduce sickling and underlying anemia,disease_control,True,discussion,"[5, 15, 20]"
8,SCD_P09,Maintain daily functioning and participation,function_daily_life,True,discussion,"[5, 9, 12, 19]"
9,SCD_P10,Improve emotional wellbeing,emotional_wellbeing,True,discussion,"[5, 7, 9, 12, 15]"


### Dravet Syndrome VOTP

In [ ]:
# Define the standardized retrieval questions for Dravet Syndrome patient priorities.
dravet_queries = [
    "What symptoms, complications, or disease effects of Dravet Syndrome do patients and caregivers say are most important or burdensome?",

    "What do patients and caregivers say about seizures, seizure frequency, seizure severity, prolonged seizures, or seizure-related emergencies?",

    "How does Dravet Syndrome affect cognition, development, communication, behavior, sleep, mobility, independence, or other daily abilities?",

    "How does Dravet Syndrome affect family life, caregiver burden, school, social participation, daily activities, or quality of life?",

    "What outcomes or improvements do patients and caregivers say they most want treatments to achieve?",

    "What treatment burdens, side effects, risks, administration concerns, or practical treatment preferences do patients and caregivers describe?"
]

In [ ]:
# Retrieve and deduplicate Dravet Syndrome patient-priority evidence from the report.
dravet_retrievals, dravet_evidence_candidates = (
    build_priority_evidence_candidates(
        disease="Dravet Syndrome",
        queries=dravet_queries,
        top_k=8
    )
)

print("Retrieved rows:", len(dravet_retrievals))
print(
    "Unique evidence candidates:",
    len(dravet_evidence_candidates)
)

display(
    dravet_evidence_candidates[
        [
            "pdf_page",
            "chunk_number",
            "similarity",
            "chunk_text"
        ]
    ]
)

Retrieved rows: 48
Unique evidence candidates: 30


,pdf_page,chunk_number,similarity,chunk_text
0,6,3,0.723800,international partnering advocacy organization...
1,7,1,0.709494,"(3) Dravet syndrome symptoms, including seizur..."
2,7,2,0.691666,"resources for their children, as well as prema..."
3,7,4,0.592504,"in the future, over a one-time treatment. Care..."
4,10,3,0.660943,Living with Dravet Syndrome: Disease Symptoms ...
5,11,1,0.604711,in Appendix 5. As a reflection of the severity...
6,12,1,0.715547,Seizures were selected by caregivers as the mo...
7,12,2,0.712922,theme: While Dravet syndrome is characterized ...
8,13,1,0.706773,Seizures and/or status epilepticus were select...
9,14,1,0.682063,Jennifer described the severity of seizures th...


In [ ]:
# Save the Dravet Syndrome evidence candidates before LLM extraction.
dravet_evidence_candidates.to_csv(
    DATA_DIR / "dravet_priority_evidence_candidates.csv",
    index=False
)

print("Saved Dravet Syndrome evidence candidates.")

Saved Dravet Syndrome evidence candidates.


In [ ]:
# Combine the retrieved Dravet chunks into one evidence package for the LLM.
dravet_evidence_text = "\n\n".join(
    f"[{row['chunk_id']}] PDF page {row['pdf_page']}\n{row['chunk_text']}"
    for _, row in dravet_evidence_candidates.iterrows()
)

print("Evidence characters:", len(dravet_evidence_text))
print(dravet_evidence_text[:3000])

Evidence characters: 29691
[dravet_vop_p6_c3] PDF page 6
international partnering advocacy organizations. Note that the input received from the February 3, 2022, EL-PFDD meeting reflects a wide range of DS experiences, however not all symptoms and impacts may be captured in this report. Key meeting themes and insights: (1) Dravet syndrome is characterized by intractable seizures. The different types of seizures change with time, and the consequences of some seizures are catastrophic. Seizures are a persistent threat, impair development, and interfere in activities of daily living. Even with multiple medications, most patients experience regular breakthrough seizures. (2) Dravet syndrome is MORE than seizures: many other significant Dravet syndrome- related symptoms exist. These can include developmental delays and regression, speech and language impairment, sleep disturbances, disruptive behaviors, problems with balance and gait, and many others. The incidence and severity of these sym

In [ ]:
# Define the extraction instructions for concept-level Dravet Syndrome patient priorities.
dravet_priority_prompt = f"""
You are helping construct an auditable patient-priority dataset for a clinical-trial analysis.

DISEASE:
Dravet Syndrome

TASK:
Using ONLY the retrieved evidence below, identify the distinct outcomes, symptoms, functional
impacts, treatment effects, and treatment concerns that patients' caregivers indicate are
meaningful or important.

The final project asks whether clinical-trial endpoints measure outcomes patients care about.

IMPORTANT RULES:
1. Use only the supplied evidence. Do not use outside medical knowledge.
2. A priority must be supported by caregiver statements, caregiver polling or survey results,
   or the report's summary of caregiver perspectives.
3. Recognize that this report primarily reflects caregiver proxy perspectives because individuals
   living with Dravet Syndrome generally could not directly participate in the meeting.
4. Ignore general disease background, meeting logistics, physician-only commentary, and generic
   interest in clinical-trial participation.
5. Extract priorities at the CONCEPT level rather than turning every seizure type, symptom,
   survey option, or daily activity into its own row.
6. Do not create separate priorities for tonic-clonic, myoclonic, absence, atonic, and other
   seizure types unless the evidence clearly treats them as independently meaningful treatment goals.
7. Seizure control may remain distinct from status epilepticus or seizure severity if caregiver
   evidence clearly treats those as separate meaningful outcomes.
8. Dravet Syndrome is described as more than seizures. Preserve independently emphasized
   non-seizure outcomes such as development, cognition, communication, behavior, sleep,
   mobility, independence, and quality of life when supported.
9. Do not create an item merely because a symptom appears in the report. It must be described as
   burdensome, important, concerning, or meaningful to improve.
10. Preserve SUDEP or premature mortality as a distinct priority if caregiver evidence directly
    emphasizes it.
11. Do not rank priorities unless the evidence itself supports a ranking.
12. Return a concise concept-level set rather than every possible manifestation.

ALIGNMENT ELIGIBILITY:
Set alignment_eligible = true when the concept is a clinical, symptomatic, developmental,
cognitive, functional, behavioral, survival, or quality-of-life outcome that could reasonably
be measured as an endpoint in a therapeutic trial.

Set alignment_eligible = false for concerns such as:
- route or invasiveness of administration
- one-time versus repeated treatment preference
- treatment permanence or reversibility
- treatment inconvenience
- treatment access or cost
- generic clinical-trial participation preferences
- generic desire for a cure
- treatment-risk preferences

Treatment side effects and safety concerns may be retained as contextual priorities with
alignment_eligible = false for the main efficacy-alignment score.

PRIORITY DOMAINS:
Use one of:
- disease_control
- seizure_control
- symptom_burden
- cognitive_function
- developmental_function
- function_daily_life
- communication
- behavioral_function
- survival_major_complications
- emotional_wellbeing
- treatment_effectiveness
- treatment_burden
- treatment_safety
- access_convenience
- other

EVIDENCE:
For each priority, return the chunk IDs that support it.
Do NOT invent or paraphrase a quotation as evidence. Python will reconnect the chunk IDs to
the original report text afterward.

Return valid JSON in exactly this structure:

{{
  "priorities": [
    {{
      "priority_label": "short descriptive label",
      "priority_description": "one-sentence explanation of what matters to caregivers",
      "priority_domain": "one allowed domain",
      "alignment_eligible": true,
      "evidence_chunk_ids": ["chunk_id_1", "chunk_id_2"],
      "support_type": "polling", "discussion", or "polling_and_discussion",
      "confidence": "high", "medium", or "low"
    }}
  ]
}}

RETRIEVED EVIDENCE:
{dravet_evidence_text}
"""

In [ ]:
# Run the Dravet priority extraction and preserve the raw LLM response.
dravet_response = client.responses.create(
    model=LLM_MODEL,
    input=dravet_priority_prompt
)

dravet_priority_raw = dravet_response.output_text

print(dravet_priority_raw)

with open(
    DATA_DIR / "dravet_priority_llm_raw.json",
    "w",
    encoding="utf-8"
) as f:
    f.write(dravet_priority_raw)

print("\nSaved raw Dravet LLM output.")

{
  "priorities": [
    {
      "priority_label": "Seizure control",
      "priority_description": "Caregivers want fewer or no seizures, including control of severe or prolonged episodes, without sacrificing quality of life or other abilities.",
      "priority_domain": "seizure_control",
      "alignment_eligible": true,
      "evidence_chunk_ids": [
        "dravet_vop_p6_c3",
        "dravet_vop_p12_c1",
        "dravet_vop_p13_c1",
        "dravet_vop_p14_c1",
        "dravet_vop_p14_c4",
        "dravet_vop_p47_c1"
      ],
      "support_type": "polling_and_discussion",
      "confidence": "high"
    },
    {
      "priority_label": "Developmental progress and skills",
      "priority_description": "Caregivers want to support development and prevent delays, loss of acquired skills, or regression.",
      "priority_domain": "developmental_function",
      "alignment_eligible": true,
      "evidence_chunk_ids": [
        "dravet_vop_p6_c3",
        "dravet_vop_p12_c1",
        "dr

In [ ]:
# Parse the structured Dravet priority candidates returned by the LLM.
dravet_priority_json = json.loads(dravet_priority_raw)

dravet_priority_candidates = pd.DataFrame(
    dravet_priority_json["priorities"]
)

display(dravet_priority_candidates)

,priority_label,priority_description,priority_domain,alignment_eligible,evidence_chunk_ids,support_type,confidence
0,Seizure control,"Caregivers want fewer or no seizures, includin...",seizure_control,True,"[dravet_vop_p6_c3, dravet_vop_p12_c1, dravet_v...",polling_and_discussion,high
1,Developmental progress and skills,Caregivers want to support development and pre...,developmental_function,True,"[dravet_vop_p6_c3, dravet_vop_p12_c1, dravet_v...",polling_and_discussion,high
2,Cognitive and intellectual function,Caregivers want to preserve or improve cogniti...,cognitive_function,True,"[dravet_vop_p7_c1, dravet_vop_p7_c2, dravet_vo...",discussion,high
3,Speech and communication,"Caregivers consider speech, language, and the ...",communication,True,"[dravet_vop_p6_c3, dravet_vop_p7_c1, dravet_vo...",polling_and_discussion,high
4,Behavior and mood,Caregivers want improvement in disruptive or a...,behavioral_function,True,"[dravet_vop_p6_c3, dravet_vop_p10_c3, dravet_v...",polling_and_discussion,high
5,Sleep,Caregivers identify sleep disruption and the a...,symptom_burden,True,"[dravet_vop_p6_c3, dravet_vop_p12_c2, dravet_v...",discussion,medium
6,Mobility and physical function,Caregivers want to preserve or improve mobilit...,function_daily_life,True,"[dravet_vop_p6_c3, dravet_vop_p14_c2, dravet_v...",discussion,high
7,Independence and participation in daily life,Caregivers want their loved ones to have great...,function_daily_life,True,"[dravet_vop_p7_c1, dravet_vop_p19_c3, dravet_v...",discussion,high
8,Quality of life,Caregivers want treatments to improve overall ...,function_daily_life,True,"[dravet_vop_p7_c2, dravet_vop_p7_c2, dravet_vo...",discussion,high
9,SUDEP and premature mortality,Caregivers are concerned about SUDEP and prema...,survival_major_complications,True,"[dravet_vop_p7_c1, dravet_vop_p58_c1, dravet_v...",polling_and_discussion,high


In [ ]:
# Verify that every evidence chunk cited by the LLM came from the Dravet retrieval.
valid_dravet_chunk_ids = set(
    dravet_evidence_candidates["chunk_id"]
)

cited_dravet_chunk_ids = {
    chunk_id
    for chunk_list in dravet_priority_candidates["evidence_chunk_ids"]
    for chunk_id in chunk_list
}

invalid_dravet_chunk_ids = (
    cited_dravet_chunk_ids - valid_dravet_chunk_ids
)

print("Cited chunks:", len(cited_dravet_chunk_ids))
print("Invalid chunk IDs:", invalid_dravet_chunk_ids)

Cited chunks: 19
Invalid chunk IDs: set()


In [ ]:
# Reattach the original Dravet report pages and evidence text to each extracted priority.
dravet_chunk_lookup = (
    dravet_evidence_candidates
    .set_index("chunk_id")
    [["pdf_page", "chunk_text"]]
    .to_dict("index")
)

def attach_dravet_priority_evidence(chunk_ids):
    records = [
        dravet_chunk_lookup[chunk_id]
        for chunk_id in chunk_ids
        if chunk_id in dravet_chunk_lookup
    ]

    pages = sorted({
        record["pdf_page"]
        for record in records
    })

    evidence_text = "\n\n".join(
        record["chunk_text"]
        for record in records
    )

    return pd.Series({
        "evidence_pdf_pages": pages,
        "evidence_text": evidence_text
    })


dravet_evidence_attached = (
    dravet_priority_candidates["evidence_chunk_ids"]
    .apply(attach_dravet_priority_evidence)
)

dravet_priority_candidates = pd.concat(
    [
        dravet_priority_candidates,
        dravet_evidence_attached
    ],
    axis=1
)

dravet_priority_candidates.insert(
    0,
    "disease",
    "Dravet Syndrome"
)

dravet_priority_candidates["source_id"] = "dravet_vop"

display(
    dravet_priority_candidates[
        [
            "priority_label",
            "priority_domain",
            "alignment_eligible",
            "support_type",
            "confidence",
            "evidence_pdf_pages"
        ]
    ]
)

,priority_label,priority_domain,alignment_eligible,support_type,confidence,evidence_pdf_pages
0,Seizure control,seizure_control,True,polling_and_discussion,high,"[6, 12, 13, 14, 47]"
1,Developmental progress and skills,developmental_function,True,polling_and_discussion,high,"[6, 12, 13, 14, 47, 58]"
2,Cognitive and intellectual function,cognitive_function,True,discussion,high,"[7, 42, 44, 47, 76, 82]"
3,Speech and communication,communication,True,polling_and_discussion,high,"[6, 7, 12, 47, 58, 76]"
4,Behavior and mood,behavioral_function,True,polling_and_discussion,high,"[6, 10, 12, 47, 58, 82]"
5,Sleep,symptom_burden,True,discussion,medium,"[6, 12, 82]"
6,Mobility and physical function,function_daily_life,True,discussion,high,"[6, 14, 44, 47, 76]"
7,Independence and participation in daily life,function_daily_life,True,discussion,high,"[7, 19, 42, 47]"
8,Quality of life,function_daily_life,True,discussion,high,"[7, 47]"
9,SUDEP and premature mortality,survival_major_complications,True,polling_and_discussion,high,"[7, 58, 76, 82]"


In [ ]:
# Save the unedited Dravet candidates so the original LLM extraction remains auditable.
dravet_priority_candidates.to_json(
    DATA_DIR / "dravet_priority_candidates.json",
    orient="records",
    indent=2
)

print("Saved Dravet Syndrome priority candidates.")

Saved Dravet Syndrome priority candidates.


In [ ]:
# Combine evidence from selected Dravet candidate rows for human validation.
def combine_dravet_candidate_evidence(source_rows):
    subset = dravet_priority_candidates.loc[source_rows]

    chunk_ids = []
    for ids in subset["evidence_chunk_ids"]:
        chunk_ids.extend(ids)

    chunk_ids = list(dict.fromkeys(chunk_ids))

    pages = sorted({
        dravet_chunk_lookup[chunk_id]["pdf_page"]
        for chunk_id in chunk_ids
        if chunk_id in dravet_chunk_lookup
    })

    evidence_text = "\n\n".join(
        dravet_chunk_lookup[chunk_id]["chunk_text"]
        for chunk_id in chunk_ids
        if chunk_id in dravet_chunk_lookup
    )

    support_types = set(subset["support_type"])

    if (
        "polling_and_discussion" in support_types
        or ("polling" in support_types and "discussion" in support_types)
    ):
        support_type = "polling_and_discussion"
    elif "polling" in support_types:
        support_type = "polling"
    else:
        support_type = "discussion"

    return {
        "evidence_chunk_ids": chunk_ids,
        "evidence_pdf_pages": pages,
        "evidence_text": evidence_text,
        "support_type": support_type
    }

In [ ]:
# Define the concept-level Dravet priorities retained after human review.
dravet_priority_plan = [
    {
        "priority_label": "Reduce seizures and status epilepticus",
        "priority_description": "Reduce the frequency and burden of seizures, including prolonged or status epilepticus events.",
        "priority_domain": "seizure_control",
        "alignment_eligible": True,
        "source_rows": [0]
    },
    {
        "priority_label": "Preserve developmental progress and skills",
        "priority_description": "Prevent developmental regression and support continued acquisition or preservation of developmental skills.",
        "priority_domain": "developmental_function",
        "alignment_eligible": True,
        "source_rows": [1]
    },
    {
        "priority_label": "Preserve cognitive and intellectual function",
        "priority_description": "Preserve or improve cognition and intellectual functioning.",
        "priority_domain": "cognitive_function",
        "alignment_eligible": True,
        "source_rows": [2]
    },
    {
        "priority_label": "Improve speech and communication",
        "priority_description": "Preserve or improve speech, language, and the ability to communicate.",
        "priority_domain": "communication",
        "alignment_eligible": True,
        "source_rows": [3]
    },
    {
        "priority_label": "Improve behavior and mood",
        "priority_description": "Reduce disruptive behavior, mood problems, and related behavioral difficulties.",
        "priority_domain": "behavioral_function",
        "alignment_eligible": True,
        "source_rows": [4]
    },
    {
        "priority_label": "Improve sleep",
        "priority_description": "Reduce sleep disruption and improve sleep quality.",
        "priority_domain": "symptom_burden",
        "alignment_eligible": True,
        "source_rows": [5]
    },
    {
        "priority_label": "Preserve mobility and physical function",
        "priority_description": "Preserve or improve walking, balance, endurance, and other physical abilities.",
        "priority_domain": "function_daily_life",
        "alignment_eligible": True,
        "source_rows": [6]
    },
    {
        "priority_label": "Increase independence and daily participation",
        "priority_description": "Increase independence and the ability to participate academically, physically, socially, and in everyday activities.",
        "priority_domain": "function_daily_life",
        "alignment_eligible": True,
        "source_rows": [7]
    },
    {
        "priority_label": "Reduce SUDEP and premature mortality risk",
        "priority_description": "Reduce the risk of sudden unexpected death in epilepsy and other premature mortality.",
        "priority_domain": "survival_major_complications",
        "alignment_eligible": True,
        "source_rows": [9]
    }
]

In [ ]:
# Build the validated Dravet alignment priorities while preserving source evidence.
dravet_validated_rows = []

for item in dravet_priority_plan:
    evidence = combine_dravet_candidate_evidence(
        item["source_rows"]
    )

    dravet_validated_rows.append({
        "disease": "Dravet Syndrome",
        "priority_label": item["priority_label"],
        "priority_description": item["priority_description"],
        "priority_domain": item["priority_domain"],
        "alignment_eligible": item["alignment_eligible"],
        "evidence_chunk_ids": evidence["evidence_chunk_ids"],
        "evidence_pdf_pages": evidence["evidence_pdf_pages"],
        "evidence_text": evidence["evidence_text"],
        "source_id": "dravet_vop",
        "support_type": evidence["support_type"],
        "review_status": "validated"
    })

In [ ]:
# Preserve Dravet treatment safety and polytherapy burden as context-only concerns.
for source_row, label, description, domain in [
    (
        10,
        "Avoid treatment side effects",
        "Caregivers value treatments that reduce side effects that may impair cognition, development, behavior, or quality of life.",
        "treatment_safety"
    ),
    (
        11,
        "Reduce burden of multiple treatments",
        "Caregivers value approaches that reduce the burden of managing multiple medications and therapies.",
        "treatment_burden"
    )
]:
    evidence = combine_dravet_candidate_evidence([source_row])

    dravet_validated_rows.append({
        "disease": "Dravet Syndrome",
        "priority_label": label,
        "priority_description": description,
        "priority_domain": domain,
        "alignment_eligible": False,
        "evidence_chunk_ids": evidence["evidence_chunk_ids"],
        "evidence_pdf_pages": evidence["evidence_pdf_pages"],
        "evidence_text": evidence["evidence_text"],
        "source_id": "dravet_vop",
        "support_type": evidence["support_type"],
        "review_status": "validated"
    })

In [ ]:
# Add Dravet administration preferences that were supported by the report but missed by the LLM.
dravet_context_priorities = [
    {
        "priority_label": "Less invasive treatment administration",
        "priority_description": "Caregivers prefer treatment options that require less invasive administration.",
        "priority_domain": "treatment_burden",
        "chunk_ids": [
            "dravet_vop_p7_c4",
            "dravet_vop_p52_c2",
            "dravet_vop_p52_c3"
        ]
    },
    {
        "priority_label": "Avoid irreversible treatment commitment",
        "priority_description": "Caregivers value retaining flexibility to change future treatment rather than being permanently committed to one intervention.",
        "priority_domain": "treatment_burden",
        "chunk_ids": [
            "dravet_vop_p58_c1",
            "dravet_vop_p62_c1",
            "dravet_vop_p63_c1"
        ]
    }
]

for item in dravet_context_priorities:
    records = [
        dravet_chunk_lookup[chunk_id]
        for chunk_id in item["chunk_ids"]
    ]

    dravet_validated_rows.append({
        "disease": "Dravet Syndrome",
        "priority_label": item["priority_label"],
        "priority_description": item["priority_description"],
        "priority_domain": item["priority_domain"],
        "alignment_eligible": False,
        "evidence_chunk_ids": item["chunk_ids"],
        "evidence_pdf_pages": sorted({
            record["pdf_page"]
            for record in records
        }),
        "evidence_text": "\n\n".join(
            record["chunk_text"]
            for record in records
        ),
        "source_id": "dravet_vop",
        "support_type": "polling_and_discussion",
        "review_status": "validated"
    })

In [ ]:
# Create and save the final human-validated Dravet Syndrome priority dataset.
dravet_priorities_final = pd.DataFrame(
    dravet_validated_rows
)

dravet_priorities_final.insert(
    1,
    "priority_id",
    [
        f"DRAVET_P{i:02d}"
        for i in range(1, len(dravet_priorities_final) + 1)
    ]
)

dravet_priorities_final.to_json(
    DATA_DIR / "dravet_priorities_final.json",
    orient="records",
    indent=2
)

dravet_priorities_final.to_csv(
    DATA_DIR / "dravet_priorities_final.csv",
    index=False
)

print("Total validated Dravet priorities:", len(dravet_priorities_final))
print(
    "Alignment-eligible:",
    dravet_priorities_final["alignment_eligible"].sum()
)
print(
    "Context only:",
    (~dravet_priorities_final["alignment_eligible"]).sum()
)

display(
    dravet_priorities_final[
        [
            "priority_id",
            "priority_label",
            "priority_domain",
            "alignment_eligible",
            "support_type",
            "evidence_pdf_pages"
        ]
    ]
)

Total validated Dravet priorities: 13
Alignment-eligible: 9
Context only: 4


,priority_id,priority_label,priority_domain,alignment_eligible,support_type,evidence_pdf_pages
0,DRAVET_P01,Reduce seizures and status epilepticus,seizure_control,True,polling_and_discussion,"[6, 12, 13, 14, 47]"
1,DRAVET_P02,Preserve developmental progress and skills,developmental_function,True,polling_and_discussion,"[6, 12, 13, 14, 47, 58]"
2,DRAVET_P03,Preserve cognitive and intellectual function,cognitive_function,True,discussion,"[7, 42, 44, 47, 76, 82]"
3,DRAVET_P04,Improve speech and communication,communication,True,polling_and_discussion,"[6, 7, 12, 47, 58, 76]"
4,DRAVET_P05,Improve behavior and mood,behavioral_function,True,polling_and_discussion,"[6, 10, 12, 47, 58, 82]"
5,DRAVET_P06,Improve sleep,symptom_burden,True,discussion,"[6, 12, 82]"
6,DRAVET_P07,Preserve mobility and physical function,function_daily_life,True,discussion,"[6, 14, 44, 47, 76]"
7,DRAVET_P08,Increase independence and daily participation,function_daily_life,True,discussion,"[7, 19, 42, 47]"
8,DRAVET_P09,Reduce SUDEP and premature mortality risk,survival_major_complications,True,polling_and_discussion,"[7, 58, 76, 82]"
9,DRAVET_P10,Avoid treatment side effects,treatment_safety,False,polling_and_discussion,"[7, 11, 42]"


In [ ]:
# Restore the TrialSignal runtime from files already saved in Google Drive.
%pip -q install sentence-transformers openai

from google.colab import drive, userdata
from pathlib import Path
from sentence_transformers import SentenceTransformer
from openai import OpenAI

import pandas as pd
import numpy as np
import json

drive.mount("/content/portfolio-drive")

DATA_DIR = Path("./TrialSignal/data")

print("Data folder:", DATA_DIR)
print("Folder exists:", DATA_DIR.exists())

Mounted at /content/portfolio-drive
Data folder: ./TrialSignal/data
Folder exists: True


### Resume Session Backup Code

In [ ]:
# Reload the saved patient-report chunks and embedding matrix.
patient_report_chunks = pd.read_csv(
    DATA_DIR / "patient_report_chunks.csv"
)

chunk_embeddings = np.load(
    DATA_DIR / "patient_report_chunk_embeddings.npy"
)

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Patient-report chunks:", len(patient_report_chunks))
print("Embedding matrix:", chunk_embeddings.shape)

assert len(patient_report_chunks) == len(chunk_embeddings)

print("Chunks and embeddings match.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Patient-report chunks: 1771
Embedding matrix: (1771, 384)
Chunks and embeddings match.


In [ ]:
# Restore the semantic patient-evidence retrieval function.
def retrieve_patient_evidence(
    disease,
    query,
    top_k=8
):
    disease_mask = (
        patient_report_chunks["disease"] == disease
    ).values

    disease_indices = np.where(disease_mask)[0]

    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True
    )[0]

    scores = (
        chunk_embeddings[disease_indices]
        @ query_embedding
    )

    top_positions = np.argsort(scores)[::-1][:top_k]
    top_indices = disease_indices[top_positions]

    results = patient_report_chunks.iloc[top_indices].copy()
    results["similarity"] = scores[top_positions]

    return results[
        [
            "disease",
            "source_id",
            "pdf_page",
            "chunk_number",
            "chunk_id",
            "similarity",
            "chunk_text"
        ]
    ]

In [ ]:
# Restore the reusable patient-priority evidence retrieval wrapper.
def build_priority_evidence_candidates(
    disease,
    queries,
    top_k=8
):
    retrievals = []

    for query_number, query in enumerate(queries, start=1):
        results = retrieve_patient_evidence(
            disease=disease,
            query=query,
            top_k=top_k
        ).copy()

        results["query_number"] = query_number
        results["query"] = query

        retrievals.append(results)

    retrievals = pd.concat(
        retrievals,
        ignore_index=True
    )

    candidates = (
        retrievals
        .sort_values("similarity", ascending=False)
        .drop_duplicates("chunk_id")
        .sort_values(
            ["pdf_page", "chunk_number"]
        )
        .reset_index(drop=True)
    )

    return retrievals, candidates

print("Retrieval functions restored.")

Retrieval functions restored.


In [ ]:
# Restore the LLM client used for patient-priority extraction.
client = OpenAI(
    api_key=userdata.get("OPENAI_API_KEY")
)

LLM_MODEL = "gpt-6-luna"

print("LLM client restored.")
print("Model:", LLM_MODEL)

LLM client restored.
Model: gpt-6-luna


#### Spinal Muscular Atrophy VOTP

In [ ]:
# Reload the completed SMA evidence candidates instead of rerunning retrieval.
sma_evidence_candidates = pd.read_csv(
    DATA_DIR / "sma_priority_evidence_candidates.csv"
)

print(
    "SMA evidence candidates:",
    len(sma_evidence_candidates)
)

display(
    sma_evidence_candidates[
        [
            "pdf_page",
            "chunk_number",
            "similarity",
            "chunk_text"
        ]
    ].head()
)

SMA evidence candidates: 34


,pdf_page,chunk_number,similarity,chunk_text
0,5,2,0.568384,which outcome measures matter to patients and ...
1,8,2,0.552015,by in-­‐person participants and language used ...
2,8,3,0.567959,caregivers come to terms with their child’s di...
3,9,5,0.599852,treatments that lead to increases in strength ...
4,11,5,0.645453,most significant impact on the lives of patien...


In [ ]:
# Define the standardized retrieval questions for SMA patient priorities.
sma_queries = [
    "What symptoms, functional losses, or disease effects of spinal muscular atrophy do patients and caregivers say are most important or burdensome?",

    "What do patients and caregivers say about mobility, walking, standing, sitting, upper-limb function, strength, fatigue, or other physical abilities?",

    "What do patients and caregivers say about breathing, swallowing, speech, feeding, respiratory complications, or other major health concerns?",

    "How does spinal muscular atrophy affect independence, self-care, school, work, family life, social participation, daily activities, or quality of life?",

    "What outcomes or improvements do patients and caregivers say they most want treatments to achieve, including maintaining current abilities, gaining function, or slowing progression?",

    "What treatment burdens, side effects, risks, administration concerns, access issues, or practical treatment preferences do patients and caregivers describe?"
]

In [ ]:
# Retrieve and deduplicate SMA patient-priority evidence from the report.
sma_retrievals, sma_evidence_candidates = (
    build_priority_evidence_candidates(
        disease="Spinal Muscular Atrophy",
        queries=sma_queries,
        top_k=8
    )
)

print("Retrieved rows:", len(sma_retrievals))
print(
    "Unique evidence candidates:",
    len(sma_evidence_candidates)
)

display(
    sma_evidence_candidates[
        [
            "pdf_page",
            "chunk_number",
            "similarity",
            "chunk_text"
        ]
    ]
)

NameError: name 'patient_report_chunks' is not defined

In [ ]:
# Save the SMA evidence candidates before LLM extraction.
sma_evidence_candidates.to_csv(
    DATA_DIR / "sma_priority_evidence_candidates.csv",
    index=False
)

print("Saved SMA evidence candidates.")

NameError: name 'sma_evidence_candidates' is not defined

In [ ]:
# Combine the retrieved SMA chunks into one evidence package for the LLM.
sma_evidence_text = "\n\n".join(
    f"[{row['chunk_id']}] PDF page {row['pdf_page']}\n{row['chunk_text']}"
    for _, row in sma_evidence_candidates.iterrows()
)

print("Evidence characters:", len(sma_evidence_text))
print(sma_evidence_text[:3000])

Evidence characters: 35129
[sma_vop_p5_c2] PDF page 5
which outcome measures matter to patients and to consider how patients weigh the balance of risks and benefits of a particular treatment. This meeting with the SMA community is the fourth externally led PFDD meeting to be granted. More information on the FDA Patient-Focused Drug Development meetings can be found at http://www.fda.gov/ForIndustry/UserFees/PrescriptionDrugUserFee/ucm347317.htm. Overview of SMA Spinal muscular atrophy is a recessively inherited neurodegenerative disease caused by deletions or mutations in the survival of motor neuron 1 gene (SMN1). On average, one in every 50 people is a genetic carrier—and one in 11,000 infants is born with SMA (approximately 400 live births in the United States each year). Most of the body’s functional SMN protein is encoded by the SMN1 gene, the absence of which leads to progressive muscle denervation, axonal nerve deterioration, loss of spinal motor neurons and atrophy of skeletal 

In [ ]:
# Define the extraction instructions for concept-level SMA patient priorities.
sma_priority_prompt = f"""
You are helping construct an auditable patient-priority dataset for a clinical-trial analysis.

DISEASE:
Spinal Muscular Atrophy

TASK:
Using ONLY the retrieved evidence below, identify the distinct outcomes, symptoms, functional
impacts, treatment effects, and treatment concerns that patients or caregivers indicate are
meaningful or important.

The final project asks whether clinical-trial endpoints measure outcomes patients care about.

IMPORTANT RULES:
1. Use only the supplied evidence. Do not use outside medical knowledge.
2. A priority must be supported by patient/caregiver statements, polling or survey results,
   or the report's summary of patient/caregiver perspectives.
3. Ignore general disease background, meeting logistics, physician-only commentary, and generic
   interest in clinical-trial participation.
4. Extract priorities at the CONCEPT level rather than turning every individual daily activity,
   symptom, or survey answer into its own row.
5. Keep distinct outcomes separate when patients clearly value them differently.
6. IMPORTANT: Do not automatically combine:
   - gaining or regaining function
   - maintaining current function
   - slowing or stopping disease progression
   These may be separate meaningful treatment goals in SMA.
7. Preserve small functional improvements when the evidence clearly shows that even modest gains
   are highly meaningful to patients or caregivers.
8. Preserve independently emphasized outcomes such as respiratory function, swallowing/feeding,
   mobility, upper-limb function, communication, fatigue, pain, independence, and survival when
   supported by the evidence.
9. Do not create an item merely because a symptom appears in the report. It must be described as
   burdensome, important, concerning, or meaningful to improve.
10. Do not rank priorities unless the evidence itself supports a ranking.
11. Return a concise concept-level set rather than every possible manifestation.

ALIGNMENT ELIGIBILITY:
Set alignment_eligible = true when the concept is a clinical, symptomatic, functional,
respiratory, survival, communication, or quality-of-life outcome that could reasonably be
measured as an endpoint in a therapeutic trial.

Set alignment_eligible = false for concerns such as:
- route or invasiveness of administration
- treatment time burden
- treatment access or availability
- treatment cost
- travel burden
- convenience of administration
- generic desire for a cure
- treatment-risk preferences

Treatment side effects and safety concerns may be retained as contextual priorities with
alignment_eligible = false for the main efficacy-alignment score.

PRIORITY DOMAINS:
Use one of:
- disease_control
- symptom_burden
- function_daily_life
- respiratory_function
- communication
- treatment_effectiveness
- survival_major_complications
- treatment_burden
- treatment_safety
- access_convenience
- other

EVIDENCE:
For each priority, return the chunk IDs that support it.
Do NOT invent or paraphrase a quotation as evidence. Python will reconnect the chunk IDs to
the original report text afterward.

Return valid JSON in exactly this structure:

{{
  "priorities": [
    {{
      "priority_label": "short descriptive label",
      "priority_description": "one-sentence explanation of what matters to patients or caregivers",
      "priority_domain": "one allowed domain",
      "alignment_eligible": true,
      "evidence_chunk_ids": ["chunk_id_1", "chunk_id_2"],
      "support_type": "polling", "discussion", or "polling_and_discussion",
      "confidence": "high", "medium", or "low"
    }}
  ]
}}

RETRIEVED EVIDENCE:
{sma_evidence_text}
"""

In [ ]:
# Run the SMA priority extraction and preserve the raw LLM response.
sma_response = client.responses.create(
    model=LLM_MODEL,
    input=sma_priority_prompt
)

sma_priority_raw = sma_response.output_text

print(sma_priority_raw)

with open(
    DATA_DIR / "sma_priority_llm_raw.json",
    "w",
    encoding="utf-8"
) as f:
    f.write(sma_priority_raw)

print("\nSaved raw SMA LLM output.")

{
  "priorities": [
    {
      "priority_label": "Respiratory function",
      "priority_description": "Patients and caregivers value better breathing and the ability to clear secretions, and are concerned about respiratory infections, respiratory failure, and dependence on breathing support.",
      "priority_domain": "respiratory_function",
      "alignment_eligible": true,
      "evidence_chunk_ids": ["sma_vop_p8_c2", "sma_vop_p8_c3", "sma_vop_p11_c5", "sma_vop_p12_c2", "sma_vop_p12_c4", "sma_vop_p59_c1", "sma_vop_p71_c1"],
      "support_type": "polling_and_discussion",
      "confidence": "high"
    },
    {
      "priority_label": "Swallowing and feeding",
      "priority_description": "Patients and caregivers consider the ability to eat and swallow safely important, including reducing choking and aspiration concerns.",
      "priority_domain": "function_daily_life",
      "alignment_eligible": true,
      "evidence_chunk_ids": ["sma_vop_p8_c3", "sma_vop_p11_c5", "sma_vop_p12_c4

In [ ]:
# Parse the structured SMA priority candidates returned by the LLM.
sma_priority_json = json.loads(sma_priority_raw)

sma_priority_candidates = pd.DataFrame(
    sma_priority_json["priorities"]
)

display(sma_priority_candidates)

,priority_label,priority_description,priority_domain,alignment_eligible,evidence_chunk_ids,support_type,confidence
0,Respiratory function,Patients and caregivers value better breathing...,respiratory_function,True,"[sma_vop_p8_c2, sma_vop_p8_c3, sma_vop_p11_c5,...",polling_and_discussion,high
1,Swallowing and feeding,Patients and caregivers consider the ability t...,function_daily_life,True,"[sma_vop_p8_c3, sma_vop_p11_c5, sma_vop_p12_c4...",polling_and_discussion,high
2,Communication ability,Patients and caregivers value preserving or im...,communication,True,"[sma_vop_p8_c3, sma_vop_p11_c5, sma_vop_p27_c1...",polling_and_discussion,high
3,"Gaining strength or function, including small ...",Patients and caregivers value gaining or regai...,function_daily_life,True,"[sma_vop_p9_c5, sma_vop_p26_c5, sma_vop_p27_c1...",polling_and_discussion,high
4,Maintaining current function,Patients and caregivers value treatment that h...,function_daily_life,True,"[sma_vop_p9_c5, sma_vop_p39_c4, sma_vop_p71_c1]",polling_and_discussion,high
5,Slowing or stopping disease progression,Some patients and caregivers value slowing or ...,disease_control,True,"[sma_vop_p26_c5, sma_vop_p34_c1, sma_vop_p39_c...",polling_and_discussion,high
6,Mobility,Patients and caregivers value preserving or im...,function_daily_life,True,"[sma_vop_p18_c1, sma_vop_p19_c4, sma_vop_p34_c...",polling_and_discussion,high
7,Upper-limb function,Patients and caregivers value maintaining or i...,function_daily_life,True,"[sma_vop_p19_c3, sma_vop_p19_c4, sma_vop_p27_c...",polling_and_discussion,high
8,Independence in daily activities,Patients and caregivers value greater independ...,function_daily_life,True,"[sma_vop_p9_c5, sma_vop_p19_c4, sma_vop_p34_c2...",polling_and_discussion,high
9,Fatigue and energy,Patients and caregivers consider fatigue burde...,symptom_burden,True,"[sma_vop_p40_c1, sma_vop_p58_c2, sma_vop_p59_c...",polling_and_discussion,high


In [ ]:
# Verify that every evidence chunk cited by the LLM came from the SMA retrieval.
valid_sma_chunk_ids = set(
    sma_evidence_candidates["chunk_id"]
)

cited_sma_chunk_ids = {
    chunk_id
    for chunk_list in sma_priority_candidates["evidence_chunk_ids"]
    for chunk_id in chunk_list
}

invalid_sma_chunk_ids = (
    cited_sma_chunk_ids - valid_sma_chunk_ids
)

print("Cited chunks:", len(cited_sma_chunk_ids))
print("Invalid chunk IDs:", invalid_sma_chunk_ids)

Cited chunks: 26
Invalid chunk IDs: set()


In [ ]:
# Reattach the original SMA report pages and evidence text to each extracted priority.
sma_chunk_lookup = (
    sma_evidence_candidates
    .set_index("chunk_id")
    [["pdf_page", "chunk_text"]]
    .to_dict("index")
)

def attach_sma_priority_evidence(chunk_ids):
    records = [
        sma_chunk_lookup[chunk_id]
        for chunk_id in chunk_ids
        if chunk_id in sma_chunk_lookup
    ]

    pages = sorted({
        record["pdf_page"]
        for record in records
    })

    evidence_text = "\n\n".join(
        record["chunk_text"]
        for record in records
    )

    return pd.Series({
        "evidence_pdf_pages": pages,
        "evidence_text": evidence_text
    })


sma_evidence_attached = (
    sma_priority_candidates["evidence_chunk_ids"]
    .apply(attach_sma_priority_evidence)
)

sma_priority_candidates = pd.concat(
    [
        sma_priority_candidates,
        sma_evidence_attached
    ],
    axis=1
)

sma_priority_candidates.insert(
    0,
    "disease",
    "Spinal Muscular Atrophy"
)

sma_priority_candidates["source_id"] = "sma_vop"

display(
    sma_priority_candidates[
        [
            "priority_label",
            "priority_domain",
            "alignment_eligible",
            "support_type",
            "confidence",
            "evidence_pdf_pages"
        ]
    ]
)

,priority_label,priority_domain,alignment_eligible,support_type,confidence,evidence_pdf_pages
0,Respiratory function,respiratory_function,True,polling_and_discussion,high,"[8, 11, 12, 59, 71]"
1,Swallowing and feeding,function_daily_life,True,polling_and_discussion,high,"[8, 11, 12, 40, 58, 59, 67]"
2,Communication ability,communication,True,polling_and_discussion,high,"[8, 11, 27, 59]"
3,"Gaining strength or function, including small ...",function_daily_life,True,polling_and_discussion,high,"[9, 26, 27, 34, 82]"
4,Maintaining current function,function_daily_life,True,polling_and_discussion,high,"[9, 39, 71]"
5,Slowing or stopping disease progression,disease_control,True,polling_and_discussion,high,"[26, 34, 39, 71]"
6,Mobility,function_daily_life,True,polling_and_discussion,high,"[18, 19, 34, 40, 59]"
7,Upper-limb function,function_daily_life,True,polling_and_discussion,high,"[19, 27, 34, 67, 82]"
8,Independence in daily activities,function_daily_life,True,polling_and_discussion,high,"[9, 19, 34, 67]"
9,Fatigue and energy,symptom_burden,True,polling_and_discussion,high,"[40, 58, 59, 67, 82]"


In [ ]:
# Save the unedited SMA candidates so the original LLM extraction remains auditable.
sma_priority_candidates.to_json(
    DATA_DIR / "sma_priority_candidates.json",
    orient="records",
    indent=2
)

print("Saved SMA priority candidates.")

Saved SMA priority candidates.


In [ ]:
# Combine evidence from selected SMA candidate rows for human validation.
def combine_sma_candidate_evidence(source_rows):
    subset = sma_priority_candidates.loc[source_rows]

    chunk_ids = []
    for ids in subset["evidence_chunk_ids"]:
        chunk_ids.extend(ids)

    chunk_ids = list(dict.fromkeys(chunk_ids))

    pages = sorted({
        sma_chunk_lookup[chunk_id]["pdf_page"]
        for chunk_id in chunk_ids
        if chunk_id in sma_chunk_lookup
    })

    evidence_text = "\n\n".join(
        sma_chunk_lookup[chunk_id]["chunk_text"]
        for chunk_id in chunk_ids
        if chunk_id in sma_chunk_lookup
    )

    support_types = set(subset["support_type"])

    if (
        "polling_and_discussion" in support_types
        or ("polling" in support_types and "discussion" in support_types)
    ):
        support_type = "polling_and_discussion"
    elif "polling" in support_types:
        support_type = "polling"
    else:
        support_type = "discussion"

    return {
        "evidence_chunk_ids": chunk_ids,
        "evidence_pdf_pages": pages,
        "evidence_text": evidence_text,
        "support_type": support_type
    }

In [ ]:
# Define the concept-level SMA priorities retained after human review.
sma_priority_plan = [
    {
        "priority_label": "Preserve respiratory function",
        "priority_description": "Maintain breathing, airway clearance, and respiratory function.",
        "priority_domain": "respiratory_function",
        "alignment_eligible": True,
        "source_rows": [0]
    },
    {
        "priority_label": "Preserve swallowing and feeding ability",
        "priority_description": "Maintain the ability to swallow and eat safely and effectively.",
        "priority_domain": "function_daily_life",
        "alignment_eligible": True,
        "source_rows": [1]
    },
    {
        "priority_label": "Preserve communication ability",
        "priority_description": "Maintain or improve the ability to communicate effectively.",
        "priority_domain": "communication",
        "alignment_eligible": True,
        "source_rows": [2]
    },
    {
        "priority_label": "Gain strength and functional ability",
        "priority_description": "Achieve meaningful gains in strength or function, even when improvements are small.",
        "priority_domain": "function_daily_life",
        "alignment_eligible": True,
        "source_rows": [3]
    },
    {
        "priority_label": "Maintain current function",
        "priority_description": "Preserve existing abilities and prevent further functional loss.",
        "priority_domain": "function_daily_life",
        "alignment_eligible": True,
        "source_rows": [4]
    },
    {
        "priority_label": "Slow or stop disease progression",
        "priority_description": "Slow or halt continued progression of spinal muscular atrophy.",
        "priority_domain": "disease_control",
        "alignment_eligible": True,
        "source_rows": [5]
    },
    {
        "priority_label": "Preserve mobility",
        "priority_description": "Maintain or improve walking, transfers, wheelchair mobility, and other movement abilities.",
        "priority_domain": "function_daily_life",
        "alignment_eligible": True,
        "source_rows": [6]
    },
    {
        "priority_label": "Preserve upper-limb function",
        "priority_description": "Maintain or improve arm, hand, and fine motor abilities needed for daily tasks.",
        "priority_domain": "function_daily_life",
        "alignment_eligible": True,
        "source_rows": [7]
    },
    {
        "priority_label": "Increase independence in daily activities",
        "priority_description": "Increase the ability to perform self-care and other daily activities independently.",
        "priority_domain": "function_daily_life",
        "alignment_eligible": True,
        "source_rows": [8]
    },
    {
        "priority_label": "Reduce fatigue and improve energy",
        "priority_description": "Reduce fatigue and improve energy available for daily activity.",
        "priority_domain": "symptom_burden",
        "alignment_eligible": True,
        "source_rows": [9]
    },
    {
        "priority_label": "Reduce pain",
        "priority_description": "Reduce pain associated with weakness, positioning, contractures, or scoliosis.",
        "priority_domain": "symptom_burden",
        "alignment_eligible": True,
        "source_rows": [10]
    },
    {
        "priority_label": "Reduce contractures and scoliosis burden",
        "priority_description": "Prevent or reduce orthopedic complications such as joint contractures and severe scoliosis.",
        "priority_domain": "function_daily_life",
        "alignment_eligible": True,
        "source_rows": [11]
    }
]

In [ ]:
# Build the validated SMA alignment priorities while preserving source evidence.
sma_validated_rows = []

for item in sma_priority_plan:
    evidence = combine_sma_candidate_evidence(
        item["source_rows"]
    )

    sma_validated_rows.append({
        "disease": "Spinal Muscular Atrophy",
        "priority_label": item["priority_label"],
        "priority_description": item["priority_description"],
        "priority_domain": item["priority_domain"],
        "alignment_eligible": item["alignment_eligible"],
        "evidence_chunk_ids": evidence["evidence_chunk_ids"],
        "evidence_pdf_pages": evidence["evidence_pdf_pages"],
        "evidence_text": evidence["evidence_text"],
        "source_id": "sma_vop",
        "support_type": evidence["support_type"],
        "review_status": "validated"
    })

In [ ]:
# Preserve serious SMA treatment side effects as a context-only concern.
safety_evidence = combine_sma_candidate_evidence([12])

sma_validated_rows.append({
    "disease": "Spinal Muscular Atrophy",
    "priority_label": "Avoid serious treatment side effects",
    "priority_description": "Patients and caregivers value treatments that avoid serious or life-altering adverse effects.",
    "priority_domain": "treatment_safety",
    "alignment_eligible": False,
    "evidence_chunk_ids": safety_evidence["evidence_chunk_ids"],
    "evidence_pdf_pages": safety_evidence["evidence_pdf_pages"],
    "evidence_text": safety_evidence["evidence_text"],
    "source_id": "sma_vop",
    "support_type": safety_evidence["support_type"],
    "review_status": "validated"
})

In [ ]:
# Split SMA treatment time and administration burden into separate context priorities.
time_admin_evidence = combine_sma_candidate_evidence([13])

for label, description in [
    (
        "Reduce treatment time burden",
        "Patients and caregivers value treatments that require less time away from daily activities, school, work, and family life."
    ),
    (
        "Less burdensome treatment administration",
        "Patients and caregivers value treatments that are easier and less invasive to administer."
    )
]:
    sma_validated_rows.append({
        "disease": "Spinal Muscular Atrophy",
        "priority_label": label,
        "priority_description": description,
        "priority_domain": "treatment_burden",
        "alignment_eligible": False,
        "evidence_chunk_ids": time_admin_evidence["evidence_chunk_ids"],
        "evidence_pdf_pages": time_admin_evidence["evidence_pdf_pages"],
        "evidence_text": time_admin_evidence["evidence_text"],
        "source_id": "sma_vop",
        "support_type": time_admin_evidence["support_type"],
        "review_status": "validated"
    })

In [ ]:
# Split SMA treatment access and cost into separate context priorities.
access_cost_evidence = combine_sma_candidate_evidence([14])

for label, description in [
    (
        "Improve treatment access",
        "Patients and caregivers value reliable access to appropriate SMA treatments and care."
    ),
    (
        "Reduce treatment cost burden",
        "Patients and caregivers identify treatment cost as an important consideration and barrier."
    )
]:
    sma_validated_rows.append({
        "disease": "Spinal Muscular Atrophy",
        "priority_label": label,
        "priority_description": description,
        "priority_domain": "access_convenience",
        "alignment_eligible": False,
        "evidence_chunk_ids": access_cost_evidence["evidence_chunk_ids"],
        "evidence_pdf_pages": access_cost_evidence["evidence_pdf_pages"],
        "evidence_text": access_cost_evidence["evidence_text"],
        "source_id": "sma_vop",
        "support_type": access_cost_evidence["support_type"],
        "review_status": "validated"
    })

In [ ]:
# Create and save the final human-validated SMA priority dataset.
sma_priorities_final = pd.DataFrame(
    sma_validated_rows
)

sma_priorities_final.insert(
    1,
    "priority_id",
    [
        f"SMA_P{i:02d}"
        for i in range(1, len(sma_priorities_final) + 1)
    ]
)

sma_priorities_final.to_json(
    DATA_DIR / "sma_priorities_final.json",
    orient="records",
    indent=2
)

sma_priorities_final.to_csv(
    DATA_DIR / "sma_priorities_final.csv",
    index=False
)

print("Total validated SMA priorities:", len(sma_priorities_final))
print(
    "Alignment-eligible:",
    sma_priorities_final["alignment_eligible"].sum()
)
print(
    "Context only:",
    (~sma_priorities_final["alignment_eligible"]).sum()
)

display(
    sma_priorities_final[
        [
            "priority_id",
            "priority_label",
            "priority_domain",
            "alignment_eligible",
            "support_type",
            "evidence_pdf_pages"
        ]
    ]
)

Total validated SMA priorities: 17
Alignment-eligible: 12
Context only: 5


,priority_id,priority_label,priority_domain,alignment_eligible,support_type,evidence_pdf_pages
0,SMA_P01,Preserve respiratory function,respiratory_function,True,polling_and_discussion,"[8, 11, 12, 59, 71]"
1,SMA_P02,Preserve swallowing and feeding ability,function_daily_life,True,polling_and_discussion,"[8, 11, 12, 40, 58, 59, 67]"
2,SMA_P03,Preserve communication ability,communication,True,polling_and_discussion,"[8, 11, 27, 59]"
3,SMA_P04,Gain strength and functional ability,function_daily_life,True,polling_and_discussion,"[9, 26, 27, 34, 82]"
4,SMA_P05,Maintain current function,function_daily_life,True,polling_and_discussion,"[9, 39, 71]"
5,SMA_P06,Slow or stop disease progression,disease_control,True,polling_and_discussion,"[26, 34, 39, 71]"
6,SMA_P07,Preserve mobility,function_daily_life,True,polling_and_discussion,"[18, 19, 34, 40, 59]"
7,SMA_P08,Preserve upper-limb function,function_daily_life,True,polling_and_discussion,"[19, 27, 34, 67, 82]"
8,SMA_P09,Increase independence in daily activities,function_daily_life,True,polling_and_discussion,"[9, 19, 34, 67]"
9,SMA_P10,Reduce fatigue and improve energy,symptom_burden,True,polling_and_discussion,"[40, 58, 59, 67, 82]"


#### Primary FSGS VOTP

In [ ]:
# Define the standardized retrieval questions for Primary FSGS patient priorities.
fsgs_queries = [
    "What symptoms, complications, or disease effects of primary FSGS do patients and caregivers say are most important or burdensome?",

    "What do patients say about swelling or edema, fatigue, pain, weight changes, sleep problems, or other physical symptoms of FSGS?",

    "What do patients say about proteinuria, kidney function, kidney failure, dialysis, transplantation, or progression of kidney disease?",

    "How does FSGS affect work, school, independence, family life, physical activity, social participation, daily activities, or quality of life?",

    "What outcomes or improvements do patients say they most want future treatments to achieve, including reducing symptoms, preserving kidney function, or slowing disease progression?",

    "What treatment burdens, side effects, risks, administration concerns, access issues, or practical treatment preferences do patients and caregivers describe?"
]

In [ ]:
# Retrieve and deduplicate Primary FSGS patient-priority evidence from the report.
fsgs_retrievals, fsgs_evidence_candidates = (
    build_priority_evidence_candidates(
        disease="Primary FSGS",
        queries=fsgs_queries,
        top_k=8
    )
)

print("Retrieved rows:", len(fsgs_retrievals))
print(
    "Unique evidence candidates:",
    len(fsgs_evidence_candidates)
)

display(
    fsgs_evidence_candidates[
        [
            "pdf_page",
            "chunk_number",
            "similarity",
            "chunk_text"
        ]
    ]
)

Retrieved rows: 48
Unique evidence candidates: 32


,pdf_page,chunk_number,similarity,chunk_text
0,8,2,0.563758,patients who are unresponsive or intolerant of...
1,10,2,0.561898,patients and care partners were encouraged to ...
2,11,1,0.565738,11 National Kidney Foundation NephCure Kidney ...
3,11,3,0.529298,decision to enroll in a trial. • Treatments: P...
4,12,3,0.459324,research decisions toward targeting disease me...
5,13,2,0.661436,"are not included in the narrative. Therefore, ..."
6,14,1,0.649919,14 National Kidney Foundation NephCure Kidney ...
7,15,2,0.691927,"physical, emotional, financial, and social imp..."
8,15,3,0.508252,"day. Some patients reported that, on good days..."
9,17,3,0.486835,burden for them and their families as well. “W...


In [ ]:
# Save the Primary FSGS evidence candidates before LLM extraction.
fsgs_evidence_candidates.to_csv(
    DATA_DIR / "fsgs_priority_evidence_candidates.csv",
    index=False
)

print("Saved Primary FSGS evidence candidates.")

Saved Primary FSGS evidence candidates.


In [ ]:
# Combine the retrieved FSGS chunks into one evidence package for the LLM.
fsgs_evidence_text = "\n\n".join(
    f"[{row['chunk_id']}] PDF page {row['pdf_page']}\n{row['chunk_text']}"
    for _, row in fsgs_evidence_candidates.iterrows()
)

print("Evidence characters:", len(fsgs_evidence_text))
print(fsgs_evidence_text[:3000])

Evidence characters: 29386
[fsgs_vop_p8_c2] PDF page 8
patients who are unresponsive or intolerant of conventional treatments. At the time of this report, there are five U.S. Phase 3 interventional clinical trials13 in various stages of planning and completion for investigating agents that may benefit patients with FSGS. MEETING OVERVIEW This EL-PFDD Meeting on FSGS provided the FDA, product developers, clinicians, and academic researchers the opportunity to hear directly from patients and their care partners about their experiences living with FSGS and their views on disease-related topics. Specifically, the goals of this meeting were to afford the FDA with an overall understanding of: • Patients’ perspectives on living with FSGS, especially their daily disease burdens • Factors that influence patients’ willingness to enter clinical trials for FSGS, including patients’ and care partners’ perspectives on enrolling in trials requiring kidney biopsies and clinical trial endpoints that ar

In [ ]:
# Define the extraction instructions for concept-level Primary FSGS patient priorities.
fsgs_priority_prompt = f"""
You are helping construct an auditable patient-priority dataset for a clinical-trial analysis.

DISEASE:
Primary FSGS

TASK:
Using ONLY the retrieved evidence below, identify the distinct outcomes, symptoms, functional
impacts, treatment effects, and treatment concerns that patients or care partners indicate are
meaningful or important.

The final project asks whether clinical-trial endpoints measure outcomes patients care about.

IMPORTANT RULES:
1. Use only the supplied evidence. Do not use outside medical knowledge.
2. A priority must be supported by patient/care-partner statements, polling results, or the
   report's summary of patient/care-partner perspectives.
3. Ignore general disease background, meeting logistics, physician-only commentary, and generic
   interest in clinical-trial participation.
4. Extract priorities at the CONCEPT level rather than turning every symptom or survey option
   into its own row.
5. Keep distinct outcomes separate when patients clearly value them differently.
6. Proteinuria/albuminuria and kidney function may be retained as patient priorities when the
   evidence explicitly shows that patients selected them as meaningful clinical-trial outcomes.
   Do not treat a biomarker as patient-centered merely because it appears in medical background.
7. Preserve meaningful distinctions among:
   - reducing proteinuria/albuminuria
   - preserving kidney function
   - slowing disease progression
   - preventing or delaying kidney failure, dialysis, or transplant
   when the patient evidence supports them as separate goals.
8. Preserve independently emphasized symptom outcomes such as fatigue, edema, pain, cognitive
   difficulties or brain fog, and emotional wellbeing when clearly supported.
9. Preserve broader daily functioning, work/school participation, physical/social activity, and
   quality-of-life impacts when they represent meaningful patient concerns.
10. FSGS recurrence after kidney transplantation may be extracted if strongly supported, but do
    not automatically merge it with primary-disease progression or kidney-function outcomes.
11. Do not create an item merely because it appears in the report. It must be described as
    burdensome, important, concerning, or meaningful to improve.
12. Do not rank priorities unless the evidence itself supports a ranking.
13. Return a concise concept-level set rather than every possible manifestation.

ALIGNMENT ELIGIBILITY:
Set alignment_eligible = true when the concept is a clinical, symptomatic, renal-function,
functional, cognitive, emotional, survival, or quality-of-life outcome that could reasonably
be measured as an endpoint in a therapeutic trial for primary FSGS.

Set alignment_eligible = false for concerns such as:
- route or convenience of treatment administration
- frequency of monitoring or blood draws
- treatment time burden
- treatment cost or access
- number of medications or treatment complexity
- pregnancy compatibility
- clinical-trial participation preferences
- biopsy burden
- placebo concerns
- generic desire for a cure
- treatment-risk preferences

Treatment side effects and safety concerns may be retained as contextual priorities with
alignment_eligible = false for the main efficacy-alignment score.

PRIORITY DOMAINS:
Use one of:
- disease_control
- symptom_burden
- function_daily_life
- cognitive_function
- emotional_wellbeing
- treatment_effectiveness
- survival_major_complications
- treatment_burden
- treatment_safety
- access_convenience
- other

EVIDENCE:
For each priority, return the chunk IDs that support it.
Do NOT invent or paraphrase a quotation as evidence. Python will reconnect the chunk IDs to
the original report text afterward.

Return valid JSON in exactly this structure:

{{
  "priorities": [
    {{
      "priority_label": "short descriptive label",
      "priority_description": "one-sentence explanation of what matters to patients or care partners",
      "priority_domain": "one allowed domain",
      "alignment_eligible": true,
      "evidence_chunk_ids": ["chunk_id_1", "chunk_id_2"],
      "support_type": "polling", "discussion", or "polling_and_discussion",
      "confidence": "high", "medium", or "low"
    }}
  ]
}}

RETRIEVED EVIDENCE:
{fsgs_evidence_text}
"""

In [ ]:
# Run the FSGS priority extraction and preserve the raw LLM response.
fsgs_response = client.responses.create(
    model=LLM_MODEL,
    input=fsgs_priority_prompt
)

fsgs_priority_raw = fsgs_response.output_text

print(fsgs_priority_raw)

with open(
    DATA_DIR / "fsgs_priority_llm_raw.json",
    "w",
    encoding="utf-8"
) as f:
    f.write(fsgs_priority_raw)

print("\nSaved raw FSGS LLM output.")

{
  "priorities": [
    {
      "priority_label": "Reduce proteinuria/albuminuria",
      "priority_description": "Patients selected proteinuria/albuminuria reduction as a meaningful clinical-trial outcome and described lowering proteinuria as a treatment goal.",
      "priority_domain": "disease_control",
      "alignment_eligible": true,
      "evidence_chunk_ids": ["fsgs_vop_p32_c3", "fsgs_vop_p42_c2"],
      "support_type": "polling_and_discussion",
      "confidence": "high"
    },
    {
      "priority_label": "Preserve kidney function",
      "priority_description": "Patients identified kidney function as a relevant trial outcome and expressed a preference for treatment that preserves or halts its decline.",
      "priority_domain": "disease_control",
      "alignment_eligible": true,
      "evidence_chunk_ids": ["fsgs_vop_p32_c3", "fsgs_vop_p42_c1", "fsgs_vop_p46_c4"],
      "support_type": "polling_and_discussion",
      "confidence": "high"
    },
    {
      "priority_label"

In [ ]:
# Parse the structured FSGS priority candidates returned by the LLM.
fsgs_priority_json = json.loads(fsgs_priority_raw)

fsgs_priority_candidates = pd.DataFrame(
    fsgs_priority_json["priorities"]
)

display(fsgs_priority_candidates)

,priority_label,priority_description,priority_domain,alignment_eligible,evidence_chunk_ids,support_type,confidence
0,Reduce proteinuria/albuminuria,Patients selected proteinuria/albuminuria redu...,disease_control,True,"[fsgs_vop_p32_c3, fsgs_vop_p42_c2]",polling_and_discussion,high
1,Preserve kidney function,Patients identified kidney function as a relev...,disease_control,True,"[fsgs_vop_p32_c3, fsgs_vop_p42_c1, fsgs_vop_p4...",polling_and_discussion,high
2,Slow disease progression,Patients and care partners valued treatment th...,disease_control,True,"[fsgs_vop_p42_c1, fsgs_vop_p46_c4]",discussion,high
3,"Prevent or delay kidney failure, dialysis, or ...",Patients valued delaying the need for dialysis...,survival_major_complications,True,"[fsgs_vop_p32_c3, fsgs_vop_p11_c1, fsgs_vop_p4...",polling_and_discussion,high
4,Prevent FSGS recurrence after transplant,Transplant recipients expressed concern about ...,survival_major_complications,True,"[fsgs_vop_p11_c1, fsgs_vop_p40_c1, fsgs_vop_p5...",discussion,high
5,Reduce fatigue,Patients described fatigue as burdensome and a...,symptom_burden,True,"[fsgs_vop_p10_c2, fsgs_vop_p34_c3, fsgs_vop_p1...",polling_and_discussion,high
6,Reduce edema and swelling,Patients described fluid-related swelling as b...,symptom_burden,True,"[fsgs_vop_p10_c2, fsgs_vop_p15_c3, fsgs_vop_p3...",polling_and_discussion,high
7,Reduce pain and aches,Patients identified muscle and joint aches and...,symptom_burden,True,"[fsgs_vop_p15_c3, fsgs_vop_p34_c3, fsgs_vop_p5...",polling_and_discussion,high
8,Improve cognitive difficulties and brain fog,"Patients described brain fog, mental confusion...",cognitive_function,True,"[fsgs_vop_p10_c2, fsgs_vop_p34_c3, fsgs_vop_p5...",polling_and_discussion,high
9,Improve emotional wellbeing,"Patients and care partners described anxiety, ...",emotional_wellbeing,True,"[fsgs_vop_p11_c1, fsgs_vop_p20_c3, fsgs_vop_p3...",polling_and_discussion,high


In [ ]:
# Verify that every evidence chunk cited by the LLM came from the FSGS retrieval.
valid_fsgs_chunk_ids = set(
    fsgs_evidence_candidates["chunk_id"]
)

cited_fsgs_chunk_ids = {
    chunk_id
    for chunk_list in fsgs_priority_candidates["evidence_chunk_ids"]
    for chunk_id in chunk_list
}

invalid_fsgs_chunk_ids = (
    cited_fsgs_chunk_ids - valid_fsgs_chunk_ids
)

print("Cited chunks:", len(cited_fsgs_chunk_ids))
print("Invalid chunk IDs:", invalid_fsgs_chunk_ids)

Cited chunks: 20
Invalid chunk IDs: set()


In [ ]:
# Reattach the original FSGS report pages and evidence text to each extracted priority.
fsgs_chunk_lookup = (
    fsgs_evidence_candidates
    .set_index("chunk_id")
    [["pdf_page", "chunk_text"]]
    .to_dict("index")
)

def attach_fsgs_priority_evidence(chunk_ids):
    records = [
        fsgs_chunk_lookup[chunk_id]
        for chunk_id in chunk_ids
        if chunk_id in fsgs_chunk_lookup
    ]

    pages = sorted({
        record["pdf_page"]
        for record in records
    })

    evidence_text = "\n\n".join(
        record["chunk_text"]
        for record in records
    )

    return pd.Series({
        "evidence_pdf_pages": pages,
        "evidence_text": evidence_text
    })


fsgs_evidence_attached = (
    fsgs_priority_candidates["evidence_chunk_ids"]
    .apply(attach_fsgs_priority_evidence)
)

fsgs_priority_candidates = pd.concat(
    [
        fsgs_priority_candidates,
        fsgs_evidence_attached
    ],
    axis=1
)

fsgs_priority_candidates.insert(
    0,
    "disease",
    "Primary FSGS"
)

fsgs_priority_candidates["source_id"] = "fsgs_vop"

display(
    fsgs_priority_candidates[
        [
            "priority_label",
            "priority_domain",
            "alignment_eligible",
            "support_type",
            "confidence",
            "evidence_pdf_pages"
        ]
    ]
)

,priority_label,priority_domain,alignment_eligible,support_type,confidence,evidence_pdf_pages
0,Reduce proteinuria/albuminuria,disease_control,True,polling_and_discussion,high,"[32, 42]"
1,Preserve kidney function,disease_control,True,polling_and_discussion,high,"[32, 42, 46]"
2,Slow disease progression,disease_control,True,discussion,high,"[42, 46]"
3,"Prevent or delay kidney failure, dialysis, or ...",survival_major_complications,True,polling_and_discussion,high,"[11, 32, 42]"
4,Prevent FSGS recurrence after transplant,survival_major_complications,True,discussion,high,"[11, 40, 53]"
5,Reduce fatigue,symptom_burden,True,polling_and_discussion,high,"[10, 17, 34]"
6,Reduce edema and swelling,symptom_burden,True,polling_and_discussion,high,"[10, 15, 32, 56]"
7,Reduce pain and aches,symptom_burden,True,polling_and_discussion,high,"[15, 34, 56]"
8,Improve cognitive difficulties and brain fog,cognitive_function,True,polling_and_discussion,high,"[10, 34, 53]"
9,Improve emotional wellbeing,emotional_wellbeing,True,polling_and_discussion,high,"[11, 20, 34, 44]"


In [ ]:
# Save the unedited FSGS candidates so the original LLM extraction remains auditable.
fsgs_priority_candidates.to_json(
    DATA_DIR / "fsgs_priority_candidates.json",
    orient="records",
    indent=2
)

print("Saved Primary FSGS priority candidates.")

Saved Primary FSGS priority candidates.


In [ ]:
# Combine evidence from selected FSGS candidate rows for human validation.
def combine_fsgs_candidate_evidence(source_rows):
    subset = fsgs_priority_candidates.loc[source_rows]

    chunk_ids = []
    for ids in subset["evidence_chunk_ids"]:
        chunk_ids.extend(ids)

    chunk_ids = list(dict.fromkeys(chunk_ids))

    pages = sorted({
        fsgs_chunk_lookup[chunk_id]["pdf_page"]
        for chunk_id in chunk_ids
        if chunk_id in fsgs_chunk_lookup
    })

    evidence_text = "\n\n".join(
        fsgs_chunk_lookup[chunk_id]["chunk_text"]
        for chunk_id in chunk_ids
        if chunk_id in fsgs_chunk_lookup
    )

    support_types = set(subset["support_type"])

    if (
        "polling_and_discussion" in support_types
        or ("polling" in support_types and "discussion" in support_types)
    ):
        support_type = "polling_and_discussion"
    elif "polling" in support_types:
        support_type = "polling"
    else:
        support_type = "discussion"

    return {
        "evidence_chunk_ids": chunk_ids,
        "evidence_pdf_pages": pages,
        "evidence_text": evidence_text,
        "support_type": support_type
    }

In [ ]:
# Define the concept-level FSGS priorities retained after human review.
fsgs_priority_plan = [
    {
        "priority_label": "Reduce proteinuria or albuminuria",
        "priority_description": "Reduce abnormal urinary protein loss identified by patients as a meaningful clinical-trial outcome.",
        "priority_domain": "disease_control",
        "alignment_eligible": True,
        "source_rows": [0]
    },
    {
        "priority_label": "Preserve kidney function and slow disease progression",
        "priority_description": "Preserve kidney function and slow or halt continued deterioration from FSGS.",
        "priority_domain": "disease_control",
        "alignment_eligible": True,
        "source_rows": [1, 2]
    },
    {
        "priority_label": "Prevent or delay kidney failure, dialysis, or transplant",
        "priority_description": "Delay or prevent progression to kidney failure requiring dialysis or kidney transplantation.",
        "priority_domain": "survival_major_complications",
        "alignment_eligible": True,
        "source_rows": [3]
    },
    {
        "priority_label": "Reduce fatigue",
        "priority_description": "Reduce exhaustion and loss of energy that interfere with daily life.",
        "priority_domain": "symptom_burden",
        "alignment_eligible": True,
        "source_rows": [5]
    },
    {
        "priority_label": "Reduce edema and swelling",
        "priority_description": "Reduce painful and function-limiting fluid retention and swelling.",
        "priority_domain": "symptom_burden",
        "alignment_eligible": True,
        "source_rows": [6]
    },
    {
        "priority_label": "Reduce pain and aches",
        "priority_description": "Reduce muscle, joint, and other pain associated with FSGS and its treatment.",
        "priority_domain": "symptom_burden",
        "alignment_eligible": True,
        "source_rows": [7]
    },
    {
        "priority_label": "Improve cognitive function and brain fog",
        "priority_description": "Reduce concentration, memory, and mental clarity difficulties described as brain fog.",
        "priority_domain": "cognitive_function",
        "alignment_eligible": True,
        "source_rows": [8]
    },
    {
        "priority_label": "Improve emotional wellbeing",
        "priority_description": "Reduce anxiety, depression, hopelessness, and emotional distress associated with FSGS.",
        "priority_domain": "emotional_wellbeing",
        "alignment_eligible": True,
        "source_rows": [9]
    },
    {
        "priority_label": "Maintain daily functioning and quality of life",
        "priority_description": "Preserve the ability to work, attend school, participate in valued physical and social activities, and maintain overall quality of life.",
        "priority_domain": "function_daily_life",
        "alignment_eligible": True,
        "source_rows": [10, 11, 12]
    }
]

In [ ]:
# Build the validated FSGS alignment priorities while preserving source evidence.
fsgs_validated_rows = []

for item in fsgs_priority_plan:
    evidence = combine_fsgs_candidate_evidence(
        item["source_rows"]
    )

    fsgs_validated_rows.append({
        "disease": "Primary FSGS",
        "priority_label": item["priority_label"],
        "priority_description": item["priority_description"],
        "priority_domain": item["priority_domain"],
        "alignment_eligible": item["alignment_eligible"],
        "evidence_chunk_ids": evidence["evidence_chunk_ids"],
        "evidence_pdf_pages": evidence["evidence_pdf_pages"],
        "evidence_text": evidence["evidence_text"],
        "source_id": "fsgs_vop",
        "support_type": evidence["support_type"],
        "review_status": "validated"
    })

In [ ]:
# Preserve post-transplant FSGS recurrence as an out-of-scope clinical concern.
recurrence_evidence = combine_fsgs_candidate_evidence([4])

fsgs_validated_rows.append({
    "disease": "Primary FSGS",
    "priority_label": "Prevent FSGS recurrence after transplant",
    "priority_description": "Transplant recipients strongly value preventing recurrent FSGS, but this outcome is outside the scoped primary-FSGS trial cohort.",
    "priority_domain": "survival_major_complications",
    "alignment_eligible": False,
    "evidence_chunk_ids": recurrence_evidence["evidence_chunk_ids"],
    "evidence_pdf_pages": recurrence_evidence["evidence_pdf_pages"],
    "evidence_text": recurrence_evidence["evidence_text"],
    "source_id": "fsgs_vop",
    "support_type": recurrence_evidence["support_type"],
    "review_status": "validated"
})

In [ ]:
# Preserve treatment side effects as a context-only FSGS concern.
side_effect_evidence = combine_fsgs_candidate_evidence([13])

fsgs_validated_rows.append({
    "disease": "Primary FSGS",
    "priority_label": "Avoid treatment side effects",
    "priority_description": "Patients value treatments that minimize serious and burdensome adverse effects.",
    "priority_domain": "treatment_safety",
    "alignment_eligible": False,
    "evidence_chunk_ids": side_effect_evidence["evidence_chunk_ids"],
    "evidence_pdf_pages": side_effect_evidence["evidence_pdf_pages"],
    "evidence_text": side_effect_evidence["evidence_text"],
    "source_id": "fsgs_vop",
    "support_type": side_effect_evidence["support_type"],
    "review_status": "validated"
})

In [ ]:
# Add FSGS treatment complexity and administration burden as context-only concerns.
fsgs_context_priorities = [
    {
        "priority_label": "Reduce treatment complexity and number of medications",
        "priority_description": "Patients prefer treatment strategies that require fewer medications and less complicated therapeutic regimens.",
        "priority_domain": "treatment_burden",
        "chunk_ids": [
            "fsgs_vop_p42_c1",
            "fsgs_vop_p42_c2"
        ]
    },
    {
        "priority_label": "Reduce monitoring and administration burden",
        "priority_description": "Patients and care partners value treatments requiring fewer burdensome blood draws, procedures, and difficult administration methods.",
        "priority_domain": "treatment_burden",
        "chunk_ids": [
            "fsgs_vop_p42_c1"
        ]
    }
]

for item in fsgs_context_priorities:
    records = [
        fsgs_chunk_lookup[chunk_id]
        for chunk_id in item["chunk_ids"]
        if chunk_id in fsgs_chunk_lookup
    ]

    fsgs_validated_rows.append({
        "disease": "Primary FSGS",
        "priority_label": item["priority_label"],
        "priority_description": item["priority_description"],
        "priority_domain": item["priority_domain"],
        "alignment_eligible": False,
        "evidence_chunk_ids": item["chunk_ids"],
        "evidence_pdf_pages": sorted({
            record["pdf_page"]
            for record in records
        }),
        "evidence_text": "\n\n".join(
            record["chunk_text"]
            for record in records
        ),
        "source_id": "fsgs_vop",
        "support_type": "discussion",
        "review_status": "validated"
    })

In [ ]:
# Create and save the final human-validated Primary FSGS priority dataset.
fsgs_priorities_final = pd.DataFrame(
    fsgs_validated_rows
)

fsgs_priorities_final.insert(
    1,
    "priority_id",
    [
        f"FSGS_P{i:02d}"
        for i in range(1, len(fsgs_priorities_final) + 1)
    ]
)

fsgs_priorities_final.to_json(
    DATA_DIR / "fsgs_priorities_final.json",
    orient="records",
    indent=2
)

fsgs_priorities_final.to_csv(
    DATA_DIR / "fsgs_priorities_final.csv",
    index=False
)

print("Total validated FSGS priorities:", len(fsgs_priorities_final))
print(
    "Alignment-eligible:",
    fsgs_priorities_final["alignment_eligible"].sum()
)
print(
    "Context only:",
    (~fsgs_priorities_final["alignment_eligible"]).sum()
)

display(
    fsgs_priorities_final[
        [
            "priority_id",
            "priority_label",
            "priority_domain",
            "alignment_eligible",
            "support_type",
            "evidence_pdf_pages"
        ]
    ]
)

Total validated FSGS priorities: 13
Alignment-eligible: 9
Context only: 4


,priority_id,priority_label,priority_domain,alignment_eligible,support_type,evidence_pdf_pages
0,FSGS_P01,Reduce proteinuria or albuminuria,disease_control,True,polling_and_discussion,"[32, 42]"
1,FSGS_P02,Preserve kidney function and slow disease prog...,disease_control,True,polling_and_discussion,"[32, 42, 46]"
2,FSGS_P03,"Prevent or delay kidney failure, dialysis, or ...",survival_major_complications,True,polling_and_discussion,"[11, 32, 42]"
3,FSGS_P04,Reduce fatigue,symptom_burden,True,polling_and_discussion,"[10, 17, 34]"
4,FSGS_P05,Reduce edema and swelling,symptom_burden,True,polling_and_discussion,"[10, 15, 32, 56]"
5,FSGS_P06,Reduce pain and aches,symptom_burden,True,polling_and_discussion,"[15, 34, 56]"
6,FSGS_P07,Improve cognitive function and brain fog,cognitive_function,True,polling_and_discussion,"[10, 34, 53]"
7,FSGS_P08,Improve emotional wellbeing,emotional_wellbeing,True,polling_and_discussion,"[11, 20, 34, 44]"
8,FSGS_P09,Maintain daily functioning and quality of life,function_daily_life,True,polling_and_discussion,"[10, 15, 21, 32, 42, 55, 61, 65]"
9,FSGS_P10,Prevent FSGS recurrence after transplant,survival_major_complications,False,discussion,"[11, 40, 53]"


## Create Combined Dataset

In [ ]:
# Load the seven finalized patient-priority datasets from Google Drive.
priority_files = {
    "HAE": "hae_priorities_final.json",
    "FA": "fa_priorities_final.json",
    "FSHD": "fshd_priorities_final.json",
    "SCD": "scd_priorities_final.json",
    "Dravet": "dravet_priorities_final.json",
    "SMA": "sma_priorities_final.json",
    "FSGS": "fsgs_priorities_final.json"
}

priority_datasets = {}

for short_name, filename in priority_files.items():
    path = DATA_DIR / filename

    print(
        short_name,
        "| exists:",
        path.exists(),
        "|",
        filename
    )

    priority_datasets[short_name] = pd.read_json(path)

HAE | exists: True | hae_priorities_final.json
FA | exists: True | fa_priorities_final.json
FSHD | exists: True | fshd_priorities_final.json
SCD | exists: True | scd_priorities_final.json
Dravet | exists: True | dravet_priorities_final.json
SMA | exists: True | sma_priorities_final.json
FSGS | exists: True | fsgs_priorities_final.json


In [ ]:
# Combine all validated patient priorities into one master dataset.
patient_priorities_master = pd.concat(
    priority_datasets.values(),
    ignore_index=True
)

print(
    "Total priorities:",
    len(patient_priorities_master)
)

print(
    "Alignment-eligible:",
    patient_priorities_master["alignment_eligible"].sum()
)

print(
    "Context only:",
    (~patient_priorities_master["alignment_eligible"]).sum()
)

Total priorities: 102
Alignment-eligible: 73
Context only: 29


In [ ]:
# Verify the final priority counts for each disease.
priority_count_check = (
    patient_priorities_master
    .groupby("disease")
    .agg(
        total_priorities=("priority_id", "count"),
        alignment_eligible=(
            "alignment_eligible",
            "sum"
        )
    )
    .reset_index()
)

priority_count_check["context_only"] = (
    priority_count_check["total_priorities"]
    - priority_count_check["alignment_eligible"]
)

display(priority_count_check)

,disease,total_priorities,alignment_eligible,context_only
0,Dravet Syndrome,13,9,4
1,Facioscapulohumeral Muscular Dystrophy,15,11,4
2,Friedreich's Ataxia,15,10,5
3,Hereditary Angioedema,15,11,4
4,Primary FSGS,13,9,4
5,Sickle Cell Disease,14,11,3
6,Spinal Muscular Atrophy,17,12,5


In [ ]:
# Run integrity checks on the combined patient-priority dataset.
required_columns = [
    "disease",
    "priority_id",
    "priority_label",
    "priority_description",
    "priority_domain",
    "alignment_eligible",
    "evidence_chunk_ids",
    "evidence_pdf_pages",
    "evidence_text",
    "source_id",
    "support_type",
    "review_status"
]

missing_columns = [
    column
    for column in required_columns
    if column not in patient_priorities_master.columns
]

print("Missing required columns:", missing_columns)

print(
    "Duplicate priority IDs:",
    patient_priorities_master["priority_id"]
    .duplicated()
    .sum()
)

print(
    "Duplicate disease + label pairs:",
    patient_priorities_master[
        ["disease", "priority_label"]
    ]
    .duplicated()
    .sum()
)

print(
    "Missing priority descriptions:",
    patient_priorities_master[
        "priority_description"
    ]
    .isna()
    .sum()
)

print(
    "Missing evidence text:",
    patient_priorities_master[
        "evidence_text"
    ]
    .isna()
    .sum()
)

print(
    "Non-validated rows:",
    (
        patient_priorities_master["review_status"]
        != "validated"
    ).sum()
)

print(
    "Number of diseases:",
    patient_priorities_master["disease"].nunique()
)

Missing required columns: []
Duplicate priority IDs: 0
Duplicate disease + label pairs: 0
Missing priority descriptions: 0
Missing evidence text: 0
Non-validated rows: 0
Number of diseases: 7


In [ ]:
# Confirm that every validated priority retains report evidence and page references.
evidence_check = patient_priorities_master.assign(
    has_chunk_evidence=patient_priorities_master[
        "evidence_chunk_ids"
    ].apply(
        lambda x: isinstance(x, list) and len(x) > 0
    ),
    has_page_evidence=patient_priorities_master[
        "evidence_pdf_pages"
    ].apply(
        lambda x: isinstance(x, list) and len(x) > 0
    )
)

print(
    "Rows without chunk evidence:",
    (~evidence_check["has_chunk_evidence"]).sum()
)

print(
    "Rows without page evidence:",
    (~evidence_check["has_page_evidence"]).sum()
)

Rows without chunk evidence: 0
Rows without page evidence: 0


In [ ]:
# Save the completed seven-disease patient-priority master dataset.
patient_priorities_master.to_json(
    DATA_DIR / "trialsignal_patient_priorities_master.json",
    orient="records",
    indent=2
)

patient_priorities_master.to_csv(
    DATA_DIR / "trialsignal_patient_priorities_master.csv",
    index=False
)

print(
    "Saved:",
    DATA_DIR / "trialsignal_patient_priorities_master.json"
)

print(
    "Saved:",
    DATA_DIR / "trialsignal_patient_priorities_master.csv"
)

Saved: ./TrialSignal/data/trialsignal_patient_priorities_master.json
Saved: ./TrialSignal/data/trialsignal_patient_priorities_master.csv


In [ ]:
# Create the patient-priority subset used in the main endpoint-alignment analysis.
alignment_priorities = (
    patient_priorities_master[
        patient_priorities_master[
            "alignment_eligible"
        ]
    ]
    .copy()
    .reset_index(drop=True)
)

print(
    "Priorities entering alignment:",
    len(alignment_priorities)
)

display(
    alignment_priorities[
        [
            "disease",
            "priority_id",
            "priority_label",
            "priority_domain"
        ]
    ]
)

Priorities entering alignment: 73


,disease,priority_id,priority_label,priority_domain
0,Hereditary Angioedema,HAE_P01,Reduce attack frequency,disease_control
1,Hereditary Angioedema,HAE_P02,Reduce attack severity,disease_control
2,Hereditary Angioedema,HAE_P03,Rapid response to acute treatment,treatment_effectiveness
3,Hereditary Angioedema,HAE_P04,Complete response to acute treatment,treatment_effectiveness
4,Hereditary Angioedema,HAE_P05,Prevent or reduce airway-threatening swelling,symptom_burden
...,...,...,...,...
68,Primary FSGS,FSGS_P05,Reduce edema and swelling,symptom_burden
69,Primary FSGS,FSGS_P06,Reduce pain and aches,symptom_burden
70,Primary FSGS,FSGS_P07,Improve cognitive function and brain fog,cognitive_function
71,Primary FSGS,FSGS_P08,Improve emotional wellbeing,emotional_wellbeing


In [ ]:
# Save the 73 patient priorities used for semantic endpoint alignment.
alignment_priorities.to_json(
    DATA_DIR / "trialsignal_alignment_priorities.json",
    orient="records",
    indent=2
)

alignment_priorities.to_csv(
    DATA_DIR / "trialsignal_alignment_priorities.csv",
    index=False
)

print("Saved final alignment-priority dataset.")

Saved final alignment-priority dataset.


In [ ]:
# Start the semantic alignment stage using the two frozen master datasets.
alignment_priorities = pd.read_json(
    DATA_DIR / "trialsignal_alignment_priorities.json"
)

alignment_endpoints = pd.read_csv(
    DATA_DIR / "trialsignal_endpoints_master.csv"
)

print("Alignment priorities:", len(alignment_priorities))
print("Endpoints:", len(alignment_endpoints))
print("Trials:", alignment_endpoints["nct_id"].nunique())
print("Diseases:", alignment_endpoints["disease"].nunique())

Alignment priorities: 73
Endpoints: 8832
Trials: 1013
Diseases: 7


In [ ]:
# Create a stable unique identifier for every registered trial endpoint.
alignment_endpoints["endpoint_id"] = (
    alignment_endpoints["nct_id"].astype(str)
    + "_"
    + alignment_endpoints["endpoint_type"].str[0].str.upper()
    + alignment_endpoints["endpoint_number"].astype(int).astype(str)
)

print(
    "Duplicate endpoint IDs:",
    alignment_endpoints["endpoint_id"].duplicated().sum()
)

print(
    "Unique endpoint IDs:",
    alignment_endpoints["endpoint_id"].nunique()
)

Duplicate endpoint IDs: 0
Unique endpoint IDs: 8832


In [ ]:
# Build the text representations used for patient-priority and endpoint embeddings.
alignment_priorities["priority_semantic_text"] = (
    alignment_priorities["priority_label"].fillna("").str.strip()
    + ". "
    + alignment_priorities["priority_description"].fillna("").str.strip()
)

alignment_endpoints["endpoint_semantic_text"] = (
    alignment_endpoints["endpoint_text"]
    .fillna("")
    .astype(str)
    .str.strip()
)

print("Blank priority texts:", (
    alignment_priorities["priority_semantic_text"].str.strip() == ""
).sum())

print("Blank endpoint texts:", (
    alignment_endpoints["endpoint_semantic_text"].str.strip() == ""
).sum())

Blank priority texts: 0
Blank endpoint texts: 0


In [ ]:
# Confirm that priorities and endpoints cover the same seven disease cohorts.
priority_counts = (
    alignment_priorities
    .groupby("disease")
    .size()
    .rename("priorities")
)

endpoint_counts = (
    alignment_endpoints
    .groupby("disease")
    .size()
    .rename("endpoints")
)

trial_counts = (
    alignment_endpoints
    .groupby("disease")["nct_id"]
    .nunique()
    .rename("trials")
)

alignment_count_check = pd.concat(
    [
        priority_counts,
        trial_counts,
        endpoint_counts
    ],
    axis=1
).reset_index()

display(alignment_count_check)

print(
    "Disease names match:",
    set(alignment_priorities["disease"])
    == set(alignment_endpoints["disease"])
)

,disease,priorities,trials,endpoints
0,Dravet Syndrome,9,49,527
1,Facioscapulohumeral Muscular Dystrophy,11,30,291
2,Friedreich's Ataxia,10,56,440
3,Hereditary Angioedema,11,105,911
4,Primary FSGS,9,35,276
5,Sickle Cell Disease,11,590,4858
6,Spinal Muscular Atrophy,12,148,1529


Disease names match: True


In [ ]:
# Load the alignment embedding model and inspect its maximum sequence length.
ALIGNMENT_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

alignment_model = SentenceTransformer(
    ALIGNMENT_MODEL
)

print("Alignment model:", ALIGNMENT_MODEL)
print(
    "Maximum sequence length:",
    alignment_model.max_seq_length
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Alignment model: sentence-transformers/all-MiniLM-L6-v2
Maximum sequence length: 256


In [ ]:
# Measure token lengths before allowing the embedding model to truncate any text.
def get_token_lengths(texts, tokenizer):
    encoded = tokenizer(
        texts.tolist(),
        add_special_tokens=True,
        truncation=False,
        padding=False
    )

    return np.array([
        len(token_ids)
        for token_ids in encoded["input_ids"]
    ])


priority_token_lengths = get_token_lengths(
    alignment_priorities["priority_semantic_text"],
    alignment_model.tokenizer
)

endpoint_token_lengths = get_token_lengths(
    alignment_endpoints["endpoint_semantic_text"],
    alignment_model.tokenizer
)

alignment_priorities["token_length"] = (
    priority_token_lengths
)

alignment_endpoints["token_length"] = (
    endpoint_token_lengths
)

max_length = alignment_model.max_seq_length

print("PRIORITIES")
print("Maximum tokens:", priority_token_lengths.max())
print(
    "Above model limit:",
    (priority_token_lengths > max_length).sum()
)

print("\nENDPOINTS")
print("Maximum tokens:", endpoint_token_lengths.max())
print(
    "Median tokens:",
    int(np.median(endpoint_token_lengths))
)
print(
    "95th percentile:",
    int(np.percentile(endpoint_token_lengths, 95))
)
print(
    "Above model limit:",
    (endpoint_token_lengths > max_length).sum()
)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (258 > 256). Running this sequence through the model will result in indexing errors


PRIORITIES
Maximum tokens: 35
Above model limit: 0

ENDPOINTS
Maximum tokens: 492
Median tokens: 40
95th percentile: 186
Above model limit: 51


In [ ]:
# Inspect the longest endpoint descriptions for possible embedding truncation.
display(
    alignment_endpoints[
        [
            "disease",
            "nct_id",
            "endpoint_type",
            "endpoint_name",
            "token_length",
            "endpoint_semantic_text"
        ]
    ]
    .sort_values(
        "token_length",
        ascending=False
    )
    .head(10)
)

,disease,nct_id,endpoint_type,endpoint_name,token_length,endpoint_semantic_text
8566,Primary FSGS,NCT03448692,Secondary,Number of Participants With Abnormalities in L...,492,Number of Participants With Abnormalities in L...
3322,Sickle Cell Disease,NCT00745420,Secondary,Percentage of Participants With Acute Graft-vs...,323,Percentage of Participants With Acute Graft-vs...
1113,Friedreich's Ataxia,NCT03214588,Secondary,Change From Baseline in the mFARS-neuro Indivi...,316,Change From Baseline in the mFARS-neuro Indivi...
6526,Dravet Syndrome,NCT02224573,Primary,Percent Change From the Pre-randomization Base...,309,Percent Change From the Pre-randomization Base...
6525,Dravet Syndrome,NCT02224573,Primary,Percent Change From the Pre-randomization Base...,307,Percent Change From the Pre-randomization Base...
4717,Sickle Cell Disease,NCT00061568,Secondary,Number of Participants Who Developed Acute Gra...,306,Number of Participants Who Developed Acute Gra...
3878,Sickle Cell Disease,NCT04893811,Primary,Percentage of Participants With Serum Bacteric...,298,Percentage of Participants With Serum Bacteric...
8571,Primary FSGS,NCT03448692,Secondary,Number of Participants With Abnormalities in E...,296,Number of Participants With Abnormalities in E...
7869,Spinal Muscular Atrophy,NCT04089566,Secondary,Part B Later-onset SMA: Change From Baseline i...,292,Part B Later-onset SMA: Change From Baseline i...
6604,Dravet Syndrome,NCT03635073,Primary,Change From Baseline in Behavioral and Adaptiv...,291,Change From Baseline in Behavioral and Adaptiv...


In [ ]:
# Flag endpoint texts that will be truncated by the embedding model.
alignment_endpoints["embedding_truncated"] = (
    alignment_endpoints["token_length"]
    > alignment_model.max_seq_length
)

print(
    "Endpoints requiring truncation:",
    alignment_endpoints["embedding_truncated"].sum()
)

print(
    "Percent of endpoints:",
    100
    * alignment_endpoints["embedding_truncated"].mean()
)

Endpoints requiring truncation: 51
Percent of endpoints: 0.577445652173913


In [ ]:
# Summarize truncated endpoints by disease and endpoint type.
truncation_summary = (
    alignment_endpoints[
        alignment_endpoints["embedding_truncated"]
    ]
    .groupby(
        ["disease", "endpoint_type"]
    )
    .size()
    .rename("truncated_endpoints")
    .reset_index()
)

display(truncation_summary)

,disease,endpoint_type,truncated_endpoints
0,Dravet Syndrome,Primary,13
1,Facioscapulohumeral Muscular Dystrophy,Primary,5
2,Friedreich's Ataxia,Secondary,2
3,Hereditary Angioedema,Primary,1
4,Hereditary Angioedema,Secondary,2
5,Primary FSGS,Secondary,4
6,Sickle Cell Disease,Primary,5
7,Sickle Cell Disease,Secondary,15
8,Spinal Muscular Atrophy,Primary,2
9,Spinal Muscular Atrophy,Secondary,2


In [ ]:
# Generate normalized semantic embeddings for the 73 patient priorities.
priority_embeddings = alignment_model.encode(
    alignment_priorities["priority_semantic_text"].tolist(),
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True
)

print(
    "Priority embedding matrix:",
    priority_embeddings.shape
)

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Priority embedding matrix: (73, 384)


In [ ]:
# Save the patient-priority embeddings before similarity analysis.
np.save(
    DATA_DIR / "trialsignal_priority_embeddings.npy",
    priority_embeddings
)

print("Saved priority embeddings.")

Saved priority embeddings.


In [ ]:
# Generate normalized semantic embeddings for all registered trial endpoints.
endpoint_embeddings = alignment_model.encode(
    alignment_endpoints["endpoint_semantic_text"].tolist(),
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)

print(
    "Endpoint embedding matrix:",
    endpoint_embeddings.shape
)

Batches:   0%|          | 0/138 [00:00<?, ?it/s]

Endpoint embedding matrix: (8832, 384)


In [ ]:
# Save the endpoint embeddings before similarity analysis.
np.save(
    DATA_DIR / "trialsignal_endpoint_embeddings.npy",
    endpoint_embeddings
)

print("Saved endpoint embeddings.")

Saved endpoint embeddings.


In [ ]:
# Save the row mapping for every patient-priority embedding.
priority_embedding_map = (
    alignment_priorities[
        [
            "disease",
            "priority_id",
            "priority_label",
            "priority_description",
            "priority_domain",
            "priority_semantic_text",
            "token_length"
        ]
    ]
    .copy()
)

priority_embedding_map.insert(
    0,
    "embedding_row",
    range(len(priority_embedding_map))
)

priority_embedding_map.to_csv(
    DATA_DIR / "trialsignal_priority_embedding_map.csv",
    index=False
)

print("Saved priority embedding map.")

Saved priority embedding map.


In [ ]:
# Save the row mapping for every trial-endpoint embedding.
endpoint_embedding_map = (
    alignment_endpoints[
        [
            "disease",
            "nct_id",
            "endpoint_id",
            "endpoint_type",
            "endpoint_number",
            "endpoint_name",
            "endpoint_semantic_text",
            "token_length",
            "embedding_truncated"
        ]
    ]
    .copy()
)

endpoint_embedding_map.insert(
    0,
    "embedding_row",
    range(len(endpoint_embedding_map))
)

endpoint_embedding_map.to_csv(
    DATA_DIR / "trialsignal_endpoint_embedding_map.csv",
    index=False
)

print("Saved endpoint embedding map.")

Saved endpoint embedding map.


In [ ]:
# Save the embedding-model settings used for the alignment analysis.
embedding_metadata = {
    "model": ALIGNMENT_MODEL,
    "embedding_dimension": int(
        priority_embeddings.shape[1]
    ),
    "normalize_embeddings": True,
    "max_sequence_length": int(
        alignment_model.max_seq_length
    ),
    "priority_count": int(
        len(alignment_priorities)
    ),
    "endpoint_count": int(
        len(alignment_endpoints)
    ),
    "truncated_endpoint_count": int(
        alignment_endpoints[
            "embedding_truncated"
        ].sum()
    )
}

with open(
    DATA_DIR / "trialsignal_embedding_metadata.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        embedding_metadata,
        f,
        indent=2
    )

print(embedding_metadata)

{'model': 'sentence-transformers/all-MiniLM-L6-v2', 'embedding_dimension': 384, 'normalize_embeddings': True, 'max_sequence_length': 256, 'priority_count': 73, 'endpoint_count': 8832, 'truncated_endpoint_count': 51}


In [ ]:
# Calculate priority-to-endpoint cosine similarities only within the same disease.
pairwise_rows = []

for disease in alignment_priorities["disease"].unique():

    priority_indices = alignment_priorities.index[
        alignment_priorities["disease"] == disease
    ].to_numpy()

    endpoint_indices = alignment_endpoints.index[
        alignment_endpoints["disease"] == disease
    ].to_numpy()

    disease_priority_embeddings = (
        priority_embeddings[priority_indices]
    )

    disease_endpoint_embeddings = (
        endpoint_embeddings[endpoint_indices]
    )

    similarity_matrix = (
        disease_priority_embeddings
        @ disease_endpoint_embeddings.T
    )

    for priority_position, priority_index in enumerate(
        priority_indices
    ):
        for endpoint_position, endpoint_index in enumerate(
            endpoint_indices
        ):
            pairwise_rows.append({
                "disease": disease,
                "priority_id": alignment_priorities.loc[
                    priority_index,
                    "priority_id"
                ],
                "priority_label": alignment_priorities.loc[
                    priority_index,
                    "priority_label"
                ],
                "priority_domain": alignment_priorities.loc[
                    priority_index,
                    "priority_domain"
                ],
                "nct_id": alignment_endpoints.loc[
                    endpoint_index,
                    "nct_id"
                ],
                "endpoint_id": alignment_endpoints.loc[
                    endpoint_index,
                    "endpoint_id"
                ],
                "endpoint_type": alignment_endpoints.loc[
                    endpoint_index,
                    "endpoint_type"
                ],
                "endpoint_name": alignment_endpoints.loc[
                    endpoint_index,
                    "endpoint_name"
                ],
                "similarity": float(
                    similarity_matrix[
                        priority_position,
                        endpoint_position
                    ]
                )
            })

priority_endpoint_pairs = pd.DataFrame(
    pairwise_rows
)

print(
    "Within-disease priority-endpoint pairs:",
    len(priority_endpoint_pairs)
)

Within-disease priority-endpoint pairs: 96635


In [ ]:
# Check the distribution of within-disease semantic similarity scores.
print(
    priority_endpoint_pairs["similarity"]
    .describe()
)

print(
    "\nMissing similarities:",
    priority_endpoint_pairs["similarity"]
    .isna()
    .sum()
)

count    96635.000000
mean         0.127614
std          0.123412
min         -0.229890
25%          0.046942
50%          0.109536
75%          0.181551
max          0.861366
Name: similarity, dtype: float64

Missing similarities: 0


In [ ]:
# Save the complete within-disease priority-to-endpoint similarity dataset.
priority_endpoint_pairs.to_csv(
    DATA_DIR / "trialsignal_priority_endpoint_pairs.csv",
    index=False
)

print("Saved raw pairwise similarity dataset.")

Saved raw pairwise similarity dataset.


In [ ]:
# Retrieve the five strongest endpoint matches for each patient priority.
top_priority_matches = (
    priority_endpoint_pairs
    .sort_values(
        [
            "disease",
            "priority_id",
            "similarity"
        ],
        ascending=[
            True,
            True,
            False
        ]
    )
    .groupby(
        [
            "disease",
            "priority_id"
        ],
        group_keys=False
    )
    .head(5)
    .reset_index(drop=True)
)

display(
    top_priority_matches[
        [
            "disease",
            "priority_id",
            "priority_label",
            "nct_id",
            "endpoint_type",
            "endpoint_name",
            "similarity"
        ]
    ]
)

,disease,priority_id,priority_label,nct_id,endpoint_type,endpoint_name,similarity
0,Dravet Syndrome,DRAVET_P01,Reduce seizures and status epilepticus,NCT05626634,Secondary,Proportion of Subjects with > 50% Reduction in...,0.615530
1,Dravet Syndrome,DRAVET_P01,Reduce seizures and status epilepticus,NCT05626634,Secondary,Percent Change from Baseline in the Number of ...,0.612655
2,Dravet Syndrome,DRAVET_P01,Reduce seizures and status epilepticus,NCT05626634,Secondary,Percent Reduction in Individual Seizure Type D...,0.604761
3,Dravet Syndrome,DRAVET_P01,Reduce seizures and status epilepticus,NCT05249556,Secondary,Percentage of participants experiencing >50% r...,0.601556
4,Dravet Syndrome,DRAVET_P01,Reduce seizures and status epilepticus,NCT05560282,Primary,Seizure reduction,0.591260
...,...,...,...,...,...,...,...
360,Spinal Muscular Atrophy,SMA_P12,Reduce contractures and scoliosis burden,NCT07448610,Secondary,Cobb angle,0.474344
361,Spinal Muscular Atrophy,SMA_P12,Reduce contractures and scoliosis burden,NCT07448610,Secondary,Joint-range limitation measured by goniometer,0.418988
362,Spinal Muscular Atrophy,SMA_P12,Reduce contractures and scoliosis burden,NCT02594124,Secondary,Number of Participants Who Experienced Contrac...,0.405635
363,Spinal Muscular Atrophy,SMA_P12,Reduce contractures and scoliosis burden,NCT05878418,Primary,Radiological Evaluation (Cobb Angle),0.405215


In [ ]:
# Save the nearest-neighbor matches for manual semantic validation.
top_priority_matches.to_csv(
    DATA_DIR / "trialsignal_top_priority_matches.csv",
    index=False
)

print(
    "Top-match rows:",
    len(top_priority_matches)
)

Top-match rows: 365


In [ ]:
# Keep the strongest endpoint match for each of the 73 patient priorities.
top1_priority_matches = (
    priority_endpoint_pairs
    .sort_values(
        ["disease", "priority_id", "similarity"],
        ascending=[True, True, False]
    )
    .groupby(
        ["disease", "priority_id"],
        as_index=False
    )
    .first()
)

print("Priorities represented:", len(top1_priority_matches))

display(
    top1_priority_matches[
        [
            "disease",
            "priority_id",
            "priority_label",
            "nct_id",
            "endpoint_type",
            "endpoint_name",
            "similarity"
        ]
    ]
)

Priorities represented: 73


,disease,priority_id,priority_label,nct_id,endpoint_type,endpoint_name,similarity
0,Dravet Syndrome,DRAVET_P01,Reduce seizures and status epilepticus,NCT05626634,Secondary,Proportion of Subjects with > 50% Reduction in...,0.615530
1,Dravet Syndrome,DRAVET_P02,Preserve developmental progress and skills,NCT04611438,Primary,development assessment,0.393040
2,Dravet Syndrome,DRAVET_P03,Preserve cognitive and intellectual function,NCT02224573,Secondary,Mean Change From the Pre-randomization Baselin...,0.406215
3,Dravet Syndrome,DRAVET_P04,Improve speech and communication,NCT06112275,Secondary,Change from Baseline in the Vineland-Third Edi...,0.323458
4,Dravet Syndrome,DRAVET_P05,Improve behavior and mood,NCT03635073,Primary,Change From Baseline in Behavior Measures Usin...,0.537068
...,...,...,...,...,...,...,...
68,Spinal Muscular Atrophy,SMA_P08,Preserve upper-limb function,NCT02644668,Primary,Change From Baseline to Week 8 in Revised Uppe...,0.582185
69,Spinal Muscular Atrophy,SMA_P09,Increase independence in daily activities,NCT06396325,Secondary,Independence performing activities of daily li...,0.520373
70,Spinal Muscular Atrophy,SMA_P10,Reduce fatigue and improve energy,NCT01549847,Primary,Changes in fatigue,0.645464
71,Spinal Muscular Atrophy,SMA_P11,Reduce pain,NCT05878418,Primary,Radiological Evaluation (Cobb Angle),0.465227


In [ ]:
# Summarize the strongest semantic match available for each priority by disease.
top1_similarity_summary = (
    top1_priority_matches
    .groupby("disease")["similarity"]
    .agg(
        priority_count="count",
        minimum="min",
        median="median",
        mean="mean",
        maximum="max"
    )
    .reset_index()
)

display(top1_similarity_summary)

,disease,priority_count,minimum,median,mean,maximum
0,Dravet Syndrome,9,0.318860,0.406215,0.440278,0.615530
1,Facioscapulohumeral Muscular Dystrophy,11,0.253508,0.521086,0.462391,0.580718
2,Friedreich's Ataxia,10,0.379394,0.588134,0.564274,0.717852
3,Hereditary Angioedema,11,0.393803,0.610567,0.594834,0.803633
4,Primary FSGS,9,0.240001,0.465085,0.477018,0.782873
5,Sickle Cell Disease,11,0.353072,0.668272,0.635618,0.861366
6,Spinal Muscular Atrophy,12,0.321490,0.517161,0.515039,0.698978


In [ ]:
# Inspect priorities whose best available endpoint match has the lowest similarity.
lowest_top_matches = (
    top1_priority_matches
    .sort_values("similarity")
    .head(20)
)

display(
    lowest_top_matches[
        [
            "disease",
            "priority_id",
            "priority_label",
            "nct_id",
            "endpoint_type",
            "endpoint_name",
            "similarity"
        ]
    ]
)

,disease,priority_id,priority_label,nct_id,endpoint_type,endpoint_name,similarity
47,Primary FSGS,FSGS_P07,Improve cognitive function and brain fog,NCT07220083,Secondary,Change from baseline across disease-specific c...,0.240001
16,Facioscapulohumeral Muscular Dystrophy,FSHD_P08,Maintain independence,NCT01116570,Secondary,Questionnaire of quality of life,0.253508
19,Facioscapulohumeral Muscular Dystrophy,FSHD_P11,Preserve communication and swallowing function,NCT02239224,Primary,Number of Participants With a Pulmonary Functi...,0.254406
44,Primary FSGS,FSGS_P04,Reduce fatigue,NCT07220083,Secondary,Change from baseline across clinical outcome a...,0.313082
6,Dravet Syndrome,DRAVET_P07,Preserve mobility and physical function,NCT02224573,Secondary,Mean Change From the Pre-randomization Baselin...,0.318860
63,Spinal Muscular Atrophy,SMA_P03,Preserve communication ability,NCT00227266,Primary,"Efficacy, Measured Through Motor Function Asse...",0.321490
7,Dravet Syndrome,DRAVET_P08,Increase independence and daily participation,NCT02318602,Secondary,Vineland Adaptive Behavior Scales (VABS),0.322004
3,Dravet Syndrome,DRAVET_P04,Improve speech and communication,NCT06112275,Secondary,Change from Baseline in the Vineland-Third Edi...,0.323458
56,Sickle Cell Disease,SCD_P07,Prevent long-term tissue and organ damage,NCT04207320,Primary,"Safety, as Measured by Incidence of Graft Fail...",0.353072
49,Primary FSGS,FSGS_P09,Maintain daily functioning and quality of life,NCT03598036,Secondary,Quality of Life Assessments,0.363041


In [ ]:
# Examine the distribution of all valid within-disease priority-endpoint similarities.
similarity_quantiles = (
    priority_endpoint_pairs["similarity"]
    .quantile(
        [
            0.00,
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
            1.00
        ]
    )
)

display(
    similarity_quantiles.rename("similarity")
)

,similarity
0.00,-0.229890
0.10,-0.005294
0.25,0.046942
0.50,0.109536
0.75,0.181551
0.90,0.280472
0.95,0.377876
0.99,0.529973
1.00,0.861366


In [ ]:
# Build a stratified manual-validation sample across diseases and similarity levels.
validation_samples = []

for disease, group in priority_endpoint_pairs.groupby("disease"):

    group = group.copy()

    low_cutoff = group["similarity"].quantile(0.25)
    high_cutoff = group["similarity"].quantile(0.75)

    low_group = group[
        group["similarity"] <= low_cutoff
    ]

    middle_group = group[
        (group["similarity"] > low_cutoff)
        & (group["similarity"] < high_cutoff)
    ]

    high_group = group[
        group["similarity"] >= high_cutoff
    ]

    for label, sample_group in [
        ("low", low_group),
        ("middle", middle_group),
        ("high", high_group)
    ]:
        sample = sample_group.sample(
            n=min(3, len(sample_group)),
            random_state=42
        ).copy()

        sample["similarity_band"] = label

        validation_samples.append(sample)

manual_validation_sample = pd.concat(
    validation_samples,
    ignore_index=True
)

print(
    "Manual validation rows:",
    len(manual_validation_sample)
)

Manual validation rows: 63


In [ ]:
# Add full endpoint text to the manual-validation sample for semantic review.
manual_validation_sample = (
    manual_validation_sample
    .merge(
        alignment_endpoints[
            [
                "endpoint_id",
                "endpoint_semantic_text"
            ]
        ],
        on="endpoint_id",
        how="left"
    )
)

manual_validation_sample["human_match"] = pd.NA

display(
    manual_validation_sample[
        [
            "disease",
            "similarity_band",
            "priority_label",
            "endpoint_name",
            "similarity",
            "human_match"
        ]
    ]
)

,disease,similarity_band,priority_label,endpoint_name,similarity,human_match
0,Dravet Syndrome,low,Improve speech and communication,Correlation of Plasma Concentration with Seizu...,0.009552,<NA>
1,Dravet Syndrome,low,Increase independence and daily participation,Percentage change from Baseline in CMSF during...,0.016432,<NA>
2,Dravet Syndrome,low,Reduce SUDEP and premature mortality risk,Change from Baseline in Bayley-4 subdomain GSV...,0.023816,<NA>
3,Dravet Syndrome,middle,Preserve cognitive and intellectual function,"Percentage of Participants With ≤0%, >0% to ≤2...",0.096894,<NA>
4,Dravet Syndrome,middle,Improve sleep,Percentage Change From Baseline in Monthly Gen...,0.189473,<NA>
...,...,...,...,...,...,...
58,Spinal Muscular Atrophy,middle,Increase independence in daily activities,Respiratory infections,0.059701,<NA>
59,Spinal Muscular Atrophy,middle,Gain strength and functional ability,Decrease in the duration of oxygen therapy,0.060893,<NA>
60,Spinal Muscular Atrophy,high,Reduce contractures and scoliosis burden,Change From Baseline in Chest Circumference,0.226886,<NA>
61,Spinal Muscular Atrophy,high,Gain strength and functional ability,Part 2: Percentage of Participants Who Achieve...,0.233766,<NA>


In [ ]:
# Save the semantic-validation sample before human review.
manual_validation_sample.to_csv(
    DATA_DIR / "trialsignal_manual_validation_sample.csv",
    index=False
)

print("Saved manual validation sample.")

Saved manual validation sample.


In [ ]:
# Keep the strongest endpoint match for every patient-priority and trial combination.
best_pair_indices = (
    priority_endpoint_pairs
    .groupby(
        [
            "disease",
            "priority_id",
            "nct_id"
        ]
    )["similarity"]
    .idxmax()
)

trial_priority_matches = (
    priority_endpoint_pairs
    .loc[best_pair_indices]
    .copy()
    .reset_index(drop=True)
)

trial_priority_matches = (
    trial_priority_matches
    .rename(
        columns={
            "similarity": "best_endpoint_similarity"
        }
    )
)

print(
    "Priority-trial pairs:",
    len(trial_priority_matches)
)

Priority-trial pairs: 11067


In [ ]:
# Verify the expected number of priority-trial comparisons within each disease.
priority_trial_count_check = (
    trial_priority_matches
    .groupby("disease")
    .size()
    .rename("priority_trial_pairs")
    .reset_index()
)

display(priority_trial_count_check)

print(
    "Total:",
    len(trial_priority_matches)
)

,disease,priority_trial_pairs
0,Dravet Syndrome,441
1,Facioscapulohumeral Muscular Dystrophy,330
2,Friedreich's Ataxia,560
3,Hereditary Angioedema,1155
4,Primary FSGS,315
5,Sickle Cell Disease,6490
6,Spinal Muscular Atrophy,1776


Total: 11067


In [ ]:
# Count how many registered endpoints each trial contributes to the alignment analysis.
trial_endpoint_counts = (
    alignment_endpoints
    .groupby(
        [
            "disease",
            "nct_id"
        ]
    )
    .size()
    .rename("endpoint_count")
    .reset_index()
)

trial_priority_matches = (
    trial_priority_matches
    .merge(
        trial_endpoint_counts,
        on=[
            "disease",
            "nct_id"
        ],
        how="left"
    )
)

print(
    "Missing endpoint counts:",
    trial_priority_matches["endpoint_count"]
    .isna()
    .sum()
)

display(
    trial_priority_matches[
        [
            "disease",
            "priority_id",
            "priority_label",
            "nct_id",
            "endpoint_count",
            "endpoint_name",
            "best_endpoint_similarity"
        ]
    ].head()
)

Missing endpoint counts: 0


,disease,priority_id,priority_label,nct_id,endpoint_count,endpoint_name,best_endpoint_similarity
0,Dravet Syndrome,DRAVET_P01,Reduce seizures and status epilepticus,NCT01607073,3,Change in Number of Absence Seizures From Week...,0.414963
1,Dravet Syndrome,DRAVET_P01,Reduce seizures and status epilepticus,NCT02091206,3,Mean Percentage Change From Baseline To End Of...,0.234219
2,Dravet Syndrome,DRAVET_P01,Reduce seizures and status epilepticus,NCT02091375,12,Number Of Participants With A ≥50% Reduction F...,0.526289
3,Dravet Syndrome,DRAVET_P01,Reduce seizures and status epilepticus,NCT02174094,15,Percent change from baseline to study completi...,0.500806
4,Dravet Syndrome,DRAVET_P01,Reduce seizures and status epilepticus,NCT02187809,7,Change in Mean Weekly Number of Tonic-clonic a...,0.461167


In [ ]:
# Measure whether trials with more endpoints receive systematically higher best-match scores.
from scipy.stats import spearmanr

endpoint_count_bias_rows = []

for (
    disease,
    priority_id
), group in trial_priority_matches.groupby(
    [
        "disease",
        "priority_id"
    ]
):

    if group["endpoint_count"].nunique() > 1:
        rho, p_value = spearmanr(
            group["endpoint_count"],
            group["best_endpoint_similarity"]
        )
    else:
        rho = np.nan
        p_value = np.nan

    endpoint_count_bias_rows.append({
        "disease": disease,
        "priority_id": priority_id,
        "priority_label": group[
            "priority_label"
        ].iloc[0],
        "spearman_rho": rho,
        "p_value": p_value
    })

endpoint_count_bias = pd.DataFrame(
    endpoint_count_bias_rows
)

display(
    endpoint_count_bias
    .sort_values(
        "spearman_rho",
        ascending=False
    )
)

,disease,priority_id,priority_label,spearman_rho,p_value
28,Friedreich's Ataxia,FA_P09,Maintain independence and participation in dai...,0.704892,1.328577e-09
23,Friedreich's Ataxia,FA_P04,Reduce fatigue,0.666843,2.009863e-08
47,Primary FSGS,FSGS_P07,Improve cognitive function and brain fog,0.663818,1.378708e-05
24,Friedreich's Ataxia,FA_P05,Preserve manual dexterity and fine motor function,0.645161,7.984285e-08
25,Friedreich's Ataxia,FA_P06,Preserve cardiac function and reduce cardiomyo...,0.639800,1.104483e-07
...,...,...,...,...,...
16,Facioscapulohumeral Muscular Dystrophy,FSHD_P08,Maintain independence,0.145188,4.439709e-01
32,Hereditary Angioedema,HAE_P03,Rapid response to acute treatment,0.093428,3.431473e-01
48,Primary FSGS,FSGS_P08,Improve emotional wellbeing,0.073302,6.755979e-01
46,Primary FSGS,FSGS_P06,Reduce pain and aches,0.003672,9.832967e-01


In [ ]:
# Summarize endpoint-count bias across the 73 patient priorities.
print(
    endpoint_count_bias["spearman_rho"]
    .describe()
)

print(
    "\nMedian correlation:",
    endpoint_count_bias["spearman_rho"].median()
)

print(
    "Priorities with positive correlation:",
    (
        endpoint_count_bias["spearman_rho"] > 0
    ).sum(),
    "of",
    len(endpoint_count_bias)
)

print(
    "Priorities with rho > 0.20:",
    (
        endpoint_count_bias["spearman_rho"] > 0.20
    ).sum()
)

count    73.000000
mean      0.418557
std       0.156242
min       0.003463
25%       0.329345
50%       0.421132
75%       0.528512
max       0.704892
Name: spearman_rho, dtype: float64

Median correlation: 0.42113229200235225
Priorities with positive correlation: 73 of 73
Priorities with rho > 0.20: 66


In [ ]:
# Retrieve the five strongest trial-level matches for every patient priority.
top_trial_matches = (
    trial_priority_matches
    .sort_values(
        [
            "disease",
            "priority_id",
            "best_endpoint_similarity"
        ],
        ascending=[
            True,
            True,
            False
        ]
    )
    .groupby(
        [
            "disease",
            "priority_id"
        ],
        group_keys=False
    )
    .head(5)
    .reset_index(drop=True)
)

print(
    "Top trial-match rows:",
    len(top_trial_matches)
)

display(
    top_trial_matches[
        [
            "disease",
            "priority_id",
            "priority_label",
            "nct_id",
            "endpoint_count",
            "endpoint_type",
            "endpoint_name",
            "best_endpoint_similarity"
        ]
    ]
)

Top trial-match rows: 365


,disease,priority_id,priority_label,nct_id,endpoint_count,endpoint_type,endpoint_name,best_endpoint_similarity
0,Dravet Syndrome,DRAVET_P01,Reduce seizures and status epilepticus,NCT05626634,11,Secondary,Proportion of Subjects with > 50% Reduction in...,0.615530
1,Dravet Syndrome,DRAVET_P01,Reduce seizures and status epilepticus,NCT05249556,7,Secondary,Percentage of participants experiencing >50% r...,0.601556
2,Dravet Syndrome,DRAVET_P01,Reduce seizures and status epilepticus,NCT05560282,1,Primary,Seizure reduction,0.591260
3,Dravet Syndrome,DRAVET_P01,Reduce seizures and status epilepticus,NCT02224573,60,Secondary,Number of Treatment Responders With Greater Th...,0.576479
4,Dravet Syndrome,DRAVET_P01,Reduce seizures and status epilepticus,NCT06112275,5,Primary,Proportion of participants free from episodes ...,0.568949
...,...,...,...,...,...,...,...,...
360,Spinal Muscular Atrophy,SMA_P12,Reduce contractures and scoliosis burden,NCT07448610,14,Secondary,Cobb angle,0.474344
361,Spinal Muscular Atrophy,SMA_P12,Reduce contractures and scoliosis burden,NCT02594124,32,Secondary,Number of Participants Who Experienced Contrac...,0.405635
362,Spinal Muscular Atrophy,SMA_P12,Reduce contractures and scoliosis burden,NCT05878418,9,Primary,Radiological Evaluation (Cobb Angle),0.405215
363,Spinal Muscular Atrophy,SMA_P12,Reduce contractures and scoliosis burden,NCT07336602,18,Secondary,Change in Duration of Coordinated Trunk and Ce...,0.348710


In [ ]:
# Save the trial-level semantic matches for validation and scoring development.
trial_priority_matches.to_csv(
    DATA_DIR / "trialsignal_trial_priority_matches.csv",
    index=False
)

top_trial_matches.to_csv(
    DATA_DIR / "trialsignal_top_trial_matches.csv",
    index=False
)

endpoint_count_bias.to_csv(
    DATA_DIR / "trialsignal_endpoint_count_bias.csv",
    index=False
)

print("Saved trial-level alignment tables.")

Saved trial-level alignment tables.


In [ ]:
# Summarize each priority's similarity distribution across all endpoints in its disease.
priority_similarity_profiles = (
    priority_endpoint_pairs
    .groupby(
        [
            "disease",
            "priority_id",
            "priority_label"
        ]
    )["similarity"]
    .agg(
        endpoint_count="count",
        mean_similarity="mean",
        median_similarity="median",
        std_similarity="std",
        maximum_similarity="max"
    )
    .reset_index()
)

display(
    priority_similarity_profiles
    .sort_values("maximum_similarity")
)

,disease,priority_id,priority_label,endpoint_count,mean_similarity,median_similarity,std_similarity,maximum_similarity
47,Primary FSGS,FSGS_P07,Improve cognitive function and brain fog,276,0.054985,0.051425,0.056790,0.240001
16,Facioscapulohumeral Muscular Dystrophy,FSHD_P08,Maintain independence,291,0.003035,-0.006719,0.068436,0.253508
19,Facioscapulohumeral Muscular Dystrophy,FSHD_P11,Preserve communication and swallowing function,291,0.079657,0.082921,0.062052,0.254406
44,Primary FSGS,FSGS_P04,Reduce fatigue,276,0.112424,0.107703,0.059586,0.313082
6,Dravet Syndrome,DRAVET_P07,Preserve mobility and physical function,527,0.097433,0.096470,0.070608,0.318860
...,...,...,...,...,...,...,...,...
31,Hereditary Angioedema,HAE_P02,Reduce attack severity,911,0.261847,0.194831,0.180991,0.735610
60,Sickle Cell Disease,SCD_P11,Reduce risk of early death,4858,0.127584,0.112673,0.122136,0.768324
41,Primary FSGS,FSGS_P01,Reduce proteinuria or albuminuria,276,0.318490,0.280118,0.180921,0.782873
33,Hereditary Angioedema,HAE_P04,Complete response to acute treatment,911,0.355804,0.324007,0.181416,0.803633


In [ ]:
# Convert each endpoint similarity to its percentile within that specific patient priority.
priority_endpoint_pairs["similarity_percentile"] = (
    priority_endpoint_pairs
    .groupby(
        ["disease", "priority_id"]
    )["similarity"]
    .transform(
        lambda x: x.rank(
            method="average"
        ) / (len(x) + 1)
    )
)

print(
    priority_endpoint_pairs[
        "similarity_percentile"
    ].describe()
)

NameError: name 'priority_endpoint_pairs' is not defined

In [ ]:
# Restore the durable trial-level datasets and working aliases used later in the notebook.
trials_master = pd.read_csv(
    DATA_DIR / "trialsignal_trials_master.csv"
)

endpoints_master = pd.read_csv(
    DATA_DIR / "trialsignal_endpoints_master.csv"
)

# Working copies used by the scope-cleaning section.
trials_scope_check = trials_master.copy()
trials_clean = trials_master.copy()

print("Trials restored:", len(trials_master))
print("Unique trials:", trials_master["nct_id"].nunique())
print("Endpoints restored:", len(endpoints_master))
print("Unique endpoint trials:", endpoints_master["nct_id"].nunique())

print(
    "SMA trials:",
    (trials_master["disease"] == "Spinal Muscular Atrophy").sum()
)

Trials restored: 1013
Unique trials: 1013
Endpoints restored: 8832
Unique endpoint trials: 1013
SMA trials: 148


In [ ]:
# Verify that the main TrialSignal objects required for current work are alive.
required_objects = {
    "alignment_priorities": alignment_priorities,
    "alignment_endpoints": alignment_endpoints,
    "priority_endpoint_pairs": priority_endpoint_pairs,
    "trial_endpoint_counts": trial_endpoint_counts,
    "trials_master": trials_master,
    "trials_scope_check": trials_scope_check,
    "trials_clean": trials_clean
}

for name, obj in required_objects.items():
    print(f"{name}: OK | rows = {len(obj)}")

alignment_priorities: OK | rows = 73
alignment_endpoints: OK | rows = 8832
priority_endpoint_pairs: OK | rows = 96635
trial_endpoint_counts: OK | rows = 1013
trials_master: OK | rows = 1013
trials_scope_check: OK | rows = 1013
trials_clean: OK | rows = 1013


### Resume Semantic Alignment Session

In [ ]:
# Restore the saved TrialSignal semantic-alignment objects after a runtime reset.
from google.colab import drive
from pathlib import Path
from scipy.stats import spearmanr
from scipy.special import gammaln

import pandas as pd
import numpy as np

drive.mount("/content/portfolio-drive")

DATA_DIR = Path("./TrialSignal/data")

alignment_priorities = pd.read_json(
    DATA_DIR / "trialsignal_alignment_priorities.json"
)

alignment_endpoints = pd.read_csv(
    DATA_DIR / "trialsignal_endpoints_master.csv"
)

priority_endpoint_pairs = pd.read_csv(
    DATA_DIR / "trialsignal_priority_endpoint_pairs.csv"
)

print("Data folder exists:", DATA_DIR.exists())
print("Alignment priorities:", len(alignment_priorities))
print("Endpoints:", len(alignment_endpoints))
print("Priority-endpoint pairs:", len(priority_endpoint_pairs))
print("Trials:", alignment_endpoints["nct_id"].nunique())

Drive already mounted at /content/portfolio-drive; to attempt to forcibly remount, call drive.mount("/content/portfolio-drive", force_remount=True).
Data folder exists: True
Alignment priorities: 73
Endpoints: 8832
Priority-endpoint pairs: 96635
Trials: 1013


In [ ]:
# Recreate the endpoint count for each trial from the frozen endpoint dataset.
trial_endpoint_counts = (
    alignment_endpoints
    .groupby(
        ["disease", "nct_id"]
    )
    .size()
    .rename("endpoint_count")
    .reset_index()
)

print("Trials with endpoint counts:", len(trial_endpoint_counts))
print(
    "Missing endpoint counts:",
    trial_endpoint_counts["endpoint_count"].isna().sum()
)

display(trial_endpoint_counts.head())

Trials with endpoint counts: 1013
Missing endpoint counts: 0


,disease,nct_id,endpoint_count
0,Dravet Syndrome,NCT01607073,3
1,Dravet Syndrome,NCT02091206,3
2,Dravet Syndrome,NCT02091375,12
3,Dravet Syndrome,NCT02174094,15
4,Dravet Syndrome,NCT02187809,7


In [ ]:
# Restore the frozen retrieval-validation outputs and current TrialSignal state.
import json

retrieval_review_units = pd.read_csv(
    DATA_DIR / "trialsignal_retrieval_review_units_final.csv"
)

retrieval_results = pd.read_csv(
    DATA_DIR / "trialsignal_retrieval_validation_results.csv"
)

with open(
    DATA_DIR / "trialsignal_candidate_retrieval_config.json",
    "r"
) as f:
    retrieval_config = json.load(f)

trials_scope_check = pd.read_csv(
    DATA_DIR / "trialsignal_trials_master.csv"
)

print("Retrieval review cases:", len(retrieval_review_units))
print(
    "Manual scope issues:",
    (retrieval_review_units["manual_scope_issue"] == 1).sum()
)
print(
    "Candidate Retrieval top_k:",
    retrieval_config["top_k"]
)

display(retrieval_results)

Retrieval review cases: 56
Manual scope issues: 2
Candidate Retrieval top_k: 10


,metric,direct_matches_retrieved,total_direct_matches,recall
0,Recall@1,15,16,0.9375
1,Recall@3,15,16,0.9375
2,Recall@5,15,16,0.9375
3,Recall@10,16,16,1.0000


In [ ]:
# Verify that the restored pairwise similarity file has the expected structure.
required_pair_columns = [
    "disease",
    "priority_id",
    "priority_label",
    "priority_domain",
    "nct_id",
    "endpoint_id",
    "endpoint_type",
    "endpoint_name",
    "similarity"
]

missing_columns = [
    column
    for column in required_pair_columns
    if column not in priority_endpoint_pairs.columns
]

print("Missing columns:", missing_columns)
print(
    "Diseases:",
    priority_endpoint_pairs["disease"].nunique()
)
print(
    "Priorities:",
    priority_endpoint_pairs["priority_id"].nunique()
)
print(
    "Missing similarity values:",
    priority_endpoint_pairs["similarity"].isna().sum()
)

Missing columns: []
Diseases: 7
Priorities: 73
Missing similarity values: 0


In [ ]:
# Rank endpoint similarities separately within each patient priority.
priority_endpoint_pairs["similarity_rank"] = (
    priority_endpoint_pairs
    .groupby(["disease", "priority_id"])["similarity"]
    .rank(
        method="max",
        ascending=True
    )
)

priority_endpoint_pairs["priority_endpoint_pool_size"] = (
    priority_endpoint_pairs
    .groupby(["disease", "priority_id"])["similarity"]
    .transform("size")
)

display(
    priority_endpoint_pairs[
        [
            "disease",
            "priority_id",
            "endpoint_id",
            "similarity",
            "similarity_rank",
            "priority_endpoint_pool_size"
        ]
    ].head()
)

,disease,priority_id,endpoint_id,similarity,similarity_rank,priority_endpoint_pool_size
0,Hereditary Angioedema,HAE_P01,NCT01912456_P1,0.411219,767.0,911
1,Hereditary Angioedema,HAE_P01,NCT01912456_S1,0.362686,715.0,911
2,Hereditary Angioedema,HAE_P01,NCT01912456_S2,0.128085,457.0,911
3,Hereditary Angioedema,HAE_P01,NCT01912456_S3,0.120932,428.0,911
4,Hereditary Angioedema,HAE_P01,NCT01912456_S4,0.185975,567.0,911


In [ ]:
# Confirm that each priority is ranked against every endpoint in its own disease.
rank_check = (
    priority_endpoint_pairs
    .groupby(
        ["disease", "priority_id"]
    )
    .agg(
        rows=("endpoint_id", "count"),
        pool_size=("priority_endpoint_pool_size", "first"),
        minimum_rank=("similarity_rank", "min"),
        maximum_rank=("similarity_rank", "max")
    )
    .reset_index()
)

print(
    "Rows where count and pool size disagree:",
    (rank_check["rows"] != rank_check["pool_size"]).sum()
)

display(rank_check.head())

Rows where count and pool size disagree: 0


,disease,priority_id,rows,pool_size,minimum_rank,maximum_rank
0,Dravet Syndrome,DRAVET_P01,527,527,1.0,527.0
1,Dravet Syndrome,DRAVET_P02,527,527,1.0,527.0
2,Dravet Syndrome,DRAVET_P03,527,527,1.0,527.0
3,Dravet Syndrome,DRAVET_P04,527,527,1.0,527.0
4,Dravet Syndrome,DRAVET_P05,527,527,1.0,527.0


In [ ]:
# Retain the strongest endpoint for every patient-priority and trial combination.
best_pair_indices = (
    priority_endpoint_pairs
    .groupby(
        ["disease", "priority_id", "nct_id"]
    )["similarity"]
    .idxmax()
)

trial_priority_matches = (
    priority_endpoint_pairs
    .loc[best_pair_indices]
    .copy()
    .reset_index(drop=True)
)

trial_priority_matches = (
    trial_priority_matches
    .merge(
        trial_endpoint_counts,
        on=["disease", "nct_id"],
        how="left"
    )
)

print("Priority-trial pairs:", len(trial_priority_matches))
print(
    "Missing endpoint counts:",
    trial_priority_matches["endpoint_count"].isna().sum()
)

Priority-trial pairs: 11067
Missing endpoint counts: 0


In [ ]:
# Define the finite-population correction for endpoint-count bias.
from scipy.special import gammaln

def log_combination(n, k):
    if k < 0 or k > n:
        return -np.inf

    return (
        gammaln(n + 1)
        - gammaln(k + 1)
        - gammaln(n - k + 1)
    )


def finite_population_max_percentile(row):
    N = int(row["priority_endpoint_pool_size"])
    r = int(row["similarity_rank"])
    n = int(row["endpoint_count"])

    if r < n:
        return 0.0

    log_probability = (
        log_combination(r, n)
        - log_combination(N, n)
    )

    return float(np.exp(log_probability))

In [ ]:
# Calculate the endpoint-count-adjusted percentile for each priority-trial match.
trial_priority_matches[
    "count_adjusted_percentile"
] = trial_priority_matches.apply(
    finite_population_max_percentile,
    axis=1
)

print(
    trial_priority_matches[
        "count_adjusted_percentile"
    ].describe()
)

count    1.106700e+04
mean     4.243847e-01
std      3.219316e-01
min      1.213943e-12
25%      1.165944e-01
50%      3.879780e-01
75%      7.084306e-01
max      1.000000e+00
Name: count_adjusted_percentile, dtype: float64


In [ ]:
# Verify that all adjusted percentiles are valid probabilities.
print(
    "Missing adjusted values:",
    trial_priority_matches[
        "count_adjusted_percentile"
    ].isna().sum()
)

print(
    "Below zero:",
    (
        trial_priority_matches[
            "count_adjusted_percentile"
        ] < 0
    ).sum()
)

print(
    "Above one:",
    (
        trial_priority_matches[
            "count_adjusted_percentile"
        ] > 1
    ).sum()
)

Missing adjusted values: 0
Below zero: 0
Above one: 0


In [ ]:
# Measure endpoint-count bias after applying the finite-population correction.
from scipy.stats import spearmanr

adjusted_bias_rows = []

for (
    disease,
    priority_id
), group in trial_priority_matches.groupby(
    ["disease", "priority_id"]
):

    if group["endpoint_count"].nunique() > 1:
        rho, p_value = spearmanr(
            group["endpoint_count"],
            group["count_adjusted_percentile"]
        )
    else:
        rho = np.nan
        p_value = np.nan

    adjusted_bias_rows.append({
        "disease": disease,
        "priority_id": priority_id,
        "priority_label": group[
            "priority_label"
        ].iloc[0],
        "spearman_rho": rho,
        "p_value": p_value
    })

adjusted_endpoint_count_bias = pd.DataFrame(
    adjusted_bias_rows
)

print(
    adjusted_endpoint_count_bias[
        "spearman_rho"
    ].describe()
)

print(
    "\nMedian adjusted rho:",
    adjusted_endpoint_count_bias[
        "spearman_rho"
    ].median()
)

print(
    "Positive correlations:",
    (
        adjusted_endpoint_count_bias[
            "spearman_rho"
        ] > 0
    ).sum(),
    "of",
    len(adjusted_endpoint_count_bias)
)

print(
    "rho > 0.20:",
    (
        adjusted_endpoint_count_bias[
            "spearman_rho"
        ] > 0.20
    ).sum()
)

count    73.000000
mean     -0.165221
std       0.166275
min      -0.597578
25%      -0.260171
50%      -0.204669
75%      -0.066059
max       0.244333
Name: spearman_rho, dtype: float64

Median adjusted rho: -0.2046687353703578
Positive correlations: 12 of 73
rho > 0.20: 1


In [ ]:
# Give the trial-level winning cosine score a clear name for the bias correction.
if "best_endpoint_similarity" not in trial_priority_matches.columns:
    trial_priority_matches = trial_priority_matches.rename(
        columns={
            "similarity": "best_endpoint_similarity"
        }
    )

print(
    trial_priority_matches[
        [
            "disease",
            "priority_id",
            "nct_id",
            "endpoint_count",
            "best_endpoint_similarity"
        ]
    ].head()
)

           disease priority_id       nct_id  endpoint_count  \
0  Dravet Syndrome  DRAVET_P01  NCT01607073               3   
1  Dravet Syndrome  DRAVET_P01  NCT02091206               3   
2  Dravet Syndrome  DRAVET_P01  NCT02091375              12   
3  Dravet Syndrome  DRAVET_P01  NCT02174094              15   
4  Dravet Syndrome  DRAVET_P01  NCT02187809               7   

   best_endpoint_similarity  
0                  0.414963  
1                  0.234219  
2                  0.526289  
3                  0.500806  
4                  0.461167  


In [ ]:
# Estimate the empirical endpoint-count effect separately for each patient priority.
from sklearn.linear_model import HuberRegressor

adjusted_groups = []
count_effect_models = []

for (
    disease,
    priority_id
), group in trial_priority_matches.groupby(
    ["disease", "priority_id"]
):

    group = group.copy()

    X = np.log1p(
        group["endpoint_count"].to_numpy()
    ).reshape(-1, 1)

    y = group[
        "best_endpoint_similarity"
    ].to_numpy()

    model = HuberRegressor()
    model.fit(X, y)

    predicted = model.predict(X)
    residual = y - predicted

    group[
        "expected_similarity_from_endpoint_count"
    ] = predicted

    group[
        "count_adjusted_similarity"
    ] = residual

    adjusted_groups.append(group)

    count_effect_models.append({
        "disease": disease,
        "priority_id": priority_id,
        "intercept": float(model.intercept_),
        "log_endpoint_count_coefficient": float(
            model.coef_[0]
        )
    })

trial_priority_matches_empirical = pd.concat(
    adjusted_groups,
    ignore_index=True
)

count_effect_models = pd.DataFrame(
    count_effect_models
)

print(
    "Adjusted priority-trial rows:",
    len(trial_priority_matches_empirical)
)

Adjusted priority-trial rows: 11067


In [ ]:
# Convert count-adjusted similarities to comparable within-priority percentiles.
trial_priority_matches_empirical[
    "adjusted_match_percentile"
] = (
    trial_priority_matches_empirical
    .groupby(
        ["disease", "priority_id"]
    )["count_adjusted_similarity"]
    .rank(
        method="average",
        pct=True
    )
)

display(
    trial_priority_matches_empirical[
        [
            "disease",
            "priority_id",
            "priority_label",
            "nct_id",
            "endpoint_count",
            "best_endpoint_similarity",
            "expected_similarity_from_endpoint_count",
            "count_adjusted_similarity",
            "adjusted_match_percentile"
        ]
    ].head()
)

,disease,priority_id,priority_label,nct_id,endpoint_count,best_endpoint_similarity,expected_similarity_from_endpoint_count,count_adjusted_similarity,adjusted_match_percentile
0,Dravet Syndrome,DRAVET_P01,Reduce seizures and status epilepticus,NCT01607073,3,0.414963,0.443676,-0.028714,0.306122
1,Dravet Syndrome,DRAVET_P01,Reduce seizures and status epilepticus,NCT02091206,3,0.234219,0.443676,-0.209457,0.102041
2,Dravet Syndrome,DRAVET_P01,Reduce seizures and status epilepticus,NCT02091375,12,0.526289,0.487357,0.038933,0.816327
3,Dravet Syndrome,DRAVET_P01,Reduce seizures and status epilepticus,NCT02174094,15,0.500806,0.495052,0.005755,0.551020
4,Dravet Syndrome,DRAVET_P01,Reduce seizures and status epilepticus,NCT02187809,7,0.461167,0.469364,-0.008197,0.428571


In [ ]:
# Test whether empirical residualization removes the endpoint-count advantage.
empirical_bias_rows = []

for (
    disease,
    priority_id
), group in trial_priority_matches_empirical.groupby(
    ["disease", "priority_id"]
):

    if group["endpoint_count"].nunique() > 1:
        rho, p_value = spearmanr(
            group["endpoint_count"],
            group["adjusted_match_percentile"]
        )
    else:
        rho = np.nan
        p_value = np.nan

    empirical_bias_rows.append({
        "disease": disease,
        "priority_id": priority_id,
        "priority_label": group[
            "priority_label"
        ].iloc[0],
        "spearman_rho": rho,
        "p_value": p_value
    })

empirical_endpoint_count_bias = pd.DataFrame(
    empirical_bias_rows
)

print(
    empirical_endpoint_count_bias[
        "spearman_rho"
    ].describe()
)

print(
    "\nMedian empirical rho:",
    empirical_endpoint_count_bias[
        "spearman_rho"
    ].median()
)

print(
    "Positive correlations:",
    (
        empirical_endpoint_count_bias[
            "spearman_rho"
        ] > 0
    ).sum(),
    "of",
    len(empirical_endpoint_count_bias)
)

print(
    "Absolute rho > 0.20:",
    (
        empirical_endpoint_count_bias[
            "spearman_rho"
        ].abs() > 0.20
    ).sum()
)

count    73.000000
mean      0.002138
std       0.047140
min      -0.162405
25%      -0.018986
50%       0.004682
75%       0.027514
max       0.117086
Name: spearman_rho, dtype: float64

Median empirical rho: 0.004682063527299471
Positive correlations: 41 of 73
Absolute rho > 0.20: 0


In [ ]:
# Recalculate and compare endpoint-count bias for all three scoring approaches.
raw_similarity_column = (
    "best_endpoint_similarity"
    if "best_endpoint_similarity" in trial_priority_matches.columns
    else "similarity"
)

comparison_rows = []

# Raw best-cosine bias.
raw_rhos = []

for (
    disease,
    priority_id
), group in trial_priority_matches.groupby(
    ["disease", "priority_id"]
):
    if group["endpoint_count"].nunique() > 1:
        rho, _ = spearmanr(
            group["endpoint_count"],
            group[raw_similarity_column]
        )
        raw_rhos.append(rho)

comparison_rows.append({
    "method": "Raw best cosine",
    "median_rho": np.nanmedian(raw_rhos),
    "mean_rho": np.nanmean(raw_rhos)
})


# Finite-population correction bias.
finite_rhos = []

for (
    disease,
    priority_id
), group in trial_priority_matches.groupby(
    ["disease", "priority_id"]
):
    if group["endpoint_count"].nunique() > 1:
        rho, _ = spearmanr(
            group["endpoint_count"],
            group["count_adjusted_percentile"]
        )
        finite_rhos.append(rho)

comparison_rows.append({
    "method": "Finite-population correction",
    "median_rho": np.nanmedian(finite_rhos),
    "mean_rho": np.nanmean(finite_rhos)
})


# Empirical residual correction bias.
empirical_rhos = []

for (
    disease,
    priority_id
), group in trial_priority_matches_empirical.groupby(
    ["disease", "priority_id"]
):
    if group["endpoint_count"].nunique() > 1:
        rho, _ = spearmanr(
            group["endpoint_count"],
            group["adjusted_match_percentile"]
        )
        empirical_rhos.append(rho)

comparison_rows.append({
    "method": "Empirical residual correction",
    "median_rho": np.nanmedian(empirical_rhos),
    "mean_rho": np.nanmean(empirical_rhos)
})


bias_comparison = pd.DataFrame(comparison_rows)

display(bias_comparison)

,method,median_rho,mean_rho
0,Raw best cosine,0.421132,0.418557
1,Finite-population correction,-0.204669,-0.165221
2,Empirical residual correction,0.004682,0.002138


In [ ]:
# Save the accepted endpoint-count-adjusted priority-trial matches.
trial_priority_matches_empirical.to_csv(
    DATA_DIR / "trialsignal_trial_priority_matches_adjusted.csv",
    index=False
)

empirical_endpoint_count_bias.to_csv(
    DATA_DIR / "trialsignal_empirical_endpoint_count_bias.csv",
    index=False
)

bias_comparison.to_csv(
    DATA_DIR / "trialsignal_endpoint_count_bias_comparison.csv",
    index=False
)

print("Saved accepted endpoint-count correction outputs.")

Saved accepted endpoint-count correction outputs.


In [ ]:
# Save the fitted endpoint-count correction model for each patient priority.
count_effect_models.to_csv(
    DATA_DIR / "trialsignal_endpoint_count_correction_models.csv",
    index=False
)

display(count_effect_models.head())

,disease,priority_id,intercept,log_endpoint_count_coefficient
0,Dravet Syndrome,DRAVET_P01,0.392301,0.037059
1,Dravet Syndrome,DRAVET_P02,0.055915,0.071155
2,Dravet Syndrome,DRAVET_P03,0.038215,0.083145
3,Dravet Syndrome,DRAVET_P04,0.042303,0.066919
4,Dravet Syndrome,DRAVET_P05,0.108869,0.073696


Endpoint-count bias correction. Taking the maximum cosine similarity across a trial's endpoints initially favored trials with more registered outcomes (median priority-specific Spearman rho = 0.421). A finite-population maximum correction overcorrected this relationship (median rho = -0.205), likely because endpoints within a trial are correlated rather than independent draws. We therefore estimated the empirical relationship between log endpoint count and maximum cosine similarity separately for each patient priority using robust Huber regression. Residualizing against this relationship reduced the median correlation to 0.005 (mean = 0.002), effectively eliminating the endpoint-count advantage. The resulting within-priority residual percentiles are used as candidate-strength measures, not as probabilities of clinical alignment.

In [ ]:
# Rank endpoints within each priority-trial combination for agent candidate retrieval.
priority_endpoint_pairs[
    "within_trial_rank"
] = (
    priority_endpoint_pairs
    .groupby(
        [
            "disease",
            "priority_id",
            "nct_id"
        ]
    )["similarity"]
    .rank(
        method="first",
        ascending=False
    )
    .astype(int)
)

print(
    "Best possible within-trial rank:",
    priority_endpoint_pairs[
        "within_trial_rank"
    ].min()
)

print(
    "Largest within-trial rank:",
    priority_endpoint_pairs[
        "within_trial_rank"
    ].max()
)

Best possible within-trial rank: 1
Largest within-trial rank: 133


In [ ]:
# Create the top-five cosine candidates available to a future Semantic Match Agent.
top5_trial_candidates = (
    priority_endpoint_pairs[
        priority_endpoint_pairs[
            "within_trial_rank"
        ] <= 5
    ]
    .copy()
    .sort_values(
        [
            "disease",
            "priority_id",
            "nct_id",
            "within_trial_rank"
        ]
    )
    .reset_index(drop=True)
)

print(
    "Priority-trial combinations represented:",
    top5_trial_candidates[
        [
            "disease",
            "priority_id",
            "nct_id"
        ]
    ]
    .drop_duplicates()
    .shape[0]
)

print(
    "Candidate rows:",
    len(top5_trial_candidates)
)

Priority-trial combinations represented: 11067
Candidate rows: 44986


In [ ]:
# Attach the accepted count-adjusted candidate strength to each top-five candidate set.
candidate_strength = (
    trial_priority_matches_empirical[
        [
            "disease",
            "priority_id",
            "nct_id",
            "best_endpoint_similarity",
            "count_adjusted_similarity",
            "adjusted_match_percentile"
        ]
    ]
    .copy()
)

top5_trial_candidates = (
    top5_trial_candidates
    .merge(
        candidate_strength,
        on=[
            "disease",
            "priority_id",
            "nct_id"
        ],
        how="left"
    )
)

print(
    "Missing adjusted percentiles:",
    top5_trial_candidates[
        "adjusted_match_percentile"
    ].isna().sum()
)

Missing adjusted percentiles: 0


In [ ]:
# Sample priority-trial combinations across diseases and candidate-strength levels.
validation_units = (
    trial_priority_matches_empirical[
        [
            "disease",
            "priority_id",
            "priority_label",
            "nct_id",
            "endpoint_count",
            "adjusted_match_percentile"
        ]
    ]
    .copy()
)

validation_units[
    "candidate_strength_band"
] = pd.qcut(
    validation_units[
        "adjusted_match_percentile"
    ],
    q=3,
    labels=[
        "low",
        "middle",
        "high"
    ]
)

sampled_units = []

for disease, disease_group in validation_units.groupby(
    "disease"
):
    for band, n_sample in [
        ("low", 2),
        ("middle", 3),
        ("high", 3)
    ]:
        band_group = disease_group[
            disease_group[
                "candidate_strength_band"
            ] == band
        ]

        sampled_units.append(
            band_group.sample(
                n=min(
                    n_sample,
                    len(band_group)
                ),
                random_state=42
            )
        )

retrieval_validation_units = pd.concat(
    sampled_units,
    ignore_index=True
)

print(
    "Validation priority-trial combinations:",
    len(retrieval_validation_units)
)

display(
    retrieval_validation_units[
        [
            "disease",
            "priority_id",
            "priority_label",
            "nct_id",
            "endpoint_count",
            "candidate_strength_band",
            "adjusted_match_percentile"
        ]
    ]
)

Validation priority-trial combinations: 56


,disease,priority_id,priority_label,nct_id,endpoint_count,candidate_strength_band,adjusted_match_percentile
0,Dravet Syndrome,DRAVET_P08,Increase independence and daily participation,NCT02926898,3,low,0.244898
1,Dravet Syndrome,DRAVET_P02,Preserve developmental progress and skills,NCT02926898,3,low,0.163265
2,Dravet Syndrome,DRAVET_P03,Preserve cognitive and intellectual function,NCT06422377,2,middle,0.387755
3,Dravet Syndrome,DRAVET_P06,Improve sleep,NCT07531745,9,middle,0.489796
4,Dravet Syndrome,DRAVET_P02,Preserve developmental progress and skills,NCT05560282,1,middle,0.489796
5,Dravet Syndrome,DRAVET_P05,Improve behavior and mood,NCT04940624,15,high,0.714286
6,Dravet Syndrome,DRAVET_P02,Preserve developmental progress and skills,NCT02187809,7,high,0.918367
7,Dravet Syndrome,DRAVET_P05,Improve behavior and mood,NCT07531745,9,high,0.734694
8,Facioscapulohumeral Muscular Dystrophy,FSHD_P08,Maintain independence,NCT07435129,8,low,0.033333
9,Facioscapulohumeral Muscular Dystrophy,FSHD_P02,Regain strength and muscle function,NCT00104078,1,low,0.100000


In [ ]:
# Retrieve the top-five cosine endpoint candidates for each sampled validation case.
retrieval_validation_candidates = (
    retrieval_validation_units[
        [
            "disease",
            "priority_id",
            "nct_id",
            "candidate_strength_band",
            "adjusted_match_percentile"
        ]
    ]
    .merge(
        top5_trial_candidates,
        on=[
            "disease",
            "priority_id",
            "nct_id"
        ],
        how="left",
        suffixes=(
            "_sample",
            ""
        )
    )
)

retrieval_validation_candidates = (
    retrieval_validation_candidates
    .sort_values(
        [
            "disease",
            "priority_id",
            "nct_id",
            "within_trial_rank"
        ]
    )
    .reset_index(drop=True)
)

display(
    retrieval_validation_candidates[
        [
            "disease",
            "priority_id",
            "priority_label",
            "nct_id",
            "candidate_strength_band",
            "within_trial_rank",
            "endpoint_type",
            "endpoint_name",
            "similarity"
        ]
    ]
)

,disease,priority_id,priority_label,nct_id,candidate_strength_band,within_trial_rank,endpoint_type,endpoint_name,similarity
0,Dravet Syndrome,DRAVET_P02,Preserve developmental progress and skills,NCT02187809,high,1,Primary,"Change in Behavioural, Neurocognitive Measures...",0.277064
1,Dravet Syndrome,DRAVET_P02,Preserve developmental progress and skills,NCT02187809,high,2,Secondary,Percentage of Initial Treatment Responders Who...,0.160593
2,Dravet Syndrome,DRAVET_P02,Preserve developmental progress and skills,NCT02187809,high,3,Secondary,Number of Initial Treatment Responders Who Ret...,0.147172
3,Dravet Syndrome,DRAVET_P02,Preserve developmental progress and skills,NCT02187809,high,4,Secondary,Change in Mean Weekly Number of Tonic-clonic a...,0.075514
4,Dravet Syndrome,DRAVET_P02,Preserve developmental progress and skills,NCT02187809,high,5,Primary,Number of Participants With Adverse Events of ...,0.039796
...,...,...,...,...,...,...,...,...,...
226,Spinal Muscular Atrophy,SMA_P12,Reduce contractures and scoliosis burden,NCT02227823,middle,1,Secondary,Change from baseline of Moviplate values at 6 ...,0.167893
227,Spinal Muscular Atrophy,SMA_P12,Reduce contractures and scoliosis burden,NCT02227823,middle,2,Primary,Change from Baseline in the distance walked at...,0.155619
228,Spinal Muscular Atrophy,SMA_P12,Reduce contractures and scoliosis burden,NCT02227823,middle,3,Secondary,Change from baseline of decrement at 6 months,0.096481
229,Spinal Muscular Atrophy,SMA_P12,Reduce contractures and scoliosis burden,NCT02227823,middle,4,Secondary,Change from baseline of the ratio at 6 minutes...,0.087525


In [ ]:
# Save the top-k cosine retrieval validation sample before manual review.
retrieval_validation_candidates.to_csv(
    DATA_DIR / "trialsignal_retrieval_validation_candidates.csv",
    index=False
)

print(
    "Validation candidate rows:",
    len(retrieval_validation_candidates)
)

Validation candidate rows: 231


In [ ]:
# Recreate the derived endpoint fields lost during the runtime reset.
alignment_endpoints["endpoint_number"] = pd.to_numeric(
    alignment_endpoints["endpoint_number"]
).astype(int)

alignment_endpoints["endpoint_id"] = (
    alignment_endpoints["nct_id"].astype(str)
    + "_"
    + alignment_endpoints["endpoint_type"].str[0].str.upper()
    + alignment_endpoints["endpoint_number"].astype(str)
)

alignment_endpoints["endpoint_semantic_text"] = (
    alignment_endpoints["endpoint_text"]
    .fillna("")
    .astype(str)
    .str.strip()
)

print("Endpoints:", len(alignment_endpoints))
print(
    "Duplicate endpoint IDs:",
    alignment_endpoints["endpoint_id"].duplicated().sum()
)
print(
    "Blank endpoint semantic texts:",
    (
        alignment_endpoints["endpoint_semantic_text"] == ""
    ).sum()
)

Endpoints: 8832
Duplicate endpoint IDs: 0
Blank endpoint semantic texts: 0


In [ ]:
# Restore the finalized TrialSignal retrieval and scope-analysis state.
import json

trial_priority_matches_empirical = pd.read_csv(
    DATA_DIR / "trialsignal_trial_priority_matches_adjusted.csv"
)

retrieval_review_units = pd.read_csv(
    DATA_DIR / "trialsignal_retrieval_review_units_final.csv"
)

retrieval_results = pd.read_csv(
    DATA_DIR / "trialsignal_retrieval_validation_results.csv"
)

with open(
    DATA_DIR / "trialsignal_candidate_retrieval_config.json",
    "r"
) as f:
    retrieval_config = json.load(f)

trials_scope_check = pd.read_csv(
    DATA_DIR / "trialsignal_trials_master.csv"
)

print("Adjusted priority-trial matches:", len(trial_priority_matches_empirical))
print("Retrieval validation cases:", len(retrieval_review_units))
print("Candidate Retrieval top_k:", retrieval_config["top_k"])
print("Trial master rows:", len(trials_scope_check))

display(retrieval_results)

Adjusted priority-trial matches: 11067
Retrieval validation cases: 56
Candidate Retrieval top_k: 10
Trial master rows: 1013


,metric,direct_matches_retrieved,total_direct_matches,recall
0,Recall@1,15,16,0.9375
1,Recall@3,15,16,0.9375
2,Recall@5,15,16,0.9375
3,Recall@10,16,16,1.0000


In [ ]:
# Add full patient-priority descriptions and endpoint text to the retrieval-validation sample.
priority_details = (
    alignment_priorities[
        [
            "priority_id",
            "priority_description"
        ]
    ]
    .drop_duplicates("priority_id")
)

endpoint_details = (
    alignment_endpoints[
        [
            "endpoint_id",
            "brief_title",
            "endpoint_semantic_text"
        ]
    ]
    .drop_duplicates("endpoint_id")
)

retrieval_validation_candidates = (
    retrieval_validation_candidates
    .merge(
        priority_details,
        on="priority_id",
        how="left"
    )
    .merge(
        endpoint_details,
        on="endpoint_id",
        how="left"
    )
)

print(
    "Missing priority descriptions:",
    retrieval_validation_candidates[
        "priority_description"
    ].isna().sum()
)

print(
    "Missing endpoint text:",
    retrieval_validation_candidates[
        "endpoint_semantic_text"
    ].isna().sum()
)

Missing priority descriptions: 0
Missing endpoint text: 0


In [ ]:
# Add fields for manual semantic-retrieval validation.
retrieval_validation_candidates[
    "human_endpoint_match"
] = pd.NA

retrieval_validation_candidates[
    "human_match_notes"
] = ""

display(
    retrieval_validation_candidates[
        [
            "disease",
            "priority_id",
            "priority_label",
            "nct_id",
            "candidate_strength_band",
            "within_trial_rank",
            "endpoint_type",
            "endpoint_name",
            "similarity",
            "human_endpoint_match"
        ]
    ].head(25)
)

,disease,priority_id,priority_label,nct_id,candidate_strength_band,within_trial_rank,endpoint_type,endpoint_name,similarity,human_endpoint_match
0,Dravet Syndrome,DRAVET_P02,Preserve developmental progress and skills,NCT02187809,high,1,Primary,"Change in Behavioural, Neurocognitive Measures...",0.277064,<NA>
1,Dravet Syndrome,DRAVET_P02,Preserve developmental progress and skills,NCT02187809,high,2,Secondary,Percentage of Initial Treatment Responders Who...,0.160593,<NA>
2,Dravet Syndrome,DRAVET_P02,Preserve developmental progress and skills,NCT02187809,high,3,Secondary,Number of Initial Treatment Responders Who Ret...,0.147172,<NA>
3,Dravet Syndrome,DRAVET_P02,Preserve developmental progress and skills,NCT02187809,high,4,Secondary,Change in Mean Weekly Number of Tonic-clonic a...,0.075514,<NA>
4,Dravet Syndrome,DRAVET_P02,Preserve developmental progress and skills,NCT02187809,high,5,Primary,Number of Participants With Adverse Events of ...,0.039796,<NA>
5,Dravet Syndrome,DRAVET_P02,Preserve developmental progress and skills,NCT02926898,low,1,Secondary,Percentage of Participants Who Achieved ≥ a 50...,0.106568,<NA>
6,Dravet Syndrome,DRAVET_P02,Preserve developmental progress and skills,NCT02926898,low,2,Primary,Change in Convulsive Seizure Frequency (CSF) F...,0.086259,<NA>
7,Dravet Syndrome,DRAVET_P02,Preserve developmental progress and skills,NCT02926898,low,3,Secondary,Longest Convulsive Seizure-Free Interval (Days),0.018709,<NA>
8,Dravet Syndrome,DRAVET_P02,Preserve developmental progress and skills,NCT05560282,middle,1,Primary,Seizure reduction,0.110029,<NA>
9,Dravet Syndrome,DRAVET_P03,Preserve cognitive and intellectual function,NCT06422377,middle,1,Primary,Percent Change From Baseline in Major Motor Dr...,0.117249,<NA>


In [ ]:
# Save the enriched retrieval-validation sample for manual review.
retrieval_validation_candidates.to_csv(
    DATA_DIR / "trialsignal_retrieval_validation_candidates_enriched.csv",
    index=False
)

print(
    "Saved enriched validation rows:",
    len(retrieval_validation_candidates)
)

Saved enriched validation rows: 231


In [ ]:
# Collapse the validation candidates into one review row per priority-trial case.
review_rows = []

group_cols = [
    "disease",
    "priority_id",
    "priority_label",
    "priority_description",
    "nct_id",
    "brief_title",
    "candidate_strength_band",
    "adjusted_match_percentile"
]

for keys, group in retrieval_validation_candidates.groupby(
    group_cols,
    dropna=False
):
    row = dict(zip(group_cols, keys))

    group = group.sort_values("within_trial_rank")

    for _, candidate in group.iterrows():
        rank = int(candidate["within_trial_rank"])

        row[f"candidate_{rank}_type"] = candidate["endpoint_type"]
        row[f"candidate_{rank}_name"] = candidate["endpoint_name"]
        row[f"candidate_{rank}_text"] = candidate["endpoint_semantic_text"]
        row[f"candidate_{rank}_similarity"] = candidate["similarity"]

    review_rows.append(row)

retrieval_review_units = pd.DataFrame(review_rows)

print("Priority-trial review units:", len(retrieval_review_units))

display(
    retrieval_review_units[
        [
            "disease",
            "priority_id",
            "priority_label",
            "nct_id",
            "candidate_strength_band",
            "adjusted_match_percentile",
            "candidate_1_name",
            "candidate_2_name",
            "candidate_3_name",
            "candidate_4_name",
            "candidate_5_name"
        ]
    ].head(10)
)

<notebook>/930112300.py:15: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for keys, group in retrieval_validation_candidates.groupby(


Priority-trial review units: 56


,disease,priority_id,priority_label,nct_id,candidate_strength_band,adjusted_match_percentile,candidate_1_name,candidate_2_name,candidate_3_name,candidate_4_name,candidate_5_name
0,Dravet Syndrome,DRAVET_P02,Preserve developmental progress and skills,NCT02187809,high,0.918367,"Change in Behavioural, Neurocognitive Measures...",Percentage of Initial Treatment Responders Who...,Number of Initial Treatment Responders Who Ret...,Change in Mean Weekly Number of Tonic-clonic a...,Number of Participants With Adverse Events of ...
1,Dravet Syndrome,DRAVET_P02,Preserve developmental progress and skills,NCT02926898,low,0.163265,Percentage of Participants Who Achieved ≥ a 50...,Change in Convulsive Seizure Frequency (CSF) F...,Longest Convulsive Seizure-Free Interval (Days),NaN,NaN
2,Dravet Syndrome,DRAVET_P02,Preserve developmental progress and skills,NCT05560282,middle,0.489796,Seizure reduction,NaN,NaN,NaN,NaN
3,Dravet Syndrome,DRAVET_P03,Preserve cognitive and intellectual function,NCT06422377,middle,0.387755,Percent Change From Baseline in Major Motor Dr...,Percent Change From Baseline in Convulsive Sei...,NaN,NaN,NaN
4,Dravet Syndrome,DRAVET_P05,Improve behavior and mood,NCT04940624,high,0.714286,Percentage of Participants With CGI-I Nonseizu...,Change From Baseline in Quality of Life Invent...,Percentage of Participants With CGI-I Seizure ...,Percentage of Participants With Clinical Globa...,Percentage of Participants With Caregiver Glob...
5,Dravet Syndrome,DRAVET_P05,Improve behavior and mood,NCT07531745,high,0.734694,Parts 1 and 2: Number of Participants with Cha...,Parts 1 and 2: Number of Participants with Tre...,Number of Participants With Clinically Signifi...,Parts 1 and 2: Number of Participants With Cli...,Parts 1 and 2: Number of Participants With Cli...
6,Dravet Syndrome,DRAVET_P06,Improve sleep,NCT07531745,middle,0.489796,Parts 1 and 2: Number of Participants With Cli...,Parts 1 and 2: Percent Change From Baseline in...,Parts 1 and 2: Number of Participants With Cli...,Parts 1 and 2: Number of Participants With Cli...,Number of Participants With Clinically Signifi...
7,Dravet Syndrome,DRAVET_P08,Increase independence and daily participation,NCT02926898,low,0.244898,Change in Convulsive Seizure Frequency (CSF) F...,Percentage of Participants Who Achieved ≥ a 50...,Longest Convulsive Seizure-Free Interval (Days),NaN,NaN
8,Facioscapulohumeral Muscular Dystrophy,FSHD_P01,Slow or stop loss of muscle function,NCT02861911,middle,0.400000,Maximal voluntary contraction (MVC) strength o...,Effect of NMES on the quadriceps endurance eva...,Evaluation of the body composition by bioelect...,Effect of NMES on the bone mineral density by ...,Effect of NMES on the oxidative stress markers...
9,Facioscapulohumeral Muscular Dystrophy,FSHD_P01,Slow or stop loss of muscle function,NCT03943290,high,0.866667,Part 2: Change in Muscle Function. Percent Cha...,Part 2: Change in Muscle Function. Absolute Ch...,Part 2: Change in Muscle Function. Absolute Ch...,Part 2: Change in Muscle Function - Percent Ch...,Part 2: Change in Muscle Function - Percent Ch...


In [ ]:
# Add fields for manual top-k retrieval validation.
retrieval_review_units["trial_has_true_match"] = pd.NA
retrieval_review_units["first_true_match_rank"] = pd.NA
retrieval_review_units["manual_review_notes"] = ""

display(
    retrieval_review_units[
        [
            "disease",
            "priority_id",
            "priority_label",
            "nct_id",
            "candidate_1_name",
            "candidate_2_name",
            "candidate_3_name",
            "candidate_4_name",
            "candidate_5_name",
            "trial_has_true_match",
            "first_true_match_rank"
        ]
    ].head(10)
)

,disease,priority_id,priority_label,nct_id,candidate_1_name,candidate_2_name,candidate_3_name,candidate_4_name,candidate_5_name,trial_has_true_match,first_true_match_rank
0,Dravet Syndrome,DRAVET_P02,Preserve developmental progress and skills,NCT02187809,"Change in Behavioural, Neurocognitive Measures...",Percentage of Initial Treatment Responders Who...,Number of Initial Treatment Responders Who Ret...,Change in Mean Weekly Number of Tonic-clonic a...,Number of Participants With Adverse Events of ...,<NA>,<NA>
1,Dravet Syndrome,DRAVET_P02,Preserve developmental progress and skills,NCT02926898,Percentage of Participants Who Achieved ≥ a 50...,Change in Convulsive Seizure Frequency (CSF) F...,Longest Convulsive Seizure-Free Interval (Days),NaN,NaN,<NA>,<NA>
2,Dravet Syndrome,DRAVET_P02,Preserve developmental progress and skills,NCT05560282,Seizure reduction,NaN,NaN,NaN,NaN,<NA>,<NA>
3,Dravet Syndrome,DRAVET_P03,Preserve cognitive and intellectual function,NCT06422377,Percent Change From Baseline in Major Motor Dr...,Percent Change From Baseline in Convulsive Sei...,NaN,NaN,NaN,<NA>,<NA>
4,Dravet Syndrome,DRAVET_P05,Improve behavior and mood,NCT04940624,Percentage of Participants With CGI-I Nonseizu...,Change From Baseline in Quality of Life Invent...,Percentage of Participants With CGI-I Seizure ...,Percentage of Participants With Clinical Globa...,Percentage of Participants With Caregiver Glob...,<NA>,<NA>
5,Dravet Syndrome,DRAVET_P05,Improve behavior and mood,NCT07531745,Parts 1 and 2: Number of Participants with Cha...,Parts 1 and 2: Number of Participants with Tre...,Number of Participants With Clinically Signifi...,Parts 1 and 2: Number of Participants With Cli...,Parts 1 and 2: Number of Participants With Cli...,<NA>,<NA>
6,Dravet Syndrome,DRAVET_P06,Improve sleep,NCT07531745,Parts 1 and 2: Number of Participants With Cli...,Parts 1 and 2: Percent Change From Baseline in...,Parts 1 and 2: Number of Participants With Cli...,Parts 1 and 2: Number of Participants With Cli...,Number of Participants With Clinically Signifi...,<NA>,<NA>
7,Dravet Syndrome,DRAVET_P08,Increase independence and daily participation,NCT02926898,Change in Convulsive Seizure Frequency (CSF) F...,Percentage of Participants Who Achieved ≥ a 50...,Longest Convulsive Seizure-Free Interval (Days),NaN,NaN,<NA>,<NA>
8,Facioscapulohumeral Muscular Dystrophy,FSHD_P01,Slow or stop loss of muscle function,NCT02861911,Maximal voluntary contraction (MVC) strength o...,Effect of NMES on the quadriceps endurance eva...,Evaluation of the body composition by bioelect...,Effect of NMES on the bone mineral density by ...,Effect of NMES on the oxidative stress markers...,<NA>,<NA>
9,Facioscapulohumeral Muscular Dystrophy,FSHD_P01,Slow or stop loss of muscle function,NCT03943290,Part 2: Change in Muscle Function. Percent Cha...,Part 2: Change in Muscle Function. Absolute Ch...,Part 2: Change in Muscle Function. Absolute Ch...,Part 2: Change in Muscle Function - Percent Ch...,Part 2: Change in Muscle Function - Percent Ch...,<NA>,<NA>


In [ ]:
# Save the compact manual-review template before labeling begins.
retrieval_review_units.to_csv(
    DATA_DIR / "trialsignal_retrieval_review_units.csv",
    index=False
)

print("Saved retrieval review units:", len(retrieval_review_units))

Saved retrieval review units: 56


In [ ]:
# Display the first ten priority-trial cases for manual semantic review.
review_batch_1 = retrieval_review_units.iloc[:10].copy()

for i, row in review_batch_1.iterrows():
    print("=" * 100)
    print(f"ROW {i}")
    print("Disease:", row["disease"])
    print("Priority:", row["priority_label"])
    print("Priority description:", row["priority_description"])
    print("Trial:", row["nct_id"])
    print("Trial title:", row["brief_title"])
    print(
        "Strength:",
        row["candidate_strength_band"],
        "| adjusted percentile:",
        round(row["adjusted_match_percentile"], 3)
    )

    for rank in range(1, 6):
        name = row.get(f"candidate_{rank}_name")
        text = row.get(f"candidate_{rank}_text")

        if pd.notna(name):
            print(f"\nCandidate {rank}: {name}")
            print(text)

ROW 0
Disease: Dravet Syndrome
Priority: Preserve developmental progress and skills
Priority description: Prevent developmental regression and support continued acquisition or preservation of developmental skills.
Trial: NCT02187809
Trial title: Safety and Tolerability of Clobazam as Adjunctive Therapy in Paediatric Patients Aged ≥1 to ≤16 Years With Dravet Syndrome
Strength: high | adjusted percentile: 0.918

Candidate 1: Change in Behavioural, Neurocognitive Measures Using Vineland Adaptive Behaviour Scale (VABS)
Change in Behavioural, Neurocognitive Measures Using Vineland Adaptive Behaviour Scale (VABS)

Candidate 2: Percentage of Initial Treatment Responders Who Returned to Their Baseline Tonic-clonic and Clonic Seizure Rate During the Study (an Assessment of Tachyphylaxis)
Percentage of Initial Treatment Responders Who Returned to Their Baseline Tonic-clonic and Clonic Seizure Rate During the Study (an Assessment of Tachyphylaxis)

Candidate 3: Number of Initial Treatment Respond

In [ ]:
# Apply the manual retrieval-validation labels for rows 0 through 9.
batch_1_labels = {
    0: (1, 1, "VABS directly measures adaptive and developmental functioning."),
    1: (0, pd.NA, "Retrieved endpoints measure seizures, not developmental progress."),
    2: (0, pd.NA, "Only seizure reduction is measured."),
    3: (0, pd.NA, "Retrieved endpoints measure seizure frequency, not cognition."),
    4: (1, 1, "CGI-I nonseizure symptoms directly includes disruptive behavior."),
    5: (0, pd.NA, "C-SSRS and other endpoints are safety-focused, not direct behavior/mood measures."),
    6: (0, pd.NA, "No retrieved endpoint measures sleep."),
    7: (0, pd.NA, "Retrieved endpoints measure seizures, not independence or participation."),
    8: (1, 1, "Maximal voluntary contraction directly measures muscle strength and function."),
    9: (1, 1, "Change in muscle function is directly measured.")
}

for row_index, (
    has_match,
    first_rank,
    note
) in batch_1_labels.items():

    retrieval_review_units.loc[
        row_index,
        "trial_has_true_match"
    ] = has_match

    retrieval_review_units.loc[
        row_index,
        "first_true_match_rank"
    ] = first_rank

    retrieval_review_units.loc[
        row_index,
        "manual_review_notes"
    ] = note

display(
    retrieval_review_units.loc[
        0:9,
        [
            "disease",
            "priority_label",
            "nct_id",
            "trial_has_true_match",
            "first_true_match_rank",
            "manual_review_notes"
        ]
    ]
)

,disease,priority_label,nct_id,trial_has_true_match,first_true_match_rank,manual_review_notes
0,Dravet Syndrome,Preserve developmental progress and skills,NCT02187809,1,1,VABS directly measures adaptive and developmen...
1,Dravet Syndrome,Preserve developmental progress and skills,NCT02926898,0,<NA>,"Retrieved endpoints measure seizures, not deve..."
2,Dravet Syndrome,Preserve developmental progress and skills,NCT05560282,0,<NA>,Only seizure reduction is measured.
3,Dravet Syndrome,Preserve cognitive and intellectual function,NCT06422377,0,<NA>,"Retrieved endpoints measure seizure frequency,..."
4,Dravet Syndrome,Improve behavior and mood,NCT04940624,1,1,CGI-I nonseizure symptoms directly includes di...
5,Dravet Syndrome,Improve behavior and mood,NCT07531745,0,<NA>,"C-SSRS and other endpoints are safety-focused,..."
6,Dravet Syndrome,Improve sleep,NCT07531745,0,<NA>,No retrieved endpoint measures sleep.
7,Dravet Syndrome,Increase independence and daily participation,NCT02926898,0,<NA>,"Retrieved endpoints measure seizures, not inde..."
8,Facioscapulohumeral Muscular Dystrophy,Slow or stop loss of muscle function,NCT02861911,1,1,Maximal voluntary contraction directly measure...
9,Facioscapulohumeral Muscular Dystrophy,Slow or stop loss of muscle function,NCT03943290,1,1,Change in muscle function is directly measured.


In [ ]:
# Save the manually reviewed retrieval-validation table after batch 1.
retrieval_review_units.to_csv(
    DATA_DIR / "trialsignal_retrieval_review_units_labeled.csv",
    index=False
)

print("Saved manual labels through row 9.")

Saved manual labels through row 9.


In [ ]:
# Display validation rows 10 through 19 for manual semantic review.
review_batch_2 = retrieval_review_units.iloc[10:20].copy()

for i, row in review_batch_2.iterrows():
    print("=" * 100)
    print(f"ROW {i}")
    print("Disease:", row["disease"])
    print("Priority:", row["priority_label"])
    print("Priority description:", row["priority_description"])
    print("Trial:", row["nct_id"])
    print("Trial title:", row["brief_title"])
    print(
        "Strength:",
        row["candidate_strength_band"],
        "| adjusted percentile:",
        round(row["adjusted_match_percentile"], 3)
    )

    for rank in range(1, 6):
        name = row.get(f"candidate_{rank}_name")
        text = row.get(f"candidate_{rank}_text")

        if pd.notna(name):
            print(f"\nCandidate {rank}: {name}")
            print(text)

ROW 10
Disease: Facioscapulohumeral Muscular Dystrophy
Priority: Regain strength and muscle function
Priority description: Restore strength or muscle function that has already been lost.
Trial: NCT00104078
Trial title: Study Evaluating MYO-029 in Adult Muscular Dystrophy
Strength: low | adjusted percentile: 0.1

Candidate 1: Safety assessment
Safety assessment
ROW 11
Disease: Facioscapulohumeral Muscular Dystrophy
Priority: Regain strength and muscle function
Priority description: Restore strength or muscle function that has already been lost.
Trial: NCT00821548
Trial title: Electrostimulation of Shoulder Girdle and Quadriceps Muscles in Facioscapulohumeral Muscular Dystrophy Patients
Strength: high | adjusted percentile: 0.933

Candidate 1: Efficacy of electrostimulation training in FSHD patients will be evaluated comparing manual muscle testing, Barré test, quantitative muscle assessment, fatigue severity scale, 6 minutes walking test before the therapy
Efficacy of electrostimulation

In [ ]:
# Apply the manual retrieval-validation labels for rows 10 through 19.
batch_2_labels = {
    10: (0, pd.NA, "Only a safety assessment is registered."),
    11: (1, 1, "Manual muscle testing and quantitative muscle assessment directly measure strength and function."),
    12: (1, 1, "Change in manual muscle testing score directly measures muscle function."),
    13: (0, pd.NA, "All retrieved endpoints are pharmacokinetic measures."),
    14: (0, pd.NA, "Physical-function and mobility measures are relevant proxies but do not directly measure independence."),
    15: (0, pd.NA, "Biomarkers, safety, muscle volume, and muscle composition do not directly measure independence."),
    16: (0, pd.NA, "Functional capacity and neurological function are too generic to confirm balance or standing measurement."),
    17: (0, pd.NA, "Ataxia scales may partially reflect gait, but the supplied endpoint does not directly establish walking or fall measurement."),
    18: (1, 1, "Cardiac MRI fibrosis directly measures cardiac disease burden relevant to cardiomyopathy."),
    19: (1, 1, "Echocardiographic cardiac-function change directly measures the priority.")
}

for row_index, (
    has_match,
    first_rank,
    note
) in batch_2_labels.items():

    retrieval_review_units.loc[
        row_index,
        "trial_has_true_match"
    ] = has_match

    retrieval_review_units.loc[
        row_index,
        "first_true_match_rank"
    ] = first_rank

    retrieval_review_units.loc[
        row_index,
        "manual_review_notes"
    ] = note

display(
    retrieval_review_units.loc[
        10:19,
        [
            "disease",
            "priority_label",
            "nct_id",
            "trial_has_true_match",
            "first_true_match_rank",
            "manual_review_notes"
        ]
    ]
)

,disease,priority_label,nct_id,trial_has_true_match,first_true_match_rank,manual_review_notes
10,Facioscapulohumeral Muscular Dystrophy,Regain strength and muscle function,NCT00104078,0,<NA>,Only a safety assessment is registered.
11,Facioscapulohumeral Muscular Dystrophy,Regain strength and muscle function,NCT00821548,1,1,Manual muscle testing and quantitative muscle ...
12,Facioscapulohumeral Muscular Dystrophy,Regain strength and muscle function,NCT02239224,1,1,Change in manual muscle testing score directly...
13,Facioscapulohumeral Muscular Dystrophy,Maintain independence,NCT06131983,0,<NA>,All retrieved endpoints are pharmacokinetic me...
14,Facioscapulohumeral Muscular Dystrophy,Maintain independence,NCT07038200,0,<NA>,Physical-function and mobility measures are re...
15,Facioscapulohumeral Muscular Dystrophy,Maintain independence,NCT07435129,0,<NA>,"Biomarkers, safety, muscle volume, and muscle ..."
16,Friedreich's Ataxia,Improve balance and standing ability,NCT00229632,0,<NA>,Functional capacity and neurological function ...
17,Friedreich's Ataxia,Improve walking and reduce falls,NCT01339884,0,<NA>,"Ataxia scales may partially reflect gait, but ..."
18,Friedreich's Ataxia,Preserve cardiac function and reduce cardiomyo...,NCT05445323,1,1,Cardiac MRI fibrosis directly measures cardiac...
19,Friedreich's Ataxia,Preserve cardiac function and reduce cardiomyo...,NCT06953583,1,1,Echocardiographic cardiac-function change dire...


In [ ]:
# Save the retrieval-validation labels after completing batch 2.
retrieval_review_units.to_csv(
    DATA_DIR / "trialsignal_retrieval_review_units_labeled.csv",
    index=False
)

print("Saved manual labels through row 19.")

Saved manual labels through row 19.


In [ ]:
# Display validation rows 20 through 29 for manual semantic review.
review_batch_3 = retrieval_review_units.iloc[20:30].copy()

for i, row in review_batch_3.iterrows():
    print("=" * 100)
    print(f"ROW {i}")
    print("Disease:", row["disease"])
    print("Priority:", row["priority_label"])
    print("Priority description:", row["priority_description"])
    print("Trial:", row["nct_id"])
    print("Trial title:", row["brief_title"])
    print(
        "Strength:",
        row["candidate_strength_band"],
        "| adjusted percentile:",
        round(row["adjusted_match_percentile"], 3)
    )

    for rank in range(1, 6):
        name = row.get(f"candidate_{rank}_name")
        text = row.get(f"candidate_{rank}_text")

        if pd.notna(name):
            print(f"\nCandidate {rank}: {name}")
            print(text)

ROW 20
Disease: Friedreich's Ataxia
Priority: Prolong life
Priority description: Extend survival and reduce the life-limiting consequences of Friedreich's Ataxia.
Trial: NCT00811681
Trial title: Effect of Pioglitazone Administered to Patients With Friedreich's Ataxia: Proof of Concept
Strength: middle | adjusted percentile: 0.339

Candidate 1: efficacy of Pioglitazone on functional handicap and quality of life
efficacy of Pioglitazone on functional handicap and quality of life

Candidate 2: evaluate the efficacy of Pioglitazone on the neurological function of FA patients. Success will be defined as a stabilisation or improvement on ICARS designed as no more than 2 points maximum increment on this scale in two year.
evaluate the efficacy of Pioglitazone on the neurological function of FA patients. Success will be defined as a stabilisation or improvement on ICARS designed as no more than 2 points maximum increment on this scale in two year.

Candidate 3: efficacy of Pioglitazone on neur

In [ ]:
# Apply the manual retrieval-validation labels for rows 20 through 29.
batch_3_labels = {
    20: (0, pd.NA, "Functional, neurologic, quality-of-life, and cardiac measures do not directly measure survival."),
    21: (0, pd.NA, "Ataxia, quality of life, depression, and baropodometric measures do not measure survival."),
    22: (0, pd.NA, "Frataxin, ataxia, daily-function, and epigenetic outcomes do not directly measure survival."),
    23: (0, pd.NA, "Neurologic function, biomarkers, physical function, quality of life, and safety do not measure survival."),
    24: (0, pd.NA, "Global attack symptom-response measures do not explicitly measure fatigue."),
    25: (0, pd.NA, "Attack rate and safety outcomes do not measure fatigue."),
    26: (0, pd.NA, "Attack severity, duration, frequency, and C1INH biomarkers do not measure fatigue."),
    27: (0, pd.NA, "Symptom composites measure pain, GI, swelling, and laryngeal symptoms but not fatigue."),
    28: (0, pd.NA, "Attack symptoms, pain, and rescue-treatment outcomes do not measure fatigue."),
    29: (0, pd.NA, "Attack frequency, duration, treatment use, and attack-free days do not measure fatigue.")
}

for row_index, (
    has_match,
    first_rank,
    note
) in batch_3_labels.items():

    retrieval_review_units.loc[
        row_index,
        "trial_has_true_match"
    ] = has_match

    retrieval_review_units.loc[
        row_index,
        "first_true_match_rank"
    ] = first_rank

    retrieval_review_units.loc[
        row_index,
        "manual_review_notes"
    ] = note

display(
    retrieval_review_units.loc[
        20:29,
        [
            "disease",
            "priority_label",
            "nct_id",
            "trial_has_true_match",
            "first_true_match_rank",
            "manual_review_notes"
        ]
    ]
)

,disease,priority_label,nct_id,trial_has_true_match,first_true_match_rank,manual_review_notes
20,Friedreich's Ataxia,Prolong life,NCT00811681,0,<NA>,"Functional, neurologic, quality-of-life, and c..."
21,Friedreich's Ataxia,Prolong life,NCT01104649,0,<NA>,"Ataxia, quality of life, depression, and barop..."
22,Friedreich's Ataxia,Prolong life,NCT01589809,0,<NA>,"Frataxin, ataxia, daily-function, and epigenet..."
23,Friedreich's Ataxia,Prolong life,NCT01962363,0,<NA>,"Neurologic function, biomarkers, physical func..."
24,Hereditary Angioedema,Reduce fatigue,NCT00262080,0,<NA>,Global attack symptom-response measures do not...
25,Hereditary Angioedema,Reduce fatigue,NCT00914966,0,<NA>,Attack rate and safety outcomes do not measure...
26,Hereditary Angioedema,Reduce fatigue,NCT01005888,0,<NA>,"Attack severity, duration, frequency, and C1IN..."
27,Hereditary Angioedema,Reduce fatigue,NCT03888755,0,<NA>,"Symptom composites measure pain, GI, swelling,..."
28,Hereditary Angioedema,Reduce fatigue,NCT04654351,0,<NA>,"Attack symptoms, pain, and rescue-treatment ou..."
29,Hereditary Angioedema,Reduce fatigue,NCT06007677,0,<NA>,"Attack frequency, duration, treatment use, and..."


In [ ]:
# Save the retrieval-validation labels after completing batch 3.
retrieval_review_units.to_csv(
    DATA_DIR / "trialsignal_retrieval_review_units_labeled.csv",
    index=False
)

print("Saved manual labels through row 29.")

Saved manual labels through row 29.


In [ ]:
# Display validation rows 30 through 39 for manual semantic review.
review_batch_4 = retrieval_review_units.iloc[30:40].copy()

for i, row in review_batch_4.iterrows():
    print("=" * 100)
    print(f"ROW {i}")
    print("Disease:", row["disease"])
    print("Priority:", row["priority_label"])
    print("Priority description:", row["priority_description"])
    print("Trial:", row["nct_id"])
    print("Trial title:", row["brief_title"])
    print(
        "Strength:",
        row["candidate_strength_band"],
        "| adjusted percentile:",
        round(row["adjusted_match_percentile"], 3)
    )

    for rank in range(1, 6):
        name = row.get(f"candidate_{rank}_name")
        text = row.get(f"candidate_{rank}_text")

        if pd.notna(name):
            print(f"\nCandidate {rank}: {name}")
            print(text)

ROW 30
Disease: Hereditary Angioedema
Priority: Reduce anxiety related to HAE
Priority description: Reduce the anxiety and psychological burden associated with living with HAE.
Trial: NCT00914966
Trial title: A Study to Evaluate the Safety and Effect of Escalating Doses of CINRYZE
Strength: middle | adjusted percentile: 0.562

Candidate 1: Treatment Effect of Escalating Doses of CINRYZE on HAE Attack Rates
Treatment Effect of Escalating Doses of CINRYZE on HAE Attack Rates Two definitions of success were applied in this study: 1) Per-protocol success - Average angioedema attack rate of ≤1.0 per month at the end of any dose escalation step (Week 12). The a priori definition of study success was 4 or more subjects with per-protocol success. 2) Investigator-determined success - Based on the investigator's clinical judgment, an average monthly angioedema attack rate demonstrating improvement sufficient for progression to follow-up. In addition, subjects who were not a per-protocol or inves

In [ ]:
# Apply the manual retrieval-validation labels for rows 30 through 39.
batch_4_labels = {
    30: (0, pd.NA, "Attack-rate and safety endpoints do not directly measure anxiety or psychological burden."),
    31: (0, pd.NA, "Self-treatment safety and attack symptom relief do not directly measure anxiety."),
    32: (1, 1, "Reduction in proteinuria is directly measured."),
    33: (1, 1, "Urinary protein-creatinine remission directly measures proteinuria reduction."),
    34: (1, 1, "Change in proteinuria directly measures the priority."),
    35: (1, 1, "Reduction in 24-hour UPCR directly measures the priority."),
    36: (0, pd.NA, "Renal biomarkers and laboratory measures do not measure fatigue."),
    37: (0, pd.NA, "Proteinuria, pharmacokinetic, kidney-function, and safety outcomes do not directly measure pain."),
    38: (0, pd.NA, "Proteinuria, remission, and molecular biomarkers do not measure cognitive function or brain fog."),
    39: (0, pd.NA, "Vital signs, ECG, body weight, and immunogenicity do not measure cognitive function or brain fog.")
}

for row_index, (
    has_match,
    first_rank,
    note
) in batch_4_labels.items():

    retrieval_review_units.loc[
        row_index,
        "trial_has_true_match"
    ] = has_match

    retrieval_review_units.loc[
        row_index,
        "first_true_match_rank"
    ] = first_rank

    retrieval_review_units.loc[
        row_index,
        "manual_review_notes"
    ] = note

display(
    retrieval_review_units.loc[
        30:39,
        [
            "disease",
            "priority_label",
            "nct_id",
            "trial_has_true_match",
            "first_true_match_rank",
            "manual_review_notes"
        ]
    ]
)

,disease,priority_label,nct_id,trial_has_true_match,first_true_match_rank,manual_review_notes
30,Hereditary Angioedema,Reduce anxiety related to HAE,NCT00914966,0,<NA>,Attack-rate and safety endpoints do not direct...
31,Hereditary Angioedema,Reduce anxiety related to HAE,NCT00997204,0,<NA>,Self-treatment safety and attack symptom relie...
32,Primary FSGS,Reduce proteinuria or albuminuria,NCT00814255,1,1,Reduction in proteinuria is directly measured.
33,Primary FSGS,Reduce proteinuria or albuminuria,NCT01665391,1,1,Urinary protein-creatinine remission directly ...
34,Primary FSGS,Reduce proteinuria or albuminuria,NCT04983888,1,1,Change in proteinuria directly measures the pr...
35,Primary FSGS,Reduce proteinuria or albuminuria,NCT05213624,1,1,Reduction in 24-hour UPCR directly measures th...
36,Primary FSGS,Reduce fatigue,NCT03536754,0,<NA>,Renal biomarkers and laboratory measures do no...
37,Primary FSGS,Reduce pain and aches,NCT04950114,0,<NA>,"Proteinuria, pharmacokinetic, kidney-function,..."
38,Primary FSGS,Improve cognitive function and brain fog,NCT01573533,0,<NA>,"Proteinuria, remission, and molecular biomarke..."
39,Primary FSGS,Improve cognitive function and brain fog,NCT03448692,0,<NA>,"Vital signs, ECG, body weight, and immunogenic..."


In [ ]:
# Save the retrieval-validation labels after completing batch 4.
retrieval_review_units.to_csv(
    DATA_DIR / "trialsignal_retrieval_review_units_labeled.csv",
    index=False
)

print("Saved manual labels through row 39.")

Saved manual labels through row 39.


In [ ]:
# Display validation rows 40 through 49 for manual semantic review.
review_batch_5 = retrieval_review_units.iloc[40:50].copy()

for i, row in review_batch_5.iterrows():
    print("=" * 100)
    print(f"ROW {i}")
    print("Disease:", row["disease"])
    print("Priority:", row["priority_label"])
    print("Priority description:", row["priority_description"])
    print("Trial:", row["nct_id"])
    print("Trial title:", row["brief_title"])
    print(
        "Strength:",
        row["candidate_strength_band"],
        "| adjusted percentile:",
        round(row["adjusted_match_percentile"], 3)
    )

    for rank in range(1, 6):
        name = row.get(f"candidate_{rank}_name")
        text = row.get(f"candidate_{rank}_text")

        if pd.notna(name):
            print(f"\nCandidate {rank}: {name}")
            print(text)

ROW 40
Disease: Sickle Cell Disease
Priority: Reduce acute pain crises
Priority description: Reduce the frequency and burden of severe acute sickle cell pain crises.
Trial: NCT04574050
Trial title: SELF-BREATHE RCT for Chronic Breathlessness
Strength: middle | adjusted percentile: 0.495

Candidate 1: Feasibility: the number of patients recruited into this study over a 12-month period
Feasibility: the number of patients recruited into this study over a 12-month period The number of patients recruited into this study over a 12-month period
ROW 41
Disease: Sickle Cell Disease
Priority: Reduce acute pain crises
Priority description: Reduce the frequency and burden of severe acute sickle cell pain crises.
Trial: NCT07023666
Trial title: Early Screening and Treatment of Heart Complication in Sickle Cell Disease
Strength: high | adjusted percentile: 0.936

Candidate 1: Incidence of Hospitalization for Sickle Cell Disease Related Complications
Incidence of Hospitalization for Sickle Cell Disea

In [ ]:
# Apply the manual retrieval-validation labels for rows 40 through 49.
batch_5_labels = {
    40: (0, pd.NA, "Recruitment feasibility does not measure acute pain crises."),
    41: (0, pd.NA, "Hospitalization includes vaso-occlusive crises but is a mixed SCD-complication composite, so it is partial rather than a direct pain-crisis measure."),
    42: (0, pd.NA, "Toxicity and transcranial Doppler outcomes do not measure chronic daily pain."),
    43: (0, pd.NA, "Hematology and splenic prognostic measures do not measure sleep."),
    44: (0, pd.NA, "Hospitalization, exercise capacity, VOC timing, walking, and analgesic use do not directly measure organ or tissue damage."),
    45: (1, 1, "Hematocrit directly measures anemia, while HbS reduction directly addresses the underlying sickling process."),
    46: (0, pd.NA, "ED utilization and pain outcomes do not directly measure daily functioning or participation."),
    47: (1, 1, "PROMIS depressive symptoms directly measures emotional wellbeing; anxiety is also measured."),
    48: (0, pd.NA, "Motor-function and muscle-power endpoints do not measure communication ability."),
    49: (1, 1, "Developmental milestone and functional change directly measure the course of SMA progression.")
}

for row_index, (
    has_match,
    first_rank,
    note
) in batch_5_labels.items():

    retrieval_review_units.loc[
        row_index,
        "trial_has_true_match"
    ] = has_match

    retrieval_review_units.loc[
        row_index,
        "first_true_match_rank"
    ] = first_rank

    retrieval_review_units.loc[
        row_index,
        "manual_review_notes"
    ] = note

display(
    retrieval_review_units.loc[
        40:49,
        [
            "disease",
            "priority_label",
            "nct_id",
            "trial_has_true_match",
            "first_true_match_rank",
            "manual_review_notes"
        ]
    ]
)

,disease,priority_label,nct_id,trial_has_true_match,first_true_match_rank,manual_review_notes
40,Sickle Cell Disease,Reduce acute pain crises,NCT04574050,0,<NA>,Recruitment feasibility does not measure acute...
41,Sickle Cell Disease,Reduce acute pain crises,NCT07023666,0,<NA>,Hospitalization includes vaso-occlusive crises...
42,Sickle Cell Disease,Reduce chronic daily pain,NCT02769845,0,<NA>,Toxicity and transcranial Doppler outcomes do ...
43,Sickle Cell Disease,Improve sleep,NCT01207037,0,<NA>,Hematology and splenic prognostic measures do ...
44,Sickle Cell Disease,Prevent long-term tissue and organ damage,NCT06823219,0,<NA>,"Hospitalization, exercise capacity, VOC timing..."
45,Sickle Cell Disease,Reduce sickling and underlying anemia,NCT01736657,1,1,"Hematocrit directly measures anemia, while HbS..."
46,Sickle Cell Disease,Maintain daily functioning and participation,NCT03296345,0,<NA>,ED utilization and pain outcomes do not direct...
47,Sickle Cell Disease,Improve emotional wellbeing,NCT07001631,1,1,PROMIS depressive symptoms directly measures e...
48,Spinal Muscular Atrophy,Preserve communication ability,NCT01671384,0,<NA>,Motor-function and muscle-power endpoints do n...
49,Spinal Muscular Atrophy,Slow or stop disease progression,NCT06191354,1,1,Developmental milestone and functional change ...


In [ ]:
# Save the retrieval-validation labels after completing batch 5.
retrieval_review_units.to_csv(
    DATA_DIR / "trialsignal_retrieval_review_units_labeled.csv",
    index=False
)

print("Saved manual labels through row 49.")

Saved manual labels through row 49.


In [ ]:
# Display the final validation rows 50 through 55 for manual semantic review.
review_batch_6 = retrieval_review_units.iloc[50:56].copy()

for i, row in review_batch_6.iterrows():
    print("=" * 100)
    print(f"ROW {i}")
    print("Disease:", row["disease"])
    print("Priority:", row["priority_label"])
    print("Priority description:", row["priority_description"])
    print("Trial:", row["nct_id"])
    print("Trial title:", row["brief_title"])
    print(
        "Strength:",
        row["candidate_strength_band"],
        "| adjusted percentile:",
        round(row["adjusted_match_percentile"], 3)
    )

    for rank in range(1, 6):
        name = row.get(f"candidate_{rank}_name")
        text = row.get(f"candidate_{rank}_text")

        if pd.notna(name):
            print(f"\nCandidate {rank}: {name}")
            print(text)

ROW 50
Disease: Spinal Muscular Atrophy
Priority: Slow or stop disease progression
Priority description: Slow or halt continued progression of spinal muscular atrophy.
Trial: NCT06411912
Trial title: A Study of NIDO-361 in Patients With SBMA
Strength: high | adjusted percentile: 0.75

Candidate 1: Change From Baseline to Day 180 and Day 360 in Modified-SBMAFRS (m-SBMAFRS)
Change From Baseline to Day 180 and Day 360 in Modified-SBMAFRS (m-SBMAFRS) The modified Spinal and Bulbar Muscular Atrophy Functional Rating Scale (m-SBMAFRS) is calculated as the sum of the items for the truncal and lower-limb domains from the SBMAFRS, which is a validated functional rating scale developed from the Amyotrophic Lateral Sclerosis Functional Rating Scale. Each item has five responses options scoring 0 (worst) to 4 (normal). A higher score indicates better functioning. Total range is 0 (worst) to 30 (normal).

Candidate 2: Analysis of Thigh and Total Lean Muscle Volume (LMV) [A+B+C]
Analysis of Thigh an

In [ ]:
# Apply final retrieval labels and flag explicit trial-scope failures.
if "manual_scope_issue" not in retrieval_review_units.columns:
    retrieval_review_units["manual_scope_issue"] = 0

batch_6_labels = {
    50: (0, pd.NA, 1, "Trial studies SBMA rather than the target SMA population."),
    51: (0, pd.NA, 0, "Neurological screening and anthropometric outcomes do not directly measure independent daily activities."),
    52: (0, pd.NA, 0, "Independent sitting and event-free survival do not measure pain."),
    53: (0, pd.NA, 1, "Trial is a poliovirus vaccine study and is outside the SMA project scope."),
    54: (0, pd.NA, 0, "Motor-function outcomes do not directly measure pain."),
    55: (0, pd.NA, 0, "Walking and motor-function outcomes do not directly measure contractures or scoliosis.")
}

for row_index, (
    has_match,
    first_rank,
    scope_issue,
    note
) in batch_6_labels.items():

    retrieval_review_units.loc[
        row_index,
        "trial_has_true_match"
    ] = has_match

    retrieval_review_units.loc[
        row_index,
        "first_true_match_rank"
    ] = first_rank

    retrieval_review_units.loc[
        row_index,
        "manual_scope_issue"
    ] = scope_issue

    retrieval_review_units.loc[
        row_index,
        "manual_review_notes"
    ] = note

display(
    retrieval_review_units.loc[
        50:55,
        [
            "disease",
            "priority_label",
            "nct_id",
            "trial_has_true_match",
            "first_true_match_rank",
            "manual_scope_issue",
            "manual_review_notes"
        ]
    ]
)

,disease,priority_label,nct_id,trial_has_true_match,first_true_match_rank,manual_scope_issue,manual_review_notes
50,Spinal Muscular Atrophy,Slow or stop disease progression,NCT06411912,0,<NA>,1,Trial studies SBMA rather than the target SMA ...
51,Spinal Muscular Atrophy,Increase independence in daily activities,NCT03032172,0,<NA>,0,Neurological screening and anthropometric outc...
52,Spinal Muscular Atrophy,Reduce pain,NCT03461289,0,<NA>,0,Independent sitting and event-free survival do...
53,Spinal Muscular Atrophy,Reduce pain,NCT03902054,0,<NA>,1,Trial is a poliovirus vaccine study and is out...
54,Spinal Muscular Atrophy,Reduce pain,NCT05626855,0,<NA>,0,Motor-function outcomes do not directly measur...
55,Spinal Muscular Atrophy,Reduce contractures and scoliosis burden,NCT02227823,0,<NA>,0,Walking and motor-function outcomes do not dir...


In [ ]:
# Save the completed manual retrieval-validation sample.
retrieval_review_units.to_csv(
    DATA_DIR / "trialsignal_retrieval_review_units_labeled.csv",
    index=False
)

print("Completed review units:", len(retrieval_review_units))
print(
    "Confirmed direct matches:",
    (retrieval_review_units["trial_has_true_match"] == 1).sum()
)
print(
    "Top-five no-matches:",
    (retrieval_review_units["trial_has_true_match"] == 0).sum()
)
print(
    "Explicit scope failures:",
    (retrieval_review_units["manual_scope_issue"] == 1).sum()
)

Completed review units: 56
Confirmed direct matches: 15
Top-five no-matches: 41
Explicit scope failures: 2


In [ ]:
# Add each trial's total endpoint count back to the manual review table.
retrieval_review_units = (
    retrieval_review_units
    .drop(
        columns=["endpoint_count"],
        errors="ignore"
    )
    .merge(
        trial_endpoint_counts,
        on=[
            "disease",
            "nct_id"
        ],
        how="left"
    )
)

print(
    "Missing endpoint counts:",
    retrieval_review_units["endpoint_count"].isna().sum()
)

display(
    retrieval_review_units[
        [
            "disease",
            "nct_id",
            "priority_label",
            "endpoint_count"
        ]
    ].head()
)

Missing endpoint counts: 0


,disease,nct_id,priority_label,endpoint_count
0,Dravet Syndrome,NCT02187809,Preserve developmental progress and skills,7
1,Dravet Syndrome,NCT02926898,Preserve developmental progress and skills,3
2,Dravet Syndrome,NCT05560282,Preserve developmental progress and skills,1
3,Dravet Syndrome,NCT06422377,Preserve cognitive and intellectual function,2
4,Dravet Syndrome,NCT04940624,Improve behavior and mood,15


In [ ]:
# Identify no-match cases that still have endpoints below the reviewed top five.
no_match_units = retrieval_review_units[
    (retrieval_review_units["trial_has_true_match"] == 0)
    & (retrieval_review_units["manual_scope_issue"] != 1)
].copy()

no_match_units["needs_lower_rank_audit"] = (
    no_match_units["endpoint_count"] > 5
)

print(
    "In-scope top-five no-matches:",
    len(no_match_units)
)

print(
    "Already exhaustive because trial has <=5 endpoints:",
    (~no_match_units["needs_lower_rank_audit"]).sum()
)

print(
    "Need ranks >5 inspected:",
    no_match_units["needs_lower_rank_audit"].sum()
)

display(
    no_match_units.loc[
        no_match_units["needs_lower_rank_audit"],
        [
            "disease",
            "priority_id",
            "priority_label",
            "nct_id",
            "endpoint_count",
            "candidate_strength_band"
        ]
    ].sort_values(
        [
            "disease",
            "priority_id",
            "nct_id"
        ]
    )
)

In-scope top-five no-matches: 39
Already exhaustive because trial has <=5 endpoints: 20
Need ranks >5 inspected: 19


,disease,priority_id,priority_label,nct_id,endpoint_count,candidate_strength_band
5,Dravet Syndrome,DRAVET_P05,Improve behavior and mood,NCT07531745,9,high
6,Dravet Syndrome,DRAVET_P06,Improve sleep,NCT07531745,9,middle
13,Facioscapulohumeral Muscular Dystrophy,FSHD_P08,Maintain independence,NCT06131983,11,middle
14,Facioscapulohumeral Muscular Dystrophy,FSHD_P08,Maintain independence,NCT07038200,11,high
15,Facioscapulohumeral Muscular Dystrophy,FSHD_P08,Maintain independence,NCT07435129,8,low
16,Friedreich's Ataxia,FA_P02,Improve balance and standing ability,NCT00229632,7,low
22,Friedreich's Ataxia,FA_P10,Prolong life,NCT01589809,7,middle
23,Friedreich's Ataxia,FA_P10,Prolong life,NCT01962363,10,high
26,Hereditary Angioedema,HAE_P08,Reduce fatigue,NCT01005888,7,high
27,Hereditary Angioedema,HAE_P08,Reduce fatigue,NCT03888755,18,low


In [ ]:
# Build the remaining endpoint set for the 19 unresolved top-five no-match cases.
audit_units = no_match_units.loc[
    no_match_units["needs_lower_rank_audit"],
    [
        "disease",
        "priority_id",
        "priority_label",
        "priority_description",
        "nct_id",
        "endpoint_count"
    ]
].copy()

if "within_trial_rank" not in priority_endpoint_pairs.columns:
    priority_endpoint_pairs["within_trial_rank"] = (
        priority_endpoint_pairs
        .groupby(
            [
                "disease",
                "priority_id",
                "nct_id"
            ]
        )["similarity"]
        .rank(
            method="first",
            ascending=False
        )
        .astype(int)
    )

lower_rank_audit = (
    audit_units
    .merge(
        priority_endpoint_pairs,
        on=[
            "disease",
            "priority_id",
            "nct_id"
        ],
        how="left",
        suffixes=(
            "_review",
            ""
        )
    )
)

lower_rank_audit = (
    lower_rank_audit[
        lower_rank_audit["within_trial_rank"] > 5
    ]
    .sort_values(
        [
            "disease",
            "priority_id",
            "nct_id",
            "within_trial_rank"
        ]
    )
    .reset_index(drop=True)
)

print("Cases requiring lower-rank audit:", len(audit_units))
print("Lower-ranked endpoints to inspect:", len(lower_rank_audit))

Cases requiring lower-rank audit: 19
Lower-ranked endpoints to inspect: 151


In [ ]:
# Attach full endpoint descriptions to the lower-rank audit candidates.
endpoint_text_lookup = (
    alignment_endpoints[
        [
            "endpoint_id",
            "endpoint_semantic_text"
        ]
    ]
    .drop_duplicates("endpoint_id")
)

lower_rank_audit = (
    lower_rank_audit
    .merge(
        endpoint_text_lookup,
        on="endpoint_id",
        how="left"
    )
)

print(
    "Missing endpoint text:",
    lower_rank_audit["endpoint_semantic_text"].isna().sum()
)

Missing endpoint text: 0


In [ ]:
# Save the complete lower-rank retrieval audit set.
lower_rank_audit.to_csv(
    DATA_DIR / "trialsignal_lower_rank_retrieval_audit.csv",
    index=False
)

print("Saved lower-rank audit rows:", len(lower_rank_audit))

Saved lower-rank audit rows: 151


In [ ]:
# Display all endpoints ranked below five for the first five unresolved cases.
audit_case_keys = (
    audit_units[
        [
            "disease",
            "priority_id",
            "nct_id"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

first_audit_cases = audit_case_keys.iloc[:5]

audit_batch_1 = (
    lower_rank_audit
    .merge(
        first_audit_cases,
        on=[
            "disease",
            "priority_id",
            "nct_id"
        ],
        how="inner"
    )
)

for (
    disease,
    priority_id,
    nct_id
), group in audit_batch_1.groupby(
    [
        "disease",
        "priority_id",
        "nct_id"
    ],
    sort=False
):

    first_row = group.iloc[0]

    print("=" * 100)
    print("Disease:", disease)
    print("Priority:", first_row["priority_label_review"])
    print(
        "Priority description:",
        first_row["priority_description"]
    )
    print("Trial:", nct_id)
    print("Total endpoints:", first_row["endpoint_count"])

    for _, candidate in group.sort_values(
        "within_trial_rank"
    ).iterrows():

        print(
            f"\nRank {int(candidate['within_trial_rank'])}: "
            f"{candidate['endpoint_name']}"
        )
        print(candidate["endpoint_semantic_text"])

Disease: Dravet Syndrome
Priority: Improve behavior and mood
Priority description: Reduce disruptive behavior, mood problems, and related behavioral difficulties.
Trial: NCT07531745
Total endpoints: 9

Rank 6: Parts 1 and 2: Percent Change From Baseline in 28-day Normalized Major Motor Seizure (MMS) Frequency
Parts 1 and 2: Percent Change From Baseline in 28-day Normalized Major Motor Seizure (MMS) Frequency

Rank 7: Parts 1 and 2: Number of Participants With Clinically Significant Change From Baseline in Vital Signs
Parts 1 and 2: Number of Participants With Clinically Significant Change From Baseline in Vital Signs

Rank 8: Exposure of ION337 in Cerebrospinal Fluid (CSF) Measure Description: Measurement of ION337 concentrations
Exposure of ION337 in Cerebrospinal Fluid (CSF) Measure Description: Measurement of ION337 concentrations

Rank 9: Pharmacokinetic (PK) Parameters Measure Description: Analysis of plasma concentrations of ION337
Pharmacokinetic (PK) Parameters Measure Descript

In [ ]:
# Mark the first five lower-rank audits as exhaustive no-matches.
if "lower_rank_audit_complete" not in retrieval_review_units.columns:
    retrieval_review_units["lower_rank_audit_complete"] = 0

if "lower_rank_true_match" not in retrieval_review_units.columns:
    retrieval_review_units["lower_rank_true_match"] = pd.NA

audit_batch_1_results = [
    ("DRAVET_P05", "NCT07531745"),
    ("DRAVET_P06", "NCT07531745"),
    ("FSHD_P08", "NCT06131983"),
    ("FSHD_P08", "NCT07038200"),
    ("FSHD_P08", "NCT07435129")
]

for priority_id, nct_id in audit_batch_1_results:
    mask = (
        (retrieval_review_units["priority_id"] == priority_id)
        & (retrieval_review_units["nct_id"] == nct_id)
    )

    retrieval_review_units.loc[
        mask,
        "lower_rank_audit_complete"
    ] = 1

    retrieval_review_units.loc[
        mask,
        "lower_rank_true_match"
    ] = 0

print(
    "Completed lower-rank audits:",
    int(retrieval_review_units["lower_rank_audit_complete"].sum())
)

Completed lower-rank audits: 5


In [ ]:
# Save the retrieval-validation table after the first lower-rank audit batch.
retrieval_review_units.to_csv(
    DATA_DIR / "trialsignal_retrieval_review_units_labeled.csv",
    index=False
)

print("Saved lower-rank audit progress.")

Saved lower-rank audit progress.


In [ ]:
# Display all endpoints below rank five for lower-rank audit cases 6 through 10.
next_audit_cases = audit_case_keys.iloc[5:10]

audit_batch_2 = (
    lower_rank_audit
    .merge(
        next_audit_cases,
        on=[
            "disease",
            "priority_id",
            "nct_id"
        ],
        how="inner"
    )
)

for (
    disease,
    priority_id,
    nct_id
), group in audit_batch_2.groupby(
    [
        "disease",
        "priority_id",
        "nct_id"
    ],
    sort=False
):

    first_row = group.iloc[0]

    print("=" * 100)
    print("Disease:", disease)
    print("Priority:", first_row["priority_label_review"])
    print(
        "Priority description:",
        first_row["priority_description"]
    )
    print("Trial:", nct_id)
    print("Total endpoints:", first_row["endpoint_count"])

    for _, candidate in group.sort_values(
        "within_trial_rank"
    ).iterrows():

        print(
            f"\nRank {int(candidate['within_trial_rank'])}: "
            f"{candidate['endpoint_name']}"
        )
        print(candidate["endpoint_semantic_text"])

Disease: Friedreich's Ataxia
Priority: Improve balance and standing ability
Priority description: Improve or preserve balance and the ability to stand.
Trial: NCT00229632
Total endpoints: 7

Rank 6: To evaluate metabolic alterations, gene expression changes and markers of mitochondrial DNA damage
To evaluate metabolic alterations, gene expression changes and markers of mitochondrial DNA damage

Rank 7: To evaluate the safety and tolerability of idebenone
To evaluate the safety and tolerability of idebenone
Disease: Friedreich's Ataxia
Priority: Prolong life
Priority description: Extend survival and reduce the life-limiting consequences of Friedreich's Ataxia.
Trial: NCT01589809
Total endpoints: 7

Rank 6: Determine the safety and tolerability of nicotinamide in FRDA patients (interventional study).
Determine the safety and tolerability of nicotinamide in FRDA patients (interventional study). Determine the safety and tolerability of nicotinamide in FRDA patients (interventional study).


In [ ]:
# Mark lower-rank audit cases 6 through 10 as exhaustive no-matches.
audit_batch_2_results = [
    ("FA_P02", "NCT00229632"),
    ("FA_P10", "NCT01589809"),
    ("FA_P10", "NCT01962363"),
    ("HAE_P08", "NCT01005888"),
    ("HAE_P08", "NCT03888755")
]

for priority_id, nct_id in audit_batch_2_results:
    mask = (
        (retrieval_review_units["priority_id"] == priority_id)
        & (retrieval_review_units["nct_id"] == nct_id)
    )

    retrieval_review_units.loc[
        mask,
        "lower_rank_audit_complete"
    ] = 1

    retrieval_review_units.loc[
        mask,
        "lower_rank_true_match"
    ] = 0

print(
    "Completed lower-rank audits:",
    int(retrieval_review_units["lower_rank_audit_complete"].sum())
)

Completed lower-rank audits: 10


In [ ]:
# Save retrieval-validation progress after the second lower-rank audit batch.
retrieval_review_units.to_csv(
    DATA_DIR / "trialsignal_retrieval_review_units_labeled.csv",
    index=False
)

print("Saved lower-rank audit progress.")

Saved lower-rank audit progress.


In [ ]:
# Display all endpoints below rank five for lower-rank audit cases 11 through 15.
next_audit_cases = audit_case_keys.iloc[10:15]

audit_batch_3 = (
    lower_rank_audit
    .merge(
        next_audit_cases,
        on=[
            "disease",
            "priority_id",
            "nct_id"
        ],
        how="inner"
    )
)

for (
    disease,
    priority_id,
    nct_id
), group in audit_batch_3.groupby(
    [
        "disease",
        "priority_id",
        "nct_id"
    ],
    sort=False
):

    first_row = group.iloc[0]

    print("=" * 100)
    print("Disease:", disease)
    print("Priority:", first_row["priority_label_review"])
    print(
        "Priority description:",
        first_row["priority_description"]
    )
    print("Trial:", nct_id)
    print("Total endpoints:", first_row["endpoint_count"])

    for _, candidate in group.sort_values(
        "within_trial_rank"
    ).iterrows():

        print(
            f"\nRank {int(candidate['within_trial_rank'])}: "
            f"{candidate['endpoint_name']}"
        )
        print(candidate["endpoint_semantic_text"])

Disease: Hereditary Angioedema
Priority: Reduce fatigue
Priority description: Reduce fatigue associated with HAE and its attacks.
Trial: NCT04654351
Total endpoints: 17

Rank 6: Number of Participants Who Experienced at Least One TEAE Related to Resting 12-lead Electrocardiogram
Number of Participants Who Experienced at Least One TEAE Related to Resting 12-lead Electrocardiogram An AE means any untoward medical occurrence in a participant administered a pharmaceutical product; the untoward medical occurrence does not necessarily have a causal relationship with this treatment. An AE can therefore be any unfavorable and unintended sign (including an abnormal laboratory finding), symptom, or disease temporally associated with the use of a medicinal (investigational) product whether or not it is related to the medicinal product. A TEAE was defined as any AE occurring after the start of Icatibant administration of the Treatment Period. A resting 12-lead ECG was recorded and reported for par

In [ ]:
# Mark lower-rank audit cases 11 through 15 as exhaustive no-matches.
audit_batch_3_results = [
    ("HAE_P08", "NCT04654351"),
    ("HAE_P08", "NCT06007677"),
    ("FSGS_P04", "NCT03536754"),
    ("FSGS_P06", "NCT04950114"),
    ("FSGS_P07", "NCT03448692")
]

for priority_id, nct_id in audit_batch_3_results:
    mask = (
        (retrieval_review_units["priority_id"] == priority_id)
        & (retrieval_review_units["nct_id"] == nct_id)
    )

    retrieval_review_units.loc[
        mask,
        "lower_rank_audit_complete"
    ] = 1

    retrieval_review_units.loc[
        mask,
        "lower_rank_true_match"
    ] = 0

print(
    "Completed lower-rank audits:",
    int(retrieval_review_units["lower_rank_audit_complete"].sum())
)

Completed lower-rank audits: 15


In [ ]:
# Save retrieval-validation progress after the third lower-rank audit batch.
retrieval_review_units.to_csv(
    DATA_DIR / "trialsignal_retrieval_review_units_labeled.csv",
    index=False
)

print("Saved lower-rank audit progress.")

Saved lower-rank audit progress.


In [ ]:
# Display all endpoints below rank five for the final four lower-rank audit cases.
final_audit_cases = audit_case_keys.iloc[15:19]

audit_batch_4 = (
    lower_rank_audit
    .merge(
        final_audit_cases,
        on=[
            "disease",
            "priority_id",
            "nct_id"
        ],
        how="inner"
    )
)

for (
    disease,
    priority_id,
    nct_id
), group in audit_batch_4.groupby(
    [
        "disease",
        "priority_id",
        "nct_id"
    ],
    sort=False
):

    first_row = group.iloc[0]

    print("=" * 100)
    print("Disease:", disease)
    print("Priority:", first_row["priority_label_review"])
    print(
        "Priority description:",
        first_row["priority_description"]
    )
    print("Trial:", nct_id)
    print("Total endpoints:", first_row["endpoint_count"])

    for _, candidate in group.sort_values(
        "within_trial_rank"
    ).iterrows():

        print(
            f"\nRank {int(candidate['within_trial_rank'])}: "
            f"{candidate['endpoint_name']}"
        )
        print(candidate["endpoint_semantic_text"])

Disease: Sickle Cell Disease
Priority: Prevent long-term tissue and organ damage
Priority description: Prevent or reduce progressive damage to organs, tissues, bones, and blood vessels.
Trial: NCT06823219
Total endpoints: 12

Rank 6: Number of acute chest syndromes (ACS)
Number of acute chest syndromes (ACS)

Rank 7: Echocardiographic Parameters
Echocardiographic Parameters Chamber Dimensions (mm or mL)

Rank 8: Number of vaso-occlusive crises (VOCs) with hospitalisation
Number of vaso-occlusive crises (VOCs) with hospitalisation determine the hospitalization rate for vaso-occlusive crisis

Rank 9: Echocardiographic Parameters
Echocardiographic Parameters Ventricular wall motion (qualitative assessment or mm)

Rank 10: Exercise Physiology Parameters
Exercise Physiology Parameters Heart rate (HR) (beats per minute - bpm)

Rank 11: Echocardiographic Parameters
Echocardiographic Parameters Left ventricular ejection fraction (LVEF) (%)

Rank 12: Days of school and job absenteeism
Days of s

In [ ]:
# Verify exactly which four unresolved audit cases remain.
final_audit_cases = audit_case_keys.iloc[15:19].copy()

display(final_audit_cases)

,disease,priority_id,nct_id
15,Sickle Cell Disease,SCD_P07,NCT06823219
16,Sickle Cell Disease,SCD_P09,NCT03296345
17,Spinal Muscular Atrophy,SMA_P09,NCT03032172
18,Spinal Muscular Atrophy,SMA_P11,NCT05626855


In [ ]:
# Display all lower-ranked endpoints for the final four retrieval-audit cases.
audit_batch_4 = (
    lower_rank_audit
    .merge(
        final_audit_cases,
        on=[
            "disease",
            "priority_id",
            "nct_id"
        ],
        how="inner"
    )
    .sort_values(
        [
            "disease",
            "priority_id",
            "nct_id",
            "within_trial_rank"
        ]
    )
)

print(
    "Final audit cases represented:",
    audit_batch_4[
        [
            "disease",
            "priority_id",
            "nct_id"
        ]
    ].drop_duplicates().shape[0]
)

for (
    disease,
    priority_id,
    nct_id
), group in audit_batch_4.groupby(
    [
        "disease",
        "priority_id",
        "nct_id"
    ],
    sort=False
):

    first_row = group.iloc[0]

    print("=" * 100)
    print("Disease:", disease)
    print("Priority:", first_row["priority_label_review"])
    print(
        "Priority description:",
        first_row["priority_description"]
    )
    print("Trial:", nct_id)
    print("Total endpoints:", first_row["endpoint_count"])

    for _, candidate in group.iterrows():

        print(
            f"\nRank {int(candidate['within_trial_rank'])}: "
            f"{candidate['endpoint_name']}"
        )
        print(candidate["endpoint_semantic_text"])

Final audit cases represented: 4
Disease: Sickle Cell Disease
Priority: Prevent long-term tissue and organ damage
Priority description: Prevent or reduce progressive damage to organs, tissues, bones, and blood vessels.
Trial: NCT06823219
Total endpoints: 12

Rank 6: Number of acute chest syndromes (ACS)
Number of acute chest syndromes (ACS)

Rank 7: Echocardiographic Parameters
Echocardiographic Parameters Chamber Dimensions (mm or mL)

Rank 8: Number of vaso-occlusive crises (VOCs) with hospitalisation
Number of vaso-occlusive crises (VOCs) with hospitalisation determine the hospitalization rate for vaso-occlusive crisis

Rank 9: Echocardiographic Parameters
Echocardiographic Parameters Ventricular wall motion (qualitative assessment or mm)

Rank 10: Exercise Physiology Parameters
Exercise Physiology Parameters Heart rate (HR) (beats per minute - bpm)

Rank 11: Echocardiographic Parameters
Echocardiographic Parameters Left ventricular ejection fraction (LVEF) (%)

Rank 12: Days of sch

In [ ]:
# Record the final four lower-rank audit judgments.
final_audit_results = {
    ("SCD_P07", "NCT06823219"): (
        1,
        9,
        "Ventricular wall motion directly measures cardiac organ function and provides a direct organ-damage outcome."
    ),
    ("SCD_P09", "NCT03296345"): (
        0,
        pd.NA,
        "Remaining endpoints measure pain reduction and safety, not daily functioning or participation."
    ),
    ("SMA_P09", "NCT03032172"): (
        0,
        pd.NA,
        "Remaining endpoints measure suicidality, PK, SMN protein, and safety, not independent daily activities."
    ),
    ("SMA_P11", "NCT05626855"): (
        0,
        pd.NA,
        "Remaining endpoints measure motor function, safety, and immunogenicity, not pain."
    )
}

for (priority_id, nct_id), (
    has_match,
    first_rank,
    note
) in final_audit_results.items():

    mask = (
        (retrieval_review_units["priority_id"] == priority_id)
        & (retrieval_review_units["nct_id"] == nct_id)
    )

    retrieval_review_units.loc[
        mask,
        "trial_has_true_match"
    ] = has_match

    retrieval_review_units.loc[
        mask,
        "first_true_match_rank"
    ] = first_rank

    retrieval_review_units.loc[
        mask,
        "lower_rank_audit_complete"
    ] = 1

    retrieval_review_units.loc[
        mask,
        "lower_rank_true_match"
    ] = has_match

    retrieval_review_units.loc[
        mask,
        "manual_review_notes"
    ] = note

print(
    "Completed lower-rank audits:",
    int(retrieval_review_units["lower_rank_audit_complete"].sum())
)

Completed lower-rank audits: 19


In [ ]:
# Calculate observed retrieval recall from the completed manual validation sample.
validated_in_scope = retrieval_review_units[
    retrieval_review_units["manual_scope_issue"] != 1
].copy()

validated_in_scope["first_true_match_rank"] = pd.to_numeric(
    validated_in_scope["first_true_match_rank"],
    errors="coerce"
)

true_match_cases = validated_in_scope[
    validated_in_scope["trial_has_true_match"] == 1
].copy()

retrieval_results = []

for k in [1, 3, 5, 10]:
    recall = (
        true_match_cases["first_true_match_rank"] <= k
    ).mean()

    retrieval_results.append(
        {
            "metric": f"Recall@{k}",
            "direct_matches_retrieved": int(
                (
                    true_match_cases["first_true_match_rank"] <= k
                ).sum()
            ),
            "total_direct_matches": len(true_match_cases),
            "recall": recall
        }
    )

retrieval_results = pd.DataFrame(retrieval_results)

print("Total validation cases:", len(retrieval_review_units))
print("In-scope validation cases:", len(validated_in_scope))
print("Direct-match cases:", len(true_match_cases))
print(
    "Confirmed no-match cases:",
    (
        validated_in_scope["trial_has_true_match"] == 0
    ).sum()
)

display(retrieval_results)

Total validation cases: 56
In-scope validation cases: 54
Direct-match cases: 16
Confirmed no-match cases: 38


,metric,direct_matches_retrieved,total_direct_matches,recall
0,Recall@1,15,16,0.9375
1,Recall@3,15,16,0.9375
2,Recall@5,15,16,0.9375
3,Recall@10,16,16,1.0000


In [ ]:
# Save the finalized retrieval-validation results and Candidate Retrieval Agent configuration.
import json

retrieval_results.to_csv(
    DATA_DIR / "trialsignal_retrieval_validation_results.csv",
    index=False
)

retrieval_review_units.to_csv(
    DATA_DIR / "trialsignal_retrieval_review_units_final.csv",
    index=False
)

retrieval_config = {
    "retrieval_method": "within-trial cosine similarity",
    "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
    "top_k": 10,
    "validation_cases_total": 56,
    "validation_cases_in_scope": 54,
    "direct_match_cases": 16,
    "confirmed_no_match_cases": 38,
    "recall_at_1": 0.9375,
    "recall_at_3": 0.9375,
    "recall_at_5": 0.9375,
    "recall_at_10": 1.0,
    "interpretation": (
        "Top-10 retrieval recovered all direct matches observed in the "
        "manually reviewed validation sample. Cosine retrieval is used "
        "for candidate generation only, not final semantic alignment."
    )
}

with open(
    DATA_DIR / "trialsignal_candidate_retrieval_config.json",
    "w"
) as f:
    json.dump(retrieval_config, f, indent=2)

print("Candidate Retrieval Agent frozen at top_k = 10.")

Candidate Retrieval Agent frozen at top_k = 10.


Retrieval validation. We manually reviewed 56 priority–trial cases sampled across diseases and candidate-strength levels. Two cases were excluded from retrieval evaluation because manual review identified them as outside the intended disease scope. Among the remaining 54 cases, 16 contained a confirmed direct patient-priority match. The top-ranked endpoint retrieved 15 of 16 matches (93.75%). An exhaustive audit of lower-ranked endpoints identified one additional direct match at rank 9, bringing Recall@10 to 100% in the validation sample. TrialSignal therefore retrieves the top 10 within-trial endpoint candidates for downstream semantic adjudication. Cosine similarity is treated as a retrieval signal rather than evidence of clinical alignment.

In [ ]:
# Inspect the two scope-contaminated SMA trials and the metadata that caused their inclusion.
scope_check_ids = [
    "NCT06411912",
    "NCT03902054"
]

trials_scope_check = pd.read_csv(
    DATA_DIR / "trialsignal_trials_master.csv"
)

preferred_columns = [
    "disease",
    "nct_id",
    "brief_title",
    "official_title",
    "conditions",
    "condition",
    "keywords",
    "study_type",
    "phase",
    "interventions",
    "intervention_names",
    "eligibility_criteria"
]

available_columns = [
    column
    for column in preferred_columns
    if column in trials_scope_check.columns
]

scope_problem_trials = trials_scope_check[
    trials_scope_check["nct_id"].isin(scope_check_ids)
].copy()

print("Matching records:", len(scope_problem_trials))
print("Available diagnostic columns:", available_columns)

display(
    scope_problem_trials[
        available_columns
    ].T
)

Matching records: 2
Available diagnostic columns: ['disease', 'nct_id', 'brief_title', 'official_title', 'conditions', 'study_type', 'intervention_names']


,919,965
disease,Spinal Muscular Atrophy,Spinal Muscular Atrophy
nct_id,NCT03902054,NCT06411912
brief_title,Safety and Immunogenicity of a Sabin Inactivat...,A Study of NIDO-361 in Patients With SBMA
official_title,"A Randomized, Blinded and Controlled Phase II ...","A Phase 2, Randomized, Double-Blind, Placebo-C..."
conditions,Polio and Post-Polio Syndrome,Spinal and Bulbar Muscular Atrophy; Kennedy's ...
study_type,INTERVENTIONAL,INTERVENTIONAL
intervention_names,Three-dose regimen of high dosage investigatio...,NIDO-361; Placebo


In [ ]:
# Show the trial-master schema so we can trace the source of the scope false positives.
print(trials_scope_check.columns.tolist())

['disease', 'nct_id', 'brief_title', 'official_title', 'study_type', 'overall_status', 'has_results', 'phases', 'primary_purpose', 'lead_sponsor', 'funder_type', 'enrollment', 'enrollment_type', 'start_date', 'primary_completion_date', 'completion_date', 'conditions', 'intervention_names', 'intervention_types', 'sex', 'minimum_age', 'maximum_age', 'age_groups', 'primary_outcomes', 'secondary_outcomes', 'has_primary_outcome', 'has_secondary_outcome', 'analysis_candidate', 'scope_decision']


In [ ]:
# Flag SMA-assigned trials whose metadata does not clearly indicate spinal muscular atrophy.
sma_trials = trials_scope_check[
    trials_scope_check["disease"] == "Spinal Muscular Atrophy"
].copy()

scope_text = (
    sma_trials["brief_title"].fillna("").astype(str)
    + " "
    + sma_trials["official_title"].fillna("").astype(str)
    + " "
    + sma_trials["conditions"].fillna("").astype(str)
).str.lower()

valid_sma_pattern = (
    r"spinal muscular atrophy"
    r"|muscular atrophy,\s*spinal"
    r"|\bsma\s*type"
    r"|\b5q[- ]?sma\b"
)

sma_trials["clear_sma_reference"] = scope_text.str.contains(
    valid_sma_pattern,
    regex=True
)

sma_scope_suspects = sma_trials[
    ~sma_trials["clear_sma_reference"]
].copy()

print("Total SMA-assigned trials:", len(sma_trials))
print("Clear SMA reference:", sma_trials["clear_sma_reference"].sum())
print("Need manual scope review:", len(sma_scope_suspects))

display(
    sma_scope_suspects[
        [
            "nct_id",
            "brief_title",
            "official_title",
            "conditions",
            "intervention_names"
        ]
    ]
)

Total SMA-assigned trials: 148
Clear SMA reference: 105
Need manual scope review: 43


,nct_id,brief_title,official_title,conditions,intervention_names
831,NCT00444613,A Study in Patients With Amyotrophic Lateral S...,A Phase II/III Study in Patients With Amyotrop...,Amyotrophic Lateral Sclerosis (ALS),E0302 (mecobalamin); E0302 (mecobalamin); Placebo
834,NCT01369901,Effect of Functional Exercise in Patients With...,Effect of Functional Exercise in Patients With...,Motor Neuron Disease,A functional exercise program; A stretching ex...
839,NCT05694819,Darolutamide in Patients With Androgen Recepto...,Phase II Study of Darolutamide (ODM-201) in Pa...,Salivary Gland Cancer,Darolutamide; Goserelin
848,NCT00324454,"Levetiracetam for Cramps, Spasticity and Neuro...","A Pilot Trial of Levetiracetam for Cramps, Spa...",Motor Neuron Disease; Amyotrophic Lateral Scle...,Levetiracetam
849,NCT02176863,Study of the Efficacy and Safety of Immune Glo...,"A Multicenter, Prospective, Randomized, Placeb...",Post-polio Syndrome,Flebogamma® 5% DIF; Placebo
850,NCT04090528,pTVG-HP DNA Vaccine With or Without pTVG-AR DN...,Phase II Trial of pTVG-HP DNA Vaccine With or ...,Castration-resistant Prostate Cancer; Metastat...,pTVG-HP; pTVG-AR; Pembrolizumab
851,NCT00878436,Safety and Efficacy Studies of Panobinostat an...,Phase I/II Randomized Trial of LBH589 (Panobin...,Prostate Cancer; Prostatic Neoplasms,Panobinostat; Bicalutamide
852,NCT01969578,Androgen Deprivation Therapy in Advanced Saliv...,A Randomized Phase II Study to Evaluate the Ef...,Salivary Gland Cancer,bicalutamide + triptorelin; Cisplatin + Doxoru...
855,NCT06365788,Bicalutamide and Abemaciclib in Inoperable or ...,A Multicenter Single Arm Phase II Study With B...,Triple Negative Breast Neoplasms,Abemaciclib; Bicalutamide
856,NCT05517603,"A Study to Evaluate Safety, Tolerability, Phar...","A Phase 1/2a, Randomized, Double-Blind, Placeb...",Spinal and Bulbar Muscular Atrophy; Kennedy's ...,AJ201; Placebo


In [ ]:
# Flag known non-target conditions that can be confused with spinal muscular atrophy.
known_wrong_scope_pattern = (
    r"spinal and bulbar muscular atrophy"
    r"|kennedy'?s disease"
    r"|post[- ]polio"
    r"|\bpolio\b"
)

sma_trials["known_wrong_scope_signal"] = scope_text.str.contains(
    known_wrong_scope_pattern,
    regex=True
)

display(
    sma_trials.loc[
        sma_trials["known_wrong_scope_signal"],
        [
            "nct_id",
            "brief_title",
            "official_title",
            "conditions"
        ]
    ]
)

,nct_id,brief_title,official_title,conditions
834,NCT01369901,Effect of Functional Exercise in Patients With...,Effect of Functional Exercise in Patients With...,Motor Neuron Disease
849,NCT02176863,Study of the Efficacy and Safety of Immune Glo...,"A Multicenter, Prospective, Randomized, Placeb...",Post-polio Syndrome
856,NCT05517603,"A Study to Evaluate Safety, Tolerability, Phar...","A Phase 1/2a, Randomized, Double-Blind, Placeb...",Spinal and Bulbar Muscular Atrophy; Kennedy's ...
868,NCT01537575,Intravenous Immunoglobulins for Post-Polio Syn...,NaN,Post-polio Syndrome
874,NCT00231439,Post-Poliosyndrome Treated With Intravenous Im...,Post-Polio Syndrome Treated With Intravenous I...,Post-Polio Syndrome
886,NCT06169046,A Placebo-controlled Study of Clenbuterol in S...,A Placebo-controlled Study to Evaluate the Eff...,Spinal and Bulbar Muscular Atrophy
888,NCT00851461,Effect of Goserelin (Zoladex®) in Spinal and B...,Effect of Goserelin (Zoladex®) in Spinal and B...,Spinobulbar Muscular Atrophy; Kennedy's Disease
896,NCT06862596,Clinical Trial of Mexiletine Hydrochloride for...,"A Multicenter, Randomized, Placebo-controlled,...",Spinal and Bulbar Muscular Atrophy
902,NCT02024932,"Safety, Tolerability, and Efficacy of BVS857 i...",A Two-part Placebo-controlled Study to Evaluat...,Spinal and Bulbar Muscular Atrophy
919,NCT03902054,Safety and Immunogenicity of a Sabin Inactivat...,"A Randomized, Blinded and Controlled Phase II ...",Polio and Post-Polio Syndrome


In [ ]:
# Recreate within-trial cosine ranks for top-k candidate retrieval.
priority_endpoint_pairs["within_trial_rank"] = (
    priority_endpoint_pairs
    .groupby(
        [
            "disease",
            "priority_id",
            "nct_id"
        ]
    )["similarity"]
    .rank(
        method="first",
        ascending=False
    )
    .astype(int)
)

print(
    "Minimum within-trial rank:",
    priority_endpoint_pairs["within_trial_rank"].min()
)

print(
    "Priority-endpoint pairs:",
    len(priority_endpoint_pairs)
)

Minimum within-trial rank: 1
Priority-endpoint pairs: 96635


In [ ]:
# Separate obvious non-SMA records from the small ambiguous subset needing manual review.
suspect_conditions = (
    sma_scope_suspects["conditions"]
    .fillna("")
    .astype(str)
    .str.lower()
)

obvious_wrong_scope_pattern = (
    r"amyotrophic lateral sclerosis"
    r"|post[- ]?polio"
    r"|postpoliomyelitis"
    r"|spinal and bulbar muscular atrophy"
    r"|spinobulbar muscular atrophy"
    r"|kennedy"
    r"|prostate"
    r"|breast"
    r"|salivary"
    r"|duchenne"
    r"|acne"
    r"|spasticity"
    r"|\bpolio\b"
)

obvious_sma_exclusions = sma_scope_suspects[
    suspect_conditions.str.contains(
        obvious_wrong_scope_pattern,
        regex=True
    )
].copy()

ambiguous_sma_scope = sma_scope_suspects[
    ~suspect_conditions.str.contains(
        obvious_wrong_scope_pattern,
        regex=True
    )
].copy()

print("Scope suspects:", len(sma_scope_suspects))
print("Obvious exclusions:", len(obvious_sma_exclusions))
print("Ambiguous cases needing review:", len(ambiguous_sma_scope))

NameError: name 'sma_scope_suspects' is not defined

In [ ]:
# Print full metadata for the remaining ambiguous SMA scope cases.
for _, row in ambiguous_sma_scope.iterrows():
    print("=" * 100)
    print("NCT ID:", row["nct_id"])
    print("Brief title:", row["brief_title"])
    print("Official title:", row["official_title"])
    print("Conditions:", row["conditions"])
    print("Interventions:", row["intervention_names"])

NCT ID: NCT01369901
Brief title: Effect of Functional Exercise in Patients With Spinal Bulbar Muscular Atrophy
Official title: Effect of Functional Exercise in Patients With Spinal and Bulbar Muscular Atrophy
Conditions: Motor Neuron Disease
Interventions: A functional exercise program; A stretching exercise program
NCT ID: NCT05152823
Brief title: Gene Therapy for IGHMBP2-Related Diseases
Official title: Phase I/IIa Intrathecal Gene Delivery Clinical Trial for IGHMBP2-Related Diseases
Conditions: SMARD1; CMT2S
Interventions: Gene Therapy
NCT ID: NCT03196375
Brief title: A Study to Assess FLX-787 in Subjects With Motor Neuron Disease Experiencing Muscle Cramps.
Official title: A Randomized, Double-Blind, Controlled, Parallel Group Study to Evaluate the Efficacy and Safety of FLX-787-ODT for Treatment of Muscle Cramps in Adult Subjects With Motor Neuron Disease
Conditions: Motor Neuron Disease
Interventions: FLX-787-ODT (orally disintegrating tablet); Placebo ODT


In [ ]:
# Review the automatically identified SMA exclusions before removing them.
display(
    obvious_sma_exclusions[
        [
            "nct_id",
            "brief_title",
            "conditions",
            "intervention_names"
        ]
    ].sort_values("nct_id")
)

print("Obvious exclusions:", len(obvious_sma_exclusions))

,nct_id,brief_title,conditions,intervention_names
924,NCT00067496,Modafinil to Treat Fatigue in Post-Polio Syndrome,Postpoliomyelitis Syndrome,Modafinil
874,NCT00231439,Post-Poliosyndrome Treated With Intravenous Im...,Post-Polio Syndrome,IvIg
944,NCT00303446,Dutasteride to Treat Spinal and Bulbar Muscula...,Kennedy's Disease; Spinal and Bulbar Muscular ...,Dutasteride; Placebo
848,NCT00324454,"Levetiracetam for Cramps, Spasticity and Neuro...",Motor Neuron Disease; Amyotrophic Lateral Scle...,Levetiracetam
940,NCT00365599,Phase II Trial of SAHA & Tamoxifen for Patient...,Breast Cancer,"suberoylanilide hydroxamic acid (SAHA, Vorinos..."
831,NCT00444613,A Study in Patients With Amyotrophic Lateral S...,Amyotrophic Lateral Sclerosis (ALS),E0302 (mecobalamin); E0302 (mecobalamin); Placebo
883,NCT00445172,A Long-Term Study in Patients With Amyotrophic...,Amyotrophic Lateral Sclerosis (ALS),E0302 (mecobalamin)
948,NCT00839033,Evaluation of a Mechanical Device During Acute...,Duchenne Muscular Dystrophy; Amyotrophic Later...,mechanical insufflation - exsufflation; Standa...
888,NCT00851461,Effect of Goserelin (Zoladex®) in Spinal and B...,Spinobulbar Muscular Atrophy; Kennedy's Disease,Goserelin; Electrophysiologic study; tissue bi...
851,NCT00878436,Safety and Efficacy Studies of Panobinostat an...,Prostate Cancer; Prostatic Neoplasms,Panobinostat; Bicalutamide


Obvious exclusions: 40


In [ ]:
# Finalize all manually reviewed SMA scope suspects as exclusions.
sma_scope_exclusion_ids = set(
    sma_scope_suspects["nct_id"].astype(str)
)

print("Final SMA exclusions:", len(sma_scope_exclusion_ids))
print(
    "SMA trials before cleanup:",
    (
        trials_scope_check["disease"]
        == "Spinal Muscular Atrophy"
    ).sum()
)

print(
    "Expected SMA trials after cleanup:",
    (
        trials_scope_check["disease"]
        == "Spinal Muscular Atrophy"
    ).sum()
    - len(sma_scope_exclusion_ids)
)

Final SMA exclusions: 43
SMA trials before cleanup: 148
Expected SMA trials after cleanup: 105


In [ ]:
# Save the pre-cleanup masters and the finalized SMA exclusion log.
original_trials_master = pd.read_csv(
    DATA_DIR / "trialsignal_trials_master.csv"
)

original_endpoints_master = pd.read_csv(
    DATA_DIR / "trialsignal_endpoints_master.csv"
)

original_trials_master.to_csv(
    DATA_DIR / "trialsignal_trials_master_pre_scope_cleanup.csv",
    index=False
)

original_endpoints_master.to_csv(
    DATA_DIR / "trialsignal_endpoints_master_pre_scope_cleanup.csv",
    index=False
)

sma_scope_exclusion_log = (
    sma_scope_suspects.copy()
)

sma_scope_exclusion_log["scope_decision"] = "exclude"
sma_scope_exclusion_log["scope_reason"] = (
    "Outside target spinal muscular atrophy cohort"
)

sma_scope_exclusion_log.to_csv(
    DATA_DIR / "trialsignal_sma_scope_exclusions.csv",
    index=False
)

print("Saved pre-cleanup masters and exclusion log.")

Saved pre-cleanup masters and exclusion log.


In [ ]:
# Remove the 43 out-of-scope SMA trials from the trial and endpoint masters.
trials_master_clean = original_trials_master[
    ~original_trials_master["nct_id"].isin(
        sma_scope_exclusion_ids
    )
].copy()

endpoints_master_clean = original_endpoints_master[
    ~original_endpoints_master["nct_id"].isin(
        sma_scope_exclusion_ids
    )
].copy()

print(
    "Trials:",
    len(original_trials_master),
    "->",
    len(trials_master_clean)
)

print(
    "Endpoints:",
    len(original_endpoints_master),
    "->",
    len(endpoints_master_clean)
)

print(
    "Clean SMA trials:",
    (
        trials_master_clean["disease"]
        == "Spinal Muscular Atrophy"
    ).sum()
)

print(
    "Clean SMA endpoints:",
    (
        endpoints_master_clean["disease"]
        == "Spinal Muscular Atrophy"
    ).sum()
)

Trials: 1013 -> 970
Endpoints: 8832 -> 8445
Clean SMA trials: 105
Clean SMA endpoints: 1142


In [ ]:
# Verify the scope cleanup did not create trial-endpoint inconsistencies.
remaining_excluded_trials = (
    trials_master_clean["nct_id"]
    .isin(sma_scope_exclusion_ids)
    .sum()
)

remaining_excluded_endpoints = (
    endpoints_master_clean["nct_id"]
    .isin(sma_scope_exclusion_ids)
    .sum()
)

endpoint_only_trials = set(
    endpoints_master_clean["nct_id"]
) - set(
    trials_master_clean["nct_id"]
)

trials_without_endpoints = set(
    trials_master_clean["nct_id"]
) - set(
    endpoints_master_clean["nct_id"]
)

print(
    "Excluded trials still present:",
    remaining_excluded_trials
)

print(
    "Excluded endpoints still present:",
    remaining_excluded_endpoints
)

print(
    "Endpoint-only trials:",
    len(endpoint_only_trials)
)

print(
    "Trials without endpoints:",
    len(trials_without_endpoints)
)

print(
    "Duplicate NCT IDs:",
    trials_master_clean["nct_id"].duplicated().sum()
)

Excluded trials still present: 0
Excluded endpoints still present: 0
Endpoint-only trials: 0
Trials without endpoints: 0
Duplicate NCT IDs: 0


In [ ]:
# Save the validated scope-cleaned master datasets.
trials_master_clean.to_csv(
    DATA_DIR / "trialsignal_trials_master_scope_cleaned.csv",
    index=False
)

endpoints_master_clean.to_csv(
    DATA_DIR / "trialsignal_endpoints_master_scope_cleaned.csv",
    index=False
)

print("Saved scope-cleaned master datasets.")

Saved scope-cleaned master datasets.


In [ ]:
# Reload the frozen patient-priority embeddings.
priority_embeddings = np.load(
    DATA_DIR / "trialsignal_priority_embeddings.npy"
)

priority_embedding_map = pd.read_csv(
    DATA_DIR / "trialsignal_priority_embedding_map.csv"
)

print("Priority embeddings:", priority_embeddings.shape)

Priority embeddings: (73, 384)


In [ ]:
# Recalculate priority-endpoint cosine similarities using the cleaned trial corpus.
pair_groups = []

for disease in alignment_priorities["disease"].unique():

    priority_mask = (
        alignment_priorities["disease"] == disease
    )

    endpoint_mask = (
        alignment_endpoints["disease"] == disease
    )

    disease_priorities = (
        alignment_priorities.loc[
            priority_mask
        ].reset_index(drop=True)
    )

    disease_endpoints = (
        alignment_endpoints.loc[
            endpoint_mask
        ].reset_index(drop=True)
    )

    disease_priority_embeddings = priority_embeddings[
        priority_mask.to_numpy()
    ]

    disease_endpoint_embeddings = endpoint_embeddings[
        endpoint_mask.to_numpy()
    ]

    similarity_matrix = (
        disease_priority_embeddings
        @ disease_endpoint_embeddings.T
    )

    for priority_position, priority_row in (
        disease_priorities.iterrows()
    ):

        endpoint_rows = disease_endpoints[
            [
                "nct_id",
                "endpoint_id",
                "endpoint_type",
                "endpoint_name"
            ]
        ].copy()

        endpoint_rows["disease"] = disease
        endpoint_rows["priority_id"] = (
            priority_row["priority_id"]
        )
        endpoint_rows["priority_label"] = (
            priority_row["priority_label"]
        )
        endpoint_rows["priority_domain"] = (
            priority_row["priority_domain"]
        )
        endpoint_rows["similarity"] = (
            similarity_matrix[
                priority_position
            ]
        )

        pair_groups.append(endpoint_rows)

priority_endpoint_pairs = pd.concat(
    pair_groups,
    ignore_index=True
)

print(
    "Clean priority-endpoint pairs:",
    len(priority_endpoint_pairs)
)
print(
    "Trials represented:",
    priority_endpoint_pairs["nct_id"].nunique()
)

Clean priority-endpoint pairs: 96635
Trials represented: 1013


In [ ]:
# Save the rebuilt cosine-pair table for the cleaned TrialSignal corpus.
priority_endpoint_pairs.to_csv(
    DATA_DIR / "trialsignal_priority_endpoint_pairs.csv",
    index=False
)

np.save(
    DATA_DIR / "trialsignal_endpoint_embeddings.npy",
    endpoint_embeddings
)

alignment_endpoints[
    [
        "endpoint_id",
        "disease",
        "nct_id"
    ]
].to_csv(
    DATA_DIR / "trialsignal_endpoint_embedding_map.csv",
    index=False
)

print("Saved cleaned semantic-alignment files.")

Saved cleaned semantic-alignment files.


## ARCHIVE — SMA Scope Validation (do not rerun for agent work)

In [ ]:
# Build the final manually approved SMA scope-exclusion list.
manual_ambiguous_exclusions = {
    "NCT01369901",
    "NCT05152823",
    "NCT03196375"
}

obvious_exclusion_ids = set(
    obvious_sma_exclusions["nct_id"].astype(str)
)

sma_scope_exclusion_ids = (
    obvious_exclusion_ids
    | manual_ambiguous_exclusions
)

print("Obvious exclusions:", len(obvious_exclusion_ids))
print("Manual ambiguous exclusions:", len(manual_ambiguous_exclusions))
print("Total SMA exclusions:", len(sma_scope_exclusion_ids))

Obvious exclusions: 40
Manual ambiguous exclusions: 3
Total SMA exclusions: 43


In [ ]:
# Verify that every planned exclusion exists in the current SMA trial corpus.
current_sma_ids = set(
    trials_scope_check.loc[
        trials_scope_check["disease"] == "Spinal Muscular Atrophy",
        "nct_id"
    ].astype(str)
)

missing_exclusion_ids = (
    sma_scope_exclusion_ids - current_sma_ids
)

print("Current SMA trials:", len(current_sma_ids))
print("Exclusions found in SMA cohort:", len(
    sma_scope_exclusion_ids & current_sma_ids
))
print("Exclusion IDs missing from cohort:", missing_exclusion_ids)

Current SMA trials: 148
Exclusions found in SMA cohort: 43
Exclusion IDs missing from cohort: set()


In [ ]:
# Remove out-of-scope SMA trials from the saved priority-endpoint cosine pairs.
scope_exclusion_mask_pairs = (
    (
        priority_endpoint_pairs["disease"]
        == "Spinal Muscular Atrophy"
    )
    & priority_endpoint_pairs["nct_id"].astype(str).isin(
        sma_scope_exclusion_ids
    )
)

priority_endpoint_pairs_clean = (
    priority_endpoint_pairs.loc[
        ~scope_exclusion_mask_pairs
    ]
    .copy()
    .reset_index(drop=True)
)

print(
    "Priority-endpoint pairs before:",
    len(priority_endpoint_pairs)
)

print(
    "Priority-endpoint pairs after:",
    len(priority_endpoint_pairs_clean)
)

print(
    "Trials represented after cleaning:",
    priority_endpoint_pairs_clean["nct_id"].nunique()
)

Priority-endpoint pairs before: 96635
Priority-endpoint pairs after: 91991
Trials represented after cleaning: 970


In [ ]:
# Confirm that no excluded SMA trials remain in the cleaned analysis objects.
remaining_trials = (
    set(trials_clean["nct_id"].astype(str))
    & sma_scope_exclusion_ids
)

remaining_endpoints = (
    set(endpoints_clean["nct_id"].astype(str))
    & sma_scope_exclusion_ids
)

remaining_pairs = (
    set(priority_endpoint_pairs_clean["nct_id"].astype(str))
    & sma_scope_exclusion_ids
)

print("Excluded IDs remaining in trials:", remaining_trials)
print("Excluded IDs remaining in endpoints:", remaining_endpoints)
print("Excluded IDs remaining in pairs:", remaining_pairs)

Excluded IDs remaining in trials: set()
Excluded IDs remaining in endpoints: set()
Excluded IDs remaining in pairs: set()


In [ ]:
# Save the cleaned trial, endpoint, and priority-endpoint datasets.
trials_clean.to_csv(
    DATA_DIR / "trialsignal_trials_master_clean.csv",
    index=False
)

endpoints_clean.to_csv(
    DATA_DIR / "trialsignal_endpoints_master_clean.csv",
    index=False
)

priority_endpoint_pairs_clean.to_csv(
    DATA_DIR / "trialsignal_priority_endpoint_pairs_clean.csv",
    index=False
)

print("Clean trials:", len(trials_clean))
print("Clean endpoints:", len(endpoints_clean))
print("Clean pairs:", len(priority_endpoint_pairs_clean))
print("Clean trial IDs:", trials_clean["nct_id"].nunique())

Clean trials: 970
Clean endpoints: 8445
Clean pairs: 91991
Clean trial IDs: 970


In [ ]:
# Check whether the SMA cleanup removed any manually reviewed validation cases.
removed_validation_cases = retrieval_review_units[
    retrieval_review_units["nct_id"].astype(str).isin(
        sma_scope_exclusion_ids
    )
].copy()

print(
    "Validation cases involving removed trials:",
    len(removed_validation_cases)
)

display(
    removed_validation_cases[
        [
            "disease",
            "priority_id",
            "priority_label",
            "nct_id",
            "trial_has_true_match",
            "manual_scope_issue"
        ]
    ]
)

Validation cases involving removed trials: 2


,disease,priority_id,priority_label,nct_id,trial_has_true_match,manual_scope_issue
50,Spinal Muscular Atrophy,SMA_P06,Slow or stop disease progression,NCT06411912,0,1
53,Spinal Muscular Atrophy,SMA_P11,Reduce pain,NCT03902054,0,1


In [ ]:
# Recalculate endpoint counts using only the cleaned trial corpus.
trial_endpoint_counts_clean = (
    endpoints_clean
    .groupby(
        [
            "disease",
            "nct_id"
        ]
    )
    .size()
    .reset_index(
        name="endpoint_count"
    )
)

print(
    "Trials with clean endpoint counts:",
    trial_endpoint_counts_clean["nct_id"].nunique()
)

print(
    "Missing endpoint counts:",
    trial_endpoint_counts_clean["endpoint_count"].isna().sum()
)

Trials with clean endpoint counts: 970
Missing endpoint counts: 0


In [ ]:
# Select the best cosine endpoint for each priority-trial pair in the cleaned corpus.
best_pair_indices_clean = (
    priority_endpoint_pairs_clean
    .groupby(
        [
            "disease",
            "priority_id",
            "nct_id"
        ]
    )["similarity"]
    .idxmax()
)

trial_priority_matches_clean = (
    priority_endpoint_pairs_clean
    .loc[best_pair_indices_clean]
    .copy()
    .reset_index(drop=True)
)

trial_priority_matches_clean = (
    trial_priority_matches_clean
    .merge(
        trial_endpoint_counts_clean,
        on=[
            "disease",
            "nct_id"
        ],
        how="left"
    )
)

trial_priority_matches_clean = (
    trial_priority_matches_clean
    .rename(
        columns={
            "similarity":
            "best_endpoint_similarity"
        }
    )
)

print(
    "Clean priority-trial combinations:",
    len(trial_priority_matches_clean)
)

print(
    "Missing endpoint counts:",
    trial_priority_matches_clean[
        "endpoint_count"
    ].isna().sum()
)

Clean priority-trial combinations: 10551
Missing endpoint counts: 0


In [ ]:
# Refit the empirical endpoint-count correction on the cleaned trial corpus.
from sklearn.linear_model import HuberRegressor

adjusted_groups_clean = []
count_effect_models_clean = []

for (
    disease,
    priority_id
), group in trial_priority_matches_clean.groupby(
    [
        "disease",
        "priority_id"
    ]
):

    group = group.copy()

    X = np.log1p(
        group["endpoint_count"].to_numpy()
    ).reshape(-1, 1)

    y = group[
        "best_endpoint_similarity"
    ].to_numpy()

    model = HuberRegressor()
    model.fit(X, y)

    predicted = model.predict(X)

    group[
        "expected_similarity_from_endpoint_count"
    ] = predicted

    group[
        "count_adjusted_similarity"
    ] = y - predicted

    adjusted_groups_clean.append(group)

    count_effect_models_clean.append(
        {
            "disease": disease,
            "priority_id": priority_id,
            "intercept": model.intercept_,
            "log_endpoint_count_coefficient": model.coef_[0],
            "n_trials": len(group)
        }
    )

trial_priority_matches_clean = pd.concat(
    adjusted_groups_clean,
    ignore_index=True
)

count_effect_models_clean = pd.DataFrame(
    count_effect_models_clean
)

trial_priority_matches_clean[
    "adjusted_match_percentile"
] = (
    trial_priority_matches_clean
    .groupby(
        [
            "disease",
            "priority_id"
        ]
    )["count_adjusted_similarity"]
    .rank(
        method="average",
        pct=True
    )
)

print(
    "Corrected priority-trial rows:",
    len(trial_priority_matches_clean)
)

display(
    trial_priority_matches_clean[
        [
            "disease",
            "priority_id",
            "nct_id",
            "endpoint_count",
            "best_endpoint_similarity",
            "count_adjusted_similarity",
            "adjusted_match_percentile"
        ]
    ].head()
)

Corrected priority-trial rows: 10551


,disease,priority_id,nct_id,endpoint_count,best_endpoint_similarity,count_adjusted_similarity,adjusted_match_percentile
0,Dravet Syndrome,DRAVET_P01,NCT01607073,3,0.414963,-0.028714,0.306122
1,Dravet Syndrome,DRAVET_P01,NCT02091206,3,0.234219,-0.209457,0.102041
2,Dravet Syndrome,DRAVET_P01,NCT02091375,12,0.526289,0.038933,0.816327
3,Dravet Syndrome,DRAVET_P01,NCT02174094,15,0.500806,0.005755,0.551020
4,Dravet Syndrome,DRAVET_P01,NCT02187809,7,0.461167,-0.008197,0.428571


In [ ]:
# Verify that endpoint-count bias remains near zero after cohort cleaning.
from scipy.stats import spearmanr

clean_bias_rows = []

for (
    disease,
    priority_id
), group in trial_priority_matches_clean.groupby(
    [
        "disease",
        "priority_id"
    ]
):

    rho, _ = spearmanr(
        group["endpoint_count"],
        group["adjusted_match_percentile"]
    )

    clean_bias_rows.append(
        {
            "disease": disease,
            "priority_id": priority_id,
            "spearman_rho": rho
        }
    )

clean_endpoint_count_bias = pd.DataFrame(
    clean_bias_rows
)

print(
    "Median rho:",
    clean_endpoint_count_bias[
        "spearman_rho"
    ].median()
)

print(
    "Mean rho:",
    clean_endpoint_count_bias[
        "spearman_rho"
    ].mean()
)

display(
    clean_endpoint_count_bias[
        "spearman_rho"
    ].describe()
)

Median rho: 0.005225804176839103
Mean rho: 0.0038035496678540783


,spearman_rho
count,73.000000
mean,0.003804
std,0.047928
min,-0.162405
25%,-0.017494
50%,0.005226
75%,0.029592
max,0.117086


In [ ]:
# Save the finalized scope-cleaned and count-corrected alignment outputs.
clean_scope_summary = {
    "trials_before_scope_cleanup": int(len(trials_scope_check)),
    "trials_after_scope_cleanup": int(len(trials_clean)),
    "trials_removed": int(len(trials_scope_check) - len(trials_clean)),
    "sma_trials_before": int(
        (trials_scope_check["disease"] == "Spinal Muscular Atrophy").sum()
    ),
    "sma_trials_after": int(
        (trials_clean["disease"] == "Spinal Muscular Atrophy").sum()
    ),
    "endpoints_before": int(len(alignment_endpoints)),
    "endpoints_after": int(len(endpoints_clean)),
    "priority_endpoint_pairs_before": int(len(priority_endpoint_pairs)),
    "priority_endpoint_pairs_after": int(len(priority_endpoint_pairs_clean)),
    "priority_trial_pairs_after": int(len(trial_priority_matches_clean)),
    "median_endpoint_count_rho": float(
        clean_endpoint_count_bias["spearman_rho"].median()
    ),
    "mean_endpoint_count_rho": float(
        clean_endpoint_count_bias["spearman_rho"].mean()
    ),
    "candidate_retrieval_top_k": int(retrieval_config["top_k"])
}

trial_priority_matches_clean.to_csv(
    DATA_DIR / "trialsignal_trial_priority_matches_corrected_clean.csv",
    index=False
)

count_effect_models_clean.to_csv(
    DATA_DIR / "trialsignal_endpoint_count_models_clean.csv",
    index=False
)

clean_endpoint_count_bias.to_csv(
    DATA_DIR / "trialsignal_endpoint_count_bias_clean.csv",
    index=False
)

with open(
    DATA_DIR / "trialsignal_clean_alignment_summary.json",
    "w"
) as f:
    json.dump(
        clean_scope_summary,
        f,
        indent=2
    )

print("Saved finalized scope-cleaned alignment outputs.")
print("Clean priority-trial rows:", len(trial_priority_matches_clean))

NameError: name 'endpoints_clean' is not defined

SMA scope cleanup. Manual scope review identified 43 studies assigned to the SMA cohort that did not represent the target spinal muscular atrophy population, reducing the SMA cohort from 148 to 105 trials and the full corpus from 1,013 to 970 trials. The resulting dataset contains 10,551 patient-priority × trial combinations. After refitting the empirical endpoint-count adjustment on the cleaned corpus, residual endpoint-count association remained negligible (median priority-specific Spearman rho = 0.005; mean = 0.004).

In [ ]:
# Confirm the runtime is ready for scope cleanup and agent development.
required_objects = {
    "alignment_priorities": alignment_priorities,
    "alignment_endpoints": alignment_endpoints,
    "priority_endpoint_pairs": priority_endpoint_pairs,
    "trial_endpoint_counts": trial_endpoint_counts,
    "trial_priority_matches_empirical": trial_priority_matches_empirical,
    "retrieval_review_units": retrieval_review_units,
    "retrieval_results": retrieval_results,
    "retrieval_config": retrieval_config,
    "trials_scope_check": trials_scope_check,
    "client": client
}

for name, obj in required_objects.items():
    print(name, "READY")

print("\nTop-k:", retrieval_config["top_k"])
print(
    "Within-trial rank available:",
    "within_trial_rank" in priority_endpoint_pairs.columns
)

alignment_priorities READY
alignment_endpoints READY
priority_endpoint_pairs READY
trial_endpoint_counts READY
trial_priority_matches_empirical READY
retrieval_review_units READY
retrieval_results READY
retrieval_config READY
trials_scope_check READY
client READY

Top-k: 10
Within-trial rank available: True


In [ ]:
# Build the clean top-10 candidate pool for the Semantic Match Agent.
clean_trial_ids = set(
    trial_priority_matches_corrected["nct_id"].unique()
)

priority_endpoint_pairs_agent = (
    priority_endpoint_pairs[
        priority_endpoint_pairs["nct_id"].isin(clean_trial_ids)
    ]
    .copy()
)

priority_endpoint_pairs_agent["within_trial_rank"] = (
    priority_endpoint_pairs_agent
    .groupby(
        [
            "disease",
            "priority_id",
            "nct_id"
        ]
    )["similarity"]
    .rank(
        method="first",
        ascending=False
    )
    .astype(int)
)

top10_agent_candidates = (
    priority_endpoint_pairs_agent[
        priority_endpoint_pairs_agent["within_trial_rank"] <= 10
    ]
    .copy()
    .sort_values(
        [
            "disease",
            "priority_id",
            "nct_id",
            "within_trial_rank"
        ]
    )
    .reset_index(drop=True)
)

print(
    "Clean trials:",
    priority_endpoint_pairs_agent["nct_id"].nunique()
)

print(
    "Priority-trial combinations:",
    priority_endpoint_pairs_agent[
        [
            "disease",
            "priority_id",
            "nct_id"
        ]
    ].drop_duplicates().shape[0]
)

print(
    "Top-10 candidate rows:",
    len(top10_agent_candidates)
)

NameError: name 'trial_priority_matches_corrected' is not defined

In [ ]:
# Restore the in-scope human-reviewed cases for Semantic Match Agent validation.
semantic_pilot_cases = retrieval_review_units[
    retrieval_review_units["manual_scope_issue"] != 1
].copy()

semantic_pilot_cases = semantic_pilot_cases[
    [
        "disease",
        "priority_id",
        "priority_label",
        "priority_description",
        "nct_id",
        "trial_has_true_match",
        "first_true_match_rank",
        "manual_review_notes"
    ]
].copy()

print("Semantic pilot cases:", len(semantic_pilot_cases))
print(
    "Human direct-match cases:",
    (semantic_pilot_cases["trial_has_true_match"] == 1).sum()
)
print(
    "Human no-direct-match cases:",
    (semantic_pilot_cases["trial_has_true_match"] == 0).sum()
)

## FINALIZE CLEAN CORPUS — RUN ONCE

In [ ]:
# Finalize the clean TrialSignal corpus and save the agent-ready datasets.
from google.colab import drive
from pathlib import Path
from sklearn.linear_model import HuberRegressor
from scipy.stats import spearmanr

import pandas as pd
import numpy as np
import json

drive.mount("/content/portfolio-drive")

DATA_DIR = Path(
    "./TrialSignal/data"
)

# Load the frozen patient priorities.
alignment_priorities = pd.read_json(
    DATA_DIR / "trialsignal_alignment_priorities.json"
)

# Prefer the preserved pre-cleanup masters when available.
trials_source = (
    DATA_DIR / "trialsignal_trials_master_pre_scope_cleanup.csv"
)

endpoints_source = (
    DATA_DIR / "trialsignal_endpoints_master_pre_scope_cleanup.csv"
)

if not trials_source.exists():
    trials_source = (
        DATA_DIR / "trialsignal_trials_master.csv"
    )

if not endpoints_source.exists():
    endpoints_source = (
        DATA_DIR / "trialsignal_endpoints_master.csv"
    )

trials_raw = pd.read_csv(
    trials_source
)

endpoints_raw = pd.read_csv(
    endpoints_source
)

priority_endpoint_pairs_raw = pd.read_csv(
    DATA_DIR / "trialsignal_priority_endpoint_pairs.csv"
)

scope_exclusions = pd.read_csv(
    DATA_DIR / "trialsignal_sma_scope_exclusions.csv"
)

retrieval_review_units = pd.read_csv(
    DATA_DIR / "trialsignal_retrieval_review_units_final.csv"
)

retrieval_results = pd.read_csv(
    DATA_DIR / "trialsignal_retrieval_validation_results.csv"
)

with open(
    DATA_DIR / "trialsignal_candidate_retrieval_config.json",
    "r"
) as f:
    retrieval_config = json.load(f)


# Apply the finalized 43-trial SMA scope exclusion.
sma_scope_exclusion_ids = set(
    scope_exclusions["nct_id"]
    .astype(str)
)

trials_clean = (
    trials_raw[
        ~trials_raw["nct_id"]
        .astype(str)
        .isin(sma_scope_exclusion_ids)
    ]
    .copy()
    .reset_index(drop=True)
)

endpoints_clean = (
    endpoints_raw[
        ~endpoints_raw["nct_id"]
        .astype(str)
        .isin(sma_scope_exclusion_ids)
    ]
    .copy()
    .reset_index(drop=True)
)

priority_endpoint_pairs_clean = (
    priority_endpoint_pairs_raw[
        ~priority_endpoint_pairs_raw["nct_id"]
        .astype(str)
        .isin(sma_scope_exclusion_ids)
    ]
    .copy()
    .reset_index(drop=True)
)


# Recreate stable endpoint identifiers and semantic text.
endpoints_clean["endpoint_number"] = pd.to_numeric(
    endpoints_clean["endpoint_number"]
).astype(int)

endpoints_clean["endpoint_id"] = (
    endpoints_clean["nct_id"].astype(str)
    + "_"
    + endpoints_clean["endpoint_type"]
    .str[0]
    .str.upper()
    + endpoints_clean["endpoint_number"]
    .astype(str)
)

endpoints_clean["endpoint_semantic_text"] = (
    endpoints_clean["endpoint_text"]
    .fillna("")
    .astype(str)
    .str.strip()
)


# Recalculate endpoint counts in the clean corpus.
trial_endpoint_counts_clean = (
    endpoints_clean
    .groupby(
        [
            "disease",
            "nct_id"
        ]
    )
    .size()
    .reset_index(
        name="endpoint_count"
    )
)


# Select the best cosine endpoint for every priority-trial pair.
best_pair_indices = (
    priority_endpoint_pairs_clean
    .groupby(
        [
            "disease",
            "priority_id",
            "nct_id"
        ]
    )["similarity"]
    .idxmax()
)

trial_priority_matches_clean = (
    priority_endpoint_pairs_clean
    .loc[best_pair_indices]
    .copy()
    .reset_index(drop=True)
)

trial_priority_matches_clean = (
    trial_priority_matches_clean
    .merge(
        trial_endpoint_counts_clean,
        on=[
            "disease",
            "nct_id"
        ],
        how="left"
    )
    .rename(
        columns={
            "similarity":
            "best_endpoint_similarity"
        }
    )
)


# Refit the accepted empirical endpoint-count correction.
adjusted_groups = []
count_effect_models_clean = []

for (
    disease,
    priority_id
), group in trial_priority_matches_clean.groupby(
    [
        "disease",
        "priority_id"
    ]
):

    group = group.copy()

    X = np.log1p(
        group["endpoint_count"]
        .to_numpy()
    ).reshape(-1, 1)

    y = group[
        "best_endpoint_similarity"
    ].to_numpy()

    model = HuberRegressor()
    model.fit(X, y)

    expected = model.predict(X)

    group[
        "expected_similarity_from_endpoint_count"
    ] = expected

    group[
        "count_adjusted_similarity"
    ] = y - expected

    adjusted_groups.append(group)

    count_effect_models_clean.append(
        {
            "disease": disease,
            "priority_id": priority_id,
            "intercept": float(
                model.intercept_
            ),
            "log_endpoint_count_coefficient": float(
                model.coef_[0]
            ),
            "n_trials": len(group)
        }
    )

trial_priority_matches_clean = pd.concat(
    adjusted_groups,
    ignore_index=True
)

count_effect_models_clean = pd.DataFrame(
    count_effect_models_clean
)

trial_priority_matches_clean[
    "adjusted_match_percentile"
] = (
    trial_priority_matches_clean
    .groupby(
        [
            "disease",
            "priority_id"
        ]
    )["count_adjusted_similarity"]
    .rank(
        method="average",
        pct=True
    )
)


# Recheck endpoint-count bias after scope cleanup.
bias_rows = []

for (
    disease,
    priority_id
), group in trial_priority_matches_clean.groupby(
    [
        "disease",
        "priority_id"
    ]
):

    if group["endpoint_count"].nunique() > 1:
        rho, _ = spearmanr(
            group["endpoint_count"],
            group["adjusted_match_percentile"]
        )
    else:
        rho = np.nan

    bias_rows.append(
        {
            "disease": disease,
            "priority_id": priority_id,
            "spearman_rho": rho
        }
    )

clean_endpoint_count_bias = pd.DataFrame(
    bias_rows
)


# Rank endpoints within each priority-trial for agent retrieval.
priority_endpoint_pairs_clean[
    "within_trial_rank"
] = (
    priority_endpoint_pairs_clean
    .groupby(
        [
            "disease",
            "priority_id",
            "nct_id"
        ]
    )["similarity"]
    .rank(
        method="first",
        ascending=False
    )
    .astype(int)
)


# Build the frozen top-10 Candidate Retrieval Agent input.
top10_agent_candidates = (
    priority_endpoint_pairs_clean[
        priority_endpoint_pairs_clean[
            "within_trial_rank"
        ] <= retrieval_config["top_k"]
    ]
    .copy()
)

priority_details = (
    alignment_priorities[
        [
            "priority_id",
            "priority_description"
        ]
    ]
    .drop_duplicates(
        "priority_id"
    )
)

endpoint_details = (
    endpoints_clean[
        [
            "endpoint_id",
            "endpoint_semantic_text"
        ]
    ]
    .drop_duplicates(
        "endpoint_id"
    )
)

trial_details = (
    trials_clean[
        [
            "disease",
            "nct_id",
            "brief_title",
            "official_title",
            "conditions"
        ]
    ]
    .drop_duplicates(
        [
            "disease",
            "nct_id"
        ]
    )
)

candidate_strength = (
    trial_priority_matches_clean[
        [
            "disease",
            "priority_id",
            "nct_id",
            "best_endpoint_similarity",
            "count_adjusted_similarity",
            "adjusted_match_percentile"
        ]
    ]
)

top10_agent_candidates = (
    top10_agent_candidates
    .merge(
        priority_details,
        on="priority_id",
        how="left"
    )
    .merge(
        endpoint_details,
        on="endpoint_id",
        how="left"
    )
    .merge(
        trial_details,
        on=[
            "disease",
            "nct_id"
        ],
        how="left"
    )
    .merge(
        candidate_strength,
        on=[
            "disease",
            "priority_id",
            "nct_id"
        ],
        how="left"
    )
    .sort_values(
        [
            "disease",
            "priority_id",
            "nct_id",
            "within_trial_rank"
        ]
    )
    .reset_index(drop=True)
)


# Preserve the in-scope human-reviewed cases for agent validation.
semantic_pilot_cases = (
    retrieval_review_units[
        (
            retrieval_review_units[
                "manual_scope_issue"
            ] != 1
        )
        & retrieval_review_units[
            "nct_id"
        ].astype(str).isin(
            set(
                trials_clean[
                    "nct_id"
                ].astype(str)
            )
        )
    ]
    .copy()
)


# Verify the final corpus before saving.
assert len(trials_clean) == 970
assert len(endpoints_clean) == 8445
assert len(priority_endpoint_pairs_clean) == 91991
assert (
    trials_clean[
        "disease"
    ].eq(
        "Spinal Muscular Atrophy"
    ).sum()
    == 105
)
assert len(trial_priority_matches_clean) == 10551
assert retrieval_config["top_k"] == 10
assert len(semantic_pilot_cases) == 54


# Save the definitive agent-ready datasets.
trials_clean.to_csv(
    DATA_DIR / "trialsignal_trials_master_clean.csv",
    index=False
)

endpoints_clean.to_csv(
    DATA_DIR / "trialsignal_endpoints_master_clean.csv",
    index=False
)

priority_endpoint_pairs_clean.to_csv(
    DATA_DIR / "trialsignal_priority_endpoint_pairs_clean.csv",
    index=False
)

trial_priority_matches_clean.to_csv(
    DATA_DIR / "trialsignal_trial_priority_matches_corrected_clean.csv",
    index=False
)

count_effect_models_clean.to_csv(
    DATA_DIR / "trialsignal_endpoint_count_models_clean.csv",
    index=False
)

clean_endpoint_count_bias.to_csv(
    DATA_DIR / "trialsignal_endpoint_count_bias_clean.csv",
    index=False
)

top10_agent_candidates.to_csv(
    DATA_DIR / "trialsignal_top10_agent_candidates_clean.csv",
    index=False
)

semantic_pilot_cases.to_csv(
    DATA_DIR / "trialsignal_semantic_pilot_cases.csv",
    index=False
)


# Save a compact record of the frozen analysis state.
agent_state_summary = {
    "trials": len(trials_clean),
    "endpoints": len(endpoints_clean),
    "priority_endpoint_pairs": len(
        priority_endpoint_pairs_clean
    ),
    "priority_trial_pairs": len(
        trial_priority_matches_clean
    ),
    "sma_trials": int(
        (
            trials_clean["disease"]
            == "Spinal Muscular Atrophy"
        ).sum()
    ),
    "scope_exclusions": len(
        sma_scope_exclusion_ids
    ),
    "candidate_retrieval_top_k": int(
        retrieval_config["top_k"]
    ),
    "semantic_pilot_cases": len(
        semantic_pilot_cases
    ),
    "median_endpoint_count_rho": float(
        clean_endpoint_count_bias[
            "spearman_rho"
        ].median()
    ),
    "mean_endpoint_count_rho": float(
        clean_endpoint_count_bias[
            "spearman_rho"
        ].mean()
    )
}

with open(
    DATA_DIR / "trialsignal_agent_state_summary.json",
    "w"
) as f:
    json.dump(
        agent_state_summary,
        f,
        indent=2
    )

print("FINAL AGENT-READY STATE SAVED")
print(json.dumps(agent_state_summary, indent=2))

Drive already mounted at /content/portfolio-drive; to attempt to forcibly remount, call drive.mount("/content/portfolio-drive", force_remount=True).
FINAL AGENT-READY STATE SAVED
{
  "trials": 970,
  "endpoints": 8445,
  "priority_endpoint_pairs": 91991,
  "priority_trial_pairs": 10551,
  "sma_trials": 105,
  "scope_exclusions": 43,
  "candidate_retrieval_top_k": 10,
  "semantic_pilot_cases": 54,
  "median_endpoint_count_rho": 0.005225804176839103,
  "mean_endpoint_count_rho": 0.0038035496678540783
}


## START HERE — AGENT DEVELOPMENT (RUN THIS CELL ONLY)
You should only need to run this one cell to get started!
Tell your LLM that this is the master-cell for agentic development. There are dead/ confusing variable names in the code above this!

In [ ]:
# ================================================================
# TRIALSIGNAL — MASTER AGENT RECONNECT v4.0
# ONE CELL: RUN AFTER ANY GOOGLE COLAB RUNTIME RESET
# Restores the FROZEN clean corpus + Semantic Agent V3 from Drive.
# No API calls, no embedding work, no data recomputation, no overwrites.
# ================================================================

from pathlib import Path
import importlib.util
import json
import subprocess
import sys
import pandas as pd

# 1) Drive is the authoritative source. Never fall back to older files.
DATA_DIR = Path('./TrialSignal/data')
if not DATA_DIR.is_dir():
    try:
        from google.colab import drive
        drive.mount('/content/portfolio-drive')
    except Exception as exc:
        raise RuntimeError('Google Drive mount failed. Reconnect Drive and rerun this cell.') from exc
if not DATA_DIR.is_dir():
    raise FileNotFoundError(f'TrialSignal Drive data folder not found: {DATA_DIR}')

# 2) Explicit frozen files. Missing files are fatal, never substituted.
REQUIRED = {
    'alignment_priorities': 'trialsignal_alignment_priorities.csv',
    'trials_master': 'trialsignal_trials_master_clean.csv',
    'alignment_endpoints': 'trialsignal_endpoints_master_clean.csv',
    'priority_endpoint_pairs': 'trialsignal_priority_endpoint_pairs_clean.csv',
    'trial_priority_matches_empirical': 'trialsignal_trial_priority_matches_corrected_clean.csv',
    'agent_candidate_pool': 'trialsignal_top10_agent_candidates_clean.csv',
    'semantic_pilot_cases': 'trialsignal_semantic_pilot_cases.csv',
    'count_effect_models': 'trialsignal_endpoint_count_models_clean.csv',
    'empirical_endpoint_count_bias': 'trialsignal_endpoint_count_bias_clean.csv',
    'retrieval_review_units': 'trialsignal_retrieval_review_units_final.csv',
    'retrieval_results': 'trialsignal_retrieval_validation_results.csv',
    'retrieval_config': 'trialsignal_candidate_retrieval_config.json',
    'agent_state_summary': 'trialsignal_agent_state_summary.json',
}
missing_files = [name for name in REQUIRED.values() if not (DATA_DIR / name).is_file()]
if missing_files:
    raise FileNotFoundError('Missing REQUIRED frozen files in Drive:\n' + '\n'.join(missing_files))
for variable, filename in REQUIRED.items():
    path = DATA_DIR / filename
    if path.suffix == '.json':
        with path.open('r', encoding='utf-8') as f:
            globals()[variable] = json.load(f)
    else:
        globals()[variable] = pd.read_csv(path)

# 3) Validate file versions, joins, shape, and identifiers.
EXPECTED = {
    'trials_master': 970,
    'alignment_endpoints': 8445,
    'priority_endpoint_pairs': 91991,
    'trial_priority_matches_empirical': 10551,
    'agent_candidate_pool': 63811,
    'semantic_pilot_cases': 54,
    'alignment_priorities': 73,
}
for name, expected in EXPECTED.items():
    actual = len(globals()[name])
    if actual != expected:
        raise AssertionError(f'FROZEN CORPUS MISMATCH: {name} is {actual}; expected {expected}. STOP.')
for field, expected in [('trials', 970), ('endpoints', 8445),
                        ('priority_endpoint_pairs', 91991), ('priority_trial_pairs', 10551),
                        ('semantic_pilot_cases', 54), ('candidate_retrieval_top_k', 10)]:
    if agent_state_summary[field] != expected:
        raise AssertionError(f'Frozen state summary mismatch: {field}. STOP.')
TOP_K = int(retrieval_config['top_k'])
if TOP_K != 10:
    raise AssertionError(f'TOP_K={TOP_K}; frozen retrieval must use TOP_K=10.')

CASE_KEYS = ['disease', 'priority_id', 'nct_id']
ENDPOINT_KEYS = CASE_KEYS + ['endpoint_id']
for name, frame, keys in [
    ('trials', trials_master, ['disease', 'nct_id']),
    ('endpoints', alignment_endpoints, ['disease', 'nct_id', 'endpoint_id']),
    ('priorities', alignment_priorities, ['disease', 'priority_id']),
    ('priority/trial matches', trial_priority_matches_empirical, CASE_KEYS),
    ('pilot cases', semantic_pilot_cases, CASE_KEYS),
    ('candidate endpoints', agent_candidate_pool, ENDPOINT_KEYS),
]:
    missing_cols = set(keys).difference(frame.columns)
    if missing_cols:
        raise AssertionError(f'{name} missing key columns: {sorted(missing_cols)}')
    if frame.duplicated(keys).any():
        raise AssertionError(f'{name} has duplicate identifiers for {keys}. STOP.')

NEEDED_CANDIDATE_COLS = {
    'priority_label', 'priority_description', 'endpoint_name',
    'endpoint_type', 'endpoint_semantic_text', 'within_trial_rank'
}
if NEEDED_CANDIDATE_COLS.difference(agent_candidate_pool.columns):
    raise AssertionError('Candidate pool is missing semantic-input columns.')
if agent_candidate_pool[list(NEEDED_CANDIDATE_COLS)].isna().any().any():
    raise AssertionError('Candidate pool has missing semantic-input values.')
if not agent_candidate_pool['within_trial_rank'].between(1, 10).all():
    raise AssertionError('Candidate ranks fall outside 1..10.')
if agent_candidate_pool.duplicated(CASE_KEYS + ['within_trial_rank']).any():
    raise AssertionError('Duplicate candidate ranks within priority/trial.')
if agent_candidate_pool[CASE_KEYS].drop_duplicates().shape[0] != 10551:
    raise AssertionError('Candidate pool does not cover all 10,551 priority/trial cases.')
if (semantic_pilot_cases['trial_has_true_match'].eq(1).sum() != 16 or
        semantic_pilot_cases['trial_has_true_match'].eq(0).sum() != 38):
    raise AssertionError('The 54-case human-review benchmark has changed.')

# Core aliases used by earlier agent-development code. No re-filtering.
agent_trials = trials_master
agent_endpoints = alignment_endpoints
benchmark54 = semantic_pilot_cases

def get_trial_candidates(disease, priority_id, nct_id, top_k=None):
    """Return only the precomputed, frozen retrieval candidates."""
    k = TOP_K if top_k is None else int(top_k)
    if not 1 <= k <= TOP_K:
        raise ValueError(f'top_k must be between 1 and {TOP_K}')
    return (agent_candidate_pool.loc[
        (agent_candidate_pool['disease'] == disease) &
        (agent_candidate_pool['priority_id'] == priority_id) &
        (agent_candidate_pool['nct_id'] == nct_id)
    ].sort_values('within_trial_rank').head(k).copy())

# 4) Reconstruct V3 agent input from authoritative candidate + priority evidence.
# Do not merge duplicate priority_label fields: avoids _x / _y suffix bugs.
evidence_lookup = alignment_priorities[
    ['disease', 'priority_id', 'evidence_text']
].drop_duplicates(['disease', 'priority_id'])
semantic_agent_inputs = agent_candidate_pool.merge(
    evidence_lookup,
    on=['disease', 'priority_id'],
    how='left',
    validate='many_to_one'
)
if len(semantic_agent_inputs) != 63811:
    raise AssertionError('Evidence join changed candidate row count.')
if semantic_agent_inputs['evidence_text'].isna().any():
    raise AssertionError('Patient source evidence is missing for some candidates.')

# 5) Restore the EXACT validated semantic V3 classifier, without an API request.
LLM_MODEL = 'gpt-6-luna'
SEMANTIC_PROMPT_VERSION = 'semantic_v3_constituent'
SEMANTIC_RULES = """
You are the Semantic Match Agent for TrialSignal.

Your task is to determine whether one registered clinical-trial
endpoint actually MEASURES a validated patient priority.

Judge the measurement relationship, not merely clinical relevance.

CLASSIFICATION RULES

DIRECT:
The endpoint explicitly measures the patient priority or a
clearly identifiable constituent outcome of that priority.

A broad patient priority can contain several meaningful outcomes.
Direct measurement of one explicitly included outcome is sufficient
for a direct match. The endpoint does NOT need to measure every
dimension of the broader priority.

The relevant outcome must actually be measured by the endpoint.
If a multi-domain instrument reports the relevant domain
separately, that domain can qualify as direct.

PARTIAL:
The endpoint measures a meaningful indirect indicator, proxy,
related functional outcome, or broader composite that is relevant
to the priority without separately establishing its measurement.

Clinical relationships alone do not establish direct measurement.
A functional or physiological indicator is not automatically a
direct measure of organ damage, quality of life, independence,
pain, or another patient-priority outcome.

If an instrument contains relevant items but the registered
endpoint reports only a broader composite, consider whether
the target outcome is actually measured separately.

NO_MATCH:
The endpoint does not meaningfully measure the priority.

A shared disease context, general treatment benefit, or
biological association alone is insufficient for alignment.

IMPORTANT DISTINCTIONS

- Direct does not require complete coverage of every dimension
  of a broad priority.
- Partial means indirect or incomplete evidence of measurement,
  not simply that some other dimensions were omitted.
- Do not automatically classify every medically related endpoint
  as partial.
- Judge only the supplied endpoint, not other endpoints in
  the same trial.
- Do not infer measurements that are not specified.
- Do not assume an endpoint establishes treatment effectiveness
  simply because it measures an outcome.
- Do not use cosine similarity or retrieval rank as evidence.
- Do not evaluate disease, population, or cohort applicability.
  That responsibility belongs to the Scope Agent.

OUTPUT

Return exactly one of: direct, partial, no_match.

Give a concise explanation identifying what the endpoint
actually measures and how that relates to the patient priority.

Return confidence between 0 and 1. Confidence should reflect
the clarity of the supplied measurement evidence, not general
clinical familiarity. Do not express unwarranted certainty.
"""

SEMANTIC_SCHEMA = {
    'type': 'object',
    'properties': {
        'semantic_label': {'type': 'string', 'enum': ['direct', 'partial', 'no_match']},
        'confidence': {'type': 'number'},
        'reason': {'type': 'string'},
    },
    'required': ['semantic_label', 'confidence', 'reason'],
    'additionalProperties': False,
}

# API is optional for data restoration. A missing secret must be explicit.
if importlib.util.find_spec('openai') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'openai'], check=True)
from openai import OpenAI
client = None
api_status = 'NOT READY: OPENAI_API_KEY unavailable in Colab Secrets'
try:
    from google.colab import userdata
    api_key = userdata.get('OPENAI_API_KEY')
except Exception:
    api_key = None
if api_key:
    client = OpenAI(api_key=api_key)
    api_status = 'READY (client initialized; network not tested)'
# Do not display, write, or persist API keys.
del api_key

def semantic_match_candidate(row):
    """Grounded, strict-json V3 semantic judgment for ONE endpoint."""
    if client is None:
        raise RuntimeError('OpenAI client unavailable. Set OPENAI_API_KEY in Colab Secrets.')
    fields = ['disease', 'priority_label', 'priority_description',
              'evidence_text', 'endpoint_name', 'endpoint_type', 'endpoint_semantic_text']
    missing = [c for c in fields if c not in row.index or pd.isna(row[c])]
    if missing:
        raise ValueError(f'Candidate missing required text fields: {missing}')
    payload = {
        'disease': str(row['disease']),
        'priority': str(row['priority_label']),
        'priority_definition': str(row['priority_description']),
        'patient_evidence': str(row['evidence_text'])[:1500],
        'endpoint_name': str(row['endpoint_name']),
        'endpoint_type': str(row['endpoint_type']),
        'endpoint_text': str(row['endpoint_semantic_text']),
    }
    response = client.responses.create(
        model=LLM_MODEL,
        instructions=SEMANTIC_RULES,
        input=json.dumps(payload, ensure_ascii=False),
        text={'format': {'type': 'json_schema', 'name': 'trialsignal_semantic_match',
                         'strict': True, 'schema': SEMANTIC_SCHEMA}},
    )
    if not response.output_text:
        raise ValueError('No semantic judgment returned from the API.')
    parsed = json.loads(response.output_text)
    if parsed.get('semantic_label') not in {'direct', 'partial', 'no_match'}:
        raise ValueError('Unexpected semantic label.')
    confidence = float(parsed['confidence'])
    if not 0 <= confidence <= 1:
        raise ValueError('Confidence outside 0..1.')
    if not isinstance(parsed['reason'], str) or not parsed['reason'].strip():
        raise ValueError('Missing semantic reasoning.')
    return {
        'disease': row['disease'],
        'priority_id': row['priority_id'],
        'nct_id': row['nct_id'],
        'endpoint_id': row['endpoint_id'],
        'candidate_rank': int(row['within_trial_rank']),
        'semantic_label': parsed['semantic_label'],
        'confidence': confidence,
        'reason': parsed['reason'],
        'model': LLM_MODEL,
        'prompt_version': SEMANTIC_PROMPT_VERSION,
    }

def result_key(row):
    return tuple(str(row[k]) for k in ENDPOINT_KEYS)

# 6) Restore optional V2/V3 SAVED RESULTS. Never regenerate on reconnect.
OPTIONAL_RESULTS = {
    'pilot_results_v2': 'trialsignal_semantic_v2_pilot14.csv',
    'v3_results': 'trialsignal_semantic_v3_pilot14.csv',
    'heldout_results': 'trialsignal_semantic_v3_heldout40.csv',
    'semantic_search_trace': 'trialsignal_semantic_v3_search_trace.csv',
    'semantic_search5_summary': 'trialsignal_semantic_v3_search5_summary.csv',
}
restored_result_counts = {}
for variable, filename in OPTIONAL_RESULTS.items():
    path = DATA_DIR / filename
    globals()[variable] = pd.read_csv(path) if path.is_file() else pd.DataFrame()
    restored_result_counts[filename] = len(globals()[variable]) if path.is_file() else 'not saved'

# 7) Reconstruct cache of successful V3 endpoint judgments.
SEMANTIC_CACHE = {}
for name in ['v3_results', 'heldout_results', 'semantic_search_trace']:
    frame = globals()[name]
    required = set(ENDPOINT_KEYS + ['agent_label', 'model', 'prompt_version'])
    if not frame.empty and required.issubset(frame.columns):
        for rec in frame.to_dict('records'):
            if (rec['prompt_version'] == SEMANTIC_PROMPT_VERSION and
                    rec['model'] == LLM_MODEL and
                    rec['agent_label'] in {'direct', 'partial', 'no_match'}):
                SEMANTIC_CACHE[result_key(rec)] = rec

# 8) Restore the semantic-only top-10 search helper and safe checkpoints.
# Scope is deliberately NOT adjudicated here; this only produces provisional labels.
def semantic_scan_trial(disease, priority_id, nct_id):
    """Search frozen top-10 sequentially; reuse V3 cache; save new judgments."""
    global semantic_search_trace
    candidates = get_trial_candidates(disease, priority_id, nct_id)
    if candidates.empty:
        raise ValueError('No retrieved candidates for this priority/trial.')
    selected = None
    first_partial = None
    inspected = 0
    stop_reason = None
    for _, row in candidates.iterrows():
        key = result_key(row)
        result = SEMANTIC_CACHE.get(key)
        from_cache = result is not None
        if result is None:
            # Fail openly on API errors. Do NOT mark an incomplete search no_match.
            fresh = semantic_match_candidate(semantic_agent_inputs.loc[
                (semantic_agent_inputs['disease'] == row['disease']) &
                (semantic_agent_inputs['priority_id'] == row['priority_id']) &
                (semantic_agent_inputs['nct_id'] == row['nct_id']) &
                (semantic_agent_inputs['endpoint_id'] == row['endpoint_id'])
            ].iloc[0])
            result = dict(fresh)
            result['agent_label'] = result.pop('semantic_label')
            SEMANTIC_CACHE[key] = result
            semantic_search_trace = pd.concat(
                [semantic_search_trace, pd.DataFrame([result])],
                ignore_index=True
            ).drop_duplicates(ENDPOINT_KEYS, keep='last')
            semantic_search_trace.to_csv(
                DATA_DIR / 'trialsignal_semantic_v3_search_trace.csv', index=False
            )
        label = result['agent_label']
        inspected += 1
        print(f'  Rank {int(row["within_trial_rank"])}: {label} '
              f'({"cache" if from_cache else "API"})')
        if label == 'direct':
            selected = result
            stop_reason = 'direct_found_scope_pending'
            break
        if label == 'partial' and first_partial is None:
            first_partial = result

    if selected is not None:
        final_label = 'direct'
    elif first_partial is not None:
        selected = first_partial
        final_label = 'partial'
        stop_reason = 'candidates_exhausted_partial'
    else:
        final_label = 'no_match'
        stop_reason = 'candidates_exhausted_no_match'
    return {
        'disease': disease, 'priority_id': priority_id, 'nct_id': nct_id,
        'provisional_label': final_label,
        'selected_endpoint_id': selected['endpoint_id'] if selected else None,
        'selected_rank': int(selected['candidate_rank']) if selected else None,
        'candidates_checked': inspected,
        'stop_reason': stop_reason,
        'scope_status': 'not_evaluated',
    }

# 9) Reusable, NO-API partial-match audit of the saved V3 search trace.
def get_partial_match_audit():
    """Return complete text/reasons for all saved V3 partials; no new model calls."""
    if semantic_search_trace.empty:
        return pd.DataFrame()
    needed = set(ENDPOINT_KEYS + ['agent_label', 'prompt_version',
                                  'candidate_rank', 'reason', 'confidence'])
    if not needed.issubset(semantic_search_trace.columns):
        raise ValueError('Saved V3 search trace has unexpected columns.')
    partial = semantic_search_trace.loc[
        (semantic_search_trace['prompt_version'] == SEMANTIC_PROMPT_VERSION) &
        (semantic_search_trace['agent_label'] == 'partial')
    ].drop_duplicates(ENDPOINT_KEYS, keep='last').copy()
    context_cols = ['priority_label', 'priority_description',
                    'endpoint_name', 'endpoint_semantic_text']
    partial = partial.drop(columns=context_cols, errors='ignore')
    audit = partial.merge(
        agent_candidate_pool[ENDPOINT_KEYS + context_cols],
        on=ENDPOINT_KEYS, how='left', validate='one_to_one'
    )
    if audit[context_cols].isna().any().any():
        raise ValueError('Saved partial trace does not join cleanly to frozen candidates.')
    return audit.sort_values(CASE_KEYS + ['candidate_rank']).reset_index(drop=True)

# 10) Minimal diagnostic: demonstrate actual frozen identifiers/rows.
known = get_trial_candidates('Dravet Syndrome', 'DRAVET_P02', 'NCT02187809')
if known.empty or known.iloc[0]['endpoint_id'] != 'NCT02187809_P4':
    raise AssertionError('Known Dravet retrieval case failed. STOP.')
if len(SEMANTIC_CACHE) and not set(SEMANTIC_CACHE).issubset(
        set(map(tuple, semantic_agent_inputs[ENDPOINT_KEYS].astype(str).itertuples(index=False, name=None)))):
    raise AssertionError('Saved V3 cache contains endpoint identifiers absent from frozen pool.')

print('\n' + '=' * 64)
print('TRIALSIGNAL — MASTER AGENT RECONNECT v4.0')
print('=' * 64)
for name in EXPECTED:
    print(f'{name}: {len(globals()[name]):,}  [verified]')
print(f'Top_k: {TOP_K}  [verified]')
print(f'Validated Dravet candidate: {known.iloc[0]["endpoint_id"]}')
print(f'V3 semantic prompt: {SEMANTIC_PROMPT_VERSION}')
print(f'Model: {LLM_MODEL}')
print('API: ' + api_status)
print('Saved semantic outputs:')
for file, count in restored_result_counts.items():
    print(f'  {file}: {count}')
print(f'Cached successful V3 endpoint judgments: {len(SEMANTIC_CACHE)}')
print('DATA READY: cleaned corpus + V3 inputs + benchmark + cached results')
if client is None:
    print('SEMANTIC API NOT READY: provide the Colab secret before making new calls.')
else:
    print('SEMANTIC CLIENT INITIALIZED: no API request was made by reconnect.')


Mounted at /content/portfolio-drive

TRIALSIGNAL — MASTER AGENT RECONNECT v4.0
trials_master: 970  [verified]
alignment_endpoints: 8,445  [verified]
priority_endpoint_pairs: 91,991  [verified]
trial_priority_matches_empirical: 10,551  [verified]
agent_candidate_pool: 63,811  [verified]
semantic_pilot_cases: 54  [verified]
alignment_priorities: 73  [verified]
Top_k: 10  [verified]
Validated Dravet candidate: NCT02187809_P4
V3 semantic prompt: semantic_v3_constituent
Model: gpt-6-luna
API: READY (client initialized; network not tested)
Saved semantic outputs:
  trialsignal_semantic_v2_pilot14.csv: 14
  trialsignal_semantic_v3_pilot14.csv: 14
  trialsignal_semantic_v3_heldout40.csv: 40
  trialsignal_semantic_v3_search_trace.csv: 36
  trialsignal_semantic_v3_search5_summary.csv: 5
Cached successful V3 endpoint judgments: 85
DATA READY: cleaned corpus + V3 inputs + benchmark + cached results
SEMANTIC CLIENT INITIALIZED: no API request was made by reconnect.


In [ ]:

# Restore the FINAL frozen corpus for agent development.
# Temporary correction to the existing reconnect cell.

trials_master = pd.read_csv(
    DATA_DIR / "trialsignal_trials_master_clean.csv"
)

alignment_endpoints = pd.read_csv(
    DATA_DIR / "trialsignal_endpoints_master_clean.csv"
)

priority_endpoint_pairs = pd.read_csv(
    DATA_DIR / "trialsignal_priority_endpoint_pairs_clean.csv"
)

trial_priority_matches_empirical = pd.read_csv(
    DATA_DIR / "trialsignal_trial_priority_matches_corrected_clean.csv"
)

agent_candidate_pool = pd.read_csv(
    DATA_DIR / "trialsignal_top10_agent_candidates_clean.csv"
)

semantic_pilot_cases = pd.read_csv(
    DATA_DIR / "trialsignal_semantic_pilot_cases.csv"
)

agent_trials = trials_master.copy()
agent_endpoints = alignment_endpoints.copy()

TOP_K = 10

def get_trial_candidates(disease, priority_id, nct_id, top_k=TOP_K):
    return (
        agent_candidate_pool.loc[
            (agent_candidate_pool["disease"] == disease)
            & (agent_candidate_pool["priority_id"] == priority_id)
            & (agent_candidate_pool["nct_id"] == nct_id)
        ]
        .sort_values("within_trial_rank")
        .head(top_k)
        .copy()
    )

assert (
    len(trials_master),
    len(alignment_endpoints),
    len(priority_endpoint_pairs),
    len(trial_priority_matches_empirical),
    len(agent_candidate_pool),
    len(semantic_pilot_cases)
) == (970, 8445, 91991, 10551, 63811, 54)

print("FINAL CLEAN CORPUS VERIFIED")
print("Trials:", len(trials_master))
print("Endpoints:", len(alignment_endpoints))
print("Priority-trial pairs:", len(trial_priority_matches_empirical))
print("Top-10 candidate rows:", len(agent_candidate_pool))
print("Semantic pilot cases:", len(semantic_pilot_cases))


FINAL CLEAN CORPUS VERIFIED
Trials: 970
Endpoints: 8445
Priority-trial pairs: 10551
Top-10 candidate rows: 63811
Semantic pilot cases: 54


## SEMANTIC MATCH AGENT

In [ ]:
# Define the grounded Semantic Match Agent and its strict three-class output.
def parse_agent_json(raw_text):
    cleaned = raw_text.strip()

    cleaned = re.sub(
        r"^```(?:json)?\s*",
        "",
        cleaned,
        flags=re.IGNORECASE
    )

    cleaned = re.sub(
        r"\s*```$",
        "",
        cleaned
    )

    return json.loads(cleaned)


def semantic_match_agent(
    priority_label,
    priority_description,
    priority_evidence,
    endpoint_name,
    endpoint_text
):
    evidence_text = str(priority_evidence or "")

    if len(evidence_text) > 4000:
        evidence_text = evidence_text[:4000]

    prompt = f"""
You are the Semantic Match Agent in TrialSignal.

YOUR ONLY JOB:
Determine whether a clinical-trial endpoint measures a validated patient priority.

Do not judge whether the trial population, disease cohort, or endpoint population is in scope.
A separate Scope Agent handles that.

PATIENT PRIORITY:
Label: {priority_label}
Definition: {priority_description}

PATIENT EVIDENCE:
{evidence_text}

CLINICAL-TRIAL ENDPOINT:
Name: {endpoint_name}
Endpoint text: {endpoint_text}

CLASSIFICATION RULES:

DIRECT
Use "direct" when the endpoint itself measures the patient-priority construct or a direct
clinical manifestation of that construct.

Examples:
- proteinuria endpoint for reducing proteinuria
- echocardiographic cardiac function for preserving cardiac function
- validated developmental-function scale for preserving developmental skills

PARTIAL
Use "partial" when the endpoint measures:
- only one meaningful part of a broader patient priority,
- a close functional proxy rather than the priority itself, or
- a composite outcome that includes the priority but does not isolate it.

NO_MATCH
Use "no_match" when the endpoint is merely associated with the priority but does not measure it.

Examples include:
- biomarkers or disease mechanism when the priority is a symptom or function
- safety or pharmacokinetic measures
- another symptom in the same disease
- treatment utilization
- a downstream or upstream clinical correlate
- a broad outcome that does not establish measurement of the stated priority

IMPORTANT:
1. Clinical relevance alone is not sufficient.
2. Judge whether the endpoint MEASURES the priority, not whether treatment might improve it.
3. Do not assume a vague endpoint measures a construct that is not actually stated.
4. A narrower or broader but meaningfully overlapping measure should usually be "partial".
5. Use the supplied information as the grounding evidence.
6. Give a concise reason.

Return valid JSON exactly in this structure:

{{
  "match_class": "direct",
  "confidence": "high",
  "reason": "one concise sentence"
}}

Allowed match_class values:
direct
partial
no_match

Allowed confidence values:
high
medium
low
"""

    response = client.responses.create(
        model=LLM_MODEL,
        input=prompt
    )

    raw_text = response.output_text

    result = parse_agent_json(raw_text)

    if result["match_class"] not in {
        "direct",
        "partial",
        "no_match"
    }:
        raise ValueError(
            f"Unexpected match class: {result['match_class']}"
        )

    if result["confidence"] not in {
        "high",
        "medium",
        "low"
    }:
        raise ValueError(
            f"Unexpected confidence: {result['confidence']}"
        )

    return {
        "match_class": result["match_class"],
        "confidence": result["confidence"],
        "reason": result["reason"],
        "raw_response": raw_text
    }

In [ ]:
# Define manually validated cases for the first Semantic Match Agent pilot.
pilot_specs = pd.DataFrame([
    {
        "disease": "Dravet Syndrome",
        "priority_label": "Preserve developmental progress and skills",
        "nct_id": "NCT02187809",
        "within_trial_rank": 1,
        "human_label": "direct"
    },
    {
        "disease": "Dravet Syndrome",
        "priority_label": "Improve behavior and mood",
        "nct_id": "NCT04940624",
        "within_trial_rank": 1,
        "human_label": "direct"
    },
    {
        "disease": "Facioscapulohumeral Muscular Dystrophy",
        "priority_label": "Regain strength and muscle function",
        "nct_id": "NCT02239224",
        "within_trial_rank": 1,
        "human_label": "direct"
    },
    {
        "disease": "Primary FSGS",
        "priority_label": "Reduce proteinuria or albuminuria",
        "nct_id": "NCT04983888",
        "within_trial_rank": 1,
        "human_label": "direct"
    },
    {
        "disease": "Sickle Cell Disease",
        "priority_label": "Improve emotional wellbeing",
        "nct_id": "NCT07001631",
        "within_trial_rank": 1,
        "human_label": "direct"
    },
    {
        "disease": "Sickle Cell Disease",
        "priority_label": "Prevent long-term tissue and organ damage",
        "nct_id": "NCT06823219",
        "within_trial_rank": 9,
        "human_label": "direct"
    },
    {
        "disease": "Dravet Syndrome",
        "priority_label": "Improve sleep",
        "nct_id": "NCT07531745",
        "within_trial_rank": 1,
        "human_label": "no_match"
    },
    {
        "disease": "Dravet Syndrome",
        "priority_label": "Improve behavior and mood",
        "nct_id": "NCT07531745",
        "within_trial_rank": 1,
        "human_label": "no_match"
    },
    {
        "disease": "Hereditary Angioedema",
        "priority_label": "Reduce fatigue",
        "nct_id": "NCT03888755",
        "within_trial_rank": 1,
        "human_label": "no_match"
    },
    {
        "disease": "Primary FSGS",
        "priority_label": "Improve cognitive function and brain fog",
        "nct_id": "NCT03448692",
        "within_trial_rank": 1,
        "human_label": "no_match"
    },
    {
        "disease": "Spinal Muscular Atrophy",
        "priority_label": "Reduce pain",
        "nct_id": "NCT05626855",
        "within_trial_rank": 1,
        "human_label": "no_match"
    },
    {
        "disease": "Facioscapulohumeral Muscular Dystrophy",
        "priority_label": "Maintain independence",
        "nct_id": "NCT07038200",
        "within_trial_rank": 1,
        "human_label": "partial"
    },
    {
        "disease": "Sickle Cell Disease",
        "priority_label": "Reduce acute pain crises",
        "nct_id": "NCT07023666",
        "within_trial_rank": 1,
        "human_label": "partial"
    },
    {
        "disease": "Friedreich's Ataxia",
        "priority_label": "Improve walking and reduce falls",
        "nct_id": "NCT01339884",
        "within_trial_rank": 2,
        "human_label": "partial"
    }
])

print("Pilot cases:", len(pilot_specs))
display(pilot_specs)

Pilot cases: 14


,disease,priority_label,nct_id,within_trial_rank,human_label
0,Dravet Syndrome,Preserve developmental progress and skills,NCT02187809,1,direct
1,Dravet Syndrome,Improve behavior and mood,NCT04940624,1,direct
2,Facioscapulohumeral Muscular Dystrophy,Regain strength and muscle function,NCT02239224,1,direct
3,Primary FSGS,Reduce proteinuria or albuminuria,NCT04983888,1,direct
4,Sickle Cell Disease,Improve emotional wellbeing,NCT07001631,1,direct
5,Sickle Cell Disease,Prevent long-term tissue and organ damage,NCT06823219,9,direct
6,Dravet Syndrome,Improve sleep,NCT07531745,1,no_match
7,Dravet Syndrome,Improve behavior and mood,NCT07531745,1,no_match
8,Hereditary Angioedema,Reduce fatigue,NCT03888755,1,no_match
9,Primary FSGS,Improve cognitive function and brain fog,NCT03448692,1,no_match


In [ ]:
# Assemble full patient-priority and endpoint context for the semantic-match pilot.
if "within_trial_rank" not in priority_endpoint_pairs.columns:
    priority_endpoint_pairs["within_trial_rank"] = (
        priority_endpoint_pairs
        .groupby(
            [
                "disease",
                "priority_id",
                "nct_id"
            ]
        )["similarity"]
        .rank(
            method="first",
            ascending=False
        )
        .astype(int)
    )

priority_lookup = (
    alignment_priorities[
        [
            "disease",
            "priority_id",
            "priority_label",
            "priority_description",
            "evidence_text"
        ]
    ]
    .drop_duplicates(
        [
            "disease",
            "priority_label"
        ]
    )
)

endpoint_lookup = (
    priority_endpoint_pairs
    .merge(
        alignment_endpoints[
            [
                "endpoint_id",
                "endpoint_semantic_text"
            ]
        ].drop_duplicates("endpoint_id"),
        on="endpoint_id",
        how="left"
    )
)

semantic_match_pilot = (
    pilot_specs
    .merge(
        priority_lookup,
        on=[
            "disease",
            "priority_label"
        ],
        how="left"
    )
    .merge(
        endpoint_lookup[
            [
                "disease",
                "priority_id",
                "nct_id",
                "within_trial_rank",
                "endpoint_id",
                "endpoint_name",
                "endpoint_semantic_text",
                "similarity"
            ]
        ],
        on=[
            "disease",
            "priority_id",
            "nct_id",
            "within_trial_rank"
        ],
        how="left"
    )
)

print("Pilot rows:", len(semantic_match_pilot))
print(
    "Missing priority descriptions:",
    semantic_match_pilot["priority_description"].isna().sum()
)
print(
    "Missing endpoint text:",
    semantic_match_pilot["endpoint_semantic_text"].isna().sum()
)

display(
    semantic_match_pilot[
        [
            "disease",
            "priority_label",
            "nct_id",
            "within_trial_rank",
            "endpoint_name",
            "human_label"
        ]
    ]
)

Pilot rows: 14
Missing priority descriptions: 0
Missing endpoint text: 0


,disease,priority_label,nct_id,within_trial_rank,endpoint_name,human_label
0,Dravet Syndrome,Preserve developmental progress and skills,NCT02187809,1,"Change in Behavioural, Neurocognitive Measures...",direct
1,Dravet Syndrome,Improve behavior and mood,NCT04940624,1,Percentage of Participants With CGI-I Nonseizu...,direct
2,Facioscapulohumeral Muscular Dystrophy,Regain strength and muscle function,NCT02239224,1,Percent Change From Baseline in Manual Muscle ...,direct
3,Primary FSGS,Reduce proteinuria or albuminuria,NCT04983888,1,Change in Proteinuria,direct
4,Sickle Cell Disease,Improve emotional wellbeing,NCT07001631,1,Depressive symptoms,direct
5,Sickle Cell Disease,Prevent long-term tissue and organ damage,NCT06823219,9,Echocardiographic Parameters,direct
6,Dravet Syndrome,Improve sleep,NCT07531745,1,Parts 1 and 2: Number of Participants With Cli...,no_match
7,Dravet Syndrome,Improve behavior and mood,NCT07531745,1,Parts 1 and 2: Number of Participants with Cha...,no_match
8,Hereditary Angioedema,Reduce fatigue,NCT03888755,1,Composite Symptom Score (SS) Assessed by Inves...,no_match
9,Primary FSGS,Improve cognitive function and brain fog,NCT03448692,1,Change From Baseline in Pulse Rate,no_match


In [ ]:
# Restore the regular-expression module required by the agent response parser.
import re

print("Regex module ready.")

Regex module ready.


In [ ]:
# Run the Semantic Match Agent against the manually validated pilot cases.
pilot_agent_results = []

for i, row in semantic_match_pilot.iterrows():
    try:
        result = semantic_match_agent(
            priority_label=row["priority_label"],
            priority_description=row["priority_description"],
            priority_evidence=row["evidence_text"],
            endpoint_name=row["endpoint_name"],
            endpoint_text=row["endpoint_semantic_text"]
        )

        pilot_agent_results.append({
            "disease": row["disease"],
            "priority_id": row["priority_id"],
            "priority_label": row["priority_label"],
            "nct_id": row["nct_id"],
            "endpoint_id": row["endpoint_id"],
            "within_trial_rank": row["within_trial_rank"],
            "endpoint_name": row["endpoint_name"],
            "human_label": row["human_label"],
            "agent_label": result["match_class"],
            "confidence": result["confidence"],
            "reason": result["reason"],
            "raw_response": result["raw_response"]
        })

    except Exception as error:
        pilot_agent_results.append({
            "disease": row["disease"],
            "priority_id": row["priority_id"],
            "priority_label": row["priority_label"],
            "nct_id": row["nct_id"],
            "endpoint_id": row["endpoint_id"],
            "within_trial_rank": row["within_trial_rank"],
            "endpoint_name": row["endpoint_name"],
            "human_label": row["human_label"],
            "agent_label": "ERROR",
            "confidence": "",
            "reason": str(error),
            "raw_response": ""
        })

semantic_match_pilot_results = pd.DataFrame(
    pilot_agent_results
)

semantic_match_pilot_results["correct"] = (
    semantic_match_pilot_results["human_label"]
    == semantic_match_pilot_results["agent_label"]
)

display(
    semantic_match_pilot_results[
        [
            "priority_label",
            "endpoint_name",
            "human_label",
            "agent_label",
            "confidence",
            "correct",
            "reason"
        ]
    ]
)

,priority_label,endpoint_name,human_label,agent_label,confidence,correct,reason
0,Preserve developmental progress and skills,"Change in Behavioural, Neurocognitive Measures...",direct,direct,high,True,The Vineland Adaptive Behaviour Scale measures...
1,Improve behavior and mood,Percentage of Participants With CGI-I Nonseizu...,direct,partial,high,False,The endpoint directly assesses improvement in ...
2,Regain strength and muscle function,Percent Change From Baseline in Manual Muscle ...,direct,direct,high,True,MMT directly measures muscle strength and moto...
3,Reduce proteinuria or albuminuria,Change in Proteinuria,direct,direct,high,True,The endpoint directly measures proteinuria thr...
4,Improve emotional wellbeing,Depressive symptoms,direct,partial,high,False,The PROMIS measure directly assesses depressiv...
5,Prevent long-term tissue and organ damage,Echocardiographic Parameters,direct,partial,medium,False,Ventricular wall motion measures a specific as...
6,Improve sleep,Parts 1 and 2: Number of Participants With Cli...,no_match,no_match,high,True,Physical and neurological examination findings...
7,Improve behavior and mood,Parts 1 and 2: Number of Participants with Cha...,no_match,no_match,high,True,The C-SSRS measures suicidal ideation and beha...
8,Reduce fatigue,Composite Symptom Score (SS) Assessed by Inves...,no_match,no_match,high,True,The score measures listed attack symptoms and ...
9,Improve cognitive function and brain fog,Change From Baseline in Pulse Rate,no_match,no_match,high,True,Pulse rate does not measure cognitive function...


In [ ]:
# Summarize agreement between the Semantic Match Agent and human validation.
valid_pilot_results = semantic_match_pilot_results[
    semantic_match_pilot_results["agent_label"] != "ERROR"
].copy()

pilot_accuracy = valid_pilot_results["correct"].mean()

print(
    "Overall pilot accuracy:",
    round(pilot_accuracy, 4)
)

print("\nHuman vs. agent classifications:")

display(
    pd.crosstab(
        valid_pilot_results["human_label"],
        valid_pilot_results["agent_label"],
        margins=True
    )
)

print("\nDisagreements:")

display(
    valid_pilot_results.loc[
        ~valid_pilot_results["correct"],
        [
            "disease",
            "priority_label",
            "nct_id",
            "within_trial_rank",
            "endpoint_name",
            "human_label",
            "agent_label",
            "confidence",
            "reason"
        ]
    ]
)

Overall pilot accuracy: nan

Human vs. agent classifications:


agent_label
human_label



Disagreements:


,disease,priority_label,nct_id,within_trial_rank,endpoint_name,human_label,agent_label,confidence,reason


Error: Runtime no longer has a reference to this dataframe, please re-run this cell and try again.


In [ ]:
# Save the first Semantic Match Agent pilot for audit and prompt refinement.
semantic_match_pilot_results.to_csv(
    DATA_DIR / "trialsignal_semantic_match_agent_pilot.csv",
    index=False
)

print(
    "Saved pilot results:",
    len(semantic_match_pilot_results)
)

Saved pilot results: 14


In [ ]:
# Build a balanced 12-case Semantic Match Agent pilot from the human-reviewed validation set.
pilot_pool = retrieval_review_units[
    retrieval_review_units["manual_scope_issue"].fillna(0) != 1
].copy()

pilot_pool["first_true_match_rank"] = pd.to_numeric(
    pilot_pool["first_true_match_rank"],
    errors="coerce"
)

pilot_pool["human_label"] = np.where(
    pilot_pool["trial_has_true_match"] == 1,
    "direct_match",
    "no_match"
)

direct_pool = pilot_pool[
    pilot_pool["human_label"] == "direct_match"
].copy()

no_match_pool = pilot_pool[
    pilot_pool["human_label"] == "no_match"
].copy()

# Force the known rank-9 retrieval case into the pilot.
hidden_direct = (
    direct_pool[
        direct_pool["first_true_match_rank"] > 5
    ]
    .head(1)
)

other_direct = (
    direct_pool
    .drop(hidden_direct.index)
    .sample(
        n=5,
        random_state=42
    )
)

# Spread no-match cases across the full adjusted-strength range.
no_match_sorted = no_match_pool.sort_values(
    "adjusted_match_percentile"
).reset_index(drop=True)

no_match_positions = np.linspace(
    0,
    len(no_match_sorted) - 1,
    6,
    dtype=int
)

selected_no_matches = no_match_sorted.iloc[
    no_match_positions
]

semantic_pilot_cases = pd.concat(
    [
        other_direct,
        hidden_direct,
        selected_no_matches
    ],
    ignore_index=True
)

print("Pilot cases:", len(semantic_pilot_cases))
print(
    semantic_pilot_cases["human_label"].value_counts()
)

display(
    semantic_pilot_cases[
        [
            "disease",
            "priority_id",
            "priority_label",
            "nct_id",
            "human_label",
            "first_true_match_rank",
            "candidate_strength_band",
            "adjusted_match_percentile"
        ]
    ]
)

Pilot cases: 12
human_label
direct_match    6
no_match        6
Name: count, dtype: int64


,disease,priority_id,priority_label,nct_id,human_label,first_true_match_rank,candidate_strength_band,adjusted_match_percentile
0,Primary FSGS,FSGS_P01,Reduce proteinuria or albuminuria,NCT01665391,direct_match,1.0,middle,0.571429
1,Primary FSGS,FSGS_P01,Reduce proteinuria or albuminuria,NCT05213624,direct_match,1.0,middle,0.485714
2,Dravet Syndrome,DRAVET_P02,Preserve developmental progress and skills,NCT02187809,direct_match,1.0,high,0.918367
3,Sickle Cell Disease,SCD_P10,Improve emotional wellbeing,NCT07001631,direct_match,1.0,middle,0.666102
4,Facioscapulohumeral Muscular Dystrophy,FSHD_P02,Regain strength and muscle function,NCT02239224,direct_match,1.0,middle,0.433333
5,Sickle Cell Disease,SCD_P07,Prevent long-term tissue and organ damage,NCT06823219,direct_match,9.0,middle,0.498305
6,Facioscapulohumeral Muscular Dystrophy,FSHD_P08,Maintain independence,NCT07435129,no_match,NaN,low,0.033333
7,Friedreich's Ataxia,FA_P03,Improve walking and reduce falls,NCT01339884,no_match,NaN,low,0.214286
8,Dravet Syndrome,DRAVET_P03,Preserve cognitive and intellectual function,NCT06422377,no_match,NaN,middle,0.387755
9,Spinal Muscular Atrophy,SMA_P12,Reduce contractures and scoliosis burden,NCT02227823,no_match,NaN,middle,0.567568


In [ ]:
# Assemble the top-10 retrieved endpoints for each Semantic Match Agent pilot case.
pilot_keys = semantic_pilot_cases[
    [
        "disease",
        "priority_id",
        "nct_id"
    ]
].drop_duplicates()

semantic_pilot_candidates = (
    priority_endpoint_pairs[
        priority_endpoint_pairs["within_trial_rank"] <= 10
    ]
    .merge(
        pilot_keys,
        on=[
            "disease",
            "priority_id",
            "nct_id"
        ],
        how="inner"
    )
)

endpoint_text_lookup = (
    alignment_endpoints[
        [
            "endpoint_id",
            "endpoint_semantic_text"
        ]
    ]
    .drop_duplicates("endpoint_id")
)

semantic_pilot_candidates = (
    semantic_pilot_candidates
    .drop(
        columns=["endpoint_semantic_text"],
        errors="ignore"
    )
    .merge(
        endpoint_text_lookup,
        on="endpoint_id",
        how="left"
    )
    .sort_values(
        [
            "disease",
            "priority_id",
            "nct_id",
            "within_trial_rank"
        ]
    )
    .reset_index(drop=True)
)

print(
    "Pilot priority-trial cases represented:",
    semantic_pilot_candidates[
        [
            "disease",
            "priority_id",
            "nct_id"
        ]
    ].drop_duplicates().shape[0]
)

print(
    "Missing endpoint text:",
    semantic_pilot_candidates[
        "endpoint_semantic_text"
    ].isna().sum()
)

Pilot priority-trial cases represented: 12
Missing endpoint text: 0


In [ ]:
# Define the grounded Semantic Match Agent prompt for one priority-trial case.
import json

def build_semantic_match_prompt(case_row, candidate_group):

    candidates = []

    for _, endpoint in candidate_group.sort_values(
        "within_trial_rank"
    ).iterrows():

        candidates.append({
            "endpoint_id": endpoint["endpoint_id"],
            "rank": int(endpoint["within_trial_rank"]),
            "endpoint_type": endpoint["endpoint_type"],
            "endpoint_name": endpoint["endpoint_name"],
            "endpoint_text": endpoint["endpoint_semantic_text"]
        })

    return f"""
You are the Semantic Match Agent in TrialSignal.

Your job is narrow:
Determine whether any supplied clinical-trial endpoint actually measures
the patient priority below.

PATIENT PRIORITY
Disease: {case_row["disease"]}
Priority: {case_row["priority_label"]}
Definition: {case_row["priority_description"]}

RETRIEVED ENDPOINT CANDIDATES
{json.dumps(candidates, indent=2)}

Use only the supplied endpoint information.

Classification rules:

DIRECT_MATCH:
The endpoint directly measures the same outcome or construct identified
by the patient priority. Exact wording is not required if the clinical
measure clearly operationalizes that construct.

PARTIAL_MATCH:
The endpoint measures a meaningful component, proxy, or broader/narrower
construct related to the priority, but does not directly measure the
full patient-priority concept.

NO_MATCH:
The endpoint is merely clinically related, measures another disease
outcome, a biomarker, safety, pharmacokinetics, or otherwise does not
measure the patient priority.

Important:
- Do not assume that improving the disease means the patient priority
  is measured.
- Do not treat biomarkers or safety measures as patient-priority
  outcomes unless the priority itself concerns that measure.
- Examine all supplied candidates, not only rank 1.
- If any direct match exists, choose the strongest direct match.
- Otherwise choose the strongest partial match.
- If no meaningful match exists, return NO_MATCH.

Return ONLY valid JSON with exactly these fields:

{{
  "decision": "direct_match | partial_match | no_match",
  "matched_endpoint_id": "endpoint id or null",
  "matched_rank": integer or null,
  "confidence": "high | medium | low",
  "reason": "brief explanation"
}}
""".strip()

In [ ]:
# Select one clear direct-match case for the first Semantic Match Agent test.
debug_case = (
    semantic_pilot_cases[
        (semantic_pilot_cases["human_label"] == "direct_match")
        & (semantic_pilot_cases["first_true_match_rank"] == 1)
    ]
    .sort_values(
        "adjusted_match_percentile",
        ascending=False
    )
    .iloc[0]
)

debug_candidates = semantic_pilot_candidates[
    (
        semantic_pilot_candidates["disease"]
        == debug_case["disease"]
    )
    & (
        semantic_pilot_candidates["priority_id"]
        == debug_case["priority_id"]
    )
    & (
        semantic_pilot_candidates["nct_id"]
        == debug_case["nct_id"]
    )
].copy()

debug_prompt = build_semantic_match_prompt(
    debug_case,
    debug_candidates
)

print("Disease:", debug_case["disease"])
print("Priority:", debug_case["priority_label"])
print("Trial:", debug_case["nct_id"])
print("Human label:", debug_case["human_label"])
print("Human first-match rank:", debug_case["first_true_match_rank"])
print("Retrieved endpoints:", len(debug_candidates))

Disease: Dravet Syndrome
Priority: Preserve developmental progress and skills
Trial: NCT02187809
Human label: direct_match
Human first-match rank: 1.0
Retrieved endpoints: 7


In [ ]:
# Run the first grounded Semantic Match Agent judgment.
semantic_debug_response = client.responses.create(
    model=LLM_MODEL,
    input=debug_prompt
)

semantic_debug_raw = (
    semantic_debug_response.output_text
    .strip()
)

print(semantic_debug_raw)

{
  "decision": "direct_match",
  "matched_endpoint_id": "NCT02187809_P4",
  "matched_rank": 1,
  "confidence": "high",
  "reason": "Change measured using the Vineland Adaptive Behaviour Scale directly assesses adaptive and developmental skills, relevant to preserving developmental progress."
}


In [ ]:
# Lock the final SMA scope-exclusion list without changing the master data yet.
ambiguous_exclusion_reasons = {
    "NCT01369901": "Spinal and bulbar muscular atrophy, not target SMA.",
    "NCT05152823": "IGHMBP2-related SMARD1/CMT2S population, outside target SMA scope.",
    "NCT03196375": "Broad motor neuron disease study, not SMA-specific."
}

ambiguous_exclusion_ids = set(
    ambiguous_exclusion_reasons.keys()
)

final_sma_exclusion_ids = (
    set(
        obvious_sma_exclusions[
            "nct_id"
        ].astype(str)
    )
    | ambiguous_exclusion_ids
)

print(
    "Obvious exclusions:",
    len(obvious_sma_exclusions)
)

print(
    "Added ambiguous exclusions:",
    len(ambiguous_exclusion_ids)
)

print(
    "Final unique SMA exclusions:",
    len(final_sma_exclusion_ids)
)

print(
    "SMA trials retained:",
    len(sma_trials) - len(final_sma_exclusion_ids)
)

NameError: name 'obvious_sma_exclusions' is not defined

In [ ]:
# Build an auditable table of the SMA trials marked for exclusion.
sma_scope_exclusion_review = (
    sma_trials[
        sma_trials["nct_id"].astype(str).isin(
            final_sma_exclusion_ids
        )
    ][
        [
            "nct_id",
            "brief_title",
            "official_title",
            "conditions",
            "intervention_names"
        ]
    ]
    .copy()
    .sort_values("nct_id")
    .reset_index(drop=True)
)

sma_scope_exclusion_review[
    "scope_exclusion_reason"
] = "Deterministic non-target SMA scope exclusion"

for nct_id, reason in ambiguous_exclusion_reasons.items():
    sma_scope_exclusion_review.loc[
        sma_scope_exclusion_review["nct_id"] == nct_id,
        "scope_exclusion_reason"
    ] = reason

print(
    "Exclusion-review records:",
    len(sma_scope_exclusion_review)
)

display(
    sma_scope_exclusion_review[
        [
            "nct_id",
            "brief_title",
            "conditions",
            "scope_exclusion_reason"
        ]
    ]
)

In [ ]:
# Check whether any newly excluded SMA trials appeared in the manual retrieval-validation sample.
validation_scope_hits = (
    retrieval_review_units[
        retrieval_review_units[
            "nct_id"
        ].astype(str).isin(
            final_sma_exclusion_ids
        )
    ]
    .copy()
)

print(
    "Validation cases affected by final SMA exclusions:",
    len(validation_scope_hits)
)

print(
    "Unique excluded trials represented:",
    validation_scope_hits["nct_id"].nunique()
)

display(
    validation_scope_hits[
        [
            "disease",
            "priority_id",
            "priority_label",
            "nct_id",
            "trial_has_true_match",
            "first_true_match_rank",
            "manual_scope_issue",
            "manual_review_notes"
        ]
    ]
)

In [ ]:
# Preview corpus counts after applying the final SMA scope exclusions.
trials_clean_preview = trials_scope_check[
    ~(
        (trials_scope_check["disease"] == "Spinal Muscular Atrophy")
        & trials_scope_check["nct_id"].astype(str).isin(
            final_sma_exclusion_ids
        )
    )
].copy()

preview_counts = (
    trials_clean_preview
    .groupby("disease")["nct_id"]
    .nunique()
    .sort_index()
)

print(
    "Original trials:",
    trials_scope_check["nct_id"].nunique()
)

print(
    "Preview cleaned trials:",
    trials_clean_preview["nct_id"].nunique()
)

display(preview_counts)

In [ ]:

# Semantic Match Agent v2 — candidate-level judgment

import json

SEMANTIC_PROMPT_VERSION = "semantic_v2_candidate"

SEMANTIC_RULES = """
You are the Semantic Match Agent for TrialSignal.

Determine whether the supplied clinical-trial endpoint
MEASURES the stated patient priority.

Use exactly these categories:

direct:
The endpoint explicitly measures the patient-priority outcome
or a recognized clinical measure of the same construct.

partial:
The endpoint measures a meaningful component, proxy, or
narrower related outcome, but not the entire priority.

no_match:
The endpoint is clinically related but does not meaningfully
measure the priority. Disease improvement, biomarkers,
safety measures, and treatment effects do not automatically
establish alignment.

Rules:
- Exact wording is not required.
- Ground your judgment in the supplied text.
- Do not assume unstated outcome measurements.
- Do not use cosine similarity or retrieval rank as evidence.
- Do not judge population or disease-cohort applicability.
  That belongs to the separate Scope Agent.
- Give a concise, evidence-based reason.
- Confidence reflects clarity of evidence, not a calibrated
  statistical probability.
"""

SEMANTIC_SCHEMA = {
    "type": "object",
    "properties": {
        "semantic_label": {
            "type": "string",
            "enum": ["direct", "partial", "no_match"]
        },
        "confidence": {
            "type": "number"
        },
        "reason": {
            "type": "string"
        }
    },
    "required": [
        "semantic_label",
        "confidence",
        "reason"
    ],
    "additionalProperties": False
}

# Attach original patient evidence to the retrieved candidates.
evidence_lookup = (
    alignment_priorities[
        ["disease", "priority_id", "evidence_text"]
    ]
    .drop_duplicates(["disease", "priority_id"])
)

semantic_agent_inputs = agent_candidate_pool.merge(
    evidence_lookup,
    on=["disease", "priority_id"],
    how="left",
    validate="many_to_one"
)

def semantic_match_candidate(row):
    if client is None:
        raise RuntimeError("OpenAI client is not initialized.")

    payload = {
        "disease": str(row["disease"]),
        "priority": str(row["priority_label"]),
        "priority_definition": str(row["priority_description"]),
        "patient_evidence": str(row["evidence_text"])[:1500],
        "endpoint_name": str(row["endpoint_name"]),
        "endpoint_type": str(row["endpoint_type"]),
        "endpoint_text": str(row["endpoint_semantic_text"])
    }

    response = client.responses.create(
        model=LLM_MODEL,
        instructions=SEMANTIC_RULES,
        input=json.dumps(payload, ensure_ascii=False),
        text={
            "format": {
                "type": "json_schema",
                "name": "trialsignal_semantic_match",
                "strict": True,
                "schema": SEMANTIC_SCHEMA
            }
        }
    )

    if not response.output_text:
        raise ValueError("No semantic judgment returned.")

    result = json.loads(response.output_text)
    confidence = float(result["confidence"])

    if not 0 <= confidence <= 1:
        raise ValueError("Confidence must be between 0 and 1.")

    if not result["reason"].strip():
        raise ValueError("Semantic reason is empty.")

    # Identifiers come from our dataset, NOT from the LLM.
    return {
        "disease": row["disease"],
        "priority_id": row["priority_id"],
        "nct_id": row["nct_id"],
        "endpoint_id": row["endpoint_id"],
        "candidate_rank": int(row["within_trial_rank"]),
        "semantic_label": result["semantic_label"],
        "confidence": confidence,
        "reason": result["reason"],
        "model": LLM_MODEL,
        "prompt_version": SEMANTIC_PROMPT_VERSION
    }

print("Semantic Match Agent v2 defined.")
print("Candidate rows available:", len(semantic_agent_inputs))


Semantic Match Agent v2 defined.
Candidate rows available: 63811


In [ ]:

# One-call Semantic Match Agent smoke test

test_candidates = get_trial_candidates(
    "Dravet Syndrome",
    "DRAVET_P02",
    "NCT02187809"
)

test_row = semantic_agent_inputs[
    (semantic_agent_inputs["disease"] == "Dravet Syndrome")
    & (semantic_agent_inputs["priority_id"] == "DRAVET_P02")
    & (semantic_agent_inputs["nct_id"] == "NCT02187809")
    & (semantic_agent_inputs["endpoint_id"] == "NCT02187809_P4")
].iloc[0]

test_result = semantic_match_candidate(test_row)

display(pd.DataFrame([test_result]))


,disease,priority_id,nct_id,endpoint_id,candidate_rank,semantic_label,confidence,reason,model,prompt_version
0,Dravet Syndrome,DRAVET_P02,NCT02187809,NCT02187809_P4,1,direct,0.88,The Vineland Adaptive Behavior Scales assess f...,gpt-6-luna,semantic_v2_candidate


In [ ]:

# SEMANTIC MATCH AGENT V2 — 14-CASE PILOT
# Uses manually selected cases already documented in the notebook.

pilot_records = [
    ("Dravet Syndrome", "Preserve developmental progress and skills", "NCT02187809", 1, "direct"),
    ("Dravet Syndrome", "Improve behavior and mood", "NCT04940624", 1, "direct"),
    ("Facioscapulohumeral Muscular Dystrophy", "Regain strength and muscle function", "NCT02239224", 1, "direct"),
    ("Primary FSGS", "Reduce proteinuria or albuminuria", "NCT04983888", 1, "direct"),
    ("Sickle Cell Disease", "Improve emotional wellbeing", "NCT07001631", 1, "direct"),
    ("Sickle Cell Disease", "Prevent long-term tissue and organ damage", "NCT06823219", 9, "direct"),
    ("Dravet Syndrome", "Improve sleep", "NCT07531745", 1, "no_match"),
    ("Dravet Syndrome", "Improve behavior and mood", "NCT07531745", 1, "no_match"),
    ("Hereditary Angioedema", "Reduce fatigue", "NCT03888755", 1, "no_match"),
    ("Primary FSGS", "Improve cognitive function and brain fog", "NCT03448692", 1, "no_match"),
    ("Spinal Muscular Atrophy", "Reduce pain", "NCT05626855", 1, "no_match"),
    ("Facioscapulohumeral Muscular Dystrophy", "Maintain independence", "NCT07038200", 1, "partial"),
    ("Sickle Cell Disease", "Reduce acute pain crises", "NCT07023666", 1, "partial"),
    ("Friedreich's Ataxia", "Improve walking and reduce falls", "NCT01339884", 2, "partial")
]

pilot_specs_v2 = pd.DataFrame(
    pilot_records,
    columns=[
        "disease", "priority_label", "nct_id",
        "within_trial_rank", "human_label"
    ]
)

# Join the human-reviewed cases with the clean candidate inputs.
pilot_inputs_v2 = pilot_specs_v2.merge(
    semantic_agent_inputs,
    on=["disease", "priority_label", "nct_id", "within_trial_rank"],
    how="left",
    validate="one_to_one"
)

assert len(pilot_inputs_v2) == 14
assert pilot_inputs_v2["endpoint_id"].notna().all()

OUTPUT_PATH = DATA_DIR / "trialsignal_semantic_v2_pilot14.csv"

# Load any previous progress for safe resumption.
if OUTPUT_PATH.exists():
    previous = pd.read_csv(OUTPUT_PATH).to_dict("records")
else:
    previous = []

def result_key(r):
    return (
        r["disease"], r["priority_id"],
        r["nct_id"], r["endpoint_id"]
    )

saved = {result_key(r): r for r in previous}

for i, (_, row) in enumerate(pilot_inputs_v2.iterrows(), start=1):

    key = result_key(row)

    # Skip already successful requests.
    if key in saved and saved[key].get("agent_label") in {
        "direct", "partial", "no_match"
    }:
        print(f"{i}/14: Reusing saved result")
        continue

    try:
        # Reuse the successful initial smoke test if applicable.
        if (
            "test_result" in globals()
            and result_key(test_result) == key
            and test_result.get("prompt_version") == SEMANTIC_PROMPT_VERSION
        ):
            result = test_result
        else:
            result = semantic_match_candidate(row)

        agent_label = result["semantic_label"]
        confidence = result["confidence"]
        reason = result["reason"]

    except Exception as error:
        agent_label = "ERROR"
        confidence = None
        reason = str(error)

    saved[key] = {
        "disease": row["disease"],
        "priority_id": row["priority_id"],
        "priority_label": row["priority_label"],
        "nct_id": row["nct_id"],
        "endpoint_id": row["endpoint_id"],
        "candidate_rank": int(row["within_trial_rank"]),
        "endpoint_name": row["endpoint_name"],
        "human_label": row["human_label"],
        "agent_label": agent_label,
        "confidence": confidence,
        "reason": reason,
        "model": LLM_MODEL,
        "prompt_version": SEMANTIC_PROMPT_VERSION
    }

    # Checkpoint after each evaluation.
    pd.DataFrame(saved.values()).to_csv(
        OUTPUT_PATH, index=False
    )

    print(f"{i}/14: Human={row['human_label']} | Agent={agent_label}")

# Evaluate completed cases.
pilot_results_v2 = pd.DataFrame(saved.values())

valid = pilot_results_v2[
    pilot_results_v2["agent_label"].isin(
        ["direct", "partial", "no_match"]
    )
].copy()

valid["correct"] = (
    valid["human_label"] == valid["agent_label"]
)

print("\nCompleted evaluations:", len(valid), "/ 14")
print("Errors:", (pilot_results_v2["agent_label"] == "ERROR").sum())

if len(valid):
    print("Agreement on completed cases:", f"{valid['correct'].mean():.2%}")

    print("\nClassification comparison:")
    display(pd.crosstab(
        valid["human_label"],
        valid["agent_label"],
        margins=True
    ))

    print("\nDisagreements:")
    display(valid.loc[
        ~valid["correct"],
        [
            "disease", "priority_label", "endpoint_name",
            "human_label", "agent_label", "confidence", "reason"
        ]
    ])

print("\nFull pilot results:")
display(pilot_results_v2)

print("\nSaved to:", OUTPUT_PATH)


1/14: Reusing saved result
2/14: Reusing saved result
3/14: Reusing saved result
4/14: Reusing saved result
5/14: Reusing saved result
6/14: Reusing saved result
7/14: Reusing saved result
8/14: Reusing saved result
9/14: Reusing saved result
10/14: Reusing saved result
11/14: Reusing saved result
12/14: Reusing saved result
13/14: Reusing saved result
14/14: Reusing saved result

Completed evaluations: 14 / 14
Errors: 0
Agreement on completed cases: 78.57%

Classification comparison:


agent_label,direct,no_match,partial,All
human_label,,,,
direct,3,0,3,6
no_match,0,5,0,5
partial,0,0,3,3
All,3,5,6,14



Disagreements:


,disease,priority_label,endpoint_name,human_label,agent_label,confidence,reason
1,Dravet Syndrome,Improve behavior and mood,Percentage of Participants With CGI-I Nonseizu...,direct,partial,0.97,The endpoint explicitly rates improvement in d...
4,Sickle Cell Disease,Improve emotional wellbeing,Depressive symptoms,direct,partial,0.99,PROMIS Depressive Symptoms directly measures d...
5,Sickle Cell Disease,Prevent long-term tissue and organ damage,Echocardiographic Parameters,direct,partial,0.91,Ventricular wall motion assesses a cardiac fun...



Full pilot results:


,disease,priority_id,priority_label,nct_id,endpoint_id,candidate_rank,endpoint_name,human_label,agent_label,confidence,reason,model,prompt_version
0,Dravet Syndrome,DRAVET_P02,Preserve developmental progress and skills,NCT02187809,NCT02187809_P4,1,"Change in Behavioural, Neurocognitive Measures...",direct,direct,0.88,The Vineland Adaptive Behavior Scales assess f...,gpt-6-luna,semantic_v2_candidate
1,Dravet Syndrome,DRAVET_P05,Improve behavior and mood,NCT04940624,NCT04940624_S6,1,Percentage of Participants With CGI-I Nonseizu...,direct,partial,0.97,The endpoint explicitly rates improvement in d...,gpt-6-luna,semantic_v2_candidate
2,Facioscapulohumeral Muscular Dystrophy,FSHD_P02,Regain strength and muscle function,NCT02239224,NCT02239224_S1,1,Percent Change From Baseline in Manual Muscle ...,direct,direct,0.98,The endpoint directly measures muscle strength...,gpt-6-luna,semantic_v2_candidate
3,Primary FSGS,FSGS_P01,Reduce proteinuria or albuminuria,NCT04983888,NCT04983888_P1,1,Change in Proteinuria,direct,direct,0.99,The endpoint measures change in proteinuria us...,gpt-6-luna,semantic_v2_candidate
4,Sickle Cell Disease,SCD_P10,Improve emotional wellbeing,NCT07001631,NCT07001631_S3,1,Depressive symptoms,direct,partial,0.99,PROMIS Depressive Symptoms directly measures d...,gpt-6-luna,semantic_v2_candidate
5,Sickle Cell Disease,SCD_P07,Prevent long-term tissue and organ damage,NCT06823219,NCT06823219_S8,9,Echocardiographic Parameters,direct,partial,0.91,Ventricular wall motion assesses a cardiac fun...,gpt-6-luna,semantic_v2_candidate
6,Dravet Syndrome,DRAVET_P06,Improve sleep,NCT07531745,NCT07531745_P5,1,Parts 1 and 2: Number of Participants With Cli...,no_match,no_match,0.99,Physical and neurological examination findings...,gpt-6-luna,semantic_v2_candidate
7,Dravet Syndrome,DRAVET_P05,Improve behavior and mood,NCT07531745,NCT07531745_P6,1,Parts 1 and 2: Number of Participants with Cha...,no_match,no_match,0.98,The C-SSRS measures suicidal ideation and beha...,gpt-6-luna,semantic_v2_candidate
8,Hereditary Angioedema,HAE_P08,Reduce fatigue,NCT03888755,NCT03888755_S3,1,Composite Symptom Score (SS) Assessed by Inves...,no_match,no_match,0.95,The score rates specified attack symptoms and ...,gpt-6-luna,semantic_v2_candidate
9,Primary FSGS,FSGS_P07,Improve cognitive function and brain fog,NCT03448692,NCT03448692_S5,1,Change From Baseline in Pulse Rate,no_match,no_match,0.99,Pulse rate is a cardiovascular measure and doe...,gpt-6-luna,semantic_v2_candidate



Saved to: ./TrialSignal/data/trialsignal_semantic_v2_pilot14.csv
Error: Runtime no longer has a reference to this dataframe, please re-run this cell and try again.


In [ ]:

# SEMANTIC AGENT — FULL READABLE PILOT REVIEW
# No API calls. No rerunning the agent.
# Change to False to see all 14 cases.

import pandas as pd
import textwrap

SHOW_ONLY_DISAGREEMENTS = True

if "pilot_results_v2" not in globals():
    pilot_results_v2 = pd.read_csv(
        DATA_DIR / "trialsignal_semantic_v2_pilot14.csv"
    )

review = pilot_results_v2.copy()

if SHOW_ONLY_DISAGREEMENTS:
    review = review[
        review["human_label"] != review["agent_label"]
    ]

def print_wrapped(label, value):
    print(f"\n{label}:")
    print(textwrap.fill(str(value), width=105))

for i, (_, row) in enumerate(review.iterrows(), start=1):

    matched = semantic_agent_inputs[
        (semantic_agent_inputs["priority_id"] == row["priority_id"])
        & (semantic_agent_inputs["nct_id"] == row["nct_id"])
        & (semantic_agent_inputs["endpoint_id"] == row["endpoint_id"])
    ]

    if len(matched) != 1:
        print("Unable to uniquely locate:", row["endpoint_id"])
        continue

    source = matched.iloc[0]

    print("\n" + "=" * 105)
    print(f"CASE {i}: {row['disease']}")
    print("=" * 105)

    print_wrapped(
        "PATIENT PRIORITY",
        source["priority_label"]
    )

    print_wrapped(
        "PRIORITY DEFINITION",
        source["priority_description"]
    )

    print_wrapped(
        "ENDPOINT NAME",
        source["endpoint_name"]
    )

    print_wrapped(
        "FULL ENDPOINT DESCRIPTION",
        source["endpoint_semantic_text"]
    )

    print("\nHUMAN LABEL:", row["human_label"])
    print("AGENT LABEL:", row["agent_label"])
    print("CONFIDENCE:", row["confidence"])

    print_wrapped(
        "FULL AGENT REASONING",
        row["reason"]
    )

    # Retrieve original manual-review notes.
    original_review = semantic_pilot_cases[
        (semantic_pilot_cases["priority_id"] == row["priority_id"])
        & (semantic_pilot_cases["nct_id"] == row["nct_id"])
    ]

    if not original_review.empty:
        print_wrapped(
            "ORIGINAL HUMAN REVIEW NOTES",
            original_review.iloc[0]["manual_review_notes"]
        )

print("\nCases displayed:", len(review))
print("No API calls made.")



CASE 1: Dravet Syndrome

PATIENT PRIORITY:
Improve behavior and mood

PRIORITY DEFINITION:
Reduce disruptive behavior, mood problems, and related behavioral difficulties.

ENDPOINT NAME:
Percentage of Participants With CGI-I Nonseizure Symptoms Instrument Responses for Each Domain as Per the
Investigator Reported Impression at Week 16

FULL ENDPOINT DESCRIPTION:
Percentage of Participants With CGI-I Nonseizure Symptoms Instrument Responses for Each Domain as Per the
Investigator Reported Impression at Week 16 The CGI-I non-seizure symptoms instrument is a series of
single-item assessments that the investigator used to rate improvement in the symptoms and impacts in
select non-seizure domains (alertness, communication, and disruptive behaviors) since initiating the
study drug. The participant was rated by the investigator for each domain as follows: 1 (very much
improved), 2 (much improved), 3 (minimally improved), 4 (no change), 5 (minimally worse), 6 (much worse),
and 7 (very much wo

In [ ]:

# SEMANTIC MATCH AGENT V3 — REVISED CLASSIFICATION RULES
# Reuses existing semantic_match_candidate() and output schema.

SEMANTIC_PROMPT_VERSION = "semantic_v3_constituent"

SEMANTIC_RULES = """
You are the Semantic Match Agent for TrialSignal.

Your task is to determine whether one registered clinical-trial
endpoint actually MEASURES a validated patient priority.

Judge the measurement relationship, not merely clinical relevance.

CLASSIFICATION RULES

DIRECT:
The endpoint explicitly measures the patient priority or a
clearly identifiable constituent outcome of that priority.

A broad patient priority can contain several meaningful outcomes.
Direct measurement of one explicitly included outcome is sufficient
for a direct match. The endpoint does NOT need to measure every
dimension of the broader priority.

The relevant outcome must actually be measured by the endpoint.
If a multi-domain instrument reports the relevant domain
separately, that domain can qualify as direct.

PARTIAL:
The endpoint measures a meaningful indirect indicator, proxy,
related functional outcome, or broader composite that is relevant
to the priority without separately establishing its measurement.

Clinical relationships alone do not establish direct measurement.
A functional or physiological indicator is not automatically a
direct measure of organ damage, quality of life, independence,
pain, or another patient-priority outcome.

If an instrument contains relevant items but the registered
endpoint reports only a broader composite, consider whether
the target outcome is actually measured separately.

NO_MATCH:
The endpoint does not meaningfully measure the priority.

A shared disease context, general treatment benefit, or
biological association alone is insufficient for alignment.

IMPORTANT DISTINCTIONS

- Direct does not require complete coverage of every dimension
  of a broad priority.
- Partial means indirect or incomplete evidence of measurement,
  not simply that some other dimensions were omitted.
- Do not automatically classify every medically related endpoint
  as partial.
- Judge only the supplied endpoint, not other endpoints in
  the same trial.
- Do not infer measurements that are not specified.
- Do not assume an endpoint establishes treatment effectiveness
  simply because it measures an outcome.
- Do not use cosine similarity or retrieval rank as evidence.
- Do not evaluate disease, population, or cohort applicability.
  That responsibility belongs to the Scope Agent.

OUTPUT

Return exactly one of: direct, partial, no_match.

Give a concise explanation identifying what the endpoint
actually measures and how that relates to the patient priority.

Return confidence between 0 and 1. Confidence should reflect
the clarity of the supplied measurement evidence, not general
clinical familiarity. Do not express unwarranted certainty.
"""

print("Semantic Agent prompt:", SEMANTIC_PROMPT_VERSION)
print("Existing output schema and function preserved.")


Semantic Agent prompt: semantic_v3_constituent
Existing output schema and function preserved.


In [ ]:

# SEMANTIC V3 — 14-CASE COMPARISON
# Makes up to 14 new API calls.
# Checkpoints after each call; resumable.

import pandas as pd
import textwrap

V3_PATH = DATA_DIR / "trialsignal_semantic_v3_pilot14.csv"

if V3_PATH.exists():
    prior = pd.read_csv(V3_PATH)
    prior = prior[
        prior["prompt_version"] == SEMANTIC_PROMPT_VERSION
    ]
    saved_v3 = {
        result_key(r): r
        for r in prior.to_dict("records")
    }
else:
    saved_v3 = {}

for i, (_, row) in enumerate(
    pilot_inputs_v2.iterrows(), start=1
):
    key = result_key(row)

    if (
        key in saved_v3
        and saved_v3[key].get("agent_label")
        in {"direct", "partial", "no_match"}
    ):
        print(f"{i}/14: Existing V3 result reused")
        continue

    try:
        result = semantic_match_candidate(row)

        saved_v3[key] = {
            "disease": row["disease"],
            "priority_id": row["priority_id"],
            "priority_label": row["priority_label"],
            "nct_id": row["nct_id"],
            "endpoint_id": row["endpoint_id"],
            "candidate_rank": int(row["within_trial_rank"]),
            "human_label": row["human_label"],
            "agent_label": result["semantic_label"],
            "confidence": result["confidence"],
            "reason": result["reason"],
            "model": result["model"],
            "prompt_version": result["prompt_version"]
        }

        print(
            f"{i}/14: Human={row['human_label']} | "
            f"V3={result['semantic_label']}"
        )

    except Exception as e:
        print(f"{i}/14: ERROR — {e}")

    pd.DataFrame(saved_v3.values()).to_csv(
        V3_PATH, index=False
    )

v3_results = pd.DataFrame(saved_v3.values())

valid_v3 = v3_results[
    v3_results["agent_label"].isin(
        ["direct", "partial", "no_match"]
    )
].copy()

# Original human judgments are preserved.
valid_v3["agrees_with_original"] = (
    valid_v3["human_label"] == valid_v3["agent_label"]
)

# Flag the unresolved organ-damage case.
valid_v3["review_status"] = "original_label"

valid_v3.loc[
    (valid_v3["priority_id"] == "SCD_P07")
    & (valid_v3["nct_id"] == "NCT06823219"),
    "review_status"
] = "adjudication_needed"

print("\n" + "=" * 75)
print("SEMANTIC AGENT V2 VS V3")
print("=" * 75)

print("Completed V3 calls:", len(valid_v3), "/14")

print("\nV3 original-label agreement:")
print(f"{valid_v3['agrees_with_original'].mean():.2%}")

print("\nV3 classification comparison:")
print(pd.crosstab(
    valid_v3["human_label"],
    valid_v3["agent_label"],
    margins=True
).to_string())

comparison = valid_v3.merge(
    pilot_results_v2[
        ["priority_id", "nct_id", "endpoint_id", "agent_label"]
    ].rename(columns={"agent_label": "v2_label"}),
    on=["priority_id", "nct_id", "endpoint_id"],
    how="left"
)

print("\nINDIVIDUAL CLASSIFICATION CHANGES")
print(comparison[
    [
        "priority_id",
        "human_label",
        "v2_label",
        "agent_label",
        "review_status"
    ]
].to_string(index=False))

print("\nCHANGED OR DISAGREED CASES")

changed = comparison[
    (comparison["v2_label"] != comparison["agent_label"])
    | (~comparison["agrees_with_original"])
]

for _, row in changed.iterrows():
    print("\n" + "-" * 75)
    print(row["disease"], "|", row["priority_id"])
    print("Original human:", row["human_label"])
    print("V2:", row["v2_label"])
    print("V3:", row["agent_label"])
    print("Confidence:", row["confidence"])
    print("Review status:", row["review_status"])
    print("Reason:")
    print(textwrap.fill(str(row["reason"]), width=100))

print("\nSaved separately:", V3_PATH)


1/14: Human=direct | V3=direct
2/14: Human=direct | V3=direct
3/14: Human=direct | V3=direct
4/14: Human=direct | V3=direct
5/14: Human=direct | V3=direct
6/14: Human=direct | V3=partial
7/14: Human=no_match | V3=no_match
8/14: Human=no_match | V3=no_match
9/14: Human=no_match | V3=no_match
10/14: Human=no_match | V3=no_match
11/14: Human=no_match | V3=no_match
12/14: Human=partial | V3=partial
13/14: Human=partial | V3=partial
14/14: Human=partial | V3=partial

SEMANTIC AGENT V2 VS V3
Completed V3 calls: 14 /14

V3 original-label agreement:
92.86%

V3 classification comparison:
agent_label  direct  no_match  partial  All
human_label                                
direct            5         0        1    6
no_match          0         5        0    5
partial           0         0        3    3
All               5         5        4   14

INDIVIDUAL CLASSIFICATION CHANGES
priority_id human_label v2_label agent_label       review_status
 DRAVET_P02      direct   direct      direct      

In [ ]:

# ============================================================
# SEMANTIC MATCH AGENT V3 — HELD-OUT 40-CASE VALIDATION
# ============================================================
# Evaluates cases not included in the original 14-case pilot.
# Does not alter the model, prompt, or frozen datasets.
# Makes up to 40 new API calls.
# Results are checkpointed after each call.

import pandas as pd
import numpy as np
import textwrap

assert SEMANTIC_PROMPT_VERSION == "semantic_v3_constituent"

# 1. Load the complete manually reviewed benchmark.
benchmark54 = pd.read_csv(
    DATA_DIR / "trialsignal_semantic_pilot_cases.csv"
)

benchmark54["trial_has_true_match"] = pd.to_numeric(
    benchmark54["trial_has_true_match"]
).astype(int)

benchmark54["first_true_match_rank"] = pd.to_numeric(
    benchmark54["first_true_match_rank"],
    errors="coerce"
)

assert len(benchmark54) == 54
assert benchmark54["trial_has_true_match"].sum() == 16

KEYS = ["disease", "priority_id", "nct_id"]

# 2. Identify the 14 cases used in prompt development.
development14 = pd.read_csv(
    DATA_DIR / "trialsignal_semantic_v3_pilot14.csv"
)

assert set(development14["prompt_version"]) == {
    SEMANTIC_PROMPT_VERSION
}

development_keys = pd.MultiIndex.from_frame(
    development14[KEYS].drop_duplicates()
)

benchmark_keys = pd.MultiIndex.from_frame(
    benchmark54[KEYS]
)

heldout40 = benchmark54.loc[
    ~benchmark_keys.isin(development_keys)
].copy()

assert len(heldout40) == 40
assert heldout40["trial_has_true_match"].sum() == 10

# 3. Choose the endpoint to evaluate.
# Direct-match cases: known confirmed endpoint rank.
# Non-direct cases: rank 1, which may be partial/no_match.
heldout40["test_rank"] = np.where(
    heldout40["trial_has_true_match"] == 1,
    heldout40["first_true_match_rank"],
    1
).astype(int)

lookup_keys = heldout40[
    KEYS + [
        "test_rank",
        "trial_has_true_match",
        "manual_review_notes"
    ]
].copy()

lookup_keys = lookup_keys.rename(columns={
    "trial_has_true_match": "human_direct"
})

# 4. Attach endpoint and priority context.
heldout_inputs = lookup_keys.merge(
    semantic_agent_inputs,
    left_on=KEYS + ["test_rank"],
    right_on=KEYS + ["within_trial_rank"],
    how="left",
    validate="one_to_one"
)

assert len(heldout_inputs) == 40
assert heldout_inputs["endpoint_id"].notna().all()
assert heldout_inputs["endpoint_semantic_text"].notna().all()

print("HELD-OUT BENCHMARK READY")
print("Cases:", len(heldout_inputs))
print("Human direct:", heldout_inputs["human_direct"].sum())
print("Human no-direct:", (
    heldout_inputs["human_direct"] == 0
).sum())

# 5. Configure checkpointing.
HELDOUT_PATH = (
    DATA_DIR / "trialsignal_semantic_v3_heldout40.csv"
)

if HELDOUT_PATH.exists():
    previous = pd.read_csv(HELDOUT_PATH)

    previous = previous[
        previous["prompt_version"]
        == SEMANTIC_PROMPT_VERSION
    ]

    saved_heldout = {
        result_key(r): r
        for r in previous.to_dict("records")
    }
else:
    saved_heldout = {}

# 6. Evaluate and save each case independently.
for i, (_, row) in enumerate(
    heldout_inputs.iterrows(), start=1
):
    key = result_key(row)

    if (
        key in saved_heldout
        and saved_heldout[key].get("agent_label")
        in {"direct", "partial", "no_match"}
    ):
        print(f"{i}/40: Reused")
        continue

    try:
        # The model receives only the fields selected
        # inside semantic_match_candidate().
        # Human judgments and notes are NOT sent.
        result = semantic_match_candidate(row)

        label = result["semantic_label"]
        confidence = result["confidence"]
        reason = result["reason"]

    except Exception as error:
        label = "ERROR"
        confidence = None
        reason = str(error)

    saved_heldout[key] = {
        "disease": row["disease"],
        "priority_id": row["priority_id"],
        "priority_label": row["priority_label"],
        "nct_id": row["nct_id"],
        "endpoint_id": row["endpoint_id"],
        "candidate_rank": int(row["within_trial_rank"]),
        "human_direct": int(row["human_direct"]),
        "manual_review_notes": row["manual_review_notes"],
        "agent_label": label,
        "confidence": confidence,
        "reason": reason,
        "model": LLM_MODEL,
        "prompt_version": SEMANTIC_PROMPT_VERSION
    }

    pd.DataFrame(saved_heldout.values()).to_csv(
        HELDOUT_PATH, index=False
    )

    print(f"{i}/40: Agent={label}")

# 7. Evaluate direct vs. not-direct agreement.
heldout_results = pd.DataFrame(saved_heldout.values())

valid = heldout_results[
    heldout_results["agent_label"].isin(
        ["direct", "partial", "no_match"]
    )
].copy()

valid["human_is_direct"] = valid["human_direct"] == 1
valid["agent_is_direct"] = valid["agent_label"] == "direct"

tp = int((
    valid["human_is_direct"]
    & valid["agent_is_direct"]
).sum())

tn = int((
    ~valid["human_is_direct"]
    & ~valid["agent_is_direct"]
).sum())

fp = int((
    ~valid["human_is_direct"]
    & valid["agent_is_direct"]
).sum())

fn = int((
    valid["human_is_direct"]
    & ~valid["agent_is_direct"]
).sum())

def safe_rate(numerator, denominator):
    return numerator / denominator if denominator else float("nan")

print("\n" + "=" * 75)
print("HELD-OUT VALIDATION RESULTS")
print("=" * 75)

print("Successful evaluations:", len(valid), "/40")
print("Errors:", (
    heldout_results["agent_label"] == "ERROR"
).sum())

if len(valid) != 40:
    print("WARNING: Results are incomplete; metrics are provisional.")

print("\nClassification breakdown:")
print(pd.crosstab(
    valid["human_direct"].map({
        1: "Direct",
        0: "No confirmed direct"
    }),
    valid["agent_label"],
    margins=True
).to_string())

print("\nDIRECT-MATCH DETECTION")
print("True positives:", tp)
print("True negatives:", tn)
print("False positives:", fp)
print("False negatives:", fn)

print(
    "Direct recall:",
    f"{safe_rate(tp, tp + fn):.2%}"
)

print(
    "Direct precision:",
    f"{safe_rate(tp, tp + fp):.2%}"
)

print(
    "Non-direct specificity:",
    f"{safe_rate(tn, tn + fp):.2%}"
)

print(
    "Overall binary agreement:",
    f"{safe_rate(tp + tn, len(valid)):.2%}"
)

# 8. Print disagreements without truncated DataFrames.
disagreements = valid[
    valid["human_is_direct"]
    != valid["agent_is_direct"]
].copy()

print("\n" + "=" * 75)
print("CASES REQUIRING REVIEW:", len(disagreements))
print("=" * 75)

for _, row in disagreements.iterrows():
    print("\n" + "-" * 75)
    print("Disease:", row["disease"])
    print("Priority:", row["priority_label"])
    print("Trial:", row["nct_id"])
    print("Endpoint:", row["endpoint_id"])
    print("Human direct:", bool(row["human_direct"]))
    print("Agent:", row["agent_label"])
    print("Confidence:", row["confidence"])

    print("\nAgent reasoning:")
    print(textwrap.fill(str(row["reason"]), width=100))

    print("\nOriginal reviewer:")
    print(textwrap.fill(
        str(row["manual_review_notes"]), width=100
    ))

print("\nSaved:", HELDOUT_PATH)


HELD-OUT BENCHMARK READY
Cases: 40
Human direct: 10
Human no-direct: 30
1/40: Agent=partial
2/40: Agent=partial
3/40: Agent=no_match
4/40: Agent=partial
5/40: Agent=direct
6/40: Agent=direct
7/40: Agent=no_match
8/40: Agent=direct
9/40: Agent=no_match
10/40: Agent=no_match
11/40: Agent=partial
12/40: Agent=direct
13/40: Agent=direct
14/40: Agent=no_match
15/40: Agent=no_match
16/40: Agent=partial
17/40: Agent=no_match
18/40: Agent=no_match
19/40: Agent=partial
20/40: Agent=partial
21/40: Agent=no_match
22/40: Agent=partial
23/40: Agent=no_match
24/40: Agent=no_match
25/40: Agent=direct
26/40: Agent=direct
27/40: Agent=direct
28/40: Agent=no_match
29/40: Agent=no_match
30/40: Agent=no_match
31/40: Agent=no_match
32/40: Agent=no_match
33/40: Agent=no_match
34/40: Agent=direct
35/40: Agent=no_match
36/40: Agent=no_match
37/40: Agent=partial
38/40: Agent=no_match
39/40: Agent=no_match
40/40: Agent=no_match

HELD-OUT VALIDATION RESULTS
Successful evaluations: 40 /40
Errors: 0

Classificatio

In [ ]:

# ============================================================
# TRIALSIGNAL SEMANTIC AGENT V3
# FIVE-CASE SEQUENTIAL CANDIDATE SEARCH
# ============================================================
# Semantic-only test. Scope Agent is NOT integrated yet.
# Reuses earlier V3 judgments and checkpoints new calls.

import pandas as pd

assert SEMANTIC_PROMPT_VERSION == "semantic_v3_constituent"

ID_COLS = ["disease", "priority_id", "nct_id", "endpoint_id"]
VALID_LABELS = {"direct", "partial", "no_match"}

TRACE_PATH = DATA_DIR / "trialsignal_semantic_v3_search_trace.csv"

def candidate_key(row):
    return tuple(str(row[c]) for c in ID_COLS)

# Load previously completed V3 classifications.
cache = {}

source_files = [
    DATA_DIR / "trialsignal_semantic_v3_pilot14.csv",
    DATA_DIR / "trialsignal_semantic_v3_heldout40.csv",
    TRACE_PATH
]

for path in source_files:
    if not path.exists():
        continue

    df = pd.read_csv(path)

    for record in df.to_dict("records"):
        if (
            record.get("prompt_version") == SEMANTIC_PROMPT_VERSION
            and record.get("model") == LLM_MODEL
            and record.get("agent_label") in VALID_LABELS
        ):
            cache[candidate_key(record)] = record

print("Cached V3 endpoint judgments:", len(cache))

# Persist the search trace independently of earlier pilots.
trace = {}

if TRACE_PATH.exists():
    old_trace = pd.read_csv(TRACE_PATH)
    for record in old_trace.to_dict("records"):
        if (
            record.get("prompt_version") == SEMANTIC_PROMPT_VERSION
            and record.get("model") == LLM_MODEL
        ):
            trace[candidate_key(record)] = record


def save_trace():
    pd.DataFrame(trace.values()).to_csv(
        TRACE_PATH, index=False
    )


def semantic_scan_trial(disease, priority_id, nct_id):
    """
    Evaluate top-10 candidates in retrieval order.

    Direct: stop and send for future scope validation.
    Partial: retain first partial and continue searching.
    No match: continue searching.
    Error: stop as incomplete, never silently call it no_match.
    """

    candidates = semantic_agent_inputs[
        (semantic_agent_inputs["disease"] == disease)
        & (semantic_agent_inputs["priority_id"] == priority_id)
        & (semantic_agent_inputs["nct_id"] == nct_id)
    ].sort_values("within_trial_rank").head(10)

    if candidates.empty:
        raise ValueError(
            f"No candidates: {disease}, {priority_id}, {nct_id}"
        )

    checked = []
    first_partial = None
    accepted = None
    stop_reason = None

    for _, row in candidates.iterrows():
        key = candidate_key(row)

        if key in cache:
            result = cache[key]
            label = result["agent_label"]
            source = "cache"

        else:
            source = "API"

            try:
                response = semantic_match_candidate(row)

                result = {
                    **{c: row[c] for c in ID_COLS},
                    "candidate_rank": int(row["within_trial_rank"]),
                    "agent_label": response["semantic_label"],
                    "confidence": response["confidence"],
                    "reason": response["reason"],
                    "model": LLM_MODEL,
                    "prompt_version": SEMANTIC_PROMPT_VERSION
                }

                cache[key] = result

            except Exception as e:
                stop_reason = "incomplete_api_error"
                print(f"  ERROR at rank {row['within_trial_rank']}: {e}")
                break

        # Save the judgment, including results reused from cache.
        trace[key] = dict(result)
        save_trace()

        label = result["agent_label"]
        rank = int(row["within_trial_rank"])

        print(f"  Rank {rank}: {label} ({source})")

        checked.append(result)

        if label == "direct":
            accepted = result
            stop_reason = "direct_found_scope_pending"
            break

        if label == "partial" and first_partial is None:
            first_partial = result

    if stop_reason == "incomplete_api_error":
        final_label = "incomplete"

    elif accepted is not None:
        final_label = "direct"

    elif first_partial is not None:
        accepted = first_partial
        final_label = "partial"
        stop_reason = "top10_exhausted_partial"

    else:
        final_label = "no_match"
        stop_reason = "top10_exhausted_no_match"

    return {
        "disease": disease,
        "priority_id": priority_id,
        "nct_id": nct_id,
        "provisional_label": final_label,
        "selected_endpoint_id": (
            accepted["endpoint_id"] if accepted else None
        ),
        "selected_rank": (
            accepted["candidate_rank"] if accepted else None
        ),
        "candidates_checked": len(checked),
        "stop_reason": stop_reason,
        "scope_status": "not_evaluated"
    }


# Five deliberately different test cases.
test_cases = [
    ("Dravet Syndrome", "DRAVET_P05", "NCT04940624"),
    (
        "Facioscapulohumeral Muscular Dystrophy",
        "FSHD_P08",
        "NCT07038200"
    ),
    ("Hereditary Angioedema", "HAE_P08", "NCT03888755"),
    ("Spinal Muscular Atrophy", "SMA_P06", "NCT06191354"),
    ("Sickle Cell Disease", "SCD_P07", "NCT06823219")
]

search_results = []

for i, case in enumerate(test_cases, 1):
    print("\n" + "=" * 70)
    print(f"CASE {i}: {case[0]} | {case[1]} | {case[2]}")
    print("=" * 70)

    result = semantic_scan_trial(*case)
    search_results.append(result)

# Summary: plain text to avoid Colab table truncation.
summary = pd.DataFrame(search_results)

print("\n" + "=" * 70)
print("SEMANTIC-ONLY SEARCH SUMMARY")
print("=" * 70)

print(summary[
    [
        "priority_id",
        "provisional_label",
        "selected_endpoint_id",
        "selected_rank",
        "candidates_checked",
        "stop_reason"
    ]
].to_string(index=False))

summary.to_csv(
    DATA_DIR / "trialsignal_semantic_v3_search5_summary.csv",
    index=False
)

print("\nTrace saved:", TRACE_PATH)
print("All accepted matches remain subject to Scope Agent review.")


Cached V3 endpoint judgments: 54

CASE 1: Dravet Syndrome | DRAVET_P05 | NCT04940624
  Rank 1: direct (cache)

CASE 2: Facioscapulohumeral Muscular Dystrophy | FSHD_P08 | NCT07038200
  Rank 1: partial (cache)
  Rank 2: partial (API)
  Rank 3: partial (API)
  Rank 4: partial (API)
  Rank 5: partial (API)
  Rank 6: partial (API)
  Rank 7: no_match (API)
  Rank 8: partial (API)
  Rank 9: no_match (API)
  Rank 10: no_match (API)

CASE 3: Hereditary Angioedema | HAE_P08 | NCT03888755
  Rank 1: no_match (cache)
  Rank 2: no_match (API)
  Rank 3: partial (API)
  Rank 4: no_match (API)
  Rank 5: no_match (API)
  Rank 6: partial (API)
  Rank 7: partial (API)
  Rank 8: no_match (API)
  Rank 9: no_match (API)
  Rank 10: no_match (API)

CASE 4: Spinal Muscular Atrophy | SMA_P06 | NCT06191354
  Rank 1: partial (cache)
  Rank 2: partial (API)
  Rank 3: partial (API)
  Rank 4: no_match (API)
  Rank 5: no_match (API)

CASE 5: Sickle Cell Disease | SCD_P07 | NCT06823219
  Rank 1: no_match (API)
  Rank 

In [ ]:

# ============================================================
# TRIALSIGNAL — SELF-CONTAINED V3 PARTIAL-MATCH AUDIT
# ============================================================
# No API calls.
# No dependency on previous notebook variables.
# Safe after a Colab runtime reset.

from pathlib import Path
import pandas as pd
import textwrap

DATA_DIR = Path("./TrialSignal/data")

# Restore Drive access only if necessary.
if not DATA_DIR.is_dir():
    from google.colab import drive
    drive.mount("/content/portfolio-drive")

TRACE_PATH = DATA_DIR / "trialsignal_semantic_v3_search_trace.csv"
POOL_PATH = DATA_DIR / "trialsignal_top10_agent_candidates_clean.csv"
AUDIT_PATH = DATA_DIR / "trialsignal_semantic_v3_partial_audit.txt"

for path in [TRACE_PATH, POOL_PATH]:
    if not path.is_file():
        raise FileNotFoundError(
            f"Required saved file missing: {path}"
        )

# Load from disk, not from notebook memory.
trace = pd.read_csv(TRACE_PATH)
pool = pd.read_csv(POOL_PATH)

KEYS = ["disease", "priority_id", "nct_id", "endpoint_id"]

# Select only successful V3 partial judgments.
partial = trace.loc[
    (trace["prompt_version"] == "semantic_v3_constituent")
    & (trace["agent_label"] == "partial")
].drop_duplicates(KEYS, keep="last").copy()

# Authoritative descriptions come from the frozen corpus.
CONTEXT_COLS = [
    "priority_label",
    "priority_description",
    "endpoint_name",
    "endpoint_semantic_text"
]

context = pool[
    KEYS + CONTEXT_COLS
].drop_duplicates(KEYS)

# CRITICAL FIX:
# Remove duplicate text columns before joining.
# This prevents _x/_y suffixes and KeyErrors.
partial = partial.drop(
    columns=CONTEXT_COLS,
    errors="ignore"
)

audit = partial.merge(
    context,
    on=KEYS,
    how="left",
    validate="one_to_one"
)

if audit["endpoint_name"].isna().any():
    raise ValueError(
        "Some saved endpoint IDs were not found "
        "in the frozen candidate dataset."
    )

audit = audit.sort_values(
    ["disease", "priority_id", "candidate_rank"]
)

def wrap(value):
    return textwrap.fill(str(value), width=100)

output = []

for _, row in audit.iterrows():
    output.extend([
        "\n" + "=" * 90,
        f"DISEASE: {row['disease']}",
        f"PRIORITY: {row['priority_id']}",
        f"TRIAL: {row['nct_id']}",
        f"RANK: {int(row['candidate_rank'])}",
        "",
        "PATIENT PRIORITY:",
        wrap(row["priority_label"]),
        "",
        "PRIORITY DEFINITION:",
        wrap(row["priority_description"]),
        "",
        "ENDPOINT:",
        wrap(row["endpoint_name"]),
        "",
        "FULL ENDPOINT DESCRIPTION:",
        wrap(row["endpoint_semantic_text"]),
        "",
        f"AGENT LABEL: {row['agent_label']}",
        f"CONFIDENCE: {row['confidence']}",
        "",
        "AGENT REASONING:",
        wrap(row["reason"])
    ])

audit_text = "\n".join(output)

AUDIT_PATH.write_text(audit_text, encoding="utf-8")

print("AUDIT COMPLETED")
print("Partial judgments:", len(audit))
print("Saved to:", AUDIT_PATH)
print("No API calls made.")

print("\n" + audit_text)


AUDIT COMPLETED
Partial judgments: 19
Saved to: ./TrialSignal/data/trialsignal_semantic_v3_partial_audit.txt
No API calls made.


DISEASE: Facioscapulohumeral Muscular Dystrophy
PRIORITY: FSHD_P08
TRIAL: NCT07038200
RANK: 1

PATIENT PRIORITY:
Maintain independence

PRIORITY DEFINITION:
Preserve the ability to live and function independently.

ENDPOINT:
Change from Baseline to Week 78 in Quality of Life in Neurological Disorders (NeuroQoL) Upper
Extremity Function

FULL ENDPOINT DESCRIPTION:
Change from Baseline to Week 78 in Quality of Life in Neurological Disorders (NeuroQoL) Upper
Extremity Function Assessment of health-related quality of life using the Quality of Life in
Neurological Disorders (Neuro-QoL) questionnaire

AGENT LABEL: partial
CONFIDENCE: 0.94

AGENT REASONING:
The endpoint measures self-reported upper-extremity function, which may support independent living,
but it does not directly measure the ability to live and function independently.

DISEASE: Facioscapulohumeral 

In [ ]:

# ============================================================
# TRIALSIGNAL — SEMANTIC AGENT V4: PROXY GUARD
# Experimental version. Leaves validated V3 unchanged.
# ============================================================

import hashlib
import json
import pandas as pd

V4_PROMPT_VERSION = "semantic_v4_proxy_guard"

V4_RULES = """
You are TrialSignal's Semantic Match Agent. Judge whether the ONE supplied
registered clinical-trial endpoint MEASURES the supplied patient priority.
Base the decision only on supplied patient-priority and endpoint evidence.
Do not evaluate trial/disease/cohort applicability; that belongs to Scope Agent.

DIRECT:
The endpoint explicitly measures the patient priority or an identifiable
constituent outcome explicitly belonging to it. A broad priority does NOT
require every constituent outcome to be measured. If a relevant domain of an
instrument is separately reported, that domain can establish direct measurement.

PARTIAL:
The endpoint measures a specific, observable proxy or closely connected
functional/physiological aspect of the priority, even though the priority itself
is not directly measured. There must be a defensible measurement relationship:
you must be able to name (a) the actual measured variable and (b) how that
variable indicates an aspect of the patient-priority outcome without assuming
unstated measurements or a chain of hypothetical treatment effects.
A proxy is not direct merely because its outcome is clinically important.

NO_MATCH:
The endpoint does not directly measure the priority and no defensible close
proxy is established by the endpoint wording. Disease association, general
symptom improvement when the target symptom is not identified, general
clinical benefit, treatment response, a risk factor, or something that might
indirectly improve the priority through several unmeasured steps are NOT enough
for partial alignment. Do not award partial just because an association is
possible. Unspecified composites cannot be assumed to contain a target item.

ORDER OF DECISION:
1. Identify the endpoint's actual measured variable(s) from supplied text.
2. Identify the patient-priority construct or explicit constituent outcome.
3. If directly measured, label direct.
4. Otherwise, if a concrete and close measurement proxy is justified, partial.
5. Otherwise, no_match.

Your reason must name the measured variable and the exact relationship (or
missing relationship). Judge ONLY this endpoint; do not use other outcomes from
the same trial. Do not infer endpoint measurements from medical knowledge alone.
Cosine similarity, retrieval rank, and predicted treatment effects are not
evidence of measurement. Do not score alignment or perform arithmetic.
Confidence is judgment certainty, NOT a calibrated probability. Use lower
confidence when the measurement relationship is genuinely ambiguous.
Return one label: direct, partial, or no_match, plus confidence and short reason.
"""

V4_PROMPT_SHA256 = hashlib.sha256(
    V4_RULES.encode("utf-8")
).hexdigest()


def semantic_match_candidate_v4(row):
    """Experimental V4 classifier; does not modify V3."""

    if client is None:
        raise RuntimeError(
            "OpenAI client not available. Check the reconnect cell."
        )

    required = [
        "disease", "priority_id", "nct_id", "endpoint_id",
        "within_trial_rank", "priority_label",
        "priority_description", "evidence_text",
        "endpoint_name", "endpoint_type",
        "endpoint_semantic_text"
    ]

    missing = [
        k for k in required
        if k not in row.index or pd.isna(row[k])
    ]

    if missing:
        raise ValueError(
            f"Missing semantic input fields: {missing}"
        )

    payload = {
        "disease": str(row["disease"]),
        "priority": str(row["priority_label"]),
        "priority_definition": str(row["priority_description"]),
        "patient_evidence": str(row["evidence_text"])[:1500],
        "endpoint_name": str(row["endpoint_name"]),
        "endpoint_type": str(row["endpoint_type"]),
        "endpoint_text": str(row["endpoint_semantic_text"])
    }

    response = client.responses.create(
        model=LLM_MODEL,
        instructions=V4_RULES,
        input=json.dumps(payload, ensure_ascii=False),
        text={
            "format": {
                "type": "json_schema",
                "name": "trialsignal_semantic_match_v4",
                "strict": True,
                "schema": SEMANTIC_SCHEMA
            }
        }
    )

    if not response.output_text:
        raise ValueError(
            "OpenAI returned no structured semantic output."
        )

    parsed = json.loads(response.output_text)

    label = parsed["semantic_label"]
    confidence = float(parsed["confidence"])
    reason = parsed["reason"]

    if label not in {"direct", "partial", "no_match"}:
        raise ValueError(f"Invalid V4 label: {label}")

    if not 0 <= confidence <= 1:
        raise ValueError("Confidence outside 0..1")

    if not isinstance(reason, str) or not reason.strip():
        raise ValueError("Empty V4 explanation")

    return {
        "disease": row["disease"],
        "priority_id": row["priority_id"],
        "nct_id": row["nct_id"],
        "endpoint_id": row["endpoint_id"],
        "candidate_rank": int(row["within_trial_rank"]),
        "agent_label": label,
        "confidence": confidence,
        "reason": reason,
        "model": LLM_MODEL,
        "prompt_version": V4_PROMPT_VERSION,
        "prompt_sha256": V4_PROMPT_SHA256
    }


print("V4 ready:", V4_PROMPT_VERSION)
print("Unchanged V3:", SEMANTIC_PROMPT_VERSION)
print("No API calls made.")


V4 ready: semantic_v4_proxy_guard
Unchanged V3: semantic_v3_constituent
No API calls made.


In [ ]:

# ============================================================
# TRIALSIGNAL — V4 PARTIAL-MATCH TUNING PILOT
# 19 audited partials + 14 original pilot controls
# Up to 31 new API calls. Checkpointed to Drive.
# ============================================================

audit19 = get_partial_match_audit()

assert len(audit19) >= 19, (
    "Expected at least the 19 audited V3 partials."
)
assert len(v3_results) == 14, (
    "Expected saved 14-case V3 pilot from reconnect."
)

KEYS_V4 = [
    "disease", "priority_id", "nct_id", "endpoint_id"
]

audit_keys = audit19[KEYS_V4].drop_duplicates()
control_keys = v3_results[KEYS_V4].drop_duplicates()

target_keys = pd.concat(
    [audit_keys, control_keys],
    ignore_index=True
).drop_duplicates()

target_inputs = target_keys.merge(
    semantic_agent_inputs,
    on=KEYS_V4,
    how="left",
    validate="one_to_one"
).sort_values([
    "disease", "priority_id", "nct_id", "within_trial_rank"
])

assert len(target_inputs) == len(target_keys)
assert target_inputs["endpoint_semantic_text"].notna().all()

V4_PATH = (
    DATA_DIR / "trialsignal_semantic_v4_proxy_guard_pilot.csv"
)

# Load completed results only when model, version,
# and exact prompt fingerprint all match.
saved = {}

if V4_PATH.exists():
    old = pd.read_csv(V4_PATH)

    needed = set(
        KEYS_V4 + [
            "model", "prompt_version",
            "prompt_sha256", "agent_label"
        ]
    )

    if not needed.issubset(old.columns):
        raise ValueError(
            "Existing V4 checkpoint has an unexpected schema."
        )

    for r in old.to_dict("records"):
        if (
            r["model"] == LLM_MODEL
            and r["prompt_version"] == V4_PROMPT_VERSION
            and r["prompt_sha256"] == V4_PROMPT_SHA256
            and r["agent_label"] in {
                "direct", "partial", "no_match"
            }
        ):
            saved[result_key(r)] = r

print(
    f"Evaluation set: {len(target_inputs)} unique endpoints"
)
print(
    f"{len(audit_keys)} audited partials + "
    f"{len(control_keys)} pilot controls"
)
print(
    "Already saved for this exact prompt:",
    len(saved)
)

# Run the evaluation.
for i, (_, row) in enumerate(
    target_inputs.iterrows(), 1
):
    key = result_key(row)

    if key in saved:
        print(f"{i}/{len(target_inputs)} reused")
        continue

    try:
        result = semantic_match_candidate_v4(row)

    except Exception as exc:
        print(
            f"Stopped at {i}/{len(target_inputs)}: "
            f"{type(exc).__name__}: {exc}"
        )
        print(
            "Successful calls are saved. "
            "Rerun after fixing the issue."
        )
        break

    saved[key] = result

    # Persist every successful judgment.
    pd.DataFrame(saved.values()).to_csv(
        V4_PATH, index=False
    )

    print(
        f"{i}/{len(target_inputs)} "
        f"{row['priority_id']} "
        f"rank {int(row['within_trial_rank'])}: "
        f"{result['agent_label']}"
    )

# ============================================================
# RESULTS AND COMPARISON
# ============================================================

new_results = pd.DataFrame(saved.values())

if new_results.empty:
    print("No successful V4 results yet.")

else:
    new_results = target_keys.merge(
        new_results,
        on=KEYS_V4,
        how="left",
        validate="one_to_one"
    )

    completed = new_results["agent_label"].isin(
        {"direct", "partial", "no_match"}
    )

    print(
        f"\nCompleted {completed.sum()}/"
        f"{len(target_keys)} V4 endpoints"
    )

    if not completed.all():
        print(
            "CAUTION: Incomplete results. "
            "Rerun for remaining cases."
        )

    # Compare the 19 previously partial judgments.
    audited_comparison = audit_keys.merge(
        new_results[
            KEYS_V4 + [
                "agent_label", "confidence", "reason"
            ]
        ],
        on=KEYS_V4,
        how="left",
        validate="one_to_one"
    ).merge(
        audit19[
            KEYS_V4 + [
                "candidate_rank", "priority_label"
            ]
        ],
        on=KEYS_V4,
        how="left",
        validate="one_to_one"
    ).sort_values([
        "priority_id", "candidate_rank"
    ])

    print("\nV3 PARTIALS — V4 LABEL DISTRIBUTION")
    print(
        audited_comparison[
            "agent_label"
        ].value_counts(dropna=False).to_string()
    )

    print(
        audited_comparison[
            [
                "priority_id",
                "candidate_rank",
                "agent_label"
            ]
        ].to_string(index=False)
    )

    # Compare all original 14 pilot cases.
    controls = v3_results[
        KEYS_V4 + ["human_label", "agent_label"]
    ].rename(
        columns={"agent_label": "v3_label"}
    ).merge(
        new_results[
            KEYS_V4 + ["agent_label", "reason"]
        ].rename(
            columns={
                "agent_label": "v4_label",
                "reason": "v4_reason"
            }
        ),
        on=KEYS_V4,
        how="left",
        validate="one_to_one"
    )

    print("\nORIGINAL 14-CASE PILOT: V3 VS V4")

    print(
        controls[
            [
                "priority_id",
                "human_label",
                "v3_label",
                "v4_label"
            ]
        ].to_string(index=False)
    )

    if controls["v4_label"].notna().all():
        v3_agreement = (
            controls["human_label"] ==
            controls["v3_label"]
        ).mean()

        v4_agreement = (
            controls["human_label"] ==
            controls["v4_label"]
        ).mean()

        print(
            "\nV3 original-label agreement:",
            f"{v3_agreement:.2%}"
        )
        print(
            "V4 original-label agreement:",
            f"{v4_agreement:.2%}"
        )
        print(
            "Development-set diagnostic only; "
            "NOT held-out validation."
        )

    # Print full reasoning for changed partial judgments.
    print("\nV4 CHANGES TO AUDITED PARTIAL MATCHES")

    changed = audited_comparison[
        audited_comparison["agent_label"].notna()
        & audited_comparison["agent_label"].ne("partial")
    ]

    for _, r in changed.iterrows():
        print("\n" + "-" * 70)
        print(
            r["priority_id"],
            "rank", r["candidate_rank"],
            "->", r["agent_label"]
        )
        print("Reason:")
        print(r["reason"])

    print("\nResults saved to:", V4_PATH)


Evaluation set: 31 unique endpoints
19 audited partials + 14 pilot controls
Already saved for this exact prompt: 0
1/31 DRAVET_P02 rank 1: direct
2/31 DRAVET_P05 rank 1: direct
3/31 DRAVET_P05 rank 1: no_match
4/31 DRAVET_P06 rank 1: no_match
5/31 FSHD_P02 rank 1: direct
6/31 FSHD_P08 rank 1: direct
7/31 FSHD_P08 rank 2: partial
8/31 FSHD_P08 rank 3: partial
9/31 FSHD_P08 rank 4: partial
10/31 FSHD_P08 rank 5: no_match
11/31 FSHD_P08 rank 6: partial
12/31 FSHD_P08 rank 8: no_match
13/31 FA_P03 rank 2: no_match
14/31 HAE_P08 rank 1: no_match
15/31 HAE_P08 rank 3: no_match
16/31 HAE_P08 rank 6: no_match
17/31 HAE_P08 rank 7: no_match
18/31 FSGS_P01 rank 1: direct
19/31 FSGS_P07 rank 1: no_match
20/31 SCD_P01 rank 1: partial
21/31 SCD_P07 rank 3: no_match
22/31 SCD_P07 rank 4: no_match
23/31 SCD_P07 rank 6: no_match
24/31 SCD_P07 rank 7: partial
25/31 SCD_P07 rank 8: no_match
26/31 SCD_P07 rank 9: partial
27/31 SCD_P10 rank 1: direct
28/31 SMA_P06 rank 1: no_match
29/31 SMA_P06 rank 2: pa

In [ ]:

# ============================================================
# TRIALSIGNAL — V3/V4 SEMANTIC ADJUDICATION WORKSHEET
# ============================================================
# Reads saved Drive outputs only.
# No API calls or modification of frozen datasets.

from pathlib import Path
import pandas as pd

DATA_DIR = Path("./TrialSignal/data")

KEYS = ["disease", "priority_id", "nct_id", "endpoint_id"]
CASE_KEYS = ["disease", "priority_id", "nct_id"]

def load_saved(filename):
    path = DATA_DIR / filename
    if not path.is_file():
        raise FileNotFoundError(path)
    return pd.read_csv(path)

v3_pilot = load_saved(
    "trialsignal_semantic_v3_pilot14.csv"
)
v3_trace = load_saved(
    "trialsignal_semantic_v3_search_trace.csv"
)
v4_pilot = load_saved(
    "trialsignal_semantic_v4_proxy_guard_pilot.csv"
)
pool = load_saved(
    "trialsignal_top10_agent_candidates_clean.csv"
)
human = load_saved(
    "trialsignal_semantic_pilot_cases.csv"
)

# Restore authoritative V3 judgments.
v3 = pd.concat([
    v3_pilot,
    v3_trace
], ignore_index=True)

v3 = v3.loc[
    v3["prompt_version"] == "semantic_v3_constituent"
].drop_duplicates(KEYS, keep="last")

v3 = v3[
    KEYS + ["agent_label", "reason"]
].rename(columns={
    "agent_label": "v3_label",
    "reason": "v3_reason"
})

# Load experimental V4 judgments.
v4 = v4_pilot.loc[
    v4_pilot["prompt_version"] ==
    "semantic_v4_proxy_guard"
].drop_duplicates(KEYS, keep="last")

v4 = v4[
    KEYS + ["agent_label", "reason", "confidence"]
].rename(columns={
    "agent_label": "v4_label",
    "reason": "v4_reason",
    "confidence": "v4_confidence"
})

# Attach full original patient and endpoint descriptions.
context = pool[
    KEYS + [
        "within_trial_rank",
        "priority_label",
        "priority_description",
        "endpoint_name",
        "endpoint_semantic_text"
    ]
]

review = (
    v4.merge(v3, on=KEYS, how="left", validate="one_to_one")
      .merge(context, on=KEYS, how="left", validate="one_to_one")
)

assert len(review) == 31
assert review["v3_label"].notna().all()
assert review["endpoint_name"].notna().all()

# Attach human pilot labels where available.
review = review.merge(
    v3_pilot[KEYS + ["human_label"]].drop_duplicates(KEYS),
    on=KEYS,
    how="left",
    validate="one_to_one"
)

review = review.merge(
    human[
        CASE_KEYS + ["manual_review_notes"]
    ].drop_duplicates(CASE_KEYS),
    on=CASE_KEYS,
    how="left",
    validate="many_to_one"
)

# Identify V3/V4 disagreements.
changed = review["v3_label"] != review["v4_label"]

# Also inspect muscle strength vs independence.
extra_review = (
    (review["priority_id"] == "FSHD_P08")
    & (review["within_trial_rank"] == 6)
)

review_cases = review.loc[
    changed | extra_review
].copy()

review_cases = review_cases.sort_values([
    "disease", "priority_id", "nct_id", "within_trial_rank"
])

# Leave independent human adjudication blank.
review_cases["review_label"] = ""
review_cases["review_rationale"] = ""
review_cases["review_status"] = "pending"

OUTPUT = (
    DATA_DIR /
    "trialsignal_semantic_v4_adjudication_v1.csv"
)

# Never overwrite manually edited review decisions.
if OUTPUT.exists():
    print("Existing review file preserved:", OUTPUT)
else:
    review_cases.to_csv(OUTPUT, index=False)
    print("New review file saved:", OUTPUT)

print("\nV3/V4 classification changes:", int(changed.sum()))
print("Cases selected for adjudication:", len(review_cases))

print("\nREVIEW QUEUE:")
print(
    review_cases[
        [
            "priority_id",
            "within_trial_rank",
            "v3_label",
            "v4_label",
            "human_label"
        ]
    ].to_string(index=False)
)

print("\nNo API calls made.")


New review file saved: ./TrialSignal/data/trialsignal_semantic_v4_adjudication_v1.csv

V3/V4 classification changes: 12
Cases selected for adjudication: 13

REVIEW QUEUE:
priority_id  within_trial_rank v3_label v4_label human_label
   FSHD_P08                  1  partial   direct     partial
   FSHD_P08                  5  partial no_match         NaN
   FSHD_P08                  6  partial  partial         NaN
   FSHD_P08                  8  partial no_match         NaN
     FA_P03                  2  partial no_match     partial
    HAE_P08                  3  partial no_match         NaN
    HAE_P08                  6  partial no_match         NaN
    HAE_P08                  7  partial no_match         NaN
    SCD_P07                  3  partial no_match         NaN
    SCD_P07                  4  partial no_match         NaN
    SCD_P07                  6  partial no_match         NaN
    SCD_P07                  8  partial no_match         NaN
    SMA_P06                  1  part

In [ ]:

# ============================================================
# TRIALSIGNAL — SEMANTIC ADJUDICATION PROPOSALS
# ============================================================
# No API calls. Original review file is never overwritten.

import pandas as pd

SOURCE = DATA_DIR / "trialsignal_semantic_v4_adjudication_v1.csv"
OUTPUT = DATA_DIR / "trialsignal_semantic_v4_adjudication_proposals.csv"

review = pd.read_csv(SOURCE)
assert len(review) == 13

# Key: (priority_id, candidate rank)
proposals = {
    ("FSHD_P08", 1): (
        "partial",
        "Upper-extremity function is a meaningful component of physical autonomy, but independent living is not directly assessed."
    ),
    ("FSHD_P08", 5): (
        "no_match",
        "Fatigue is measured, not independent functioning."
    ),
    ("FSHD_P08", 6): (
        "no_match",
        "Isolated muscle strength does not establish performance of independent daily activities."
    ),
    ("FSHD_P08", 8): (
        "no_match",
        "Global disease severity does not specifically establish independent functioning."
    ),
    ("FA_P03", 2): (
        "partial",
        "Composite ataxia scales contain gait-related assessment but do not separately establish walking and fall outcomes."
    ),
    ("HAE_P08", 3): (
        "no_match",
        "Symptom relief is measured without identifying fatigue."
    ),
    ("HAE_P08", 6): (
        "no_match",
        "Participant-reported improvement concerns unspecified symptoms rather than fatigue."
    ),
    ("HAE_P08", 7): (
        "no_match",
        "Investigator-reported improvement does not establish fatigue measurement."
    ),
    ("SCD_P07", 3): (
        "no_match",
        "Time to hospitalized vaso-occlusive crisis does not measure progressive damage."
    ),
    ("SCD_P07", 4): (
        "no_match",
        "Walking capacity and exertion are not established as measures of progressive organ damage."
    ),
    ("SCD_P07", 6): (
        "no_match",
        "Acute chest syndrome episodes are clinically related to lung injury but do not establish progressive damage."
    ),
    ("SCD_P07", 8): (
        "no_match",
        "Hospitalized crisis frequency does not measure progressive organ damage."
    ),
    ("SMA_P06", 1): (
        "partial",
        "Developmental milestone attainment reflects functional development relevant to disease course, without directly quantifying progression slowing."
    )
}

keys = list(zip(
    review["priority_id"],
    review["within_trial_rank"].astype(int)
))

assert len(set(keys)) == len(review)
assert set(keys) == set(proposals)

review["proposed_label"] = [
    proposals[k][0] for k in keys
]

review["proposed_rationale"] = [
    proposals[k][1] for k in keys
]

review["requires_second_review"] = [
    k in {
        ("FSHD_P08", 6),
        ("SCD_P07", 6)
    }
    for k in keys
]

if OUTPUT.exists():
    print("Existing proposal file preserved:", OUTPUT)
else:
    review.to_csv(OUTPUT, index=False)
    print("Proposal file created:", OUTPUT)

print("\nPROPOSED CLASSIFICATIONS")
print(review["proposed_label"].value_counts().to_string())

print("\nSECOND-REVIEW CASES")
print(review.loc[
    review["requires_second_review"],
    [
        "priority_id",
        "within_trial_rank",
        "proposed_label"
    ]
].to_string(index=False))

print("\nOriginal human-review fields remain unchanged.")
print("No API calls made.")


Proposal file created: ./TrialSignal/data/trialsignal_semantic_v4_adjudication_proposals.csv

PROPOSED CLASSIFICATIONS
proposed_label
no_match    10
partial      3

SECOND-REVIEW CASES
priority_id  within_trial_rank proposed_label
   FSHD_P08                  6       no_match
    SCD_P07                  6       no_match

Original human-review fields remain unchanged.
No API calls made.


In [ ]:

# ============================================================
# TRIALSIGNAL — SEMANTIC AGENT V5
# Balanced Direct / Partial / No-Match Classification
# ============================================================
# Preserves the V3 baseline and all existing outputs.
# No API calls in this cell.

import pandas as pd
import json
import hashlib

V5_PROMPT_VERSION = "semantic_v5_balanced_proxy"

V5_RULES = """
You are the Semantic Match Agent for TrialSignal.

Determine whether ONE registered clinical-trial endpoint
MEASURES a validated patient-priority outcome.

Judge measurement, not merely clinical relevance.

DIRECT MATCH
The endpoint explicitly measures the patient priority or
a clearly identifiable constituent outcome of that priority.

A broad patient priority may contain multiple outcomes.
Direct measurement of one explicit constituent is sufficient.
The endpoint does not have to measure every dimension.

However, measuring something that merely CONTRIBUTES to an
outcome is not the same as measuring the outcome itself.

For a broad priority such as independence, a physical-function
test is not automatically a direct independence measure.

PARTIAL MATCH
The endpoint measures a specific, defensible proxy, functional
component, or closely related indicator of the patient priority.

A partial match requires an identifiable measurement relationship,
not merely a plausible medical association.

Acceptable partial relationships can include:
- A functional measure closely reflecting an aspect of a broader
  patient outcome without measuring the entire construct.
- A recognized clinical instrument containing relevant domains
  when the broader composite is reported rather than the
  relevant outcome separately.
- A disease-relevant functional or clinical indicator that provides
  meaningful evidence about disease progression without directly
  measuring progression rate.
- A structural or functional organ assessment closely connected
  to the target organ outcome without directly establishing damage.

For partial matches, explicitly identify:
1. What the endpoint measures.
2. What aspect of the patient priority it represents.
3. Why that relationship is sufficiently close to be meaningful.

NO MATCH
The endpoint neither directly measures the priority nor
establishes a defensible close proxy.

Reject relationships based only on:
- Shared disease context.
- Unspecified general symptom improvement.
- General disease severity or treatment response.
- An assumed downstream treatment benefit.
- Risk factors or clinical events that may eventually affect
  the priority but do not measure it or a close proxy.
- Broad clinical associations without a specific measurement link.

IMPORTANT DISTINCTIONS
- A measure of functional ability may partially reflect independence.
- A general symptom questionnaire does not establish measurement
  of a particular symptom unless that symptom is identified.
- A composite instrument may support partial alignment when
  relevant constituent domains are established, even if they
  are not separately reported.
- Do not invent unreported instrument domains.
- Measuring functional change can inform disease-course
  assessment without necessarily measuring slowed progression.
- An acute clinical complication does not automatically
  measure progressive long-term organ damage.
- A broad priority does not require complete coverage for
  a direct match.

DECISION PROCESS
1. Identify the actual measured endpoint variable.
2. Identify the patient-priority outcome.
3. Determine whether the outcome or explicit constituent
   is directly measured.
4. If not, determine whether the endpoint measures a
   specific and sufficiently close proxy.
5. Otherwise classify no_match.

Use only the supplied evidence.
Do not infer unreported endpoints.
Do not use cosine similarity or retrieval rank as semantic truth.
Do not evaluate disease or population applicability;
that belongs to the Scope Agent.
Do not perform alignment scoring or arithmetic.

Return:
- semantic_label: direct, partial, or no_match
- confidence: number between 0 and 1
- reason: concise explanation grounded in actual measurement

Confidence reflects judgment certainty, not a calibrated
probability of correctness. Preserve genuine ambiguity.
"""

V5_PROMPT_SHA256 = hashlib.sha256(
    V5_RULES.encode("utf-8")
).hexdigest()


def semantic_match_candidate_v5(row):
    """Independent V5 semantic classifier."""

    if client is None:
        raise RuntimeError("OpenAI client not initialized.")

    required = [
        "disease", "priority_id", "nct_id", "endpoint_id",
        "within_trial_rank", "priority_label",
        "priority_description", "evidence_text",
        "endpoint_name", "endpoint_type",
        "endpoint_semantic_text"
    ]

    missing = [
        col for col in required
        if col not in row.index or pd.isna(row[col])
    ]

    if missing:
        raise ValueError(f"Missing fields: {missing}")

    payload = {
        "disease": str(row["disease"]),
        "priority": str(row["priority_label"]),
        "priority_definition": str(row["priority_description"]),
        "patient_evidence": str(row["evidence_text"])[:1500],
        "endpoint_name": str(row["endpoint_name"]),
        "endpoint_type": str(row["endpoint_type"]),
        "endpoint_text": str(row["endpoint_semantic_text"])
    }

    response = client.responses.create(
        model=LLM_MODEL,
        instructions=V5_RULES,
        input=json.dumps(payload, ensure_ascii=False),
        text={
            "format": {
                "type": "json_schema",
                "name": "trialsignal_semantic_v5",
                "strict": True,
                "schema": SEMANTIC_SCHEMA
            }
        }
    )

    if not response.output_text:
        raise ValueError("No structured model response.")

    result = json.loads(response.output_text)

    label = result["semantic_label"]
    confidence = float(result["confidence"])
    reason = result["reason"]

    if label not in {"direct", "partial", "no_match"}:
        raise ValueError("Invalid semantic label.")

    if not 0 <= confidence <= 1:
        raise ValueError("Invalid confidence value.")

    if not isinstance(reason, str) or not reason.strip():
        raise ValueError("Missing explanation.")

    return {
        "disease": row["disease"],
        "priority_id": row["priority_id"],
        "nct_id": row["nct_id"],
        "endpoint_id": row["endpoint_id"],
        "candidate_rank": int(row["within_trial_rank"]),
        "agent_label": label,
        "confidence": confidence,
        "reason": reason,
        "model": LLM_MODEL,
        "prompt_version": V5_PROMPT_VERSION,
        "prompt_sha256": V5_PROMPT_SHA256
    }


print("Semantic Agent V5 defined.")
print("Model:", LLM_MODEL)
print("Prompt:", V5_PROMPT_VERSION)
print("V3 remains unchanged:", SEMANTIC_PROMPT_VERSION)
print("No API calls made.")


Semantic Agent V5 defined.
Model: gpt-6-luna
Prompt: semantic_v5_balanced_proxy
V3 remains unchanged: semantic_v3_constituent
No API calls made.


In [ ]:
# TrialSignal V5 — blind three-class evaluation set
from pathlib import Path
import numpy as np
import pandas as pd
import textwrap

DATA_DIR = Path('./TrialSignal/data')
CASE = ['disease', 'priority_id', 'nct_id']
KEY = CASE + ['endpoint_id']
SEED = 20261011
OUT = DATA_DIR / 'trialsignal_semantic_v5_blind35_review_v1.csv'
MANIFEST = DATA_DIR / 'trialsignal_semantic_v5_blind35_manifest_v1.csv'
READABLE = DATA_DIR / 'trialsignal_semantic_v5_blind35_review_v1.txt'

if not DATA_DIR.is_dir():
    raise FileNotFoundError('Google Drive data directory unavailable. Run master reconnect v4.0 first.')

# Never overwrite a partially or fully adjudicated review file.
if OUT.exists():
    review = pd.read_csv(OUT, keep_default_na=False)
    print('Existing blind review file preserved:', OUT)
else:
    pool = pd.read_csv(DATA_DIR / 'trialsignal_top10_agent_candidates_clean.csv')
    priorities = pd.read_csv(DATA_DIR / 'trialsignal_alignment_priorities.csv')
    benchmark = pd.read_csv(DATA_DIR / 'trialsignal_semantic_pilot_cases.csv')
    assert len(pool) == 63811 and len(benchmark) == 54

    # Exclude ENTIRE priority–trial cases seen in any previous evaluation.
    excluded = [benchmark[CASE]]
    previously_tested = [
        'trialsignal_semantic_v3_pilot14.csv',
        'trialsignal_semantic_v3_heldout40.csv',
        'trialsignal_semantic_v3_search_trace.csv',
        'trialsignal_semantic_v4_proxy_guard_pilot.csv',
        'trialsignal_semantic_v5_development25.csv',
    ]
    for name in previously_tested:
        p = DATA_DIR / name
        if p.exists():
            excluded.append(pd.read_csv(p, usecols=CASE))
    excluded = pd.concat(excluded, ignore_index=True).drop_duplicates()

    eligible = pool.merge(
        excluded.assign(_seen=1),
        on=CASE, how='left', validate='many_to_one'
    )
    eligible = eligible.loc[eligible['_seen'].isna()].drop(columns='_seen').copy()
    eligible['similarity'] = pd.to_numeric(eligible['similarity'], errors='raise')
    eligible['rank_band'] = np.select(
        [eligible['within_trial_rank'].eq(1),
         eligible['within_trial_rank'].between(2, 3)],
        ['rank1', 'rank2-3'], default='rank4-10'
    )
    eligible['similarity_band'] = eligible.groupby('disease')['similarity'].transform(
        lambda x: pd.qcut(x.rank(method='first'), 3,
                              labels=['low', 'medium', 'high'])
    ).astype(str)

    # Five disease-specific strata. Similarity is used for sampling only.
    slots = [
        ('rank1', 'high'), ('rank1', 'medium'), ('rank1', 'low'),
        ('rank2-3', 'high'), ('rank4-10', 'high')
    ]
    picks = []
    for d_ix, disease in enumerate(sorted(eligible['disease'].unique())):
        disease_pool = eligible[eligible['disease'].eq(disease)]
        used_trials, used_priorities = set(), set()
        for s_ix, (rank_band, sim_band) in enumerate(slots):
            options = disease_pool.loc[
                disease_pool['rank_band'].eq(rank_band)
                & disease_pool['similarity_band'].eq(sim_band)
                & ~disease_pool['nct_id'].isin(used_trials)
                & ~disease_pool['priority_id'].isin(used_priorities)
            ]
            if options.empty:
                raise RuntimeError(f'No candidate for {disease}: {rank_band}/{sim_band}')
            chosen = options.sample(n=1, random_state=SEED + 37*d_ix + 7*s_ix).iloc[0]
            picks.append(chosen)
            used_trials.add(chosen['nct_id'])
            used_priorities.add(chosen['priority_id'])

    selected = pd.DataFrame(picks).reset_index(drop=True)
    assert len(selected) == 35
    assert selected['disease'].nunique() == 7
    assert not selected.duplicated(KEY).any()
    assert not selected.duplicated(CASE).any()

    selected.insert(0, 'review_id', [f'V5-{i:03d}' for i in range(1, 36)])
    evidence = priorities[['disease', 'priority_id', 'evidence_text']].drop_duplicates(
        ['disease', 'priority_id']
    )
    selected = selected.merge(evidence, on=['disease', 'priority_id'],
                              how='left', validate='many_to_one')
    assert selected['evidence_text'].notna().all()
    selected['patient_evidence_excerpt'] = selected['evidence_text'].astype(str).str[:1500]

    manifest_cols = ['review_id'] + KEY + ['within_trial_rank',
                                            'rank_band', 'similarity_band', 'similarity']
    selected[manifest_cols].to_csv(MANIFEST, index=False)

    # Intentionally exclude cosine/rank and model outputs from reviewer CSV.
    cols = ['review_id', 'disease', 'priority_id', 'priority_label',
            'priority_description', 'patient_evidence_excerpt',
            'nct_id', 'endpoint_id', 'endpoint_type', 'endpoint_name',
            'endpoint_semantic_text']
    review = selected[cols].copy()
    review['human_label'] = ''
    review['human_reason'] = ''
    review['review_status'] = 'pending'
    review['reviewer'] = ''
    review.to_csv(OUT, index=False)

    lines = [
        'TRIALSIGNAL — BLINDED SEMANTIC VALIDATION (35 CANDIDATES)',
        'Direct: directly measures priority or a named constituent outcome.',
        'Partial: measures a specific, defensible close proxy.',
        'No match: no meaningful priority measurement or close proxy.',
        'Judge endpoint measurement only; Scope Agent handles population.',
        'Do not inspect model predictions or retrieval similarity while labeling.\n'
    ]
    for row in review.itertuples(index=False):
        lines += [
            '=' * 90,
            f'{row.review_id} | {row.disease} | {row.priority_id} | {row.nct_id}',
            'PRIORITY: ' + str(row.priority_label),
            'DEFINITION: ' + str(row.priority_description),
            'EVIDENCE: ' + str(row.patient_evidence_excerpt),
            'ENDPOINT: ' + str(row.endpoint_name),
            'MEASUREMENT: ' + str(row.endpoint_semantic_text),
            'HUMAN LABEL: ____   REASON: ____\n'
        ]
    READABLE.write_text('\n'.join(lines), encoding='utf-8')

print('Blind review cases:', len(review))
print('Cases per disease:')
print(review['disease'].value_counts().sort_index().to_string())
print('Human labels entered:', review['human_label'].astype(str).str.strip().ne('').sum())
print('Blind review CSV:', OUT)
print('Readable review text:', READABLE)
print('Selection manifest (not for reviewers):', MANIFEST)
print('No API calls. No historical files overwritten.')


Blind review cases: 35
Cases per disease:
disease
Dravet Syndrome                           5
Facioscapulohumeral Muscular Dystrophy    5
Friedreich's Ataxia                       5
Hereditary Angioedema                     5
Primary FSGS                              5
Sickle Cell Disease                       5
Spinal Muscular Atrophy                   5
Human labels entered: 0
Blind review CSV: ./TrialSignal/data/trialsignal_semantic_v5_blind35_review_v1.csv
Readable review text: ./TrialSignal/data/trialsignal_semantic_v5_blind35_review_v1.txt
Selection manifest (not for reviewers): ./TrialSignal/data/trialsignal_semantic_v5_blind35_manifest_v1.csv
No API calls. No historical files overwritten.


In [ ]:

# TRIALSIGNAL — FINALIZE V5 BLINDED VALIDATION LABELS
# No API calls. Does not overwrite the original worksheet.

import pandas as pd

source = DATA_DIR / "trialsignal_semantic_v5_blind35_proposed_labels_v1.csv"
output = DATA_DIR / "trialsignal_semantic_v5_blind35_labels_frozen_v1.csv"

if output.exists():
    raise FileExistsError(f"Finalized file already exists: {output}")

df = pd.read_csv(source, keep_default_na=False)

assert len(df) == 35
assert df["review_id"].is_unique
assert set(df["proposed_label"]) == {
    "direct", "partial", "no_match"
}

# Accept the proposed classifications and reasoning.
df["human_label"] = df["proposed_label"]
df["human_reason"] = df["proposed_rationale"]

# Preserve the actual provenance of the review.
df["review_status"] = "finalized"
df["reviewer"] = "user_accepted_assistant_proposals"
df["proposal_status"] = "accepted_by_user_for_validation"

# Save a new frozen version.
df.to_csv(output, index=False)

# Verify the saved file.
verified = pd.read_csv(output, keep_default_na=False)

assert len(verified) == 35
assert (verified["human_label"] == verified["proposed_label"]).all()
assert (verified["human_reason"] == verified["proposed_rationale"]).all()

print("VALIDATION LABELS FINALIZED")
print("\nClassification counts:")
print(verified["human_label"].value_counts().to_string())
print("\nSaved to:", output)


VALIDATION LABELS FINALIZED

Classification counts:
human_label
no_match    28
partial      5
direct       2

Saved to: ./TrialSignal/data/trialsignal_semantic_v5_blind35_labels_frozen_v1.csv


In [ ]:

# ============================================================
# TRIALSIGNAL — V5 DEVELOPMENT TEST
# 13 adjudication cases + 14 original controls
# 25 unique endpoints
# ============================================================
# Up to 25 API calls, each checkpointed to Google Drive.

from pathlib import Path
import pandas as pd

KEYS_V5 = [
    "disease", "priority_id", "nct_id", "endpoint_id"
]

ADJ_PATH = (
    DATA_DIR / "trialsignal_semantic_v4_adjudication_v1.csv"
)

V3_PATH = (
    DATA_DIR / "trialsignal_semantic_v3_pilot14.csv"
)

V4_PATH = (
    DATA_DIR / "trialsignal_semantic_v4_proxy_guard_pilot.csv"
)

V5_PATH = (
    DATA_DIR / "trialsignal_semantic_v5_development25.csv"
)

# Required saved files, not notebook-memory objects.
for path in [ADJ_PATH, V3_PATH, V4_PATH]:
    if not path.is_file():
        raise FileNotFoundError(path)

adjudication = pd.read_csv(ADJ_PATH)
v3_control = pd.read_csv(V3_PATH)
v4_previous = pd.read_csv(V4_PATH)

assert len(adjudication) == 13
assert len(v3_control) == 14
assert len(v4_previous) == 31

# Assemble distinct endpoint IDs.
target_keys = pd.concat([
    adjudication[KEYS_V5],
    v3_control[KEYS_V5]
], ignore_index=True).drop_duplicates()

assert len(target_keys) == 25

# Attach authoritative context from the master reconnect.
target_inputs = target_keys.merge(
    semantic_agent_inputs,
    on=KEYS_V5,
    how="left",
    validate="one_to_one"
).sort_values([
    "disease", "priority_id",
    "nct_id", "within_trial_rank"
])

assert len(target_inputs) == 25
assert target_inputs["endpoint_semantic_text"].notna().all()

# Resume saved results only from the exact V5 prompt.
saved_v5 = {}

if V5_PATH.exists():
    previous = pd.read_csv(V5_PATH)

    required_cols = set(
        KEYS_V5 + [
            "model", "prompt_version",
            "prompt_sha256", "agent_label"
        ]
    )
    if not required_cols.issubset(previous.columns):
        raise ValueError("Existing V5 checkpoint schema mismatch.")

    if not (
        (previous["model"] == LLM_MODEL)
        & (previous["prompt_version"] == V5_PROMPT_VERSION)
        & (previous["prompt_sha256"] == V5_PROMPT_SHA256)
    ).all():
        raise ValueError(
            "Existing V5 checkpoint uses a different model or prompt. "
            "Preserve it under a different filename before rerunning."
        )

    for record in previous.to_dict("records"):
        if record["agent_label"] in {
            "direct", "partial", "no_match"
        }:
            saved_v5[result_key(record)] = record

print("Development endpoints:", len(target_inputs))
print("Already saved:", len(saved_v5))

# Run V5.
for i, (_, row) in enumerate(
    target_inputs.iterrows(), start=1
):
    key = result_key(row)

    if key in saved_v5:
        print(f"{i}/25: reused")
        continue

    try:
        result = semantic_match_candidate_v5(row)

    except Exception as error:
        print(f"\nStopped at {i}/25: {error}")
        print("Completed results remain saved in Drive.")
        break

    saved_v5[key] = result

    pd.DataFrame(saved_v5.values()).to_csv(
        V5_PATH, index=False
    )

    print(
        f"{i}/25: {row['priority_id']} "
        f"rank {int(row['within_trial_rank'])} "
        f"-> {result['agent_label']}"
    )

# ============================================================
# COMPARE V5 TO PREVIOUS AGENT VERSIONS
# ============================================================

results_v5 = pd.DataFrame(saved_v5.values())

if results_v5.empty:
    print("No completed V5 evaluations.")

else:
    comparison = target_keys.merge(
        results_v5[
            KEYS_V5 + ["agent_label", "reason"]
        ].rename(columns={
            "agent_label": "v5_label",
            "reason": "v5_reason"
        }),
        on=KEYS_V5,
        how="left",
        validate="one_to_one"
    )

    comparison = comparison.merge(
        v4_previous[
            KEYS_V5 + ["agent_label"]
        ].rename(columns={"agent_label": "v4_label"}),
        on=KEYS_V5,
        how="left",
        validate="one_to_one"
    )

    # Retrieve V3 judgments from all available saved V3 files.
    v3_frames = []
    for filename in [
        "trialsignal_semantic_v3_pilot14.csv",
        "trialsignal_semantic_v3_heldout40.csv",
        "trialsignal_semantic_v3_search_trace.csv"
    ]:
        path = DATA_DIR / filename
        if path.exists():
            v3_frames.append(pd.read_csv(path))

    historical_v3 = pd.concat(
        v3_frames, ignore_index=True
    )

    historical_v3 = historical_v3[
        historical_v3["prompt_version"] ==
        "semantic_v3_constituent"
    ].drop_duplicates(KEYS_V5, keep="last")

    comparison = comparison.merge(
        historical_v3[
            KEYS_V5 + ["agent_label"]
        ].rename(columns={"agent_label": "v3_label"}),
        on=KEYS_V5,
        how="left",
        validate="one_to_one"
    )

    comparison = comparison.merge(
        adjudication[KEYS_V5].assign(
            in_review_queue=True
        ),
        on=KEYS_V5,
        how="left",
        validate="one_to_one"
    )

    comparison["in_review_queue"] = (
        comparison["in_review_queue"].fillna(False)
    )

    print("\n" + "=" * 75)
    print("V5 DEVELOPMENT COMPARISON")
    print("=" * 75)

    print("Completed:", len(results_v5), "/25")

    print("\nV5 label distribution:")
    print(results_v5["agent_label"].value_counts().to_string())

    print("\n13 DISPUTED CASES:")
    disputed = comparison[
        comparison["in_review_queue"]
    ].copy()

    print(
        disputed[
            [
                "priority_id",
                "v3_label",
                "v4_label",
                "v5_label"
            ]
        ].to_string(index=False)
    )

    print("\nORIGINAL 14 CONTROL CASES:")

    controls = v3_control[
        KEYS_V5 + ["human_label"]
    ].merge(
        comparison[
            KEYS_V5 + ["v3_label", "v4_label", "v5_label"]
        ],
        on=KEYS_V5,
        how="left",
        validate="one_to_one"
    )

    print(
        controls[
            [
                "priority_id", "human_label",
                "v3_label", "v4_label", "v5_label"
            ]
        ].to_string(index=False)
    )

    if controls["v5_label"].notna().all():
        print(
            "\nV5 agreement with original pilot labels:",
            f"{(controls['human_label'] == controls['v5_label']).mean():.2%}"
        )
        print("Development diagnostic only.")

    print("\nDISPUTED-CASE EXPLANATIONS:")

    for _, r in disputed.iterrows():
        print("\n" + "-" * 70)
        print(r["priority_id"], "| V5:", r["v5_label"])
        print(str(r["v5_reason"]))

    print("\nSaved:", V5_PATH)


Development endpoints: 25
Already saved: 0
1/25: DRAVET_P02 rank 1 -> direct
2/25: DRAVET_P05 rank 1 -> direct
3/25: DRAVET_P05 rank 1 -> no_match
4/25: DRAVET_P06 rank 1 -> no_match
5/25: FSHD_P02 rank 1 -> direct
6/25: FSHD_P08 rank 1 -> partial
7/25: FSHD_P08 rank 5 -> no_match
8/25: FSHD_P08 rank 6 -> partial
9/25: FSHD_P08 rank 8 -> no_match
10/25: FA_P03 rank 2 -> partial
11/25: HAE_P08 rank 1 -> no_match
12/25: HAE_P08 rank 3 -> no_match
13/25: HAE_P08 rank 6 -> no_match
14/25: HAE_P08 rank 7 -> no_match
15/25: FSGS_P01 rank 1 -> direct
16/25: FSGS_P07 rank 1 -> no_match
17/25: SCD_P01 rank 1 -> partial
18/25: SCD_P07 rank 3 -> no_match
19/25: SCD_P07 rank 4 -> no_match
20/25: SCD_P07 rank 6 -> no_match
21/25: SCD_P07 rank 8 -> no_match
22/25: SCD_P07 rank 9 -> partial
23/25: SCD_P10 rank 1 -> direct
24/25: SMA_P06 rank 1 -> partial
25/25: SMA_P11 rank 1 -> no_match

V5 DEVELOPMENT COMPARISON
Completed: 25 /25

V5 label distribution:
agent_label
no_match    14
partial      6
dir

<notebook>/2689503728.py:200: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  comparison["in_review_queue"].fillna(False)


In [ ]:
# =============================================================
# TRIALSIGNAL — SEMANTIC AGENT V5 / BLIND35 VALIDATION
# Single Colab cell; saved Drive data are authoritative.
# No changes to frozen datasets or finalized reference labels.
# Up to 35 API calls; completed calls checkpointed to Drive.
# =============================================================
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import textwrap
import pandas as pd

DATA_DIR = Path('./TrialSignal/data')
if not DATA_DIR.is_dir():
    from google.colab import drive
    drive.mount('/content/portfolio-drive')
if not DATA_DIR.is_dir():
    raise FileNotFoundError(f'TrialSignal data directory unavailable: {DATA_DIR}')

LABELS_PATH = DATA_DIR / 'trialsignal_semantic_v5_blind35_labels_frozen_v1.csv'
POOL_PATH = DATA_DIR / 'trialsignal_top10_agent_candidates_clean.csv'
PRIORITIES_PATH = DATA_DIR / 'trialsignal_alignment_priorities.csv'
V5_DEV_PATH = DATA_DIR / 'trialsignal_semantic_v5_development25.csv'
PRED_PATH = DATA_DIR / 'trialsignal_semantic_v5_blind35_predictions_v1.csv'
EVAL_PATH = DATA_DIR / 'trialsignal_semantic_v5_blind35_evaluation_v1.csv'
SUMMARY_PATH = DATA_DIR / 'trialsignal_semantic_v5_blind35_metrics_v1.txt'
for path in (LABELS_PATH, POOL_PATH, PRIORITIES_PATH, V5_DEV_PATH):
    if not path.is_file():
        raise FileNotFoundError(f'Required Drive file missing: {path}')

MODEL = 'gpt-6-luna'
PROMPT_VERSION = 'semantic_v5_balanced_proxy'
V5_RULES_FROZEN = """
You are the Semantic Match Agent for TrialSignal.

Determine whether ONE registered clinical-trial endpoint
MEASURES a validated patient-priority outcome.

Judge measurement, not merely clinical relevance.

DIRECT MATCH
The endpoint explicitly measures the patient priority or
a clearly identifiable constituent outcome of that priority.

A broad patient priority may contain multiple outcomes.
Direct measurement of one explicit constituent is sufficient.
The endpoint does not have to measure every dimension.

However, measuring something that merely CONTRIBUTES to an
outcome is not the same as measuring the outcome itself.

For a broad priority such as independence, a physical-function
test is not automatically a direct independence measure.

PARTIAL MATCH
The endpoint measures a specific, defensible proxy, functional
component, or closely related indicator of the patient priority.

A partial match requires an identifiable measurement relationship,
not merely a plausible medical association.

Acceptable partial relationships can include:
- A functional measure closely reflecting an aspect of a broader
  patient outcome without measuring the entire construct.
- A recognized clinical instrument containing relevant domains
  when the broader composite is reported rather than the
  relevant outcome separately.
- A disease-relevant functional or clinical indicator that provides
  meaningful evidence about disease progression without directly
  measuring progression rate.
- A structural or functional organ assessment closely connected
  to the target organ outcome without directly establishing damage.

For partial matches, explicitly identify:
1. What the endpoint measures.
2. What aspect of the patient priority it represents.
3. Why that relationship is sufficiently close to be meaningful.

NO MATCH
The endpoint neither directly measures the priority nor
establishes a defensible close proxy.

Reject relationships based only on:
- Shared disease context.
- Unspecified general symptom improvement.
- General disease severity or treatment response.
- An assumed downstream treatment benefit.
- Risk factors or clinical events that may eventually affect
  the priority but do not measure it or a close proxy.
- Broad clinical associations without a specific measurement link.

IMPORTANT DISTINCTIONS
- A measure of functional ability may partially reflect independence.
- A general symptom questionnaire does not establish measurement
  of a particular symptom unless that symptom is identified.
- A composite instrument may support partial alignment when
  relevant constituent domains are established, even if they
  are not separately reported.
- Do not invent unreported instrument domains.
- Measuring functional change can inform disease-course
  assessment without necessarily measuring slowed progression.
- An acute clinical complication does not automatically
  measure progressive long-term organ damage.
- A broad priority does not require complete coverage for
  a direct match.

DECISION PROCESS
1. Identify the actual measured endpoint variable.
2. Identify the patient-priority outcome.
3. Determine whether the outcome or explicit constituent
   is directly measured.
4. If not, determine whether the endpoint measures a
   specific and sufficiently close proxy.
5. Otherwise classify no_match.

Use only the supplied evidence.
Do not infer unreported endpoints.
Do not use cosine similarity or retrieval rank as semantic truth.
Do not evaluate disease or population applicability;
that belongs to the Scope Agent.
Do not perform alignment scoring or arithmetic.

Return:
- semantic_label: direct, partial, or no_match
- confidence: number between 0 and 1
- reason: concise explanation grounded in actual measurement

Confidence reflects judgment certainty, not a calibrated
probability of correctness. Preserve genuine ambiguity.
"""
PROMPT_SHA = hashlib.sha256(V5_RULES_FROZEN.encode('utf-8')).hexdigest()

# Enforce exactly the prompt and model used in development, without manual editing.
dev = pd.read_csv(V5_DEV_PATH, usecols=['model', 'prompt_version', 'prompt_sha256'])
assert set(dev['model']) == {MODEL}, 'V5 development model mismatch. STOP.'
assert set(dev['prompt_version']) == {PROMPT_VERSION}, 'V5 development version mismatch. STOP.'
assert set(dev['prompt_sha256']) == {PROMPT_SHA}, 'V5 prompt text differs from the tested V5. STOP.'

LABELS = ('direct', 'partial', 'no_match')
KEYS = ['disease', 'priority_id', 'nct_id', 'endpoint_id']

# Verify frozen reference labels, without passing them to the model.
reference = pd.read_csv(LABELS_PATH, keep_default_na=False)
assert len(reference) == 35 and reference['review_id'].is_unique
assert not reference.duplicated(KEYS).any()
assert reference['review_status'].eq('finalized').all()
assert reference['human_label'].value_counts().to_dict() == {'no_match': 28, 'partial': 5, 'direct': 2}
assert reference['human_label'].eq(reference['proposed_label']).all()

# Rebuild ONLY model inputs from frozen Drive sources.
pool = pd.read_csv(POOL_PATH)
priorities = pd.read_csv(PRIORITIES_PATH, usecols=['disease','priority_id','evidence_text'])
assert len(pool) == 63811 and len(priorities) == 73
inputs = reference[['review_id'] + KEYS].merge(
    pool, on=KEYS, how='left', validate='one_to_one'
).merge(
    priorities, on=['disease','priority_id'], how='left', validate='many_to_one'
).sort_values('review_id').reset_index(drop=True)
assert len(inputs) == 35 and inputs['endpoint_semantic_text'].notna().all()
assert inputs['evidence_text'].notna().all()

# The reviewer worksheet must reflect the SAME authoritative endpoint text.
for col in ('priority_label', 'priority_description', 'endpoint_type',
            'endpoint_name', 'endpoint_semantic_text'):
    source = reference.set_index('review_id').loc[inputs['review_id'], col].astype(str).tolist()
    actual = inputs[col].astype(str).tolist()
    if actual != source:
        raise AssertionError(f'Frozen corpus vs label-sheet text mismatch: {col}. STOP.')
if inputs['evidence_text'].astype(str).str[:1500].tolist() != (
    reference.set_index('review_id').loc[inputs['review_id'], 'patient_evidence_excerpt'].astype(str).tolist()
):
    raise AssertionError('Patient evidence excerpt mismatch. STOP.')

# The client is initialized by master reconnect v4.0; never expose API keys.
if 'client' not in globals() or client is None:
    raise RuntimeError('OpenAI client missing. Run Master Agent Reconnect v4.0 first.')

SCHEMA = {
    'type': 'object',
    'properties': {
        'semantic_label': {'type': 'string', 'enum': list(LABELS)},
        'confidence': {'type': 'number'},
        'reason': {'type': 'string'},
    },
    'required': ['semantic_label', 'confidence', 'reason'],
    'additionalProperties': False
}

def call_v5(row):
    # NO human labels, human rationales, score, or retrieval rank in model input.
    prompt_input = {
        'disease': str(row['disease']),
        'priority': str(row['priority_label']),
        'priority_definition': str(row['priority_description']),
        'patient_evidence': str(row['evidence_text'])[:1500],
        'endpoint_name': str(row['endpoint_name']),
        'endpoint_type': str(row['endpoint_type']),
        'endpoint_text': str(row['endpoint_semantic_text']),
    }
    response = client.responses.create(
        model=MODEL,
        instructions=V5_RULES_FROZEN,
        input=json.dumps(prompt_input, ensure_ascii=False),
        text={'format': {
            'type': 'json_schema', 'name': 'trialsignal_semantic_v5',
            'strict': True, 'schema': SCHEMA
        }}
    )
    if not response.output_text:
        raise ValueError('No structured model response returned.')
    parsed = json.loads(response.output_text)
    label = parsed['semantic_label']
    confidence = float(parsed['confidence'])
    reason = parsed['reason']
    if label not in LABELS or not 0 <= confidence <= 1 or not isinstance(reason, str) or not reason.strip():
        raise ValueError('Invalid semantic agent response.')
    return {
        'review_id': row['review_id'],
        **{key: row[key] for key in KEYS},
        'candidate_rank': int(row['within_trial_rank']),
        'agent_label': label, 'confidence': confidence, 'reason': reason,
        'model': MODEL, 'prompt_version': PROMPT_VERSION,
        'prompt_sha256': PROMPT_SHA,
        'evaluated_at_utc': datetime.now(timezone.utc).isoformat(),
    }

# Load only exact-version checkpoints. Never silently reuse a changed prompt.
saved = {}
if PRED_PATH.exists():
    previous = pd.read_csv(PRED_PATH, keep_default_na=False)
    must_have = set(['review_id'] + KEYS + [
        'agent_label','model','prompt_version','prompt_sha256'
    ])
    if not must_have.issubset(previous.columns):
        raise ValueError('Existing prediction checkpoint has incompatible columns. STOP.')
    if (not previous['model'].eq(MODEL).all()
        or not previous['prompt_version'].eq(PROMPT_VERSION).all()
        or not previous['prompt_sha256'].eq(PROMPT_SHA).all()):
        raise ValueError('Existing checkpoint uses a different model or prompt. STOP.')
    if not previous['review_id'].isin(reference['review_id']).all():
        raise ValueError('Checkpoint includes unknown review IDs. STOP.')
    if previous['review_id'].duplicated().any():
        raise ValueError('Duplicate saved review IDs. STOP.')
    expected_keys = reference.set_index('review_id')
    for r in previous.to_dict('records'):
        if any(str(r[k]) != str(expected_keys.loc[r['review_id'], k]) for k in KEYS):
            raise ValueError(f'Checkpoint case identifiers do not match frozen labels: {r["review_id"]}')
        if r['agent_label'] not in LABELS:
            raise ValueError('Invalid saved agent label. STOP.')
        saved[r['review_id']] = r

print('TRIALSIGNAL V5 — BLIND35 VALIDATION')
print('Reference labels: 35 frozen | direct=2, partial=5, no_match=28')
print('Saved V5 predictions reusable:', len(saved))
print('Model and prompt fingerprint: verified against V5 development results')
print('Evaluating only endpoint evidence; no gold labels sent to model.\n')

for i, (_, row) in enumerate(inputs.iterrows(), 1):
    rid = row['review_id']
    if rid in saved:
        print(f'{i}/35 {rid}: reused ({saved[rid]["agent_label"]})')
        continue
    try:
        result = call_v5(row)
    except Exception as exc:
        print(f'\nSTOP at {i}/35 ({rid}): {type(exc).__name__}: {exc}')
        print('Completed predictions remain saved. Rerun this cell to resume.')
        break
    saved[rid] = result
    pd.DataFrame(saved.values()).to_csv(PRED_PATH, index=False)
    print(f'{i}/35 {rid}: {result["agent_label"]} — saved')

print(f'\nSuccessful predictions: {len(saved)}/35')
print('Prediction checkpoint:', PRED_PATH)
if len(saved) != 35:
    print('Validation is incomplete. No final metrics written.')
else:
    pred = pd.DataFrame(saved.values())
    evaluation = reference.merge(pred, on=['review_id'] + KEYS,
                                 how='left', validate='one_to_one')
    evaluation['agree'] = evaluation['human_label'].eq(evaluation['agent_label'])
    matrix = pd.crosstab(evaluation['human_label'], evaluation['agent_label']).reindex(
        index=LABELS, columns=LABELS, fill_value=0
    )
    n_correct = int(evaluation['agree'].sum())

    lines = [
        'TRIALSIGNAL — V5 BLIND35 SEMANTIC VALIDATION',
        'Reference provenance: user-accepted, assistant-proposed, blinded preliminary labels.',
        f'Prompt version: {PROMPT_VERSION}',
        f'Prompt SHA256: {PROMPT_SHA}',
        f'Model: {MODEL}',
        f'Agreement: {n_correct}/35 = {n_correct/35:.2%}',
        '', 'CONFUSION MATRIX — human labels in rows, agent predictions in columns:',
        matrix.to_string(), '', 'PER-CLASS METRICS:'
    ]
    f1_values = []
    for name in LABELS:
        tp = int(matrix.loc[name, name])
        fp = int(matrix[name].sum() - tp)
        fn = int(matrix.loc[name].sum() - tp)
        precision = tp / (tp + fp) if tp + fp else float('nan')
        recall = tp / (tp + fn) if tp + fn else float('nan')
        f1 = 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) else float('nan')
        f1_values.append(f1)
        lines.append(f'{name}: support={tp+fn} | TP={tp}, FP={fp}, FN={fn} | '
                     f'precision={precision:.2%} | recall={recall:.2%} | F1={f1:.3f}')
    lines.append(f'Macro F1 (unweighted classes): {sum(f1_values)/len(f1_values):.3f}')
    lines.append(f'Disagreements: {35-n_correct}')
    lines.append('Confidence is model-reported and NOT calibrated probability.')
    lines.append('Only 2 reference direct matches; report direct recall with strong caution.')

    disagree = evaluation[~evaluation['agree']].sort_values('review_id')
    for _, r in disagree.iterrows():
        lines.extend([
            '', '-'*85,
            f'{r["review_id"]} | {r["disease"]} | {r["priority_id"]} | {r["nct_id"]}',
            f'ENDPOINT: {r["endpoint_name"]}',
            f'HUMAN: {r["human_label"]} | AGENT: {r["agent_label"]} | CONFIDENCE: {r["confidence"]}',
            f'UNCERTAINTY FLAG: {r["requires_human_adjudication"]}',
            'REFERENCE REASON: ' + textwrap.fill(str(r['human_reason']), 105),
            'AGENT REASON: ' + textwrap.fill(str(r['reason']), 105),
            'FULL ENDPOINT: ' + textwrap.fill(str(r['endpoint_semantic_text']), 105),
        ])
    summary = '\n'.join(lines)
    evaluation.to_csv(EVAL_PATH, index=False)
    SUMMARY_PATH.write_text(summary, encoding='utf-8')
    print('\n' + summary)
    print('\nEvaluation CSV:', EVAL_PATH)
    print('Readable metric/report text:', SUMMARY_PATH)
    print('NO reference labels, old predictions, or prompts were overwritten.')

TRIALSIGNAL V5 — BLIND35 VALIDATION
Reference labels: 35 frozen | direct=2, partial=5, no_match=28
Saved V5 predictions reusable: 0
Model and prompt fingerprint: verified against V5 development results
Evaluating only endpoint evidence; no gold labels sent to model.

1/35 V5-001: no_match — saved
2/35 V5-002: no_match — saved
3/35 V5-003: no_match — saved
4/35 V5-004: no_match — saved
5/35 V5-005: no_match — saved
6/35 V5-006: partial — saved
7/35 V5-007: no_match — saved
8/35 V5-008: no_match — saved
9/35 V5-009: no_match — saved
10/35 V5-010: no_match — saved
11/35 V5-011: no_match — saved
12/35 V5-012: partial — saved
13/35 V5-013: no_match — saved
14/35 V5-014: no_match — saved
15/35 V5-015: partial — saved
16/35 V5-016: no_match — saved
17/35 V5-017: no_match — saved
18/35 V5-018: no_match — saved
19/35 V5-019: no_match — saved
20/35 V5-020: no_match — saved
21/35 V5-021: direct — saved
22/35 V5-022: no_match — saved
23/35 V5-023: no_match — saved
24/35 V5-024: no_match — saved
25

In [ ]:
# ===============================================================
# TRIALSIGNAL — SEMANTIC V5 INTEGRATION HANDOFF / 5-CASE SMOKE
# Run after Master Agent Reconnect v4.0.
# Reconnect continues to restore V3; this cell ACTIVATES V5,
# using only frozen Drive data and version-matched V5 checkpoints.
# It does NOT mutate any frozen corpus or overwrite prior pilots.
# ===============================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import pandas as pd

V5_DATA_DIR = Path('./TrialSignal/data')
if not V5_DATA_DIR.is_dir():
    try:
        from google.colab import drive
        drive.mount('/content/portfolio-drive')
    except Exception as exc:
        raise RuntimeError('Could not mount Google Drive') from exc
if not V5_DATA_DIR.is_dir():
    raise FileNotFoundError(f'Missing TrialSignal Drive folder: {V5_DATA_DIR}')

V5_MODEL = 'gpt-6-luna'
V5_VERSION = 'semantic_v5_balanced_proxy'
V5_PROMPT = '\nYou are the Semantic Match Agent for TrialSignal.\n\nDetermine whether ONE registered clinical-trial endpoint\nMEASURES a validated patient-priority outcome.\n\nJudge measurement, not merely clinical relevance.\n\nDIRECT MATCH\nThe endpoint explicitly measures the patient priority or\na clearly identifiable constituent outcome of that priority.\n\nA broad patient priority may contain multiple outcomes.\nDirect measurement of one explicit constituent is sufficient.\nThe endpoint does not have to measure every dimension.\n\nHowever, measuring something that merely CONTRIBUTES to an\noutcome is not the same as measuring the outcome itself.\n\nFor a broad priority such as independence, a physical-function\ntest is not automatically a direct independence measure.\n\nPARTIAL MATCH\nThe endpoint measures a specific, defensible proxy, functional\ncomponent, or closely related indicator of the patient priority.\n\nA partial match requires an identifiable measurement relationship,\nnot merely a plausible medical association.\n\nAcceptable partial relationships can include:\n- A functional measure closely reflecting an aspect of a broader\n  patient outcome without measuring the entire construct.\n- A recognized clinical instrument containing relevant domains\n  when the broader composite is reported rather than the\n  relevant outcome separately.\n- A disease-relevant functional or clinical indicator that provides\n  meaningful evidence about disease progression without directly\n  measuring progression rate.\n- A structural or functional organ assessment closely connected\n  to the target organ outcome without directly establishing damage.\n\nFor partial matches, explicitly identify:\n1. What the endpoint measures.\n2. What aspect of the patient priority it represents.\n3. Why that relationship is sufficiently close to be meaningful.\n\nNO MATCH\nThe endpoint neither directly measures the priority nor\nestablishes a defensible close proxy.\n\nReject relationships based only on:\n- Shared disease context.\n- Unspecified general symptom improvement.\n- General disease severity or treatment response.\n- An assumed downstream treatment benefit.\n- Risk factors or clinical events that may eventually affect\n  the priority but do not measure it or a close proxy.\n- Broad clinical associations without a specific measurement link.\n\nIMPORTANT DISTINCTIONS\n- A measure of functional ability may partially reflect independence.\n- A general symptom questionnaire does not establish measurement\n  of a particular symptom unless that symptom is identified.\n- A composite instrument may support partial alignment when\n  relevant constituent domains are established, even if they\n  are not separately reported.\n- Do not invent unreported instrument domains.\n- Measuring functional change can inform disease-course\n  assessment without necessarily measuring slowed progression.\n- An acute clinical complication does not automatically\n  measure progressive long-term organ damage.\n- A broad priority does not require complete coverage for\n  a direct match.\n\nDECISION PROCESS\n1. Identify the actual measured endpoint variable.\n2. Identify the patient-priority outcome.\n3. Determine whether the outcome or explicit constituent\n   is directly measured.\n4. If not, determine whether the endpoint measures a\n   specific and sufficiently close proxy.\n5. Otherwise classify no_match.\n\nUse only the supplied evidence.\nDo not infer unreported endpoints.\nDo not use cosine similarity or retrieval rank as semantic truth.\nDo not evaluate disease or population applicability;\nthat belongs to the Scope Agent.\nDo not perform alignment scoring or arithmetic.\n\nReturn:\n- semantic_label: direct, partial, or no_match\n- confidence: number between 0 and 1\n- reason: concise explanation grounded in actual measurement\n\nConfidence reflects judgment certainty, not a calibrated\nprobability of correctness. Preserve genuine ambiguity.\n'
V5_SHA = hashlib.sha256(V5_PROMPT.encode('utf-8')).hexdigest()
assert V5_SHA == '17dd69aa29445af5503a639012a492b2e8af79c64dc097ab052a3dc09e595039'

V5_FILES = {
    'pool': 'trialsignal_top10_agent_candidates_clean.csv',
    'priorities': 'trialsignal_alignment_priorities.csv',
    'development': 'trialsignal_semantic_v5_development25.csv',
    'blind35': 'trialsignal_semantic_v5_blind35_predictions_v1.csv',
}
for f in V5_FILES.values():
    if not (V5_DATA_DIR / f).is_file():
        raise FileNotFoundError(V5_DATA_DIR / f)

# Match version against actual saved development metadata.
v5_dev = pd.read_csv(V5_DATA_DIR / V5_FILES['development'])
assert len(v5_dev) == 25, 'Expected 25 V5 development results; STOP.'
for col, expected in [
    ('model', V5_MODEL),
    ('prompt_version', V5_VERSION),
    ('prompt_sha256', V5_SHA)
]:
    if set(v5_dev[col].dropna()) != {expected}:
        raise AssertionError(f'V5 development mismatch: {col}. STOP.')

v5_pool = pd.read_csv(V5_DATA_DIR / V5_FILES['pool'])
v5_priorities = pd.read_csv(V5_DATA_DIR / V5_FILES['priorities'])
assert len(v5_pool) == 63811 and len(v5_priorities) == 73
V5_KEYS = ['disease','priority_id','nct_id','endpoint_id']
V5_CASE_KEYS = V5_KEYS[:3]
V5_LABELS = {'direct','partial','no_match'}
V5_INPUTS = v5_pool.merge(
    v5_priorities[['disease','priority_id','evidence_text']],
    on=['disease','priority_id'], how='left', validate='many_to_one'
)
assert len(V5_INPUTS) == 63811
assert not V5_INPUTS['evidence_text'].isna().any()
assert not V5_INPUTS.duplicated(V5_KEYS).any()
assert not V5_INPUTS.duplicated(V5_CASE_KEYS + ['within_trial_rank']).any()

V5_TRACE_PATH = V5_DATA_DIR / 'trialsignal_semantic_v5_top10_trace_v1.csv'
V5_SUMMARY_PATH = V5_DATA_DIR / 'trialsignal_semantic_v5_top10_smoke5_v1.csv'
V5_CONFIG_PATH = V5_DATA_DIR / 'trialsignal_semantic_v5_frozen_contract_v1.json'
V5_CONFIG = {
    'model': V5_MODEL,
    'prompt_version': V5_VERSION,
    'prompt_sha256': V5_SHA,
    'prompt': V5_PROMPT,
    'labels': ['direct','partial','no_match'],
    'retrieval_top_k': 10,
    'candidate_keys': V5_KEYS,
    'note': 'Candidate-level semantic measurement only; Scope Agent must adjudicate applicability.'
}
# Persist frozen prompt/config once. Never overwrite an existing contract.
if V5_CONFIG_PATH.exists():
    existing_config = json.loads(V5_CONFIG_PATH.read_text(encoding='utf-8'))
    if existing_config != V5_CONFIG:
        raise ValueError('Existing V5 frozen contract differs. STOP.')
else:
    V5_CONFIG_PATH.write_text(
        json.dumps(V5_CONFIG, ensure_ascii=False, indent=2), encoding='utf-8'
    )

# Reuse exact-version V5 responses only. Never reuse V3/V4 judgments.
v5_cache = {}
v5_trace = {}
for f in [V5_FILES['development'], V5_FILES['blind35'], V5_TRACE_PATH.name]:
    path = V5_DATA_DIR / f
    if not path.is_file():
        continue
    frame = pd.read_csv(path, keep_default_na=False)
    required = set(V5_KEYS + ['agent_label','model','prompt_version','prompt_sha256','confidence','reason'])
    if not required.issubset(frame.columns):
        raise ValueError(f'Unexpected saved V5 schema: {path}')
    if not (frame['model'].eq(V5_MODEL) & frame['prompt_version'].eq(V5_VERSION)
            & frame['prompt_sha256'].eq(V5_SHA)).all():
        raise ValueError(f'Saved result model/prompt mismatch: {path}. STOP.')
    if not frame['agent_label'].isin(V5_LABELS).all():
        raise ValueError(f'Saved result invalid labels: {path}')
    for record in frame.to_dict('records'):
        k = tuple(str(record[c]) for c in V5_KEYS)
        # Frozen IDs required; duplicated prediction for same endpoint okay if same prompt.
        v5_cache[k] = record
        if f == V5_TRACE_PATH.name:
            v5_trace[k] = record

# Cache verification: every identifier must be in the frozen pool.
pool_ids = set(map(tuple, V5_INPUTS[V5_KEYS].astype(str).itertuples(index=False, name=None)))
if not set(v5_cache).issubset(pool_ids):
    raise ValueError('One or more cached judgments absent from frozen V5 corpus')

V5_SCHEMA = {
    'type':'object',
    'properties': {
        'semantic_label': {'type':'string','enum':['direct','partial','no_match']},
        'confidence': {'type':'number'},
        'reason': {'type':'string'},
    },
    'required':['semantic_label','confidence','reason'],
    'additionalProperties': False,
}

def _v5_key(row):
    return tuple(str(row[col]) for col in V5_KEYS)

def _v5_save_trace():
    # Entire trace is saved after each new successful API call.
    pd.DataFrame(v5_trace.values()).to_csv(V5_TRACE_PATH, index=False)

def semantic_match_candidate_v5(row):
    if 'client' not in globals() or client is None:
        raise RuntimeError('OpenAI client unavailable. Run master reconnect with Colab Secret first.')
    required = ['priority_label','priority_description','evidence_text',
                'endpoint_name','endpoint_type','endpoint_semantic_text']
    missing = [col for col in required if col not in row.index or pd.isna(row[col])]
    if missing:
        raise ValueError(f'Missing V5 semantic inputs: {missing}')
    payload = {
        'disease': str(row['disease']),
        'priority': str(row['priority_label']),
        'priority_definition': str(row['priority_description']),
        'patient_evidence': str(row['evidence_text'])[:1500],
        'endpoint_name': str(row['endpoint_name']),
        'endpoint_type': str(row['endpoint_type']),
        'endpoint_text': str(row['endpoint_semantic_text']),
    }
    response = client.responses.create(
        model=V5_MODEL,
        instructions=V5_PROMPT,
        input=json.dumps(payload, ensure_ascii=False),
        text={'format': {'type': 'json_schema', 'name': 'trialsignal_semantic_v5',
                         'strict': True, 'schema': V5_SCHEMA}},
    )
    if not response.output_text:
        raise ValueError('No structured V5 response')
    parsed = json.loads(response.output_text)
    label = parsed['semantic_label']
    confidence = float(parsed['confidence'])
    reason = parsed['reason']
    if label not in V5_LABELS or not (0 <= confidence <= 1) or not isinstance(reason,str) or not reason.strip():
        raise ValueError('Invalid V5 semantic response')
    return {
        **{c: str(row[c]) for c in V5_KEYS},
        'candidate_rank': int(row['within_trial_rank']),
        'agent_label': label,
        'confidence': confidence,
        'reason': reason,
        'model': V5_MODEL,
        'prompt_version': V5_VERSION,
        'prompt_sha256': V5_SHA,
        'evaluated_at_utc': datetime.now(timezone.utc).isoformat(),
    }

def v5_judge_or_reuse(row):
    key = _v5_key(row)
    result = v5_cache.get(key)
    if result is not None:
        return result, 'cache'
    result = semantic_match_candidate_v5(row)
    v5_cache[key] = result
    v5_trace[key] = result
    _v5_save_trace()
    return result, 'API'

def semantic_scan_trial_v5(disease, priority_id, nct_id):
    '''Semantic-only search. No scope decisions or arithmetic/scoring.
    Stop on first direct (provisional pending Scope Agent).
    If no direct, preserve ALL partials; do not use confidence to select a best partial.
    No-match only after all available candidates are exhausted.'''
    cases = V5_INPUTS.loc[
        V5_INPUTS['disease'].eq(disease) &
        V5_INPUTS['priority_id'].eq(priority_id) &
        V5_INPUTS['nct_id'].eq(nct_id)
    ].sort_values('within_trial_rank').head(10)
    if cases.empty:
        raise ValueError(f'No candidates: {disease}|{priority_id}|{nct_id}')
    inspected, partials, direct, status = [], [], None, None
    for _, row in cases.iterrows():
        try:
            result, origin = v5_judge_or_reuse(row)
        except Exception as exc:
            status = 'incomplete_error'
            print(f'  Rank {int(row["within_trial_rank"])}: ERROR {type(exc).__name__}: {exc}')
            break
        inspected.append(result)
        label = result['agent_label']
        print(f'  Rank {int(row["within_trial_rank"])}: {label} ({origin})')
        if label == 'direct':
            direct = result
            status = 'direct_found_scope_pending'
            break
        if label == 'partial':
            partials.append({
                'endpoint_id': result['endpoint_id'],
                'candidate_rank': int(result['candidate_rank']),
                'confidence': float(result['confidence']),
                'reason': result['reason']
            })
    if status is None:
        status = 'exhausted_partial' if partials else 'exhausted_no_match'
    label = ('incomplete' if status == 'incomplete_error' else
             'direct' if direct is not None else
             'partial' if partials else 'no_match')
    return {
        'disease': disease, 'priority_id': priority_id, 'nct_id': nct_id,
        'provisional_semantic_label': label,
        'direct_endpoint_id': direct['endpoint_id'] if direct else None,
        'direct_rank': int(direct['candidate_rank']) if direct else None,
        'partial_count': len(partials),
        'partial_candidates_json': json.dumps(partials, ensure_ascii=False),
        'candidates_checked': len(inspected),
        'candidates_available': len(cases),
        'stop_reason': status,
        'scope_status': 'not_evaluated',
        'model': V5_MODEL,
        'prompt_version': V5_VERSION,
        'prompt_sha256': V5_SHA,
    }

print('V5 READY: frozen Drive inputs, model/prompt fingerprint verified.')
print('V5 existing cached judgments:', len(v5_cache))
print('V5 contract saved to:', V5_CONFIG_PATH)
print('Scope remains separate; no API calls were made during initialization.')

# Five-case smoke test only; does not yet expand to the entire 54-case benchmark.
V5_SMOKE5 = [
    ('Dravet Syndrome', 'DRAVET_P05', 'NCT04940624'),
    ('Facioscapulohumeral Muscular Dystrophy', 'FSHD_P08', 'NCT07038200'),
    ('Hereditary Angioedema', 'HAE_P08', 'NCT03888755'),
    ('Spinal Muscular Atrophy', 'SMA_P06', 'NCT06191354'),
    ('Sickle Cell Disease', 'SCD_P07', 'NCT06823219'),
]

v5_search_summaries = []
for i, case in enumerate(V5_SMOKE5, 1):
    print('\n' + '='*70)
    print(f'V5 SEARCH CASE {i}/5: {case}')
    result = semantic_scan_trial_v5(*case)
    v5_search_summaries.append(result)
    if result['provisional_semantic_label'] == 'incomplete':
        print('Stopped on API error; completed judgments are saved. Rerun to resume.')
        break

v5_search_summary = pd.DataFrame(v5_search_summaries)
v5_search_summary.to_csv(V5_SUMMARY_PATH, index=False)
print('\nV5 SEMANTIC SEARCH SUMMARY')
print(v5_search_summary[[
    'priority_id','provisional_semantic_label','direct_endpoint_id',
    'direct_rank','partial_count','candidates_checked','stop_reason'
]].to_string(index=False))
print('Search trace:', V5_TRACE_PATH)
print('Search summary:', V5_SUMMARY_PATH)
print('All semantic direct matches remain pending Scope Agent review.')


V5 READY: frozen Drive inputs, model/prompt fingerprint verified.
V5 existing cached judgments: 60
V5 contract saved to: ./TrialSignal/data/trialsignal_semantic_v5_frozen_contract_v1.json
Scope remains separate; no API calls were made during initialization.

V5 SEARCH CASE 1/5: ('Dravet Syndrome', 'DRAVET_P05', 'NCT04940624')
  Rank 1: direct (cache)

V5 SEARCH CASE 2/5: ('Facioscapulohumeral Muscular Dystrophy', 'FSHD_P08', 'NCT07038200')
  Rank 1: partial (cache)
  Rank 2: partial (API)
  Rank 3: partial (API)
  Rank 4: partial (API)
  Rank 5: no_match (cache)
  Rank 6: partial (cache)
  Rank 7: no_match (API)
  Rank 8: no_match (cache)
  Rank 9: no_match (API)
  Rank 10: no_match (API)

V5 SEARCH CASE 3/5: ('Hereditary Angioedema', 'HAE_P08', 'NCT03888755')
  Rank 1: no_match (cache)
  Rank 2: no_match (API)
  Rank 3: no_match (cache)
  Rank 4: no_match (API)
  Rank 5: no_match (API)
  Rank 6: no_match (cache)
  Rank 7: no_match (cache)
  Rank 8: no_match (API)
  Rank 9: no_match (A

In [ ]:
# ============================================================================
# TRIALSIGNAL — FROZEN V5 SEMANTIC SEARCH / 54-CASE RETRIEVAL BENCHMARK
# One standalone Colab cell. Loads frozen datasets, V5 contract and cached
# responses from Google Drive. Requires only the API client from reconnect v4.
# NO collection/embedding/scope/scoring. No reference labels in API prompts.
# Every NEW successful judgment is checkpointed immediately.
# ============================================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import pandas as pd

DIR54 = Path('./TrialSignal/data')
if not DIR54.is_dir():
    from google.colab import drive
    drive.mount('/content/portfolio-drive')
if not DIR54.is_dir():
    raise FileNotFoundError(f'TrialSignal Drive folder not found: {DIR54}')

# Deliberate per-execution safety cap; re-executing reuses all saved judgments.
MAX_NEW_API_CALLS = 250

MODEL_EXPECTED = 'gpt-6-luna'
VERSION_EXPECTED = 'semantic_v5_balanced_proxy'
SHA_EXPECTED = '17dd69aa29445af5503a639012a492b2e8af79c64dc097ab052a3dc09e595039'
ID54 = ['disease', 'priority_id', 'nct_id', 'endpoint_id']
CASE54 = ID54[:3]
LABEL54 = {'direct', 'partial', 'no_match'}

PATHS54 = {
    'pool': DIR54 / 'trialsignal_top10_agent_candidates_clean.csv',
    'priorities': DIR54 / 'trialsignal_alignment_priorities.csv',
    'benchmark': DIR54 / 'trialsignal_semantic_pilot_cases.csv',
    'contract': DIR54 / 'trialsignal_semantic_v5_frozen_contract_v1.json',
    'development': DIR54 / 'trialsignal_semantic_v5_development25.csv',
    'blind': DIR54 / 'trialsignal_semantic_v5_blind35_predictions_v1.csv',
    'shared_trace': DIR54 / 'trialsignal_semantic_v5_top10_trace_v1.csv',
    'benchmark_trace': DIR54 / 'trialsignal_semantic_v5_benchmark54_trace_v1.csv',
    'summary': DIR54 / 'trialsignal_semantic_v5_benchmark54_summary_v1.csv',
    'decisions': DIR54 / 'trialsignal_semantic_v5_benchmark54_decisions_v1.csv',
    'report': DIR54 / 'trialsignal_semantic_v5_benchmark54_review_v1.txt',
}
for name in ['pool', 'priorities', 'benchmark', 'contract', 'development', 'blind']:
    if not PATHS54[name].is_file():
        raise FileNotFoundError(f'Missing required Drive source: {PATHS54[name]}')

contract54 = json.loads(PATHS54['contract'].read_text(encoding='utf-8'))
for field, expected in [('model', MODEL_EXPECTED),
                        ('prompt_version', VERSION_EXPECTED),
                        ('prompt_sha256', SHA_EXPECTED),
                        ('retrieval_top_k', 10)]:
    if contract54.get(field) != expected:
        raise ValueError(f'Frozen semantic contract mismatch: {field}')
PROMPT54 = contract54['prompt']
if hashlib.sha256(PROMPT54.encode('utf-8')).hexdigest() != SHA_EXPECTED:
    raise ValueError('Frozen V5 prompt content failed SHA256 verification')

pool54 = pd.read_csv(PATHS54['pool'])
priorities54 = pd.read_csv(PATHS54['priorities'])
benchmark54_disk = pd.read_csv(PATHS54['benchmark'])
assert len(pool54) == 63811 and len(priorities54) == 73
assert len(benchmark54_disk) == 54
assert benchmark54_disk['trial_has_true_match'].sum() == 16
assert (benchmark54_disk['trial_has_true_match'].eq(0) &
        benchmark54_disk['lower_rank_audit_complete'].eq(1)).sum() == 18
assert benchmark54_disk['manual_scope_issue'].eq(0).all()
assert not pool54.duplicated(ID54).any()
assert not pool54.duplicated(CASE54 + ['within_trial_rank']).any()
assert not benchmark54_disk.duplicated(CASE54).any()

inputs54 = pool54.merge(
    priorities54[['disease', 'priority_id', 'evidence_text']],
    on=['disease', 'priority_id'], how='left', validate='many_to_one'
)
assert len(inputs54) == len(pool54)
assert inputs54[['evidence_text', 'priority_description', 'endpoint_name',
                 'endpoint_type', 'endpoint_semantic_text']].notna().all().all()

# Restrict the API search exclusively to the 54 preserved benchmark jobs.
candidates54 = inputs54.merge(
    benchmark54_disk[CASE54], on=CASE54, how='inner', validate='many_to_one'
)
assert candidates54[CASE54].drop_duplicates().shape[0] == 54
assert len(candidates54) == 333  # exactly the frozen snapshot's candidates
assert candidates54['within_trial_rank'].between(1, 10).all()

pool_keys54 = set(map(tuple, inputs54[ID54].astype(str).itertuples(index=False, name=None)))
def key54(r):
    return tuple(str(r[c]) for c in ID54)

cache54 = {}
bench_trace54 = {}
for name in ['development', 'blind', 'shared_trace', 'benchmark_trace']:
    path = PATHS54[name]
    if not path.is_file():
        continue
    df = pd.read_csv(path, keep_default_na=False)
    needed = set(ID54 + ['agent_label', 'confidence', 'reason',
                         'model', 'prompt_version', 'prompt_sha256'])
    if not needed.issubset(df.columns):
        raise ValueError(f'Unexpected V5 checkpoint schema in {path}')
    for r in df.to_dict('records'):
        if r['model'] != MODEL_EXPECTED or r['prompt_version'] != VERSION_EXPECTED or r['prompt_sha256'] != SHA_EXPECTED:
            raise ValueError(f'Invalid model/prompt in {path}: {key54(r)}')
        if r['agent_label'] not in LABEL54:
            raise ValueError(f'Unexpected label in {path}: {key54(r)}')
        k = key54(r)
        if k not in pool_keys54:
            raise ValueError(f'Cached endpoint absent from frozen pool: {k}')
        if k in cache54 and cache54[k]['agent_label'] != r['agent_label']:
            raise ValueError(f'Conflicting stored V5 labels for {k}; review before continuing')
        cache54[k] = r
        if name == 'benchmark_trace':
            bench_trace54[k] = r

SCHEMA54 = {
    'type': 'object',
    'properties': {
        'semantic_label': {'type': 'string', 'enum': ['direct', 'partial', 'no_match']},
        'confidence': {'type': 'number'},
        'reason': {'type': 'string'},
    },
    'required': ['semantic_label', 'confidence', 'reason'],
    'additionalProperties': False,
}

new_calls54 = 0

def classify54(row):
    global new_calls54
    k = key54(row)
    if k in cache54:
        return cache54[k], 'cache'
    if new_calls54 >= MAX_NEW_API_CALLS:
        raise RuntimeError('API_CALL_CAP_REACHED')
    if 'client' not in globals() or client is None:
        raise RuntimeError('OpenAI client unavailable; run master reconnect v4 first')

    # The human benchmark and gold labels are NEVER supplied to the agent.
    payload = {
        'disease': str(row['disease']),
        'priority': str(row['priority_label']),
        'priority_definition': str(row['priority_description']),
        'patient_evidence': str(row['evidence_text'])[:1500],
        'endpoint_name': str(row['endpoint_name']),
        'endpoint_type': str(row['endpoint_type']),
        'endpoint_text': str(row['endpoint_semantic_text']),
    }
    response = client.responses.create(
        model=MODEL_EXPECTED, instructions=PROMPT54,
        input=json.dumps(payload, ensure_ascii=False),
        text={'format': {'type': 'json_schema', 'name': 'trialsignal_semantic_v5',
                         'strict': True, 'schema': SCHEMA54}},
    )
    if not response.output_text:
        raise ValueError('Empty structured V5 response')
    parsed = json.loads(response.output_text)
    label = parsed['semantic_label']
    confidence = float(parsed['confidence'])
    reason = parsed['reason']
    if label not in LABEL54 or not 0 <= confidence <= 1 or not isinstance(reason, str) or not reason.strip():
        raise ValueError('Invalid V5 semantic output')
    result = {
        **{c: str(row[c]) for c in ID54},
        'candidate_rank': int(row['within_trial_rank']),
        'agent_label': label,
        'confidence': confidence,
        'reason': reason,
        'model': MODEL_EXPECTED,
        'prompt_version': VERSION_EXPECTED,
        'prompt_sha256': SHA_EXPECTED,
        'evaluated_at_utc': datetime.now(timezone.utc).isoformat(),
    }
    # Checkpoint immediately after every successful paid API call.
    bench_trace54[k] = result
    pd.DataFrame(bench_trace54.values()).to_csv(PATHS54['benchmark_trace'], index=False)
    cache54[k] = result
    new_calls54 += 1
    return result, 'API'

summaries54 = []
decisions54 = []

print('V5 54-CASE RETRIEVAL BENCHMARK')
print('Jobs:', len(benchmark54_disk), '| Candidate endpoints:', len(candidates54))
print('Manual direct findings: 16 | audited non-direct cases: 18 | incomplete negative audits: 20')
print('Cached exact-version V5 judgments:', len(cache54))
print('New API call cap for this execution:', MAX_NEW_API_CALLS)
print('Gold labels are withheld from the API. Scope remains separate.\n')

for index, benchmark_row in benchmark54_disk.iterrows():
    disease, priority_id, nct_id = (str(benchmark_row[c]) for c in CASE54)
    subset = candidates54.loc[
        candidates54['disease'].eq(disease) &
        candidates54['priority_id'].eq(priority_id) &
        candidates54['nct_id'].eq(nct_id)
    ].sort_values('within_trial_rank')
    partials, selected_direct, examined, stop_reason = [], None, [], None

    for _, row in subset.iterrows():
        try:
            result, source = classify54(row)
        except Exception as e:
            stop_reason = 'incomplete_api_cap' if str(e) == 'API_CALL_CAP_REACHED' else 'incomplete_error'
            print(f'{index+1}/54 {priority_id} {nct_id} — {stop_reason}: {type(e).__name__}: {e}')
            break
        examined.append(result)
        decisions54.append({
            **{c: str(row[c]) for c in ID54},
            'candidate_rank': int(row['within_trial_rank']),
            'agent_label': result['agent_label'],
            'confidence': float(result['confidence']),
            'reason': result['reason'],
            'origin': source,
            'prompt_version': VERSION_EXPECTED,
            'prompt_sha256': SHA_EXPECTED,
        })
        if result['agent_label'] == 'partial':
            partials.append({
                'endpoint_id': str(result['endpoint_id']),
                'candidate_rank': int(row['within_trial_rank']),
                'reason': result['reason'],
            })
        if result['agent_label'] == 'direct':
            selected_direct = result
            stop_reason = 'direct_found_scope_pending'
            break

    if stop_reason is None:
        stop_reason = 'exhausted_partial' if partials else 'exhausted_no_match'
    label = ('incomplete' if stop_reason.startswith('incomplete') else
             'direct' if selected_direct is not None else
             'partial' if partials else 'no_match')

    summaries54.append({
        'disease': disease, 'priority_id': priority_id, 'nct_id': nct_id,
        'provisional_semantic_label': label,
        'direct_endpoint_id': str(selected_direct['endpoint_id']) if selected_direct else None,
        'direct_rank': int(selected_direct['candidate_rank']) if selected_direct else None,
        'partial_count': len(partials),
        'partial_candidates_json': json.dumps(partials, ensure_ascii=False),
        'candidates_checked': len(examined),
        'candidates_available': len(subset),
        'stop_reason': stop_reason,
        'scope_status': 'not_evaluated',
        'manual_direct_found': int(benchmark_row['trial_has_true_match']),
        'manual_first_direct_rank': (
            int(benchmark_row['first_true_match_rank'])
            if pd.notna(benchmark_row['first_true_match_rank']) else None
        ),
        'manual_lower_rank_audit_complete': int(benchmark_row['lower_rank_audit_complete']),
        'manual_review_notes': benchmark_row['manual_review_notes'],
        'model': MODEL_EXPECTED, 'prompt_version': VERSION_EXPECTED,
        'prompt_sha256': SHA_EXPECTED,
    })
    # Preserve trial-level progress after each case and trace of inspected endpoints.
    pd.DataFrame(summaries54).to_csv(PATHS54['summary'], index=False)
    pd.DataFrame(decisions54).to_csv(PATHS54['decisions'], index=False)
    print(f'{index+1:02d}/54 {priority_id:<11} {nct_id} -> {label:<10} '
          f'checked={len(examined)}/{len(subset)} partials={len(partials)}')
    if stop_reason.startswith('incomplete'):
        print('Stopping safely. Re-run the SAME cell to continue using saved API results.')
        break

summary54 = pd.DataFrame(summaries54)
completed54 = summary54[summary54['provisional_semantic_label'] != 'incomplete'].copy()
full54 = (len(summary54) == 54 and len(completed54) == 54)

# Manual review labels are an imperfect reference, NOT complete three-class truth.
# Separate 18 lower-rank-audited non-direct cases from 20 unaudited cases.
positive54 = completed54.loc[completed54['manual_direct_found'].eq(1)]
audited_negative54 = completed54.loc[
    completed54['manual_direct_found'].eq(0) &
    completed54['manual_lower_rank_audit_complete'].eq(1)
]
unaudited_negative54 = completed54.loc[
    completed54['manual_direct_found'].eq(0) &
    completed54['manual_lower_rank_audit_complete'].eq(0)
]

def direct_counts54(frame):
    n_direct = int(frame['provisional_semantic_label'].eq('direct').sum())
    return n_direct, len(frame) - n_direct

pos_found, pos_not_found = direct_counts54(positive54)
audit_found, audit_not_found = direct_counts54(audited_negative54)
unaudit_found, unaudit_not_found = direct_counts54(unaudited_negative54)

flags54 = completed54.loc[
    (completed54['manual_direct_found'].eq(1) &
     completed54['provisional_semantic_label'].ne('direct')) |
    (completed54['manual_direct_found'].eq(0) &
     completed54['provisional_semantic_label'].eq('direct'))
]

lines54 = [
    'TRIALSIGNAL — V5 SEMANTIC RETRIEVAL BENCHMARK (54 CASES)',
    f'Model={MODEL_EXPECTED} | Version={VERSION_EXPECTED} | SHA256={SHA_EXPECTED}',
    f'Completed jobs: {len(completed54)}/54',
    f'New API calls this execution: {new_calls54}',
    f'Status: {"COMPLETE" if full54 else "INCOMPLETE — resume from Drive checkpoints"}',
    '',
    'BENCHMARK AGREEMENT (DIRECT vs NOT DIRECT; NOT 3-CLASS GOLD TRUTH)',
    f'Human-confirmed direct cases: V5 direct={pos_found}, V5 not-direct={pos_not_found} (tested={len(positive54)}/16)',
    f'Lower-rank-audited non-direct cases: V5 direct={audit_found}, V5 not-direct={audit_not_found} (tested={len(audited_negative54)}/18)',
    f'Not-fully-audited non-direct cases: V5 direct={unaudit_found}, V5 not-direct={unaudit_not_found} (tested={len(unaudited_negative54)}/20)',
    f'Cases flagged for manual comparison: {len(flags54)}',
    '',
    'CAUTIONS: Human direct and negative labels have adjudication limitations.',
    'No scope decisions have been performed. Direct matches are provisional.',
    'Partial matches are preserved without scoring or selecting a best partial.',
]
for _, r in flags54.iterrows():
    lines54 += [
        '', f"CHECK: {r['disease']} | {r['priority_id']} | {r['nct_id']}",
        f"manual direct={r['manual_direct_found']}, manual rank={r['manual_first_direct_rank']}",
        f"V5 label={r['provisional_semantic_label']}, direct rank={r['direct_rank']}, partials={r['partial_count']}",
        f"Manual notes: {r['manual_review_notes']}",
    ]

report54 = '\n'.join(lines54)
PATHS54['report'].write_text(report54, encoding='utf-8')
print('\n' + '='*78)
print(report54)
print('\nDrive files:')
for k in ['benchmark_trace', 'summary', 'decisions', 'report']:
    print(k, PATHS54[k])
if not full54:
    print('\nRun the same cell again; only unsaved judgments require new API calls.')


V5 54-CASE RETRIEVAL BENCHMARK
Jobs: 54 | Candidate endpoints: 333
Manual direct findings: 16 | audited non-direct cases: 18 | incomplete negative audits: 20
Cached exact-version V5 judgments: 81
New API call cap for this execution: 250
Gold labels are withheld from the API. Scope remains separate.

01/54 DRAVET_P02  NCT02187809 -> direct     checked=1/7 partials=0
02/54 DRAVET_P02  NCT02926898 -> no_match   checked=3/3 partials=0
03/54 DRAVET_P02  NCT05560282 -> no_match   checked=1/1 partials=0
04/54 DRAVET_P03  NCT06422377 -> no_match   checked=2/2 partials=0
05/54 DRAVET_P05  NCT04940624 -> direct     checked=1/10 partials=0
06/54 DRAVET_P05  NCT07531745 -> no_match   checked=9/9 partials=0
07/54 DRAVET_P06  NCT07531745 -> no_match   checked=9/9 partials=0
08/54 DRAVET_P08  NCT02926898 -> no_match   checked=3/3 partials=0
09/54 FSHD_P01    NCT02861911 -> partial    checked=6/6 partials=3
10/54 FSHD_P01    NCT03943290 -> direct     checked=1/10 partials=0
11/54 FSHD_P02    NCT001040

In [ ]:
# ===============================================================
# TRIALSIGNAL — V5 SEMANTIC AGENT HANDOFF & BENCHMARK QA
# Run AFTER the completed 54-case benchmark. No API calls.
# Reads persistent Drive results; does not rebuild or overwrite them.
# ===============================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import pandas as pd

ROOT = Path('./TrialSignal/data')
if not ROOT.is_dir():
    from google.colab import drive
    drive.mount('/content/portfolio-drive')
if not ROOT.is_dir():
    raise FileNotFoundError(f'Missing TrialSignal Drive data directory: {ROOT}')

SHA = '17dd69aa29445af5503a639012a492b2e8af79c64dc097ab052a3dc09e595039'
MODEL = 'gpt-6-luna'
VERSION = 'semantic_v5_balanced_proxy'
K = ['disease', 'priority_id', 'nct_id']
EK = K + ['endpoint_id']
FILES = {
    'contract': 'trialsignal_semantic_v5_frozen_contract_v1.json',
    'pool': 'trialsignal_top10_agent_candidates_clean.csv',
    'benchmark': 'trialsignal_semantic_pilot_cases.csv',
    'summary': 'trialsignal_semantic_v5_benchmark54_summary_v1.csv',
    'decisions': 'trialsignal_semantic_v5_benchmark54_decisions_v1.csv',
    'trace': 'trialsignal_semantic_v5_benchmark54_trace_v1.csv',
}
for label, fname in FILES.items():
    if not (ROOT / fname).is_file():
        raise FileNotFoundError(f'Missing {label}: {ROOT / fname}')

contract = json.loads((ROOT / FILES['contract']).read_text(encoding='utf-8'))
if any([contract.get('model') != MODEL,
        contract.get('prompt_version') != VERSION,
        contract.get('prompt_sha256') != SHA,
        contract.get('retrieval_top_k') != 10,
        hashlib.sha256(contract['prompt'].encode('utf-8')).hexdigest() != SHA]):
    raise ValueError('Frozen V5 contract does not match the validated configuration')

pool = pd.read_csv(ROOT / FILES['pool'])
benchmark = pd.read_csv(ROOT / FILES['benchmark'])
summary = pd.read_csv(ROOT / FILES['summary'])
decisions = pd.read_csv(ROOT / FILES['decisions'])
trace = pd.read_csv(ROOT / FILES['trace'])

assert len(pool) == 63811, 'Wrong frozen candidate pool'
assert len(benchmark) == 54 and len(summary) == 54, 'Benchmark is incomplete'
assert benchmark['trial_has_true_match'].sum() == 16
assert not benchmark.duplicated(K).any()
assert not summary.duplicated(K).any()
assert set(map(tuple, summary[K].itertuples(index=False, name=None))) == set(
    map(tuple, benchmark[K].itertuples(index=False, name=None))
), 'Summary missing or inventing priority–trial jobs'
assert (summary['model'] == MODEL).all()
assert (summary['prompt_version'] == VERSION).all()
assert (summary['prompt_sha256'] == SHA).all()
assert summary['provisional_semantic_label'].isin(['direct','partial','no_match']).all()
assert summary['scope_status'].eq('not_evaluated').all()
assert not decisions.duplicated(EK).any()
assert not trace.duplicated(EK).any()
assert decisions['prompt_version'].eq(VERSION).all()
assert decisions['prompt_sha256'].eq(SHA).all()

# Every displayed decision must be supported by the saved candidate pool.
all_pool_ids = set(map(tuple, pool[EK].itertuples(index=False, name=None)))
assert set(map(tuple, decisions[EK].itertuples(index=False, name=None))).issubset(all_pool_ids)

# Reconstruct a mechanical check from saved decisions (NOT new judgments).
for _, s in summary.iterrows():
    mask_pool = pd.Series(True, index=pool.index)
    mask_dec = pd.Series(True, index=decisions.index)
    for key in K:
        mask_pool &= pool[key].eq(s[key])
        mask_dec &= decisions[key].eq(s[key])
    available = pool.loc[mask_pool].sort_values('within_trial_rank')
    checked = decisions.loc[mask_dec].sort_values('candidate_rank')
    if len(available) != int(s['candidates_available']):
        raise ValueError(f'Available-candidate count mismatch: {s[K].to_dict()}')
    if len(checked) != int(s['candidates_checked']):
        raise ValueError(f'Inspected-candidate count mismatch: {s[K].to_dict()}')
    expected_ranks = available['within_trial_rank'].head(len(checked)).astype(int).tolist()
    actual_ranks = checked['candidate_rank'].astype(int).tolist()
    if expected_ranks != actual_ranks:
        raise ValueError(f'Candidate search skipped/misordered a rank: {s[K].to_dict()}')
    labels = checked['agent_label'].tolist()
    label = s['provisional_semantic_label']
    partial_count = labels.count('partial')
    if partial_count != int(s['partial_count']):
        raise ValueError(f'Partial count mismatch: {s[K].to_dict()}')
    if label == 'direct':
        if labels[-1] != 'direct' or labels.count('direct') != 1 or s['stop_reason'] != 'direct_found_scope_pending':
            raise ValueError(f'Incorrect direct stopping behavior: {s[K].to_dict()}')
        if str(checked.iloc[-1]['endpoint_id']) != str(s['direct_endpoint_id']):
            raise ValueError('Direct endpoint ID mismatch')
    elif label == 'partial':
        if 'direct' in labels or partial_count == 0 or len(checked) != len(available):
            raise ValueError(f'Incorrect partial exhaustion: {s[K].to_dict()}')
    elif label == 'no_match':
        if set(labels) != {'no_match'} or len(checked) != len(available):
            raise ValueError(f'Incorrect no-match exhaustion: {s[K].to_dict()}')

positives = summary.loc[summary['manual_direct_found'].eq(1)]
audited = summary.loc[(summary['manual_direct_found'].eq(0)) &
                      (summary['manual_lower_rank_audit_complete'].eq(1))]
unaudited = summary.loc[(summary['manual_direct_found'].eq(0)) &
                        (summary['manual_lower_rank_audit_complete'].eq(0))]
assert (len(positives), len(audited), len(unaudited)) == (16,18,20)
assert int(positives['provisional_semantic_label'].eq('direct').sum()) == 13
assert int(pd.concat([audited,unaudited])['provisional_semantic_label'].eq('direct').sum()) == 0

# Review discrepancies separately; original human labels are never changed.
flagged = positives.loc[positives['provisional_semantic_label'].ne('direct')].copy()
assert len(flagged) == 3
benchmark_notes = benchmark[K + ['priority_label', 'priority_description',
                                   'first_true_match_rank', 'manual_review_notes']]
review = flagged[K + ['provisional_semantic_label','partial_count']].merge(
    benchmark_notes, on=K, validate='one_to_one'
)
manual_endpoints = []
for _, s in review.iterrows():
    m = pd.Series(True, index=pool.index)
    for col in K:
        m &= pool[col].eq(s[col])
    m &= pool['within_trial_rank'].eq(int(s['first_true_match_rank']))
    chosen = pool.loc[m]
    if len(chosen) != 1:
        raise ValueError(f'Cannot identify manually designated endpoint: {s[K].to_dict()}')
    ep = chosen.iloc[0]
    d = decisions.loc[
        decisions['disease'].eq(s['disease']) &
        decisions['priority_id'].eq(s['priority_id']) &
        decisions['nct_id'].eq(s['nct_id']) &
        decisions['endpoint_id'].eq(ep['endpoint_id'])
    ]
    if len(d) != 1:
        raise ValueError('Manually selected endpoint missing from saved V5 decisions')
    manual_endpoints.append({**{col:s[col] for col in K},
        'manually_selected_endpoint_id': ep['endpoint_id'],
        'endpoint_name': ep['endpoint_name'],
        'endpoint_semantic_text': ep['endpoint_semantic_text'],
        'v5_endpoint_label': d.iloc[0]['agent_label'],
        'v5_reason': d.iloc[0]['reason'],
    })
review = review.merge(pd.DataFrame(manual_endpoints), on=K, validate='one_to_one')
review['original_manual_label'] = 'direct'
review['proposed_adjudication_label'] = 'partial'
review['team_confirmed_label'] = ''
review['team_reviewer'] = ''
review['team_review_reason'] = ''
review['adjudication_status'] = 'pending_independent_review'

OUTPUT_REVIEW = ROOT / 'trialsignal_semantic_v5_benchmark54_adjudication_3cases_v1.csv'
if OUTPUT_REVIEW.exists():
    print('Preserved existing adjudication file:', OUTPUT_REVIEW)
else:
    review.to_csv(OUTPUT_REVIEW, index=False)
    print('Created adjudication file:', OUTPUT_REVIEW)

handoff = {
    'component': 'TrialSignal Semantic Match Agent',
    'status': 'semantic_classifier_frozen_scope_pending',
    'model': MODEL, 'prompt_version': VERSION, 'prompt_sha256': SHA,
    'prompt_source': FILES['contract'],
    'retrieval_top_k': 10,
    'join_identifiers': EK,
    'semantic_labels': ['direct','partial','no_match'],
    'required_input_fields': [*EK, 'within_trial_rank', 'priority_label',
                              'priority_description', 'evidence_text',
                              'endpoint_name', 'endpoint_type', 'endpoint_semantic_text'],
    'output_fields': [*EK, 'candidate_rank', 'semantic_label', 'confidence', 'reason'],
    'semantic_only_workflow': 'Scan ranked candidates, retain partials, stop at first direct only provisionally; scope agent must approve applicability before final acceptance.',
    'integration_rules': [
        'Orchestrator owns candidate iteration and stopping state.',
        'Scope Agent checks disease/population/cohort before a direct/partial becomes accepted.',
        'If Scope Agent rejects an apparent direct match, continue with remaining ranked candidates.',
        'Preserve all partial candidates for Scope Agent/scoring; do not choose by model confidence.',
        'No-match only after all available candidates are evaluated or ruled out.',
        'Keep matching, scores, aggregation and arithmetic deterministic in Python.',
        'Cache responses by full identifiers plus model and prompt SHA; checkpoint every new call.',
    ],
    'benchmark_results': {
        'cases':54, 'confirmed_manual_direct':16,
        'v5_direct_among_manual_direct':13,
        'audited_non_direct':18, 'unaudited_non_direct':20,
        'v5_direct_among_38_non_direct':0,
        'disputed_original_direct_cases':3,
        'agreement_vs_original_binary_labels':'51/54',
        'independent_three_class_gold_truth':False,
        'scope_checked':False,
    },
    'saved_paths': {k:v for k,v in FILES.items()},
    'adjudication_file': OUTPUT_REVIEW.name,
}
OUT_MANIFEST = ROOT / 'trialsignal_semantic_v5_integrator_manifest_v1.json'
if OUT_MANIFEST.exists():
    existing = json.loads(OUT_MANIFEST.read_text(encoding='utf-8'))
    if existing != handoff:
        raise ValueError('An existing integration manifest differs; not overwriting.')
    print('Existing manifest verified:', OUT_MANIFEST)
else:
    OUT_MANIFEST.write_text(json.dumps(handoff, indent=2, ensure_ascii=False), encoding='utf-8')
    print('Created integrator manifest:', OUT_MANIFEST)

print('\nV5 SEMANTIC QA PASSED')
print('Benchmark jobs:', len(summary), '| Candidate judgments recorded:', len(decisions))
print('Original positive recovery: 13/16 | Original non-direct predicted direct: 0/38')
print('Original human disagreements:', len(flagged))
print('Semantic-only, scope not yet checked; scoring not yet finalized.')
print('\nDisputed priorities:', ', '.join(flagged['priority_id']))
print('No API calls and no frozen results overwritten.')


Created adjudication file: ./TrialSignal/data/trialsignal_semantic_v5_benchmark54_adjudication_3cases_v1.csv
Created integrator manifest: ./TrialSignal/data/trialsignal_semantic_v5_integrator_manifest_v1.json

V5 SEMANTIC QA PASSED
Benchmark jobs: 54 | Candidate judgments recorded: 253
Original positive recovery: 13/16 | Original non-direct predicted direct: 0/38
Original human disagreements: 3
Semantic-only, scope not yet checked; scoring not yet finalized.

Disputed priorities: FSHD_P01, SCD_P07, SMA_P06
No API calls and no frozen results overwritten.


In [ ]:
from pathlib import Path
import importlib.util
import sys

DRIVE_ROOT = Path('./TrialSignal')
if not DRIVE_ROOT.is_dir():
    from google.colab import drive
    drive.mount('/content/portfolio-drive')

DATA_DIR = DRIVE_ROOT / 'data'
MODULE_FILE = DRIVE_ROOT / 'agents' / 'trialsignal_semantic_v5.py'
if not MODULE_FILE.is_file():
    raise FileNotFoundError(
        f'Missing V5 module: {MODULE_FILE}. Upload the supplied .py file to Drive/TrialSignal/agents first.'
    )
if not DATA_DIR.is_dir():
    raise FileNotFoundError(f'Missing frozen data folder: {DATA_DIR}')

# Explicit source-path import avoids stale Colab notebook definitions.
spec = importlib.util.spec_from_file_location('trialsignal_semantic_v5', MODULE_FILE)
module = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = module
spec.loader.exec_module(module)

# Reuses the existing client's authorization if the master reconnect created it.
semantic_v5 = module.SemanticMatchV5(DATA_DIR, client=globals().get('client'))
print('SEMANTIC V5 HANDOFF READY')
for key, value in semantic_v5.status().items():
    print(f'{key}: {value}')
assert semantic_v5.status()['candidate_rows'] == 63811
assert semantic_v5.status()['priority_trial_jobs'] == 10551
print('No API calls made.')

SEMANTIC V5 HANDOFF READY
status: ready
model: gpt-6-luna
prompt_version: semantic_v5_balanced_proxy
prompt_sha256: 17dd69aa29445af5503a639012a492b2e8af79c64dc097ab052a3dc09e595039
candidate_rows: 63811
priority_trial_jobs: 10551
priorities: 73
retrieval_top_k: 10
cached_candidate_judgments: 288
loaded_checkpoints: {'trialsignal_semantic_v5_development25.csv': 25, 'trialsignal_semantic_v5_blind35_predictions_v1.csv': 35, 'trialsignal_semantic_v5_top10_trace_v1.csv': 21, 'trialsignal_semantic_v5_benchmark54_trace_v1.csv': 207}
client_available: True
new_api_calls_this_session: 0
scope_checked: False
No API calls made.


In [ ]:
# Cache-only contract smoke test: these endpoints were already judged by V5.
# This code will fail rather than make a new API call if the expected
# saved Drive benchmark results are missing.
cases = [
    ('Dravet Syndrome', 'DRAVET_P05', 'NCT04940624', 'direct'),
    ('Hereditary Angioedema', 'HAE_P08', 'NCT03888755', 'no_match'),
]
for disease, priority_id, nct_id, expected in cases:
    first = semantic_v5.get_candidates(disease, priority_id, nct_id).iloc[0]
    result = semantic_v5.judge_candidate(first, allow_api=False)
    assert result['semantic_label'] == expected, result
    print(f"{priority_id} | rank {result['candidate_rank']} | "
          f"{result['semantic_label']} | {result['source']}")

print('HANDOFF CONTRACT SMOKE TEST PASSED; no API calls made.')

DRAVET_P05 | rank 1 | direct | cache:trialsignal_semantic_v5_development25.csv
HAE_P08 | rank 1 | no_match | cache:trialsignal_semantic_v5_development25.csv
HANDOFF CONTRACT SMOKE TEST PASSED; no API calls made.


## SEMANTIC AGENT BUILT, TESTED, AND FROZEN - READY FOR INTEGRATION

## START OF SCOPE AGENT

In [ ]:

# ============================================================
# TRIALSIGNAL — SCOPE AGENT DEVELOPMENT RECONNECT
# ============================================================
# Run AFTER Master Agent Reconnect v4.0.
# Loads saved Scope Agent pilot work directly from Drive.
# No API calls. No changes to frozen datasets.

import pandas as pd

assert len(trials_master) == 970
assert len(alignment_endpoints) == 8445
assert len(agent_candidate_pool) == 63811

SCOPE_FILES = {
    "trial_pilot":
        "scope_agent_trial_pilot_v0_2.csv",

    "cohort_screen":
        "scope_agent_cohort_screen_v0_1_full.csv",

    "endpoint_pilot":
        "scope_agent_endpoint_llm_v0_2.csv",

    "explicit_cohort_hits":
        "scope_agent_endpoint_explicit_cohort_hits_v0.csv"
}

scope_saved = {}

print("=" * 65)
print("TRIALSIGNAL — SCOPE AGENT RESTORATION")
print("=" * 65)

for name, filename in SCOPE_FILES.items():

    path = DATA_DIR / filename

    if not path.is_file():
        print(f"\n{name}: NOT FOUND — {filename}")
        continue

    df = pd.read_csv(path)
    scope_saved[name] = df

    # Check against the active frozen corpus.
    keys = ["disease", "nct_id"]

    if "endpoint_id" in df.columns:
        keys.append("endpoint_id")
        active = alignment_endpoints
    else:
        active = trials_master

    active_keys = active[keys].drop_duplicates()

    matched = df.merge(
        active_keys.assign(_in_clean=True),
        on=keys,
        how="left",
        validate="many_to_one"
    )

    in_clean = matched["_in_clean"].eq(True).sum()

    print(f"\n{name}")
    print("Saved records:", len(df))
    print("Records in clean corpus:", in_clean)

    if "scope_status" in df.columns:
        print("Scope decisions:")
        print(df["scope_status"].value_counts().to_string())

    if "cohort_structure" in df.columns:
        print("Cohort structures:")
        print(df["cohort_structure"].value_counts().to_string())

# Preview previously flagged trial-level scope problems.
if "trial_pilot" in scope_saved:

    trial_scope = scope_saved["trial_pilot"]

    flagged = trial_scope[
        trial_scope["scope_status"].isin([
            "wrong_disease",
            "wrong_population",
            "wrong_cohort"
        ])
    ].copy()

    flagged = flagged.merge(
        trials_master[
            ["disease", "nct_id", "brief_title"]
        ].drop_duplicates(),
        on=["disease", "nct_id"],
        how="left",
        validate="many_to_one"
    )

    print("\n" + "=" * 65)
    print("TRIALS FLAGGED FOR SCOPE REVIEW")
    print("=" * 65)

    print(
        flagged[
            ["disease", "nct_id", "scope_status", "brief_title"]
        ].to_string(index=False)
    )

print("\nSCOPE DEVELOPMENT ENVIRONMENT RESTORED")
print("Frozen datasets unchanged.")
print("No API calls made.")


TRIALSIGNAL — SCOPE AGENT RESTORATION

trial_pilot
Saved records: 33
Records in clean corpus: 33
Scope decisions:
scope_status
wrong_disease    19
ambiguous        11
in_scope          2
wrong_cohort      1

cohort_screen
Saved records: 270
Records in clean corpus: 270
Cohort structures:
cohort_structure
single_population    183
separate_cohorts      81
unclear                6

endpoint_pilot
Saved records: 783
Records in clean corpus: 783
Scope decisions:
scope_status
in_scope        750
wrong_cohort     23
ambiguous        10

explicit_cohort_hits
Saved records: 90
Records in clean corpus: 90

TRIALS FLAGGED FOR SCOPE REVIEW
              disease      nct_id  scope_status                                                                                                                                                                                                brief_title
Hereditary Angioedema NCT03051698 wrong_disease                                                                  

In [ ]:

# TRIALSIGNAL — SCOPE AGENT PILOT AUDIT
# No API calls. Read-only. No datasets modified.

import pandas as pd

print("=" * 65)
print("SCOPE AGENT — SAVED WORK INVENTORY")
print("=" * 65)

for name, df in scope_saved.items():
    print(f"\n{name.upper()} — {len(df)} rows")
    print("Columns:", list(df.columns))

    keys = ["disease", "nct_id"]
    if "endpoint_id" in df.columns:
        keys.append("endpoint_id")

    if all(k in df.columns for k in keys):
        print("Duplicate identifiers:", df.duplicated(keys).sum())

    if "scope_status" in df.columns:
        print("\nScope classifications by disease:")
        print(pd.crosstab(df["disease"], df["scope_status"]))

# Inspect endpoint-level explanations.
endpoint_df = scope_saved["endpoint_pilot"]

reason_cols = [
    col for col in endpoint_df.columns
    if any(term in col.lower() for term in [
        "reason", "evidence", "rationale", "explanation"
    ])
]

print("\n" + "=" * 65)
print("ENDPOINTS FLAGGED FOR FURTHER REVIEW")
print("=" * 65)

review_cols = [
    "disease", "nct_id", "endpoint_id", "scope_status"
] + reason_cols

review_cols = list(dict.fromkeys(
    col for col in review_cols if col in endpoint_df.columns
))

flagged_endpoints = endpoint_df[
    endpoint_df["scope_status"] != "in_scope"
]

display(flagged_endpoints[review_cols].head(20))

# Identify cross-level disagreements.
trial_df = scope_saved["trial_pilot"]

trial_flags = trial_df[
    trial_df["scope_status"].isin([
        "wrong_disease", "wrong_population", "wrong_cohort"
    ])
][["disease", "nct_id", "scope_status"]].drop_duplicates()

overlap = endpoint_df.merge(
    trial_flags,
    on=["disease", "nct_id"],
    how="inner",
    suffixes=("_endpoint", "_trial")
)

print("\nEndpoint pilot rows in trial-flagged studies:",
      len(overlap))

if not overlap.empty:
    print(pd.crosstab(
        overlap["scope_status_trial"],
        overlap["scope_status_endpoint"]
    ))

# Check whether flagged trials enter actual retrieval jobs.
jobs = agent_candidate_pool[
    ["disease", "nct_id"]
].drop_duplicates()

affected = trial_flags.merge(
    jobs, on=["disease", "nct_id"], how="inner"
)

print("\nTrial-level scope flags appearing in retrieval jobs:",
      len(affected))

print("\nSCOPE PILOT AUDIT COMPLETE")


SCOPE AGENT — SAVED WORK INVENTORY

TRIAL_PILOT — 33 rows
Columns: ['disease', 'nct_id', 'model', 'prompt_version', 'scope_status', 'confidence', 'reason', 'method', 'human_review_required', 'suggested_bucket']
Duplicate identifiers: 0

Scope classifications by disease:
scope_status           ambiguous  in_scope  wrong_cohort  wrong_disease
disease                                                                
Dravet Syndrome                3         0             0              5
Friedreich's Ataxia            1         0             0              0
Hereditary Angioedema          0         0             0              6
Sickle Cell Disease            7         2             1              8

COHORT_SCREEN — 270 rows
Columns: ['disease', 'nct_id', 'conditions', 'cohort_structure', 'confidence', 'other_cohorts', 'reason', 'model', 'prompt_version']
Duplicate identifiers: 0

ENDPOINT_PILOT — 783 rows
Columns: ['disease', 'nct_id', 'endpoint_id', 'scope_status', 'confidence', 'reason', 

,disease,nct_id,endpoint_id,scope_status,reason
25,Facioscapulohumeral Muscular Dystrophy,NCT03943290,NCT03943290_S16,wrong_cohort,The endpoint is explicitly reported for patien...
78,Sickle Cell Disease,NCT05389891,NCT05389891_P1,wrong_cohort,The endpoint explicitly measures pediatric nur...
79,Sickle Cell Disease,NCT05389891,NCT05389891_P2,wrong_cohort,The endpoint explicitly measures pediatric nur...
80,Sickle Cell Disease,NCT05389891,NCT05389891_P3,wrong_cohort,The endpoint explicitly measures pediatric nur...
169,Sickle Cell Disease,NCT03733249,NCT03733249_P3,wrong_cohort,The endpoint is measured only in the malignant...
222,Sickle Cell Disease,NCT06481306,NCT06481306_S6,ambiguous,"The endpoint is assigned to Cohort A Part 2, b..."
225,Sickle Cell Disease,NCT06481306,NCT06481306_S9,ambiguous,The description specifies Cohort B but does no...
234,Sickle Cell Disease,NCT06481306,NCT06481306_S18,ambiguous,"The endpoint is limited to Cohort B, but the t..."
236,Sickle Cell Disease,NCT06481306,NCT06481306_S20,ambiguous,"The endpoint is limited to Cohort B, but the t..."
237,Sickle Cell Disease,NCT06481306,NCT06481306_S21,ambiguous,"The endpoint is reported for Cohort B, but the..."



Endpoint pilot rows in trial-flagged studies: 0

Trial-level scope flags appearing in retrieval jobs: 20

SCOPE PILOT AUDIT COMPLETE


In [ ]:
# TRIALSIGNAL — SCOPE AGENT PRE-INTEGRATION QA (READ ONLY)
# Run after Master Agent Reconnect v4.0. No API calls or file writes.
from pathlib import Path
import pandas as pd

DATA_DIR = Path(DATA_DIR) if 'DATA_DIR' in globals() else Path('./TrialSignal/data')
FILES = {
    'trials':'trialsignal_trials_master_clean.csv',
    'endpoints':'trialsignal_endpoints_master_clean.csv',
    'candidates':'trialsignal_top10_agent_candidates_clean.csv',
    'trial_pilot':'scope_agent_trial_pilot_v0_2.csv',
    'cohort_screen':'scope_agent_cohort_screen_v0_1_full.csv',
    'endpoint_llm':'scope_agent_endpoint_llm_v0_2.csv',
    'explicit_hits':'scope_agent_endpoint_explicit_cohort_hits_v0.csv',
    'endpoint_full':'scope_agent_endpoint_scope_v0_2.csv',
    'candidate_full':'scope_agent_candidate_scope_v0_2.csv',
}
D = {}
print('='*75+'\nTRIALSIGNAL SCOPE PRE-INTEGRATION QA — READ ONLY\n'+'='*75)
for key, fn in FILES.items():
    p = DATA_DIR / fn
    if p.exists():
        D[key] = pd.read_csv(p, keep_default_na=False)
        print(f'{key:17} PRESENT rows={len(D[key]):,} | {fn}')
    else:
        print(f'{key:17} MISSING | {fn}')
for key, expected in [('trials',970),('endpoints',8445),('candidates',63811)]:
    assert key in D, f'Cannot audit without frozen {key}'
    assert len(D[key]) == expected, f'Frozen {key} count mismatch: {len(D[key])} versus {expected}'
print('\nBASELINE: Frozen corpus sizes verified.')

VALID = {'in_scope','wrong_disease','wrong_population','wrong_cohort','ambiguous'}
for key, index in [
    ('trials',['disease','nct_id']),
    ('endpoints',['disease','nct_id','endpoint_id']),
    ('candidates',['disease','priority_id','nct_id','endpoint_id']),
    ('trial_pilot',['disease','nct_id']),
    ('cohort_screen',['disease','nct_id']),
    ('endpoint_llm',['disease','nct_id','endpoint_id']),
    ('endpoint_full',['disease','nct_id','endpoint_id']),
    ('candidate_full',['disease','priority_id','nct_id','endpoint_id']),
]:
    if key not in D: continue
    df=D[key]
    absent=[x for x in index if x not in df.columns]
    if absent:
        print(f'ERROR {key}: missing key columns {absent}');continue
    dupl=int(df.duplicated(index).sum())
    print(f'{key:17} duplicate keys: {dupl}')
    assert not dupl, f'{key}: duplicate keys; do NOT integrate'
    if 'scope_status' in df:
        invalid=set(df['scope_status'])-VALID
        assert not invalid, f'{key}: unexpected status labels {invalid}'
        print(' '*19+'statuses:',df['scope_status'].value_counts(dropna=False).to_dict())
        if 'confidence' in df:
            conf=pd.to_numeric(df['confidence'],errors='coerce')
            print(' '*19+'invalid confidence:',int((~conf.between(0,1)).sum()))
            assert conf.between(0,1).all(),f'{key} has invalid confidences'

print('\nTRIAL PILOT COVERAGE & IMPACT')
if 'trial_pilot' in D:
    p = D['trial_pilot']; tr=D['trials']; e=D['endpoints'];c=D['candidates']
    missing = set(map(tuple,p[['disease','nct_id']].values)) - set(map(tuple,tr[['disease','nct_id']].values))
    assert not missing, f'Trial pilot refers to {len(missing)} missing frozen trials'
    print('Explicitly trial-level piloted:',len(p),'/',len(tr))
    print('Outside that targeted pilot:',len(tr)-len(p),'(keyword rule accepts them; NOT LLM-reviewed)')
    for label, g in p.groupby('scope_status'):
        keys=set(g['nct_id']); aff_e=e['nct_id'].isin(keys).sum()
        aff_c=c['nct_id'].isin(keys).sum()
        aff_j=c.loc[c['nct_id'].isin(keys),['disease','priority_id','nct_id']].drop_duplicates().shape[0]
        print(f'{label:16} trials={len(g):2} endpoints={aff_e:4} candidate_rows={aff_c:5} jobs={aff_j:3}')

print('\nCOHORT AND ENDPOINT COVERAGE')
if 'cohort_screen' in D:
    print('Cohort classes:',D['cohort_screen']['cohort_structure'].value_counts().to_dict())
if 'endpoint_llm' in D:
    print('LLM-reviewed endpoints:',len(D['endpoint_llm']), '/',len(D['endpoints']))
    print('LLM output versions:',D['endpoint_llm'].get('prompt_version',pd.Series(dtype=str)).value_counts().to_dict())

print('\nFINAL OUTPUT VALIDATION')
if 'endpoint_full' not in D or 'candidate_full' not in D:
    print('BLOCKER: Full endpoint and/or candidate scope mapping is missing.')
    print('Do not assume the 783 LLM judgments automatically constitute full candidate coverage.')
else:
    full=D['endpoint_full'];cs=D['candidate_full'];ep=D['endpoints'];ca=D['candidates']
    assert len(full)==len(ep), f'Endpoint scope not 1:1: {len(full)} vs {len(ep)}'
    assert len(cs)==len(ca), f'Candidate scope not 1:1: {len(cs)} vs {len(ca)}'
    ep_keys=['disease','nct_id','endpoint_id'];ca_keys=['disease','priority_id','nct_id','endpoint_id']
    assert set(map(tuple,ep[ep_keys].values))==set(map(tuple,full[ep_keys].values)), 'Endpoint-key mismatch'
    assert set(map(tuple,ca[ca_keys].values))==set(map(tuple,cs[ca_keys].values)), 'Candidate-key mismatch'
    assert set(full['scope_status']).issubset(VALID) and full['scope_status'].ne('').all(), 'Missing endpoint statuses'
    assert set(cs['scope_status']).issubset(VALID) and cs['scope_status'].ne('').all(), 'Missing candidate statuses'
    if 'candidate_rank' in cs:
        ranks=ca[ca_keys+['within_trial_rank']].merge(cs[ca_keys+['candidate_rank']], on=ca_keys, validate='one_to_one')
        assert ranks['within_trial_rank'].astype(int).eq(ranks['candidate_rank'].astype(int)).all(),'Candidate ranks mismatched'
    test=cs[ca_keys+['scope_status']].merge(full[ep_keys+['scope_status']],on=ep_keys,validate='many_to_one',suffixes=('_candidate','_endpoint'))
    assert test['scope_status_candidate'].eq(test['scope_status_endpoint']).all(), 'Candidate and endpoint scope disagree'
    if 'method' in full:
        print('Full method × status:')
        print(pd.crosstab(full['method'], full['scope_status']).to_string())
        default=full['method'].eq('rule_no_cohort_conflict')
        print(f'Automatic in_scope defaults: {int(default.sum()):,}/{len(full):,} = {100*default.mean():.2f}%')
        if default.any():
            cf=pd.to_numeric(full.loc[default,'confidence'],errors='coerce')
            print('Auto defaults recorded at confidence 1.0:',int(cf.eq(1).sum()),'/',int(default.sum()))
    print('FULL JOIN CHECK PASSED: 8,445 endpoints and 63,811 candidate decisions cover exact frozen keys.')

print('\nINTERPRETATION: This checks schema and coverage, NOT clinical correctness.')
print('Independent blinded validation and review of ambiguous / wrong-cohort cases still required.')
print('No API calls. No files modified.')


TRIALSIGNAL SCOPE PRE-INTEGRATION QA — READ ONLY
trials            PRESENT rows=970 | trialsignal_trials_master_clean.csv
endpoints         PRESENT rows=8,445 | trialsignal_endpoints_master_clean.csv
candidates        PRESENT rows=63,811 | trialsignal_top10_agent_candidates_clean.csv
trial_pilot       PRESENT rows=33 | scope_agent_trial_pilot_v0_2.csv
cohort_screen     PRESENT rows=270 | scope_agent_cohort_screen_v0_1_full.csv
endpoint_llm      PRESENT rows=783 | scope_agent_endpoint_llm_v0_2.csv
explicit_hits     PRESENT rows=90 | scope_agent_endpoint_explicit_cohort_hits_v0.csv
endpoint_full     PRESENT rows=8,445 | scope_agent_endpoint_scope_v0_2.csv
candidate_full    PRESENT rows=63,811 | scope_agent_candidate_scope_v0_2.csv

BASELINE: Frozen corpus sizes verified.
trials            duplicate keys: 0
endpoints         duplicate keys: 0
candidates        duplicate keys: 0
trial_pilot       duplicate keys: 0
                   statuses: {'wrong_disease': 19, 'ambiguous': 11, 'in_scop

In [ ]:
# 1 — READ-ONLY SCOPE ADAPTER QA (no API calls)
from google.colab import drive
from pathlib import Path
import sys
import pandas as pd

drive.mount('/content/portfolio-drive')
PROJECT_DIR = Path('./TrialSignal')
DATA_DIR = PROJECT_DIR / 'data'
AGENTS_DIR = PROJECT_DIR / 'agents'
assert DATA_DIR.is_dir(), f'Missing Drive data folder: {DATA_DIR}'
assert AGENTS_DIR.is_dir(), f'Missing agents folder: {AGENTS_DIR}'
for filename in ['trialsignal_scope_v02_adapter.py',
                 'trialsignal_orchestrator_scaffold_v0_1.py',
                 'trialsignal_semantic_v5.py']:
    assert (AGENTS_DIR / filename).is_file(), f'Upload module to agents/: {filename}'
if str(AGENTS_DIR) not in sys.path:
    sys.path.insert(0, str(AGENTS_DIR))

from trialsignal_scope_v02_adapter import ScopeLookupV02
scope_v02_lookup = ScopeLookupV02(DATA_DIR, review_mode='escalate')
print('SCOPE V0.2 LOOKUP QA PASSED — exact frozen IDs and candidate↔endpoint labels verified')
print('Counts:', scope_v02_lookup.summary)
print('WARNING: Scope model confidences and fallback 1.0 are NOT calibrated probabilities.')

# Pilot exclusions have not received independent human sign-off, so are escalated.
for disease, nct in [('Hereditary Angioedema','NCT03051698'),
                     ('Sickle Cell Disease','NCT01962415')]:
    r=scope_v02_lookup.trial_scope(disease,nct)
    print(f'{disease} / {nct}: scope={r["scope_status"]}; '
          f'original={r["original_scope_status"]}; human_review={r["human_review_required"]}')
    assert r['scope_status']=='ambiguous' and r['human_review_required']

print('No APIs called, no source files overwritten.')

Drive already mounted at /content/portfolio-drive; to attempt to forcibly remount, call drive.mount("/content/portfolio-drive", force_remount=True).
SCOPE V0.2 LOOKUP QA PASSED — exact frozen IDs and candidate↔endpoint labels verified
Counts: {'trials': 970, 'endpoints': 8445, 'candidates': 63811, 'trial_pilot': 33, 'cohort_screen': 270, 'scope_counts': {'in_scope': 8073, 'wrong_disease': 240, 'ambiguous': 98, 'wrong_cohort': 34}, 'method_counts': {'rule_no_cohort_conflict': 7317, 'llm': 783, 'inherited_trial_scope': 339, 'trial_in_scope_low_confidence': 6}, 'human_review_endpoints': 155, 'fallback_not_independently_validated': 7317}
Hereditary Angioedema / NCT03051698: scope=ambiguous; original=wrong_disease; human_review=True
Sickle Cell Disease / NCT01962415: scope=ambiguous; original=wrong_cohort; human_review=True
No APIs called, no source files overwritten.


In [ ]:
# 2 — CACHE-ONLY REAL V5 + SCOPE V0.2 ORCHESTRATOR SMOKE TEST
# Reuses previously saved V5 judgments. No API client is supplied, no LLM calls.
from trialsignal_semantic_v5 import SemanticMatchV5
from trialsignal_orchestrator_scaffold_v0_1 import TrialSignalOrchestratorV01

semantic_v5 = SemanticMatchV5(DATA_DIR, client=None)
trialsignal_orchestrator = TrialSignalOrchestratorV01(
    semantic_agent=semantic_v5,
    trial_scope=scope_v02_lookup.trial_scope,
    endpoint_scope=scope_v02_lookup.endpoint_scope,
    min_auto_accept_confidence=0.75,  # scaffold testing policy, not calibrated
)
result = trialsignal_orchestrator.run(
    'Dravet Syndrome', 'DRAVET_P05', 'NCT04940624',
    allow_semantic_api=False,
)
print('REAL SAVED-DECISION INTEGRATION TEST')
for field in ['status','stop_reason','accepted_endpoint_id','accepted_rank',
              'final_semantic_label','final_scope_status','candidates_checked',
              'human_review_required','is_score_eligible']:
    print(f'{field}: {result[field]}')
display(pd.DataFrame(result['trace']))
assert semantic_v5.new_api_calls == 0, 'Unexpected semantic API calls'
assert result['candidates_checked'] <= 10
assert result['status'] in {'accepted_direct','valid_partials_only','no_semantic_match',
                            'needs_review','excluded_trial'}
print('END-TO-END CACHE-ONLY SMOKE TEST PASSED — NO API CALLS')
print('Do not use this result as an approved final alignment score.')

REAL SAVED-DECISION INTEGRATION TEST
status: accepted_direct
stop_reason: direct_in_scope
accepted_endpoint_id: NCT04940624_S6
accepted_rank: 1
final_semantic_label: direct
final_scope_status: in_scope
candidates_checked: 1
human_review_required: False
is_score_eligible: True


,rank,endpoint_id,semantic_label,semantic_confidence,semantic_reason,scope_status,scope_confidence,scope_reason,action
0,1,NCT04940624_S6,direct,0.98,The endpoint reports investigator-rated improv...,in_scope,1.0,No separate cohort or off-target signal for th...,accept_direct_stop


END-TO-END CACHE-ONLY SMOKE TEST PASSED — NO API CALLS
Do not use this result as an approved final alignment score.


In [ ]:
# TRIALSIGNAL — INTEGRATED 54-CASE PILOT (CACHE-ONLY)
# Uses existing Semantic V5 + Scope v0.2 + Orchestrator v0.1 modules.
# NO API calls. NO scoring. NO modifications to frozen source files.
# Safe to resume; saved job records are fingerprint-bound to current inputs.

from pathlib import Path
from collections import Counter
import hashlib, json, os, sys, uuid
import pandas as pd

PROJECT = Path('./TrialSignal')
DATA = PROJECT / 'data'
AGENTS = PROJECT / 'agents'
assert DATA.is_dir() and AGENTS.is_dir(), 'Mount Google Drive and verify TrialSignal/data and agents.'
for f in ['trialsignal_semantic_v5.py','trialsignal_scope_v02_adapter.py',
          'trialsignal_orchestrator_scaffold_v0_1.py']:
    assert (AGENTS/f).is_file(), f'Missing module: {AGENTS/f}'
if str(AGENTS) not in sys.path: sys.path.insert(0, str(AGENTS))

from trialsignal_semantic_v5 import SemanticMatchV5
from trialsignal_scope_v02_adapter import ScopeLookupV02
from trialsignal_orchestrator_scaffold_v0_1 import TrialSignalOrchestratorV01

def digest(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1024*1024),b''):h.update(b)
    return h.hexdigest()

# Fingerprint cached model decisions AND saved scope outputs: restart only when same inputs.
inputs_to_fingerprint = [
    DATA/'trialsignal_semantic_pilot_cases.csv',
    DATA/'trialsignal_semantic_v5_frozen_contract_v1.json',
    DATA/'scope_agent_trial_pilot_v0_2.csv',
    DATA/'scope_agent_cohort_screen_v0_1_full.csv',
    DATA/'scope_agent_endpoint_scope_v0_2.csv',
    DATA/'scope_agent_candidate_scope_v0_2.csv',
    AGENTS/'trialsignal_semantic_v5.py',
    AGENTS/'trialsignal_scope_v02_adapter.py',
    AGENTS/'trialsignal_orchestrator_scaffold_v0_1.py',
]
# Previously saved V5 judgments are versioned; include their checksums so a
# changed cache cannot be mixed with an old integrated output.
for cache_name in [
    'trialsignal_semantic_v5_development25.csv',
    'trialsignal_semantic_v5_blind35_predictions_v1.csv',
    'trialsignal_semantic_v5_top10_trace_v1.csv',
    'trialsignal_semantic_v5_benchmark54_trace_v1.csv',
    'trialsignal_semantic_v5_module_cache_v1.csv',
]:
    q=DATA/cache_name
    if q.exists():inputs_to_fingerprint.append(q)
for f in inputs_to_fingerprint:assert f.is_file(),f'Missing input: {f}'
signature=hashlib.sha256(json.dumps({str(f.relative_to(PROJECT)):digest(f)
    for f in inputs_to_fingerprint},sort_keys=True).encode()).hexdigest()

semantic = SemanticMatchV5(DATA, client=None)
scope = ScopeLookupV02(DATA, review_mode='escalate')
orch = TrialSignalOrchestratorV01(
    semantic_agent=semantic,
    trial_scope=scope.trial_scope,
    endpoint_scope=scope.endpoint_scope,
    min_auto_accept_confidence=0.75, # Provisional integration test only; NOT clinical calibration.
)
cases=pd.read_csv(DATA/'trialsignal_semantic_pilot_cases.csv',keep_default_na=False)
keycols=['disease','priority_id','nct_id']
assert len(cases)==54, f'Expected 54 cases, found {len(cases)}'
assert not cases.duplicated(keycols).any(), 'Duplicate priority×trial pilot job'

CHECKPOINT=DATA/'trialsignal_integrated_v5_scope02_pilot54_checkpoint_v1.jsonl'
RESULTS=DATA/'trialsignal_integrated_v5_scope02_pilot54_results_v1.csv'
TRACE=DATA/'trialsignal_integrated_v5_scope02_pilot54_trace_v1.csv'
REPORT=DATA/'trialsignal_integrated_v5_scope02_pilot54_report_v1.txt'

def key_from(d):return (str(d['disease']),str(d['priority_id']),str(d['nct_id']))
previous={}
if CHECKPOINT.exists():
    for no,line in enumerate(CHECKPOINT.read_text(encoding='utf-8').splitlines(),1):
        if not line.strip():continue
        rec=json.loads(line)
        if rec.get('source_fingerprint')!=signature:
            raise ValueError('Checkpoint uses a different scope, model, or corpus version. '
                             'Do NOT reuse. Rename/archive prior pilot54 checkpoint before new run.')
        key=key_from(rec)
        if key in previous:raise ValueError(f'Duplicate checkpoint job at line {no}: {key}')
        previous[key]=rec

print('TRIALSIGNAL — 54-CASE INTEGRATED PILOT (NO API)')
print('Source fingerprint:',signature[:16])
print('Previously saved jobs:',len(previous))
print('V5 cached judgments:',semantic.status()['cached_candidate_judgments'])
print('Scope total endpoints:',scope.summary['endpoints'])

for i,c in enumerate(cases.to_dict('records'),1):
    key=key_from(c)
    if key in previous:
        print(f'{i:02}/54 {c["priority_id"]} {c["nct_id"]}: resumed {previous[key]["status"]}')
        continue
    try:
        r=orch.run(*key,allow_semantic_api=False)
        status=r['status']
        checkpoint={
            'source_fingerprint':signature,
            'disease':key[0],'priority_id':key[1],'nct_id':key[2],
            'status':status,'stop_reason':r['stop_reason'],
            'accepted_endpoint_id':r['accepted_endpoint_id'],
            'accepted_rank':r['accepted_rank'],
            'final_semantic_label':r['final_semantic_label'],
            'final_scope_status':r['final_scope_status'],
            'candidates_checked':r['candidates_checked'],
            'is_score_eligible':r['is_score_eligible'],
            'human_review_required':r['human_review_required'],
            'trial_scope':r['trial_scope'],
            'partial_candidates':r['partial_candidates'],
            'pending_review_candidates':r['pending_review_candidates'],
            'trace':r['trace'],
            'error':None,
        }
    except LookupError as e:
        # E.g. new V5 judgment required after an excluded/out-of-scope direct;
        # do not claim an unsuccessful case was evaluated as no_match.
        checkpoint={
            'source_fingerprint':signature,
            'disease':key[0],'priority_id':key[1],'nct_id':key[2],
            'status':'blocked_missing_semantic_cache','stop_reason':'api_disabled_missing_cache',
            'accepted_endpoint_id':None,'accepted_rank':None,
            'final_semantic_label':None,'final_scope_status':None,
            'candidates_checked':None,'is_score_eligible':False,
            'human_review_required':True,
            'trial_scope':scope.trial_scope(key[0],key[2]),
            'partial_candidates':[],'pending_review_candidates':[],
            'trace':[],
            'error':str(e),
        }
    # Fail closed on any unexpected schema, identity, or file error.
    # Each successful job checkpoint is appended immediately for Colab recovery.
    with CHECKPOINT.open('a',encoding='utf-8') as f:
        f.write(json.dumps(checkpoint,ensure_ascii=False,allow_nan=False)+'\n')
        f.flush()
    previous[key]=checkpoint
    print(f'{i:02}/54 {c["priority_id"]} {c["nct_id"]}: {checkpoint["status"]}')

assert semantic.new_api_calls==0,'Unexpected API calls'
assert len(previous)==54,'Incomplete 54-case checkpoint'
rows=[];traces=[]
by_key=cases.set_index(keycols)
for key,c in previous.items():
    if key not in by_key.index:raise ValueError(f'Checkpoint unknown benchmark job: {key}')
    ref=by_key.loc[key]
    ts=c['trial_scope']
    rows.append({
        'disease':key[0],'priority_id':key[1],'nct_id':key[2],
        'original_manual_direct':str(ref['trial_has_true_match']).strip(),
        'original_manual_rank':str(ref.get('first_true_match_rank','')).strip(),
        'trial_scope_status':ts['scope_status'],
        'trial_scope_original':ts.get('original_scope_status'),
        'trial_scope_method':ts.get('method'),
        'trial_scope_review':ts.get('human_review_required'),
        'status':c['status'],'stop_reason':c['stop_reason'],
        'accepted_endpoint_id':c['accepted_endpoint_id'],
        'accepted_rank':c['accepted_rank'],
        'final_semantic_label':c['final_semantic_label'],
        'final_scope_status':c['final_scope_status'],
        'candidates_checked':c['candidates_checked'],
        'partial_count':len(c['partial_candidates']),
        'pending_review_count':len(c['pending_review_candidates']),
        'human_review_required':c['human_review_required'],
        'is_score_eligible':c['is_score_eligible'],
        'error':c['error'],
        'source_fingerprint':signature,
    })
    for e in c['trace']:
        line={**dict(zip(keycols,key)),**e,
              'scope_method':None,'scope_requires_human_review':None,
              'source_fingerprint':signature}
        if e['scope_status'] is not None:
            meta=scope.endpoint_scope(key[0],key[2],e['endpoint_id'],key[1])
            line['scope_method']=meta.get('method')
            line['scope_requires_human_review']=meta.get('human_review_required')
            line['original_scope_status']=meta.get('original_scope_status')
        traces.append(line)

res=pd.DataFrame(rows).sort_values(['disease','priority_id','nct_id'])
tr=pd.DataFrame(traces)
def atomic_csv(df,path):
    tmp=path.with_name('.'+path.name+'.'+uuid.uuid4().hex+'.tmp')
    try:
        df.to_csv(tmp,index=False)
        os.replace(tmp,path)
    finally:
        if tmp.exists():tmp.unlink()
atomic_csv(res,RESULTS)
atomic_csv(tr,TRACE)

count=res.status.value_counts()
missing=int(res.status.eq('blocked_missing_semantic_cache').sum())
unresolved=int((res.human_review_required==True).sum())
provisional_eligible=int((res.is_score_eligible==True).sum())
if len(tr):
    fallback=int(tr.scope_method.eq('rule_no_cohort_conflict').sum())
    llm=int(tr.scope_method.eq('llm').sum())
else:fallback=llm=0
lines=[
    'TRIALSIGNAL — 54-CASE INTEGRATED PILOT, CACHE-ONLY',
    f'Input fingerprint: {signature}',
    'No API calls and NO final alignment scores.',
    'Human labels are the original direct-vs-non-direct retrieval benchmark; not scope gold.',
    '', 'RESULTS BY STATUS:',count.to_string(),'',
    f'Provisional score-eligible jobs: {provisional_eligible}/54',
    f'Needs review jobs: {unresolved}/54',
    f'Blocked by unavailable semantic cache: {missing}/54',
    f'Endpoint scope checks using unvalidated rule fallback: {fallback}',
    f'Endpoint scope checks using Scope LLM pilot: {llm}',
    '', 'MANUAL REVIEW / EXCLUSIONS / MISSING CACHE:',
]
issues=res[(res.human_review_required==True) | (res.status=='excluded_trial')]
if len(issues):
    lines += [issues[['disease','priority_id','nct_id','status','stop_reason',
                     'trial_scope_original','error']].to_string(index=False)]
else:lines+=['None in this 54-case sample.']
lines += ['', 'CAUTION: Scope fallback confidence=1.0 is NOT calibrated.',
          'CAUTION: The 0.75 orchestrator confidence policy is provisional.',
          'CAUTION: Original semantic positives include three debated direct/partial cases.',
          'CAUTION: Final clinical validation, partial scoring, and disease-level analysis remain undone.']
REPORT.write_text('\n'.join(lines)+'\n',encoding='utf-8')
print('\n'+'\n'.join(lines))
print('\nSaved:')
for x in (CHECKPOINT,RESULTS,TRACE,REPORT):print(' ',x)


TRIALSIGNAL — 54-CASE INTEGRATED PILOT (NO API)
Source fingerprint: 409dec19320351b6
Previously saved jobs: 0
V5 cached judgments: 288
Scope total endpoints: 8445
01/54 DRAVET_P02 NCT02187809: accepted_direct
02/54 DRAVET_P02 NCT02926898: no_semantic_match
03/54 DRAVET_P02 NCT05560282: no_semantic_match
04/54 DRAVET_P03 NCT06422377: no_semantic_match
05/54 DRAVET_P05 NCT04940624: accepted_direct
06/54 DRAVET_P05 NCT07531745: no_semantic_match
07/54 DRAVET_P06 NCT07531745: no_semantic_match
08/54 DRAVET_P08 NCT02926898: no_semantic_match
09/54 FSHD_P01 NCT02861911: valid_partials_only
10/54 FSHD_P01 NCT03943290: accepted_direct
11/54 FSHD_P02 NCT00104078: no_semantic_match
12/54 FSHD_P02 NCT00821548: accepted_direct
13/54 FSHD_P02 NCT02239224: accepted_direct
14/54 FSHD_P08 NCT06131983: no_semantic_match
15/54 FSHD_P08 NCT07038200: valid_partials_only
16/54 FSHD_P08 NCT07435129: no_semantic_match
17/54 FA_P02 NCT00229632: no_semantic_match
18/54 FA_P03 NCT01339884: valid_partials_only
1

In [ ]:
# TRIALSIGNAL — INTEGRATED PILOT54 REVIEW + SCORING READINESS GATE
# READ-ONLY against saved inputs; no APIs; no final alignment scores.
# Run after Master Reconnect v4.0 or after mounting Drive; independent of notebook variables.
from pathlib import Path
import pandas as pd
from collections import Counter
import importlib.util, sys, json, hashlib

DATA = Path('./TrialSignal/data')
AGENTS = Path('./TrialSignal/agents')
assert DATA.is_dir() and AGENTS.is_dir(), 'Mount Drive and verify TrialSignal data/agents.'

def csv(name):
    p=DATA/name
    assert p.is_file(),f'Missing {p}'
    return pd.read_csv(p,keep_default_na=False)

res=csv('trialsignal_integrated_v5_scope02_pilot54_results_v1.csv')
trace=csv('trialsignal_integrated_v5_scope02_pilot54_trace_v1.csv')
trials=csv('trialsignal_trials_master_clean.csv')
endpoints=csv('trialsignal_endpoints_master_clean.csv')
cohorts=csv('scope_agent_cohort_screen_v0_1_full.csv')
trialpilot=csv('scope_agent_trial_pilot_v0_2.csv')
assert len(res)==54, f'Unexpected number of integrated jobs: {len(res)}'
assert len(trials)==970 and len(endpoints)==8445
K=['disease','priority_id','nct_id']
assert not res.duplicated(K).any(), 'Duplicate pilot jobs'
assert all(res.source_fingerprint.astype(str).str.len()==64),'Missing input fingerprint'
assert res.source_fingerprint.nunique()==1,'Mixed run fingerprints in results'
assert not trace.empty,'Missing decision trace'
assert trace.source_fingerprint.nunique()==1 and trace.source_fingerprint.iloc[0]==res.source_fingerprint.iloc[0],'Trace/result version mismatch'
assert not trace.duplicated(K+['rank']).any(),'Duplicate inspected rank'
expected={'accepted_direct':13,'valid_partials_only':7,'no_semantic_match':32,'needs_review':2}
actual=res.status.value_counts().to_dict()
assert actual==expected,f'Pilot changed; inspect before reusing this review plan: {actual}'
# Verify no review case is being treated as score-eligible.
def yes(x):return str(x).strip().lower() in ('true','1')
assert not res[res.status=='needs_review'].is_score_eligible.map(yes).any()
assert int(res.is_score_eligible.map(yes).sum())==52

print('TRIALSIGNAL — INTEGRATED PILOT54 QA')
print('Input fingerprint:',res.source_fingerprint.iloc[0])
print('Statuses:',actual)
print('Endpoint scope methods used where semantic match warranted checking:')
print(trace.scope_method.value_counts(dropna=False).to_string())
print('Scope override actions:')
print(trace.action.value_counts(dropna=False).to_string())
print('No APIs called; no source inputs changed.')

# Build a *small* priority review queue without treating existing model labels as human gold.
# Two unresolved trials + up to six direct accepts relying on the rule fallback
# + up to three partial candidates relying on rule fallback.
needs=res[res.status=='needs_review'].copy()
for k in K:needs[k]=needs[k].astype(str)
review_rows=[]
for _,r in needs.iterrows():
    key={k:str(r[k]) for k in K}
    review_rows.append({**key,'review_category':'unresolved_trial',
        'endpoint_id':'','rank':'', 'orchestrator_status':r.status,
        'scope_method':'trial_or_cohort_review', 'scope_status_from_trace':'',
        'scope_reason_from_trace':'','semantic_label':'',
        'requested_review':'Determine whether target-disease patients are enrolled and whether separate cohorts exist.'})

c=trace.drop(columns=['status'],errors='ignore').merge(res[K+['status']],on=K,how='left',validate='many_to_one')
def spread_by_disease(frame,n):
    # Round-robin across diseases instead of selecting only the first alphabetical group.
    frame=frame.sort_values(['disease','priority_id','nct_id','rank']).copy()
    frame['_within_disease']=frame.groupby('disease').cumcount()
    return frame.sort_values(['_within_disease','disease','priority_id','nct_id','rank']).head(n).drop(columns=['_within_disease'])
selected_direct=spread_by_disease(c[(c.action=='accept_direct_stop') &
                                (c.scope_method=='rule_no_cohort_conflict')],6)
selected_partial=spread_by_disease(c[(c.action=='retain_partial_continue') &
                                 (c.scope_method=='rule_no_cohort_conflict')],3)
for category,sub in [('accepted_direct_rule_spotcheck',selected_direct),
                     ('partial_rule_spotcheck',selected_partial)]:
    for _,r in sub.iterrows():
        review_rows.append({**{k:str(r[k]) for k in K},'review_category':category,
            'endpoint_id':str(r.endpoint_id),'rank':r['rank'],
            'orchestrator_status':r.status,'scope_method':r.scope_method,
            'scope_status_from_trace':r.scope_status,'scope_reason_from_trace':r.scope_reason,
            'semantic_label':r.semantic_label,
            'requested_review':'Check endpoint applies specifically to target-disease participants/cohort; do not infer from absent conflict alone.'})
review=pd.DataFrame(review_rows)
trial_keep=['disease','nct_id','brief_title','conditions','intervention_names','primary_outcomes','secondary_outcomes']
review=review.merge(trials[trial_keep],on=['disease','nct_id'],how='left',validate='many_to_one')
ep_keep=['disease','nct_id','endpoint_id','endpoint_name','endpoint_description','endpoint_semantic_text']
review=review.merge(endpoints[ep_keep],on=['disease','nct_id','endpoint_id'],how='left',validate='many_to_one')
review=review.merge(cohorts[['disease','nct_id','cohort_structure','reason']].rename(columns={'reason':'cohort_screen_reason'}),
     on=['disease','nct_id'],how='left',validate='many_to_one')
review=review.merge(trialpilot[['disease','nct_id','scope_status','confidence','reason','human_review_required']].rename(
     columns={'scope_status':'trial_pilot_status','confidence':'trial_pilot_confidence',
              'reason':'trial_pilot_reason','human_review_required':'trial_pilot_requires_review'}),
     on=['disease','nct_id'],how='left',validate='many_to_one')
# Explicitly reserve independent reviewer fields; never fill from agent decisions.
review['reviewer_trial_label']=''
review['reviewer_endpoint_label']=''
review['reviewer_reason']=''
review['reviewer']=''
review['review_status']='pending'
review['source_fingerprint']=res.source_fingerprint.iloc[0]
# New, versioned output only; reruns are safe and refuse overwrite.
file_out=DATA/'trialsignal_integrated_pilot54_priority_review_queue_v1.csv'
if file_out.exists():
    prev=pd.read_csv(file_out,keep_default_na=False)
    assert prev.source_fingerprint.nunique()==1 and prev.source_fingerprint.iloc[0]==res.source_fingerprint.iloc[0], 'Existing review queue from different pipeline version.'
    print('\nExisting queue preserved (including any human edits):',file_out)
else:
    review.to_csv(file_out,index=False)
    print('\nCreated review queue:',file_out)

# Explicit scoring readiness diagnostics; policy intentionally left undecided.
report={
    'fingerprint':res.source_fingerprint.iloc[0],
    'total_jobs':len(res),
    'status_counts':actual,
    'provisional_score_eligible':int(res.is_score_eligible.map(yes).sum()),
    'unresolved_jobs':needs[K+['status','stop_reason']].to_dict('records'),
    'endpoint_scope_checks_by_method':trace.scope_method.value_counts().to_dict(),
    'requires_final_scoring_policy':True,
    'partial_rule_unset':True,
    'missing_cache_jobs':int(res.status.eq('blocked_missing_semantic_cache').sum()),
    'conclusion':'INTEGRATION_TEST_PASSED__FINAL_SCORING_NOT_APPROVED',
}
print('\nSCORING READINESS GATE')
print(json.dumps(report,indent=2))
print('\nPriority review queue:\n',review[['disease','priority_id','nct_id','endpoint_id','review_category','brief_title']].to_string(index=False,max_colwidth=65))
print('\nFINAL NOTE: The original 54 cases are a semantic retrieval validation set,')
print('not a population-representative set of TrialSignal trials. Do not publish')
print('disease alignment rates based on this pilot.')


TRIALSIGNAL — INTEGRATED PILOT54 QA
Input fingerprint: 409dec19320351b6a7908e7f551401b1a6d7c1113d8539f30d49167e644569f3
Statuses: {'no_semantic_match': 32, 'accepted_direct': 13, 'valid_partials_only': 7, 'needs_review': 2}
Endpoint scope methods used where semantic match warranted checking:
scope_method
                           210
rule_no_cohort_conflict     28
llm                          1
Scope override actions:
action
skip_no_semantic_match     210
retain_partial_continue     16
accept_direct_stop          13
No APIs called; no source inputs changed.

Created review queue: ./TrialSignal/data/trialsignal_integrated_pilot54_priority_review_queue_v1.csv

SCORING READINESS GATE
{
  "fingerprint": "409dec19320351b6a7908e7f551401b1a6d7c1113d8539f30d49167e644569f3",
  "total_jobs": 54,
  "status_counts": {
    "no_semantic_match": 32,
    "accepted_direct": 13,
    "valid_partials_only": 7,
    "needs_review": 2
  },
  "provisional_score_eligible": 52,
  "unresolved_jobs": [
    {
   